# 07 · Mudanças na intensidade carbônica da Iatan 2

Experimento de Thiago Menezes. Com os registros horários da EPA (6065/2, 2011–2025), estima a intensidade de emissão, identifica mudanças retrospectivas e calibra com dados de 2017 o fator de pós-correção F. O modelo de geração do 07 é distinto dos modelos A–D do 06.

Execute 00 para preparar o projeto. O 07 reutiliza os 15 arquivos anuais com SHA-256 verificado. As previsões de 2025, a reconciliação das datas, a calibração e as métricas são gravadas em `outputs/07_intensidade_carbonica/`. O 08 cruza essas saídas com a rodada anual de 06. Cada nova execução do 07 recalcula os arquivos desta pasta.

As datas são selecionadas com CO₂ de 2025. As comparações descrevem uma pós-correção retrospectiva e não identificam manutenções nem demonstram ganho prospectivo.

## Fundamentação e mapa de rastreabilidade metodológica

Este notebook integra abordagens **estabelecidas na literatura** e **hipóteses operacionais específicas do estudo**. O mapa abaixo orienta a leitura científica do código.

| Componente implementado | Referências de fundamento | Natureza da aplicação |
|---|---|---|
| EPA/CEMS, interpretação e limites dos indicadores | EPA (2022, p. 8, *Hourly Data*) | Fonte e interpretação dos registros; CAMPD não oferece a identificação detalhada de substituição disponível no FACT/FTP |
| Intensidade de CO₂ em função da geração e operação | Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11) | Motivação física; análise descritiva da unidade |
| Segmentação de regimes e seleção aproximada do número de quebras | Bai e Perron (2003, §3, pp. 3–6); Schwarz (1978, §1, p. 461); Truong et al. (2020, §2.2 e §5.1.1) | Implementação própria de programação dinâmica, custo ponderado e BIC **aproximado** |
| Comparação pré/pós em faixas de carga | Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11) | Controle descritivo de condições operacionais comparáveis |
| XGBoost e avaliação temporal | Chen e Guestrin (2016, §§2.1–2.2, p. 786); Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3) | Algoritmo de referência e cuidados com o tempo |
| Adaptação à instabilidade temporal | Gama et al. (2014, pp. 44:2 e 44:4) | Motivação conceitual; **não** é detecção online |
| Bootstrap de séries dependentes | Künsch (1989, pp. 1217–1218) | Intervalos condicionais, em blocos calendários |
| Inferência após seleção de mudanças | Jewell et al. (2022, §1, pp. 1082–1084) | Limitação reconhecida; o método de inferência seletiva do artigo **não** é aplicado |

**Contribuição experimental própria:** usar uma mudança candidata de 2017 para calibrar um fator multiplicativo \(F\), obtido por diferenças de medianas das log-razões entre CO₂ observado e previsto em faixas comparáveis de geração, e testar sua aplicação única (M1) ou cumulativa (M2) após mudanças detectadas retrospectivamente em 2025. **Nenhuma referência listada estabelece que o fator deve ser idêntico em eventos distintos ou confirma manutenções na usina.**

A bibliografia completa, com links e DOI, está ao final. Os métodos do código e as saídas executadas permanecem intactos.

## 01 · Ambiente e diretórios

Carrega a raiz preparada pelo 00 e registra as versões efetivamente utilizadas.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

if os.environ.get("IATAN_PROJECT_ROOT"):
    root = Path(os.environ["IATAN_PROJECT_ROOT"]).expanduser()
elif drive is not None:
    delivery = Path(os.environ.get("IATAN_DELIVERY_ROOT",
        "/content/drive/MyDrive/Ciência de Dados - Iatan - Process Mining e Experimentos/Iatan 2025 - entrega revisada"))
    root = delivery / "execucao"
else:
    root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "configs/iatan_inputs.json").is_file()), Path.cwd())
if not all((root / name).is_file() for name in ("iatan/colab.py", "iatan/carbon_tracking.py", "requirements-iatan.txt")):
    raise FileNotFoundError("Execute a versão atual do notebook 00 ou indique sua raiz em IATAN_PROJECT_ROOT.")
os.chdir(root)
sys.path.insert(0, str(Path.cwd()))
PROJECT_ROOT = Path(".")
from iatan.colab import ensure_dependencies
ensure_dependencies(PROJECT_ROOT, install=drive is not None)

from importlib import metadata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

PLANT_ID, UNIT_ID, ANO_INICIAL, ANO_FINAL = 6065, "2", 2011, 2025
OUTPUT_ROOT = PROJECT_ROOT / "outputs/07_intensidade_carbonica"
DIR_DADOS = OUTPUT_ROOT / "dados"
DIR_TABELAS = OUTPUT_ROOT / "tabelas"
DIR_FIGURAS = OUTPUT_ROOT / "figuras"
DIR_RELATORIOS = OUTPUT_ROOT / "relatorios"
DIR_AUDITORIA = OUTPUT_ROOT / "auditoria"
for folder in (DIR_DADOS, DIR_TABELAS, DIR_FIGURAS, DIR_RELATORIOS, DIR_AUDITORIA):
    folder.mkdir(parents=True, exist_ok=True)
AMBIENTE = {pkg: metadata.version(pkg) for pkg in
            ("numpy", "pandas", "matplotlib", "scikit-learn", "xgboost", "pyarrow")}
AMBIENTE["python"] = sys.version.split()[0]
(DIR_AUDITORIA / "ambiente.json").write_text(json.dumps(AMBIENTE, indent=2), encoding="utf-8")
print("Unidade EPA 6065/2. Resultados:", OUTPUT_ROOT)

## 02 · Dados horários da EPA

Reutiliza os 15 arquivos anuais do projeto, conferidos com o manifesto do 06. Reconstitui os campos horários de Iatan 2, incluindo registros sem operação. A coluna de carga é mantida com seu valor publicado; a interpretação de unidades está na seção 04.

In [ ]:
# Insumos congelados do 06. A soma dos 15 arquivos cobre o calendário inteiro.
from iatan.colab import restore_multiyear
import hashlib

restore_multiyear(PROJECT_ROOT, download=True)
campos = {"Facility ID": "facilityId", "Unit ID": "unitId", "Operating Time": "opTime",
          "Gross Load (MW)": "grossLoad", "CO2 Mass (short tons)": "co2Mass",
          "CO2 Mass Measure Indicator": "co2MassMeasureFlg", "timestamp": "timestamp"}
partes = []
for ano in range(ANO_INICIAL, ANO_FINAL + 1):
    origem = PROJECT_ROOT / f"data/interim/03A04_multiyear/iatan_unit2_{ano}.parquet"
    partes.append(pd.read_parquet(origem, columns=list(campos)).rename(columns=campos))
df_raw = pd.concat(partes, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
calendario = pd.date_range("2011-01-01", "2025-12-31 23:00", freq="h")
if (not pd.DatetimeIndex(df_raw["timestamp"]).equals(calendario)
    or not df_raw["facilityId"].astype(str).eq("6065").all()
    or not df_raw["unitId"].astype(str).eq("2").all()):
    raise ValueError("Calendário ou unidade divergente dos insumos preservados.")
BASE = DIR_DADOS / "iatan2_hourly_raw_2011_2025.parquet"
df_raw.to_parquet(BASE, index=False)
print(len(df_raw), "registros horários de Iatan 2.")

## 03 — Auditoria de integridade

Audita cobertura horária, operação, dados ausentes e sinalizadores de medição do CO₂. Não confundir dados substituídos com medições diretas.

---

**Fundamento — EPA (2022, p. 8, *Hourly Data*).** Os registros horários de CO₂ podem envolver medições válidas ou valores substituídos segundo procedimentos regulatórios. A auditoria de cobertura, duplicidades, operação e sinalizadores disponíveis é útil para detectar inconsistências; a ausência de registro não deve ser confundida com desligamento.

**Limitação dos indicadores CAMPD (EPA, 2022, p. 8).** O CAMPD não disponibiliza a identificação detalhada de valores horários substituídos: o FTP da EPA dispõe de indicador de medido/substituído, enquanto o FACT/FACT API disponibiliza códigos de método de determinação (*MODC*). Assim, o campo `co2MassMeasureFlg` e quaisquer outras *flags* presentes na extração deste estudo **não equivalem a uma auditoria MODC/FTP**, não certificam medição direta para cada hora e não excluem substituições. A presente análise é diagnóstica e não comprova qualidade metrológica individual de todos os registros.

In [ ]:
# ===== ORIGEM: CÉLULA 03 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 03/20 — AUDITORIA DA BASE HISTÓRICA
# Notebook 07 — MUDANÇAS NA INTENSIDADE CARBÔNICA
# IATAN 2 | EPA 6065/2 | 2011–2025
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CARREGAMENTO INDEPENDENTE DA MEMÓRIA
# ------------------------------------------------------------

BASE = (
    OUTPUT_ROOT / "dados" /
    "iatan2_hourly_raw_2011_2025.parquet"
)

if not BASE.is_file():
    raise FileNotFoundError(
        f"Base histórica não encontrada: {BASE}. "
        "Execute a Célula 02 antes de continuar."
    )

df_auditoria = pd.read_parquet(BASE)

print("=" * 65)
print("CÉLULA 03/20 — AUDITORIA HISTÓRICA")
print("=" * 65)

print("Registros:", f"{len(df_auditoria):,}")
print("Colunas:", len(df_auditoria.columns))
print("\nEsquema original:")
display(pd.DataFrame({
    "coluna": df_auditoria.columns,
    "tipo": df_auditoria.dtypes.astype(str).values
}))

# ------------------------------------------------------------
# 2. VERIFICAÇÃO DOS CAMPOS OBRIGATÓRIOS
# ------------------------------------------------------------

campos = [
    "facilityId", "unitId", "timestamp",
    "opTime", "grossLoad", "co2Mass",
    "co2MassMeasureFlg"
]

faltantes = sorted(set(campos) - set(df_auditoria.columns))

if faltantes:
    raise KeyError(
        f"Campos obrigatórios ausentes: {faltantes}"
    )

# Cópia de trabalho: não altera o Parquet original.
d = df_auditoria[campos].copy()

d["timestamp"] = pd.to_datetime(
    d["timestamp"], errors="coerce"
)

for coluna in ["opTime", "grossLoad", "co2Mass"]:
    d[coluna] = pd.to_numeric(
        d[coluna], errors="coerce"
    )

# ------------------------------------------------------------
# 3. IDENTIDADE E INTEGRIDADE TEMPORAL
# ------------------------------------------------------------

identidade_ok = (
    pd.to_numeric(
        d["facilityId"], errors="coerce"
    ).eq(6065).all()
    and d["unitId"].astype(str).str.strip().eq("2").all()
)

grade = pd.date_range(
    "2011-01-01 00:00",
    "2025-12-31 23:00",
    freq="h"
)

datas_validas = pd.DatetimeIndex(
    d["timestamp"].dropna()
)

n_invalidos = int(d["timestamp"].isna().sum())
n_duplicados = int(
    d["timestamp"].dropna().duplicated().sum()
)
n_ausentes = len(grade.difference(datas_validas))
n_extras = len(datas_validas.difference(grade))

integridade = pd.DataFrame([{
    "identidade_6065_2": bool(identidade_ok),
    "registros": len(d),
    "horas_esperadas": len(grade),
    "timestamps_invalidos": n_invalidos,
    "timestamps_duplicados": n_duplicados,
    "horas_ausentes": n_ausentes,
    "horas_extras": n_extras
}])

print("\nINTEGRIDADE TEMPORAL E IDENTIDADE")
display(integridade)

if (
    not identidade_ok
    or n_invalidos
    or n_duplicados
    or n_ausentes
    or n_extras
):
    raise ValueError(
        "A integridade histórica falhou. "
        "Não prosseguir antes de investigar."
    )

# ------------------------------------------------------------
# 4. CLASSIFICAÇÃO DO ESTADO OPERACIONAL
# ------------------------------------------------------------

# Não imputamos Operating Time.
# 0: desligada
# 1: operando em alguma fração da hora
# NA: estado desconhecido

d["is_on"] = pd.Series(
    pd.NA, index=d.index, dtype="Int64"
)

d.loc[d["opTime"].eq(0), "is_on"] = 0
d.loc[d["opTime"].gt(0), "is_on"] = 1

d["ano"] = d["timestamp"].dt.year

# ------------------------------------------------------------
# 5. AUDITORIA ANUAL
# ------------------------------------------------------------

resumos = []

for ano, grupo in d.groupby("ano", sort=True):

    on = grupo["is_on"].eq(1).fillna(False)
    off = grupo["is_on"].eq(0).fillna(False)

    resumos.append({
        "ano": int(ano),
        "horas": len(grupo),
        "horas_on": int(on.sum()),
        "horas_off": int(off.sum()),
        "estado_desconhecido": int(
            grupo["is_on"].isna().sum()
        ),
        "operacao_parcial": int(
            grupo["opTime"].between(
                0, 1, inclusive="neither"
            ).sum()
        ),
        "opTime_ausente": int(
            grupo["opTime"].isna().sum()
        ),
        "opTime_invalido": int(
            (
                grupo["opTime"].lt(0)
                | grupo["opTime"].gt(1)
            ).sum()
        ),
        "geracao_ausente_on": int(
            (on & grupo["grossLoad"].isna()).sum()
        ),
        "co2_ausente_on": int(
            (on & grupo["co2Mass"].isna()).sum()
        ),
        "geracao_nao_positiva_on": int(
            (on & grupo["grossLoad"].le(0)).sum()
        ),
        "co2_negativo": int(
            grupo["co2Mass"].lt(0).sum()
        )
    })

auditoria_anual = pd.DataFrame(resumos)

print("\nAUDITORIA ANUAL")
display(auditoria_anual)

# ------------------------------------------------------------
# 6. INDICADORES DE MEDIÇÃO DO CO2
# ------------------------------------------------------------

indicadores = (
    d["co2MassMeasureFlg"]
    .astype("string")
    .fillna("(ausente)")
    .value_counts(dropna=False)
    .rename_axis("indicador")
    .reset_index(name="horas")
)

print("\nINDICADORES DE MEDIÇÃO DE CO2")
display(indicadores)

# ------------------------------------------------------------
# 7. ESTATÍSTICAS DAS VARIÁVEIS ORIGINAIS
# ------------------------------------------------------------

print("\nESTATÍSTICAS DESCRITIVAS")
display(
    d[["opTime", "grossLoad", "co2Mass"]]
    .describe()
    .round(3)
)

# ------------------------------------------------------------
# 8. EXPORTAÇÃO DA AUDITORIA
# ------------------------------------------------------------

DIR_AUDITORIA = OUTPUT_ROOT / "auditoria"
DIR_AUDITORIA.mkdir(parents=True, exist_ok=True)

auditoria_anual.to_csv(
    DIR_AUDITORIA / "auditoria_anual.csv",
    index=False
)

indicadores.to_csv(
    DIR_AUDITORIA / "indicadores_co2.csv",
    index=False
)

integridade.to_csv(
    DIR_AUDITORIA / "integridade_temporal.csv",
    index=False
)

relatorio = {
    "notebook": "07_mudancas_intensidade_carbonica",
    "etapa": "auditoria_historica",
    "facility_id": 6065,
    "unit_id": "2",
    "periodo": [2011, 2025],
    "registros": int(len(d)),
    "integridade_temporal": True,
    "horas_on": int(d["is_on"].eq(1).sum()),
    "horas_off": int(d["is_on"].eq(0).sum()),
    "horas_estado_desconhecido": int(
        d["is_on"].isna().sum()
    ),
    "execucao_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "observacao": (
        "Auditoria não destrutiva. "
        "Nenhum valor ausente foi imputado."
    )
}

with open(
    DIR_AUDITORIA / "auditoria_historica.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        relatorio,
        arquivo,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 65)
print("CÉLULA 03/20 — AUDITORIA CONCLUÍDA")
print("=" * 65)
print("Arquivos salvos em:", DIR_AUDITORIA)
print("Base original preservada.")

## 04 · Potência, energia, massa e intensidade

O extrato congelado identifica a potência bruta em MW. Em cada hora, a energia gerada é potência × duração de operação (0 a 1 h). O total oficial da EPA para 6065/2 em 2025 permite conferir a interpretação do campo. Os 77 registros parciais distinguem as duas convenções. A massa horária de CO₂ já soma o total oficial e recebe somente a conversão de short tons para toneladas métricas. A intensidade é a razão entre massa e energia quando há operação, energia positiva e massa não negativa. Para uma classe de potência usa-se o campo em MW.

O guia da EPA (2022, p. 8, *Hourly Data*) distingue os produtos de dados; esta conciliação verifica quais transformações reproduzem o agregado oficial do produto utilizado. Akpan e Fuls (2021, §3.3, pp. 8–9) relacionam intensidade e condições de carga em unidades a carvão.

A [definição dos campos da EPA](https://github.com/US-EPA-CAMD/easey-common/blob/9f7c67c1dfecb7541db6c3e61ccd8eb21eff560e/src/constants/property-metadata.constants.ts#L2100-L2116) especifica MW para a série horária e MWh para o agregado. A [resposta anual preservada](../../reproducibility/iatan/epa_anual_6065_2_2025.json) documenta os valores da conciliação.

In [ ]:
# ===== ORIGEM: CÉLULA 04 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 04/20 — PADRONIZAÇÃO E INTENSIDADE CARBÔNICA
# Notebook 07 — IATAN 2 | 2011–2025
# ============================================================
#
# OBJETIVOS:
# 1. Padronizar as variáveis necessárias à pesquisa.
# 2. Converter short tons de CO2 para toneladas métricas.
# 3. Calcular a energia bruta horária (MWh).
# 4. Calcular a intensidade carbônica (tCO2/MWh).
# 5. Documentar horas não elegíveis.
# 6. Preservar integralmente a grade histórica.
#
# Não aplica filtros estatísticos, imputação ou heat rate.
# ============================================================

import numpy as np
import pandas as pd
import json

from datetime import datetime, timezone
from IPython.display import display

# ------------------------------------------------------------
# 1. CARREGAMENTO DA BASE ORIGINAL
# ------------------------------------------------------------

BASE_ORIGINAL = (
    OUTPUT_ROOT / "dados" /
    "iatan2_hourly_raw_2011_2025.parquet"
)

if not BASE_ORIGINAL.exists():
    raise FileNotFoundError(
        "Base histórica ausente. Execute a Célula 02."
    )

raw = pd.read_parquet(BASE_ORIGINAL)

colunas_necessarias = [
    "facilityId", "unitId", "timestamp",
    "opTime", "grossLoad", "co2Mass",
    "co2MassMeasureFlg"
]

ausentes = set(colunas_necessarias) - set(raw.columns)

if ausentes:
    raise KeyError(
        f"Colunas ausentes: {sorted(ausentes)}"
    )

# ------------------------------------------------------------
# 2. PADRONIZAÇÃO
# ------------------------------------------------------------

df = pd.DataFrame({
    "facility_id": pd.to_numeric(
        raw["facilityId"], errors="coerce"
    ),
    "unit_id": raw["unitId"].astype("string").str.strip(),
    "timestamp": pd.to_datetime(
        raw["timestamp"], errors="coerce"
    ),
    "tempo_operacao_h": pd.to_numeric(
        raw["opTime"], errors="coerce"
    ),
    # Potência bruta na hora, em MW.
    "potencia_informada_mw": pd.to_numeric(
        raw["grossLoad"], errors="coerce"
    ),
    "co2_short_tons": pd.to_numeric(
        raw["co2Mass"], errors="coerce"
    ),
    "indicador_medicao_co2": (
        raw["co2MassMeasureFlg"].astype("string")
    )
})

df = df.sort_values("timestamp").reset_index(drop=True)

if (
    len(df) != 131496
    or not df["facility_id"].eq(6065).all()
    or not df["unit_id"].eq("2").all()
    or df["timestamp"].isna().any()
    or df["timestamp"].duplicated().any()
):
    raise ValueError(
        "Identidade ou integridade temporal inválida."
    )

# ------------------------------------------------------------
# 3. ESTADO OPERACIONAL
# ------------------------------------------------------------

df["is_on"] = pd.Series(
    pd.NA, index=df.index, dtype="Int64"
)

df.loc[df["tempo_operacao_h"].eq(0), "is_on"] = 0
df.loc[df["tempo_operacao_h"].gt(0), "is_on"] = 1

# ------------------------------------------------------------
# 4. CONVERSÃO DE UNIDADES
# ------------------------------------------------------------

FATOR_SHORT_TON = 0.90718474

df["co2_t"] = (
    df["co2_short_tons"] * FATOR_SHORT_TON
)

# Reconciliação com a resposta anual oficial EPA 6065/2.
df["carga_bruta_mw"] = df["potencia_informada_mw"]
df["geracao_mwh"] = df["carga_bruta_mw"] * df["tempo_operacao_h"]
controle = json.loads((PROJECT_ROOT / "reproducibility/iatan/epa_anual_6065_2_2025.json").read_text(encoding="utf-8"))
em_2025 = df[df["timestamp"].dt.year == controle["year"]]
anual = {"geracao_mwh": em_2025["geracao_mwh"].sum(),
         "co2_short_tons": em_2025["co2_short_tons"].sum(),
         "soma_campo_mw_sem_ponderar": em_2025["carga_bruta_mw"].sum(),
         "tempo_operacao_h": em_2025["tempo_operacao_h"].sum(),
         "horas_on": int(em_2025["tempo_operacao_h"].gt(0).sum())}
for nome, campo, tolerancia in [("geracao_mwh", "grossLoad", 0.005),
                                ("co2_short_tons", "co2Mass", 0.0005),
                                ("tempo_operacao_h", "sumOpTime", 0.005),
                                ("horas_on", "countOpTime", 0)]:
    if not np.isclose(anual[nome], controle[campo], atol=tolerancia, rtol=0):
        raise ValueError(f"Agregado anual divergente no campo {nome}.")
pd.DataFrame([anual]).to_csv(DIR_TABELAS / "04_conciliacao_anual_2025.csv", index=False)

# 5. CRITÉRIOS DE ELEGIBILIDADE
# ------------------------------------------------------------

operacao_valida = (
    df["tempo_operacao_h"].gt(0)
    & df["tempo_operacao_h"].le(1)
)

geracao_valida = (
    df["geracao_mwh"].notna()
    & np.isfinite(df["geracao_mwh"])
    & df["geracao_mwh"].gt(0)
)

co2_valido = (
    df["co2_t"].notna()
    & np.isfinite(df["co2_t"])
    & df["co2_t"].ge(0)
)

elegivel = (
    operacao_valida
    & geracao_valida
    & co2_valido
)

df["elegivel_intensidade"] = elegivel

# Motivos hierárquicos e mutuamente exclusivos
condicoes = [
    df["tempo_operacao_h"].isna(),
    df["tempo_operacao_h"].eq(0),
    (
        df["tempo_operacao_h"].lt(0)
        | df["tempo_operacao_h"].gt(1)
    ),
    ~geracao_valida,
    ~co2_valido
]

motivos = [
    "tempo_operacao_ausente",
    "unidade_desligada",
    "tempo_operacao_invalido",
    "geracao_ausente_ou_nao_positiva",
    "co2_ausente_ou_invalido"
]

df["situacao_intensidade"] = np.select(
    condicoes,
    motivos,
    default="valida"
)

# ------------------------------------------------------------
# 6. INTENSIDADE HORÁRIA
# ------------------------------------------------------------

df["intensidade_t_mwh"] = np.nan

df.loc[
    elegivel, "intensidade_t_mwh"
] = (
    df.loc[elegivel, "co2_t"]
    / df.loc[elegivel, "geracao_mwh"]
)

# Não remover valores extremos nesta etapa.
# A avaliação estatística ocorrerá posteriormente.

# ------------------------------------------------------------
# 7. AUDITORIA DA VARIÁVEL CALCULADA
# ------------------------------------------------------------

resumo_situacoes = (
    df["situacao_intensidade"]
    .value_counts()
    .rename_axis("situacao")
    .reset_index(name="horas")
)

resumo_anual = (
    df.assign(ano=df["timestamp"].dt.year)
    .groupby("ano", as_index=False)
    .agg(
        horas=("timestamp", "size"),
        horas_elegiveis=("elegivel_intensidade", "sum"),
        geracao_mwh_elegivel=(
            "geracao_mwh",
            lambda s: s[df.loc[s.index, "elegivel_intensidade"]].sum()
        ),
        co2_t_elegivel=(
            "co2_t",
            lambda s: s[df.loc[s.index, "elegivel_intensidade"]].sum()
        )
    )
)

# Razão de totais, não média simples das intensidades
resumo_anual["intensidade_agregada_t_mwh"] = (
    resumo_anual["co2_t_elegivel"]
    / resumo_anual["geracao_mwh_elegivel"].replace(0, np.nan)
)

print("=" * 65)
print("CÉLULA 04/20 — INTENSIDADE CARBÔNICA")
print("=" * 65)

print("\nCLASSIFICAÇÃO DAS HORAS")
display(resumo_situacoes)

print("\nRESUMO ANUAL PRELIMINAR")
display(resumo_anual.round(4))

print("\nDISTRIBUIÇÃO DA INTENSIDADE HORÁRIA")
display(
    df.loc[elegivel, "intensidade_t_mwh"]
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .round(5)
)

# ------------------------------------------------------------
# 8. VERIFICAÇÕES FINAIS
# ------------------------------------------------------------

assert len(df) == len(raw)
assert int(df["elegivel_intensidade"].sum()) == int(
    df["intensidade_t_mwh"].notna().sum()
)
assert np.isfinite(
    df.loc[elegivel, "intensidade_t_mwh"]
).all()
assert (df.loc[elegivel, "geracao_mwh"] > 0).all()

# ------------------------------------------------------------
# 9. EXPORTAÇÃO
# ------------------------------------------------------------

ARQUIVO_ANALITICO = (
    DIR_DADOS /
    "iatan2_intensidade_horaria_2011_2025.parquet"
)

df.to_parquet(
    ARQUIVO_ANALITICO,
    index=False
)

resumo_situacoes.to_csv(
    DIR_TABELAS / "situacoes_intensidade.csv",
    index=False
)

resumo_anual.to_csv(
    DIR_TABELAS / "resumo_anual_preliminar.csv",
    index=False
)

metadados = {
    "unidade": "Iatan 2",
    "facility_id": 6065,
    "unit_id": "2",
    "periodo": [2011, 2025],
    "registros": int(len(df)),
    "horas_elegiveis": int(elegivel.sum()),
    "fator_short_ton_para_t": FATOR_SHORT_TON,
    "formula_geracao": "grossLoad (MW) × opTime (h)",
    "formula_carga_operacional": "grossLoad (MW)",
    "fonte_interpretacao": "EPA, resposta anual 6065/2/2025 e metadados do extrato horário",
    "formula_intensidade": "co2_t / geracao_mwh",
    "heat_rate_utilizado": False,
    "imputacao_realizada": False,
    "execucao_utc": datetime.now(timezone.utc).isoformat()
}

with open(
    DIR_RELATORIOS / "metodologia_intensidade.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadados,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 65)
print("CÉLULA 04/20 CONCLUÍDA")
print("=" * 65)
print("Base analítica:", ARQUIVO_ANALITICO)
print("Horas preservadas:", f"{len(df):,}")
print("Horas elegíveis:", f"{int(elegivel.sum()):,}")

## 05 — Evolução anual e mensal

Caracteriza a série histórica de IC pela razão entre somas de CO₂ e geração, não pela média simples das razões horárias. Produz tabelas e gráficos anuais e mensais.

---

**Escolha estatística do estudo — EPA (2022, p. 8, *Hourly Data*); Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11).** A intensidade agregada do período é \(\sum_t CO_{2,t}/\sum_t G_t\). Essa razão corresponde à média das intensidades horárias ponderada pela geração, e não à média aritmética de \(CO_{2,t}/G_t\); evita atribuir peso excessivo às horas de baixa geração. É uma definição algébrica usada aqui para agregação, não um resultado experimental dos artigos.

In [ ]:
# ===== ORIGEM: CÉLULA 05 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 05/20 — EVOLUÇÃO ANUAL DA INTENSIDADE CARBÔNICA
# Notebook 07 — IATAN 2 | 2011–2025
# ============================================================
#
# OBJETIVOS:
# 1. Calcular a intensidade anual como razão de totais.
# 2. Comparar geração, emissões e intensidade.
# 3. Quantificar variações anuais.
# 4. Identificar anos que merecem investigação.
# 5. Exportar tabela e figura.
#
# Não identifica formalmente pontos de mudança.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CARREGAMENTO DA BASE ANALÍTICA
# ------------------------------------------------------------

ARQUIVO_ANALITICO = (
    OUTPUT_ROOT / "dados" /
    "iatan2_intensidade_horaria_2011_2025.parquet"
)

if not ARQUIVO_ANALITICO.is_file():
    raise FileNotFoundError(
        "Execute a Célula 04 antes de continuar."
    )

df = pd.read_parquet(ARQUIVO_ANALITICO)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["ano"] = df["timestamp"].dt.year

if len(df) != 131496:
    raise ValueError("Quantidade histórica inesperada.")

# ------------------------------------------------------------
# 2. SELEÇÃO DAS HORAS ELEGÍVEIS
# ------------------------------------------------------------

validos = df.loc[
    df["elegivel_intensidade"].eq(True)
].copy()

if validos.empty:
    raise ValueError("Nenhuma hora elegível encontrada.")

# ------------------------------------------------------------
# 3. AGREGAÇÃO ANUAL PONDERADA PELA GERAÇÃO
# ------------------------------------------------------------

anual = (
    validos.groupby("ano", as_index=False)
    .agg(
        horas_validas=("timestamp", "size"),
        geracao_mwh=("geracao_mwh", "sum"),
        co2_t=("co2_t", "sum")
    )
)

anual["intensidade_t_mwh"] = (
    anual["co2_t"] / anual["geracao_mwh"]
)

# Total de horas e disponibilidade dos registros
horas_totais = (
    df.groupby("ano")
    .size()
    .rename("horas_calendario")
    .reset_index()
)

anual = anual.merge(
    horas_totais,
    on="ano",
    how="left",
    validate="one_to_one"
)

anual["cobertura_elegivel_pct"] = (
    100 * anual["horas_validas"]
    / anual["horas_calendario"]
)

# Variação percentual da intensidade anual
anual["variacao_intensidade_pct"] = (
    anual["intensidade_t_mwh"].pct_change() * 100
)

# Unidades para visualização
anual["geracao_twh"] = anual["geracao_mwh"] / 1e6
anual["co2_mt"] = anual["co2_t"] / 1e6

# ------------------------------------------------------------
# 4. VALIDAÇÕES
# ------------------------------------------------------------

assert len(anual) == 15
assert anual["ano"].tolist() == list(range(2011, 2026))
assert anual["geracao_mwh"].gt(0).all()
assert anual["intensidade_t_mwh"].notna().all()

print("=" * 65)
print("CÉLULA 05/20 — EVOLUÇÃO ANUAL")
print("=" * 65)

display(
    anual[[
        "ano",
        "horas_validas",
        "cobertura_elegivel_pct",
        "geracao_twh",
        "co2_mt",
        "intensidade_t_mwh",
        "variacao_intensidade_pct"
    ]].round(4)
)

# ------------------------------------------------------------
# 5. GRÁFICO DA INTENSIDADE ANUAL
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5.5))

ax.plot(
    anual["ano"],
    anual["intensidade_t_mwh"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Iatan 2 — Intensidade carbônica anual (2011–2025)"
)
ax.set_xlabel("Ano")
ax.set_ylabel("Intensidade (tCO₂/MWh)")
ax.set_xticks(anual["ano"])
ax.grid(alpha=0.25)

fig.tight_layout()

DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

fig.savefig(
    DIR_FIGURAS / "intensidade_anual_2011_2025.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. GRÁFICO DE GERAÇÃO E EMISSÕES
# ------------------------------------------------------------

fig, ax1 = plt.subplots(figsize=(12, 5.5))

ax1.plot(
    anual["ano"],
    anual["geracao_twh"],
    marker="o",
    linewidth=2,
    label="Geração bruta"
)

ax1.set_xlabel("Ano")
ax1.set_ylabel("Geração bruta (TWh)")
ax1.set_xticks(anual["ano"])

ax2 = ax1.twinx()

ax2.plot(
    anual["ano"],
    anual["co2_mt"],
    marker="s",
    linestyle="--",
    linewidth=2,
    color="tab:orange",
    label="Emissões de CO₂"
)

ax2.set_ylabel("Emissões de CO₂ (Mt)")

ax1.grid(alpha=0.25)

linhas1, labels1 = ax1.get_legend_handles_labels()
linhas2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    linhas1 + linhas2,
    labels1 + labels2,
    loc="best"
)

ax1.set_title(
    "Iatan 2 — Geração bruta e emissões anuais"
)

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "geracao_emissoes_anuais.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. DESTAQUES EXPLORATÓRIOS
# ------------------------------------------------------------

maior_reducao = anual.loc[
    anual["variacao_intensidade_pct"].idxmin()
]

maior_aumento = anual.loc[
    anual["variacao_intensidade_pct"].idxmax()
]

print("\nDESTAQUES EXPLORATÓRIOS")

print(
    f"Maior redução anual: {int(maior_reducao['ano'])} "
    f"({maior_reducao['variacao_intensidade_pct']:.2f}%)"
)

print(
    f"Maior aumento anual: {int(maior_aumento['ano'])} "
    f"({maior_aumento['variacao_intensidade_pct']:.2f}%)"
)

print(
    "\nEssas variações não constituem, isoladamente, "
    "evidência de mudança estrutural."
)

# ------------------------------------------------------------
# 8. EXPORTAÇÃO
# ------------------------------------------------------------

DIR_TABELAS.mkdir(parents=True, exist_ok=True)

ARQUIVO_ANUAL = (
    DIR_TABELAS /
    "intensidade_anual_2011_2025.csv"
)

anual.to_csv(
    ARQUIVO_ANUAL,
    index=False
)

print("\n" + "=" * 65)
print("CÉLULA 05/20 — CONCLUÍDA")
print("=" * 65)
print("Tabela:", ARQUIVO_ANUAL)
print("Figuras:", DIR_FIGURAS)


# ===== ORIGEM: CÉLULA 06 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 06/20 — EVOLUÇÃO MENSAL DA INTENSIDADE CARBÔNICA
# Notebook 07 — IATAN 2 | EPA 6065/2 | 2011–2025
# ============================================================
#
# OBJETIVOS:
# 1. Construir a série mensal de 180 meses.
# 2. Calcular intensidade como razão de totais.
# 3. Comparar intensidade, carga e utilização.
# 4. Examinar os períodos 2012, 2016–2018 e 2023–2025.
# 5. Exportar tabelas e gráficos.
#
# Não detecta formalmente quebras estruturais.
# Não pressupõe manutenção.
# Não utiliza heat rate.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. DIRETÓRIOS E CARREGAMENTO
# ------------------------------------------------------------

ARQUIVO_ANALITICO = (
    OUTPUT_ROOT / "dados" /
    "iatan2_intensidade_horaria_2011_2025.parquet"
)

DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
DIR_TABELAS.mkdir(parents=True, exist_ok=True)

if not ARQUIVO_ANALITICO.is_file():
    raise FileNotFoundError(
        "Base analítica não encontrada. "
        "Execute a Célula 04."
    )

dados = pd.read_parquet(ARQUIVO_ANALITICO)

dados["timestamp"] = pd.to_datetime(
    dados["timestamp"], errors="coerce"
)

if (
    len(dados) != 131496
    or dados["timestamp"].isna().any()
    or dados["timestamp"].duplicated().any()
):
    raise ValueError(
        "Integridade da base histórica inválida."
    )

dados["mes"] = (
    dados["timestamp"].dt.to_period("M").dt.to_timestamp()
)

dados["ano"] = dados["timestamp"].dt.year

# ------------------------------------------------------------
# 2. INDICADORES OPERACIONAIS MENSAIS
# ------------------------------------------------------------

operacao = (
    dados.groupby("mes")
    .agg(
        horas_calendario=("timestamp", "size"),
        horas_on=("is_on", lambda s: int(s.eq(1).sum())),
        horas_off=("is_on", lambda s: int(s.eq(0).sum())),
        horas_elegiveis=(
            "elegivel_intensidade",
            "sum"
        ),
        tempo_operacao_total_h=(
            "tempo_operacao_h",
            "sum"
        )
    )
)

operacao["utilizacao_calendario_pct"] = (
    100 * operacao["tempo_operacao_total_h"]
    / operacao["horas_calendario"]
)

operacao["cobertura_elegivel_pct"] = (
    100 * operacao["horas_elegiveis"]
    / operacao["horas_calendario"]
)

# ------------------------------------------------------------
# 3. AGREGAÇÃO MENSAL DAS HORAS ELEGÍVEIS
# ------------------------------------------------------------

validos = dados.loc[
    dados["elegivel_intensidade"].eq(True)
].copy()

mensal_validos = (
    validos.groupby("mes")
    .agg(
        geracao_mwh=("geracao_mwh", "sum"),
        co2_t=("co2_t", "sum"),
        carga_media_mw=("carga_bruta_mw", "mean"),
        carga_mediana_mw=("carga_bruta_mw", "median"),
        intensidade_mediana_horaria=(
            "intensidade_t_mwh",
            "median"
        )
    )
)

mensal = operacao.join(
    mensal_validos,
    how="left"
).sort_index()

# A intensidade mensal é uma razão de somas.
# Não é a média das intensidades horárias.

mensal["intensidade_t_mwh"] = (
    mensal["co2_t"]
    / mensal["geracao_mwh"].replace(0, np.nan)
)

mensal["geracao_gwh"] = mensal["geracao_mwh"] / 1000
mensal["co2_kt"] = mensal["co2_t"] / 1000

mensal["variacao_mensal_pct"] = (
    mensal["intensidade_t_mwh"].pct_change(
        fill_method=None
    ) * 100
)

mensal["ano"] = mensal.index.year
mensal["numero_mes"] = mensal.index.month

# ------------------------------------------------------------
# 4. VERIFICAÇÃO DE COBERTURA
# ------------------------------------------------------------

meses_esperados = pd.date_range(
    "2011-01-01",
    "2025-12-01",
    freq="MS"
)

if not mensal.index.equals(meses_esperados):
    raise ValueError(
        "A série mensal não contém exatamente "
        "os 180 meses esperados."
    )

# Meses sem geração válida são mantidos como NaN.
# Nunca transformar ausência de operação em intensidade zero.

print("=" * 70)
print("CÉLULA 06/20 — EVOLUÇÃO MENSAL")
print("=" * 70)

print(f"Meses no histórico: {len(mensal)}")
print(
    "Meses com intensidade calculável:",
    mensal["intensidade_t_mwh"].notna().sum()
)
print(
    "Meses sem intensidade calculável:",
    mensal["intensidade_t_mwh"].isna().sum()
)

# ------------------------------------------------------------
# 5. GRÁFICO HISTÓRICO — 180 MESES
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 5.5))

ax.plot(
    mensal.index,
    mensal["intensidade_t_mwh"],
    linewidth=1.7,
    marker=".",
    markersize=3,
    label="Intensidade mensal"
)

# Suavização apenas para visualização.
# Não é usada como série de detecção de quebras.
mensal["media_movel_3m"] = (
    mensal["intensidade_t_mwh"]
    .rolling(window=3, min_periods=3)
    .mean()
)

ax.plot(
    mensal.index,
    mensal["media_movel_3m"],
    linewidth=2.2,
    linestyle="--",
    label="Média móvel de 3 meses"
)

ax.set_title(
    "Iatan 2 — Intensidade carbônica mensal (2011–2025)"
)
ax.set_ylabel("Intensidade (tCO₂/MWh)")
ax.set_xlabel("Ano")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "intensidade_mensal_2011_2025.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. INTENSIDADE, CARGA E UTILIZAÇÃO
# ------------------------------------------------------------

fig, eixos = plt.subplots(
    3, 1,
    figsize=(15, 10),
    sharex=True
)

eixos[0].plot(
    mensal.index,
    mensal["intensidade_t_mwh"],
    linewidth=1.7
)
eixos[0].set_ylabel("tCO₂/MWh")
eixos[0].set_title("Intensidade carbônica")
eixos[0].grid(alpha=0.25)

eixos[1].plot(
    mensal.index,
    mensal["carga_media_mw"],
    linewidth=1.7
)
eixos[1].set_ylabel("MW")
eixos[1].set_title("Carga média nas horas elegíveis")
eixos[1].grid(alpha=0.25)

eixos[2].plot(
    mensal.index,
    mensal["utilizacao_calendario_pct"],
    linewidth=1.7
)
eixos[2].set_ylabel("Utilização (%)")
eixos[2].set_xlabel("Ano")
eixos[2].set_title(
    "Tempo de operação / horas do calendário"
)
eixos[2].grid(alpha=0.25)

fig.suptitle(
    "Iatan 2 — Intensidade e regime operacional",
    fontsize=14
)

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "intensidade_carga_utilizacao_mensal.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. JANELAS EXPLORATÓRIAS
# ------------------------------------------------------------

# Janelas amplas: não representam datas de intervenção.
janelas = {
    "Episódio A — 2012": ("2011-01", "2013-12"),
    "Episódio B — 2016–2018": ("2015-01", "2019-12"),
    "Episódio C — 2023–2025": ("2022-01", "2025-12")
}

fig, eixos = plt.subplots(
    3, 1,
    figsize=(14, 12)
)

for ax, (titulo, (inicio, fim)) in zip(
    eixos, janelas.items()
):

    trecho = mensal.loc[inicio:fim]

    ax.plot(
        trecho.index,
        trecho["intensidade_t_mwh"],
        marker="o",
        markersize=3,
        linewidth=1.6,
        label="Intensidade"
    )

    ax.plot(
        trecho.index,
        trecho["media_movel_3m"],
        linestyle="--",
        linewidth=1.8,
        label="Média móvel 3m"
    )

    ax.set_title(titulo)
    ax.set_ylabel("tCO₂/MWh")
    ax.grid(alpha=0.25)
    ax.legend()

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "janelas_exploratorias_mensais.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 8. VARIAÇÕES MENSAIS
# ------------------------------------------------------------

# Comparações exigem intensidade válida em meses
# consecutivos e cobertura mínima de horas elegíveis.
# O limiar de 20% é apenas diagnóstico exploratório.
COBERTURA_MINIMA = 20.0

pares_validos = (
    mensal["intensidade_t_mwh"].notna()
    & mensal["intensidade_t_mwh"].shift(1).notna()
    & mensal["cobertura_elegivel_pct"].ge(
        COBERTURA_MINIMA
    )
    & mensal["cobertura_elegivel_pct"].shift(1).ge(
        COBERTURA_MINIMA
    )
)

variacoes = mensal.loc[
    pares_validos,
    [
        "intensidade_t_mwh",
        "variacao_mensal_pct",
        "carga_media_mw",
        "cobertura_elegivel_pct"
    ]
].copy()

print("\nMAIORES REDUÇÕES MENSAIS")
display(
    variacoes.nsmallest(
        10, "variacao_mensal_pct"
    ).round(4)
)

print("\nMAIORES AUMENTOS MENSAIS")
display(
    variacoes.nlargest(
        10, "variacao_mensal_pct"
    ).round(4)
)

# ------------------------------------------------------------
# 9. RESUMO DAS JANELAS CANDIDATAS
# ------------------------------------------------------------

resumo_janelas = []

for nome, (inicio, fim) in janelas.items():

    trecho = mensal.loc[inicio:fim]
    ativos = trecho.loc[
        trecho["geracao_mwh"].gt(0)
        & trecho["co2_t"].notna()
    ]

    intensidade_agregada = (
        ativos["co2_t"].sum()
        / ativos["geracao_mwh"].sum()
        if not ativos.empty else np.nan
    )

    resumo_janelas.append({
        "janela": nome,
        "meses": len(trecho),
        "meses_com_intensidade": int(
            trecho["intensidade_t_mwh"].notna().sum()
        ),
        "intensidade_agregada_t_mwh": intensidade_agregada,
        "intensidade_minima_mensal": (
            trecho["intensidade_t_mwh"].min()
        ),
        "intensidade_maxima_mensal": (
            trecho["intensidade_t_mwh"].max()
        ),
        "carga_media_mw": (
            ativos["carga_media_mw"].mean()
            if not ativos.empty else np.nan
        )
    })

resumo_janelas = pd.DataFrame(resumo_janelas)

print("\nRESUMO EXPLORATÓRIO DAS JANELAS")
display(resumo_janelas.round(4))

# ------------------------------------------------------------
# 10. EXPORTAÇÃO
# ------------------------------------------------------------

mensal_exportar = (
    mensal.reset_index()
    .rename(columns={"index": "mes"})
)

ARQUIVO_MENSAL = (
    DIR_TABELAS /
    "intensidade_mensal_2011_2025.csv"
)

mensal_exportar.to_csv(
    ARQUIVO_MENSAL,
    index=False
)

resumo_janelas.to_csv(
    DIR_TABELAS / "janelas_exploratorias.csv",
    index=False
)

variacoes.to_csv(
    DIR_TABELAS / "variacoes_mensais_exploratorias.csv"
)

print("\n" + "=" * 70)
print("CÉLULA 06/20 — CONCLUÍDA")
print("=" * 70)

print("Tabela mensal:", ARQUIVO_MENSAL)
print("Figuras:", DIR_FIGURAS)
print("Meses preservados:", len(mensal))
print(
    "Nenhuma quebra estrutural foi "
    "fixada antecipadamente."
)

## 06 — Efeito da carga operacional

Analisa a dependência da IC em relação à carga e calcula resíduos ajustados. O ajuste usa todo o histórico: **descritivo, não preditivo fora da amostra**.

---

**Justificativa física — Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11).** O fator de emissão de termelétricas a carvão pode variar com o nível de carga e com o regime de operação. Assim, mudanças na distribuição das cargas podem simular alterações de intensidade caso não sejam consideradas. O modelo flexível ajustado no código para descrever a relação com carga é **exploratório, treinado com a série inteira**; seus resíduos não constituem previsões fora da amostra, nem identificam causas de mudanças.

In [ ]:
# ===== ORIGEM: CÉLULA 07 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 07/20 — CARGA, INTENSIDADE E MUDANÇAS EXPLORATÓRIAS
# Notebook 07 — IATAN 2 | EPA 6065/2 | 2011–2025
# ============================================================
#
# OBJETIVOS:
# 1. Investigar a relação entre carga e intensidade.
# 2. Comparar a intensidade em faixas de carga.
# 3. Ajustar uma relação não linear de referência.
# 4. Calcular desvios mensais após controle da carga.
# 5. Localizar períodos candidatos de mudança.
#
# Não identifica causalidade.
# Não utiliza heat rate.
# Não fixa o número de eventos.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

ARQUIVO_ANALITICO = (
    OUTPUT_ROOT / "dados" /
    "iatan2_intensidade_horaria_2011_2025.parquet"
)

DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
DIR_TABELAS.mkdir(parents=True, exist_ok=True)
DIR_DADOS.mkdir(parents=True, exist_ok=True)

if not ARQUIVO_ANALITICO.is_file():
    raise FileNotFoundError(
        "Base analítica não encontrada. "
        "Execute a Célula 04."
    )

df = pd.read_parquet(ARQUIVO_ANALITICO)

df["timestamp"] = pd.to_datetime(df["timestamp"])

# ------------------------------------------------------------
# 2. SELEÇÃO DE OBSERVAÇÕES VÁLIDAS
# ------------------------------------------------------------

dados = df.loc[
    df["elegivel_intensidade"].eq(True)
].copy()

dados = dados.loc[
    np.isfinite(dados["intensidade_t_mwh"])
    & np.isfinite(dados["carga_bruta_mw"])
    & dados["carga_bruta_mw"].gt(0)
    & dados["geracao_mwh"].gt(0)
].copy()

dados["ano"] = dados["timestamp"].dt.year

dados["mes"] = (
    dados["timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ------------------------------------------------------------
# 3. DIAGNÓSTICO DE CARGA
# ------------------------------------------------------------

print("=" * 70)
print("CÉLULA 07/20 — CARGA E INTENSIDADE")
print("=" * 70)

print("\nDISTRIBUIÇÃO DA CARGA")

display(
    dados["carga_bruta_mw"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
    .round(2)
)

# ------------------------------------------------------------
# 4. COMPARAÇÃO POR FAIXAS DE CARGA
# ------------------------------------------------------------

# Faixas fixas para permitir comparação entre anos.
# Não são quantis recalculados em cada período.

FAIXAS = [0, 300, 500, 600, 700, 800, 900, np.inf]

ROTULOS = [
    "0–300",
    "300–500",
    "500–600",
    "600–700",
    "700–800",
    "800–900",
    ">900"
]

dados["faixa_carga"] = pd.cut(
    dados["carga_bruta_mw"],
    bins=FAIXAS,
    labels=ROTULOS,
    include_lowest=False,
    right=True
)

resumo_faixas = (
    dados.groupby(
        "faixa_carga",
        observed=True
    )
    .agg(
        horas=("timestamp", "size"),
        geracao_mwh=("geracao_mwh", "sum"),
        co2_t=("co2_t", "sum"),
        carga_media_mw=("carga_bruta_mw", "mean")
    )
    .reset_index()
)

resumo_faixas["intensidade_t_mwh"] = (
    resumo_faixas["co2_t"]
    / resumo_faixas["geracao_mwh"]
)

print("\nINTENSIDADE POR FAIXA DE CARGA")
display(resumo_faixas.round(4))

# ------------------------------------------------------------
# 5. MODELO NÃO LINEAR DE REFERÊNCIA
# ------------------------------------------------------------

# Estimamos a massa de CO2 por unidade de tempo como
# função da carga, usando ponderação por tempo de operação.
#
# Depois convertemos a previsão em intensidade esperada.
#
# Essa abordagem evita ajustar diretamente razões
# extremas produzidas por geração próxima de zero.

X = dados[["carga_bruta_mw"]].to_numpy()

# Massa de CO2 por hora efetiva de operação
y = (
    dados["co2_t"].to_numpy()
    / dados["tempo_operacao_h"].to_numpy()
)

pesos = dados["tempo_operacao_h"].to_numpy()

transformador = SplineTransformer(
    n_knots=7,
    degree=3,
    knots="quantile",
    extrapolation="constant",
    include_bias=False
)

X_spline = transformador.fit_transform(X)

modelo = Ridge(alpha=10.0)

modelo.fit(
    X_spline,
    y,
    sample_weight=pesos
)

# Previsão de massa por hora efetiva
co2_taxa_prevista = modelo.predict(X_spline)

# Evita previsão fisicamente negativa
co2_taxa_prevista = np.maximum(
    co2_taxa_prevista, 0
)

dados["co2_t_previsto"] = (
    co2_taxa_prevista
    * dados["tempo_operacao_h"].to_numpy()
)

dados["intensidade_esperada_t_mwh"] = (
    dados["co2_t_previsto"]
    / dados["geracao_mwh"]
)

dados["residuo_co2_t"] = (
    dados["co2_t"]
    - dados["co2_t_previsto"]
)

# ------------------------------------------------------------
# 6. CURVA DE REFERÊNCIA POR CARGA
# ------------------------------------------------------------

carga_min = max(
    1.0,
    float(dados["carga_bruta_mw"].quantile(0.01))
)

carga_max = float(
    dados["carga_bruta_mw"].quantile(0.99)
)

grade_carga = np.linspace(
    carga_min,
    carga_max,
    250
).reshape(-1, 1)

taxa_grade = np.maximum(
    modelo.predict(
        transformador.transform(grade_carga)
    ),
    0
)

intensidade_grade = (
    taxa_grade / grade_carga.ravel()
)

fig, ax = plt.subplots(figsize=(11, 5.5))

# Amostra determinística para visualização.
amostra = dados.sample(
    n=min(5000, len(dados)),
    random_state=42
)

ax.scatter(
    amostra["carga_bruta_mw"],
    amostra["intensidade_t_mwh"],
    s=7,
    alpha=0.15,
    label="Observações horárias"
)

ax.plot(
    grade_carga.ravel(),
    intensidade_grade,
    linewidth=2.5,
    color="black",
    label="Relação de referência"
)

# Escala visual limitada ao percentil 99,5.
# Os dados extremos permanecem no modelo e na base.
limite_visual = float(
    dados["intensidade_t_mwh"].quantile(0.995)
)

ax.set_ylim(
    0,
    max(1.05, limite_visual * 1.1)
)

ax.set_xlabel("Carga bruta (MW)")
ax.set_ylabel("Intensidade (tCO₂/MWh)")
ax.set_title(
    "Iatan 2 — Relação entre carga e intensidade"
)
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "relacao_carga_intensidade.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. DESVIOS MENSAIS CONTROLANDO A CARGA
# ------------------------------------------------------------

# O desvio é calculado em massa e agregado mensalmente.
# Depois dividimos pela geração mensal.
#
# Isso equivale à diferença entre intensidade observada
# e intensidade prevista para a carga efetivamente operada.

mensal_residuos = (
    dados.groupby("mes")
    .agg(
        horas=("timestamp", "size"),
        geracao_mwh=("geracao_mwh", "sum"),
        co2_t=("co2_t", "sum"),
        co2_t_previsto=("co2_t_previsto", "sum"),
        residuo_co2_t=("residuo_co2_t", "sum"),
        carga_media_mw=("carga_bruta_mw", "mean")
    )
)

mensal_residuos["intensidade_observada"] = (
    mensal_residuos["co2_t"]
    / mensal_residuos["geracao_mwh"]
)

mensal_residuos["intensidade_esperada"] = (
    mensal_residuos["co2_t_previsto"]
    / mensal_residuos["geracao_mwh"]
)

mensal_residuos["desvio_t_mwh"] = (
    mensal_residuos["residuo_co2_t"]
    / mensal_residuos["geracao_mwh"]
)

# Reindexação: preservar os 180 meses, inclusive os
# nove meses sem intensidade calculável.

grade_mensal = pd.date_range(
    "2011-01-01",
    "2025-12-01",
    freq="MS"
)

mensal_residuos = mensal_residuos.reindex(
    grade_mensal
)

mensal_residuos.index.name = "mes"

# ------------------------------------------------------------
# 8. GRÁFICO DE DESVIOS AJUSTADOS
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 5.5))

ax.plot(
    mensal_residuos.index,
    mensal_residuos["desvio_t_mwh"],
    marker=".",
    markersize=4,
    linewidth=1.6,
    label="Desvio mensal ajustado"
)

ax.axhline(
    0,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.set_title(
    "Iatan 2 — Desvios de intensidade após ajuste pela carga"
)
ax.set_xlabel("Ano")
ax.set_ylabel("Desvio (tCO₂/MWh)")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "desvios_mensais_ajustados.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 9. COMPARAÇÃO DOS TRÊS EPISÓDIOS
# ------------------------------------------------------------

JANELAS = {
    "A — 2011–2013": ("2011-01", "2013-12"),
    "B — 2015–2019": ("2015-01", "2019-12"),
    "C — 2022–2025": ("2022-01", "2025-12")
}

fig, eixos = plt.subplots(
    3, 1,
    figsize=(14, 11)
)

for ax, (nome, (inicio, fim)) in zip(
    eixos, JANELAS.items()
):

    trecho = mensal_residuos.loc[inicio:fim]

    ax.plot(
        trecho.index,
        trecho["desvio_t_mwh"],
        marker="o",
        markersize=3,
        linewidth=1.6
    )

    ax.axhline(
        0,
        linestyle="--",
        color="black",
        linewidth=1
    )

    ax.set_title(nome)
    ax.set_ylabel("Desvio (tCO₂/MWh)")
    ax.grid(alpha=0.25)

fig.suptitle(
    "Iatan 2 — Episódios candidatos após controle da carga",
    fontsize=14
)

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "episodios_ajustados_carga.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 10. VARIAÇÕES DO DESVIO MENSAL
# ------------------------------------------------------------

# Comparar somente meses consecutivos observados.
# Evita tratar lacunas de operação como mudanças.

mensal_residuos["variacao_desvio"] = (
    mensal_residuos["desvio_t_mwh"].diff()
)

pares = (
    mensal_residuos["desvio_t_mwh"].notna()
    & mensal_residuos["desvio_t_mwh"].shift(1).notna()
)

comparacoes = mensal_residuos.loc[
    pares
].copy()

print("\nMAIORES REDUÇÕES DO DESVIO AJUSTADO")

display(
    comparacoes.nsmallest(
        12, "variacao_desvio"
    )[[
        "horas",
        "carga_media_mw",
        "intensidade_observada",
        "intensidade_esperada",
        "desvio_t_mwh",
        "variacao_desvio"
    ]].round(5)
)

print("\nMAIORES AUMENTOS DO DESVIO AJUSTADO")

display(
    comparacoes.nlargest(
        12, "variacao_desvio"
    )[[
        "horas",
        "carga_media_mw",
        "intensidade_observada",
        "intensidade_esperada",
        "desvio_t_mwh",
        "variacao_desvio"
    ]].round(5)
)

# ------------------------------------------------------------
# 11. EXPORTAÇÃO
# ------------------------------------------------------------

ARQUIVO_MENSAL_AJUSTADO = (
    DIR_TABELAS /
    "intensidade_mensal_ajustada_carga.csv"
)

ARQUIVO_HORARIO_AJUSTADO = (
    DIR_DADOS /
    "iatan2_residuos_horarios_carga.parquet"
)

mensal_residuos.reset_index().to_csv(
    ARQUIVO_MENSAL_AJUSTADO,
    index=False
)

dados[[
    "timestamp",
    "carga_bruta_mw",
    "tempo_operacao_h",
    "geracao_mwh",
    "co2_t",
    "intensidade_t_mwh",
    "co2_t_previsto",
    "intensidade_esperada_t_mwh",
    "residuo_co2_t"
]].to_parquet(
    ARQUIVO_HORARIO_AJUSTADO,
    index=False
)

print("\n" + "=" * 70)
print("CÉLULA 07/20 — CONCLUÍDA")
print("=" * 70)

print("Horas utilizadas:", f"{len(dados):,}")
print(
    "Meses com desvio calculável:",
    int(mensal_residuos["desvio_t_mwh"].notna().sum())
)
print("Tabela:", ARQUIVO_MENSAL_AJUSTADO)
print("Base horária:", ARQUIVO_HORARIO_AJUSTADO)
print("Figuras:", DIR_FIGURAS)

print(
    "\nATENÇÃO: os desvios são exploratórios. "
    "O modelo foi ajustado com todo o histórico, "
    "portanto não constitui uma avaliação preditiva "
    "fora da amostra."
)

## 07 — Segmentação mensal e robustez

Aplica segmentação por programação dinâmica/BIC e verifica estabilidade com alterações de parâmetros e faixas de carga. As quebras são candidatas, não manutenção confirmada.

---

**Fundamentos — Bai e Perron (2003, §3, pp. 3–6); Truong, Oudre e Vayatis (2020); Schwarz (1978, §1, p. 461).** A detecção retrospectiva de mudanças pode ser formulada como minimização do custo de segmentação, com restrições ao número e ao tamanho dos segmentos. Este notebook implementa **programação dinâmica e SSE ponderada**, selecionando o número de quebras por uma expressão de **BIC aproximado**. Como os pesos e as restrições alteram as condições probabilísticas usuais, o valor resultante é usado como critério **exploratório**, não como teste formal do BIC nem como implementação idêntica à de Bai–Perron. Não foi implementado o algoritmo PELT.

In [ ]:
# ===== ORIGEM: CÉLULA 08 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 08/20 — DETECÇÃO EXPLORATÓRIA DE MUDANÇAS
# Notebook 07 — IATAN 2 | 2011–2025
# ============================================================
#
# MÉTODO:
# Segmentação de médias por programação dinâmica.
# Critério: soma ponderada dos quadrados + BIC.
#
# A série analisada é o desvio mensal de intensidade
# após ajuste pela carga (Célula 07).
#
# PRECAUÇÕES:
# - Não preencher meses sem dados.
# - Não utilizar média simples de intensidades horárias.
# - Exigir duração e cobertura mínimas por segmento.
# - Comparar 0 a 6 pontos de mudança.
# - Não atribuir causalidade à manutenção.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

ARQUIVO_ENTRADA = (
    DIR_TABELAS /
    "intensidade_mensal_ajustada_carga.csv"
)

if not ARQUIVO_ENTRADA.is_file():
    raise FileNotFoundError(
        "Arquivo da Célula 07 não encontrado."
    )

MAX_QUEBRAS = 6

# Duração mínima de cada regime, em meses de calendário.
MIN_MESES_SEGMENTO = 9

# Mínimo de meses com dados por segmento.
MIN_MESES_OBSERVADOS = 6

# Mínimo de horas elegíveis por mês para participar
# da estimação de mudanças.
MIN_HORAS_MES = 100

# Impede que meses de operação muito curta dominem
# a análise. Não modifica os dados originais.
PESO_MAX_RELATIVO = 3.0

DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
DIR_TABELAS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. CARREGAMENTO E VERIFICAÇÃO
# ------------------------------------------------------------

mensal = pd.read_csv(
    ARQUIVO_ENTRADA,
    parse_dates=["mes"]
)

mensal = (
    mensal.set_index("mes")
    .sort_index()
)

grade = pd.date_range(
    "2011-01-01",
    "2025-12-01",
    freq="MS"
)

if not mensal.index.equals(grade):
    raise ValueError(
        "A série deve conter exatamente 180 meses."
    )

colunas = [
    "desvio_t_mwh",
    "geracao_mwh",
    "horas",
    "intensidade_observada",
    "intensidade_esperada"
]

faltantes = set(colunas) - set(mensal.columns)

if faltantes:
    raise KeyError(
        f"Colunas ausentes: {sorted(faltantes)}"
    )

# ------------------------------------------------------------
# 3. MESES ADMISSÍVEIS
# ------------------------------------------------------------

admissivel = (
    mensal["desvio_t_mwh"].notna()
    & np.isfinite(mensal["desvio_t_mwh"])
    & mensal["geracao_mwh"].gt(0)
    & mensal["horas"].ge(MIN_HORAS_MES)
)

serie = mensal["desvio_t_mwh"].to_numpy(dtype=float)

# Pesos proporcionais à geração mensal, com limite
# superior para evitar concentração excessiva.
geracao = mensal["geracao_mwh"].to_numpy(dtype=float)

pesos = np.zeros(len(mensal), dtype=float)

geracao_valida = geracao[admissivel.to_numpy()]

if len(geracao_valida) < 30:
    raise ValueError(
        "Quantidade insuficiente de meses admissíveis."
    )

mediana_geracao = float(
    np.median(geracao_valida)
)

pesos[admissivel.to_numpy()] = np.minimum(
    geracao[admissivel.to_numpy()] / mediana_geracao,
    PESO_MAX_RELATIVO
)

y = np.where(
    admissivel.to_numpy(),
    serie,
    0.0
)

w = pesos

print("=" * 70)
print("CÉLULA 08/20 — DETECÇÃO DE MUDANÇAS")
print("=" * 70)

print("Meses no calendário:", len(mensal))
print("Meses admissíveis:", int(admissivel.sum()))
print("Meses não admissíveis:", int((~admissivel).sum()))
print("Duração mínima:", MIN_MESES_SEGMENTO, "meses")
print("Máximo de quebras:", MAX_QUEBRAS)

# ------------------------------------------------------------
# 4. SOMAS ACUMULADAS PARA SEGMENTAÇÃO
# ------------------------------------------------------------

n = len(y)

W = np.r_[0.0, np.cumsum(w)]
WY = np.r_[0.0, np.cumsum(w * y)]
WY2 = np.r_[0.0, np.cumsum(w * y * y)]

NOBS = np.r_[
    0,
    np.cumsum(admissivel.to_numpy().astype(int))
]

def estatisticas_segmento(i, j):
    """
    Segmento [i, j), com j exclusivo.
    Retorna peso, observações, média e custo SSE.
    """

    duracao = j - i
    observacoes = int(NOBS[j] - NOBS[i])

    if (
        duracao < MIN_MESES_SEGMENTO
        or observacoes < MIN_MESES_OBSERVADOS
    ):
        return None

    sw = W[j] - W[i]

    if sw <= 0:
        return None

    swy = WY[j] - WY[i]
    swy2 = WY2[j] - WY2[i]

    media = swy / sw

    sse = max(
        0.0,
        swy2 - (swy * swy / sw)
    )

    return sw, observacoes, media, sse

# ------------------------------------------------------------
# 5. PROGRAMAÇÃO DINÂMICA
# ------------------------------------------------------------

MAX_SEGMENTOS = MAX_QUEBRAS + 1

dp = np.full(
    (MAX_SEGMENTOS + 1, n + 1),
    np.inf
)

anterior = np.full(
    (MAX_SEGMENTOS + 1, n + 1),
    -1,
    dtype=int
)

dp[0, 0] = 0.0

for k in range(1, MAX_SEGMENTOS + 1):

    for j in range(1, n + 1):

        for i in range(0, j):

            if not np.isfinite(dp[k - 1, i]):
                continue

            stats = estatisticas_segmento(i, j)

            if stats is None:
                continue

            custo = dp[k - 1, i] + stats[3]

            if custo < dp[k, j]:
                dp[k, j] = custo
                anterior[k, j] = i

# ------------------------------------------------------------
# 6. COMPARAÇÃO DOS MODELOS PELO BIC
# ------------------------------------------------------------

# BIC aproximado para médias segmentadas:
# N*ln(SSE/N) + p*ln(N)
#
# p = número de médias dos segmentos + número de
# posições de quebra. Penalização conservadora.
#
# A ponderação altera a interpretação probabilística
# do BIC; usamos o critério como seleção exploratória.

N = int(admissivel.sum())

modelos = []

for k in range(1, MAX_SEGMENTOS + 1):

    sse = dp[k, n]

    if not np.isfinite(sse):
        continue

    numero_quebras = k - 1
    parametros = k + numero_quebras

    bic = (
        N * np.log(max(sse / N, 1e-15))
        + parametros * np.log(N)
    )

    modelos.append({
        "segmentos": k,
        "quebras": numero_quebras,
        "sse_ponderada": sse,
        "bic_aproximado": bic
    })

comparacao = pd.DataFrame(modelos)

if comparacao.empty:
    raise ValueError(
        "Nenhuma segmentação viável encontrada."
    )

melhor = comparacao.loc[
    comparacao["bic_aproximado"].idxmin()
]

K_OTIMO = int(melhor["segmentos"])

print("\nCOMPARAÇÃO DOS MODELOS")
display(comparacao.round(5))

print(
    "\nNúmero selecionado de quebras:",
    K_OTIMO - 1
)

# ------------------------------------------------------------
# 7. RECONSTRUÇÃO DOS SEGMENTOS
# ------------------------------------------------------------

segmentos = []

j = n

for k in range(K_OTIMO, 0, -1):

    i = int(anterior[k, j])

    if i < 0:
        raise RuntimeError(
            "Falha ao reconstruir a segmentação."
        )

    segmentos.append((i, j))
    j = i

segmentos.reverse()

if segmentos[0][0] != 0 or segmentos[-1][1] != n:
    raise RuntimeError(
        "Segmentação não cobre todo o calendário."
    )

# ------------------------------------------------------------
# 8. REGIMES ESTIMADOS
# ------------------------------------------------------------

regimes = []
serie_ajustada = np.full(n, np.nan)

for identificador, (i, j) in enumerate(segmentos):

    sw, nobs, media, sse = estatisticas_segmento(i, j)

    serie_ajustada[i:j] = media

    regimes.append({
        "regime_id": identificador,
        "inicio": mensal.index[i],
        "fim": mensal.index[j - 1],
        "meses_calendario": j - i,
        "meses_observados": nobs,
        "media_desvio_t_mwh": media,
        "peso_total": sw,
        "sse_ponderada": sse
    })

regimes_df = pd.DataFrame(regimes)

print("\nREGIMES ESTIMADOS")
display(regimes_df.round(5))

# ------------------------------------------------------------
# 9. DATAS CANDIDATAS E MAGNITUDES
# ------------------------------------------------------------

eventos = []

for idx in range(1, len(segmentos)):

    inicio_regime = segmentos[idx][0]

    media_antes = regimes_df.loc[
        idx - 1, "media_desvio_t_mwh"
    ]

    media_depois = regimes_df.loc[
        idx, "media_desvio_t_mwh"
    ]

    delta = media_depois - media_antes

    eventos.append({
        "evento_id": idx,
        "mes_candidato": mensal.index[inicio_regime],
        "media_antes": media_antes,
        "media_depois": media_depois,
        "delta_t_mwh": delta,
        "direcao": (
            "reducao" if delta < 0
            else "aumento" if delta > 0
            else "sem_alteracao"
        )
    })

eventos_df = pd.DataFrame(
    eventos,
    columns=[
        "evento_id",
        "mes_candidato",
        "media_antes",
        "media_depois",
        "delta_t_mwh",
        "direcao"
    ]
)

print("\nPONTOS DE MUDANÇA CANDIDATOS")
display(eventos_df.round(5))

# ------------------------------------------------------------
# 10. GRÁFICO DA SEGMENTAÇÃO
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(
    mensal.index,
    mensal["desvio_t_mwh"],
    marker=".",
    markersize=4,
    linewidth=1.3,
    label="Desvio mensal observado"
)

ax.plot(
    mensal.index,
    serie_ajustada,
    linewidth=2.5,
    color="black",
    label="Média estimada por regime"
)

ax.axhline(
    0,
    linestyle=":",
    color="gray",
    linewidth=1
)

for evento in eventos:

    data = evento["mes_candidato"]

    ax.axvline(
        data,
        linestyle="--",
        linewidth=1.2,
        color="tab:red",
        alpha=0.8
    )

ax.set_title(
    "Iatan 2 — Segmentação exploratória dos desvios "
    "de intensidade ajustados pela carga"
)

ax.set_xlabel("Ano")
ax.set_ylabel("Desvio (tCO₂/MWh)")
ax.grid(alpha=0.2)
ax.legend()

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "segmentacao_mensal_ajustada_carga.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 11. GRÁFICO DO CRITÉRIO DE SELEÇÃO
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(
    comparacao["quebras"],
    comparacao["bic_aproximado"],
    marker="o",
    linewidth=2
)

ax.axvline(
    K_OTIMO - 1,
    linestyle="--",
    color="tab:red",
    label="Modelo selecionado"
)

ax.set_xlabel("Número de quebras")
ax.set_ylabel("BIC aproximado")
ax.set_title("Comparação da complexidade dos modelos")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS / "comparacao_bic_quebras.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 12. EXPORTAÇÃO
# ------------------------------------------------------------

comparacao.to_csv(
    DIR_TABELAS / "comparacao_modelos_quebras.csv",
    index=False
)

regimes_df.to_csv(
    DIR_TABELAS / "regimes_mensais_estimados.csv",
    index=False
)

eventos_df.to_csv(
    DIR_TABELAS / "eventos_mensais_candidatos.csv",
    index=False
)

mensal_saida = mensal.copy()

mensal_saida["regime_estimado"] = serie_ajustada
mensal_saida["mes_admissivel"] = admissivel

mensal_saida.reset_index().to_csv(
    DIR_TABELAS / "serie_mensal_segmentada.csv",
    index=False
)

print("\n" + "=" * 70)
print("CÉLULA 08/20 — CONCLUÍDA")
print("=" * 70)

print("Meses analisados:", N)
print("Regimes selecionados:", K_OTIMO)
print("Quebras candidatas:", K_OTIMO - 1)
print("Tabelas:", DIR_TABELAS)
print("Figuras:", DIR_FIGURAS)

print(
    "\nAs datas são candidatas mensais, não datas "
    "confirmadas de intervenções ou manutenções."
)


# ===== ORIGEM: CÉLULA 09 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 09/20 — ROBUSTEZ DA SEGMENTAÇÃO MENSAL
# Notebook 07 — IATAN 2 | EPA 6065/2 | 2011–2025
# ============================================================
#
# OBJETIVOS:
# 1. Testar a estabilidade das quebras mensais.
# 2. Variar duração mínima e cobertura operacional.
# 3. Comparar esquemas de ponderação.
# 4. Variar o número máximo de quebras.
# 5. Reestimar desvios usando cargas comparáveis.
# 6. Produzir ranking exploratório de estabilidade.
#
# Não atribui causalidade à manutenção.
# Não utiliza heat rate.
# Não define datas horárias nesta etapa.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from IPython.display import display

# ------------------------------------------------------------
# 1. ENTRADAS
# ------------------------------------------------------------

ARQUIVO_MENSAL = (
    DIR_TABELAS / "intensidade_mensal_ajustada_carga.csv"
)

ARQUIVO_HORARIO = (
    OUTPUT_ROOT / "dados" /
    "iatan2_intensidade_horaria_2011_2025.parquet"
)

for arquivo in [ARQUIVO_MENSAL, ARQUIVO_HORARIO]:
    if not arquivo.is_file():
        raise FileNotFoundError(
            f"Arquivo necessário não encontrado: {arquivo}"
        )

DIR_TABELAS.mkdir(parents=True, exist_ok=True)
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

mensal_base = pd.read_csv(
    ARQUIVO_MENSAL,
    parse_dates=["mes"]
).set_index("mes").sort_index()

grade = pd.date_range(
    "2011-01-01", "2025-12-01", freq="MS"
)

if not mensal_base.index.equals(grade):
    raise ValueError(
        "A série mensal precisa conter os 180 meses."
    )

horario = pd.read_parquet(ARQUIVO_HORARIO)

horario["timestamp"] = pd.to_datetime(
    horario["timestamp"]
)

horario = horario.loc[
    horario["elegivel_intensidade"].eq(True)
].copy()

horario["mes"] = (
    horario["timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ------------------------------------------------------------
# 2. FUNÇÃO DE SEGMENTAÇÃO
# ------------------------------------------------------------

def segmentar_serie(
    tabela,
    min_meses=9,
    min_observados=6,
    min_horas=100,
    max_quebras=6,
    ponderacao="geracao",
    peso_max=3.0
):
    """
    Segmentação de médias por programação dinâmica.

    As quebras ocorrem em fronteiras mensais.
    Lacunas permanecem no calendário.

    Retorna:
      - tabela de modelos
      - modelo selecionado
      - datas das quebras
      - segmentos estimados
    """

    serie = tabela["desvio_t_mwh"].to_numpy(
        dtype=float
    )

    geracao = tabela["geracao_mwh"].to_numpy(
        dtype=float
    )

    horas = tabela["horas"].to_numpy(
        dtype=float
    )

    n = len(tabela)

    admissivel = (
        np.isfinite(serie)
        & np.isfinite(geracao)
        & (geracao > 0)
        & np.isfinite(horas)
        & (horas >= min_horas)
    )

    if admissivel.sum() < 30:
        return None

    w = np.zeros(n, dtype=float)

    if ponderacao == "geracao":

        referencia = np.median(
            geracao[admissivel]
        )

        w[admissivel] = np.minimum(
            geracao[admissivel] / referencia,
            peso_max
        )

    elif ponderacao == "horas":

        referencia = np.median(
            horas[admissivel]
        )

        w[admissivel] = np.minimum(
            horas[admissivel] / referencia,
            peso_max
        )

    elif ponderacao == "uniforme":

        w[admissivel] = 1.0

    else:
        raise ValueError(
            f"Ponderação desconhecida: {ponderacao}"
        )

    y = np.where(admissivel, serie, 0.0)

    W = np.r_[0.0, np.cumsum(w)]
    WY = np.r_[0.0, np.cumsum(w * y)]
    WY2 = np.r_[0.0, np.cumsum(w * y * y)]

    NOBS = np.r_[
        0,
        np.cumsum(admissivel.astype(int))
    ]

    def estatisticas(i, j):

        duracao = j - i
        observados = int(NOBS[j] - NOBS[i])

        if (
            duracao < min_meses
            or observados < min_observados
        ):
            return None

        sw = W[j] - W[i]

        if sw <= 0:
            return None

        swy = WY[j] - WY[i]
        swy2 = WY2[j] - WY2[i]

        media = swy / sw

        sse = max(
            0.0,
            swy2 - swy * swy / sw
        )

        return media, sse, observados

    KMAX = max_quebras + 1

    dp = np.full(
        (KMAX + 1, n + 1),
        np.inf
    )

    anterior = np.full(
        (KMAX + 1, n + 1),
        -1,
        dtype=int
    )

    dp[0, 0] = 0.0

    # Programação dinâmica
    for k in range(1, KMAX + 1):

        for j in range(k * min_meses, n + 1):

            inicio_minimo = (
                (k - 1) * min_meses
            )

            fim_maximo = j - min_meses

            for i in range(
                inicio_minimo,
                fim_maximo + 1
            ):

                if not np.isfinite(
                    dp[k - 1, i]
                ):
                    continue

                stats = estatisticas(i, j)

                if stats is None:
                    continue

                custo = (
                    dp[k - 1, i] + stats[1]
                )

                if custo < dp[k, j]:
                    dp[k, j] = custo
                    anterior[k, j] = i

    # Seleção exploratória pelo BIC aproximado
    N = int(admissivel.sum())

    modelos = []

    for k in range(1, KMAX + 1):

        sse = dp[k, n]

        if not np.isfinite(sse):
            continue

        parametros = 2 * k - 1

        bic = (
            N * np.log(
                max(sse / N, 1e-15)
            )
            + parametros * np.log(N)
        )

        modelos.append({
            "segmentos": k,
            "quebras": k - 1,
            "sse": sse,
            "bic": bic
        })

    if not modelos:
        return None

    modelos_df = pd.DataFrame(modelos)

    melhor = modelos_df.loc[
        modelos_df["bic"].idxmin()
    ]

    k_otimo = int(
        melhor["segmentos"]
    )

    # Reconstrução
    segmentos = []
    j = n

    for k in range(k_otimo, 0, -1):

        i = int(anterior[k, j])

        if i < 0:
            raise RuntimeError(
                "Falha na reconstrução."
            )

        segmentos.append((i, j))
        j = i

    segmentos.reverse()

    datas_quebras = [
        tabela.index[i]
        for i, _ in segmentos[1:]
    ]

    regimes = []

    for idx, (i, j) in enumerate(segmentos):

        media, sse, obs = estatisticas(i, j)

        regimes.append({
            "regime_id": idx,
            "inicio": tabela.index[i],
            "fim": tabela.index[j - 1],
            "meses": j - i,
            "meses_observados": obs,
            "media_desvio": media
        })

    return {
        "modelos": modelos_df,
        "melhor": melhor,
        "datas": datas_quebras,
        "regimes": pd.DataFrame(regimes),
        "n_admissiveis": N
    }

# ------------------------------------------------------------
# 3. CENÁRIOS DE SENSIBILIDADE
# ------------------------------------------------------------

DURACOES = [6, 9, 12, 18]
MIN_HORAS = [100, 200, 300]
PONDERACOES = [
    "geracao", "horas", "uniforme"
]
MAX_QUEBRAS = [3, 4, 6]

cenarios = list(product(
    DURACOES,
    MIN_HORAS,
    PONDERACOES,
    MAX_QUEBRAS
))

print("=" * 70)
print("CÉLULA 09/20 — ROBUSTEZ DA SEGMENTAÇÃO")
print("=" * 70)
print("Cenários principais:", len(cenarios))

resultados = []
quebras_cenarios = []

for numero, (
    duracao,
    horas_min,
    ponderacao,
    max_q
) in enumerate(cenarios, start=1):

    resultado = segmentar_serie(
        mensal_base,
        min_meses=duracao,
        min_observados=max(
            4, int(np.ceil(duracao * 0.6))
        ),
        min_horas=horas_min,
        max_quebras=max_q,
        ponderacao=ponderacao
    )

    if resultado is None:
        continue

    identificador = f"C{numero:03d}"

    resultados.append({
        "cenario": identificador,
        "duracao_minima": duracao,
        "horas_minimas": horas_min,
        "ponderacao": ponderacao,
        "max_quebras": max_q,
        "quebras_selecionadas": len(
            resultado["datas"]
        ),
        "meses_admissiveis": (
            resultado["n_admissiveis"]
        ),
        "bic": float(
            resultado["melhor"]["bic"]
        ),
        "atingiu_limite": (
            len(resultado["datas"]) == max_q
        )
    })

    for data in resultado["datas"]:

        quebras_cenarios.append({
            "cenario": identificador,
            "data": data,
            "duracao_minima": duracao,
            "horas_minimas": horas_min,
            "ponderacao": ponderacao,
            "max_quebras": max_q
        })

resultados_df = pd.DataFrame(resultados)
quebras_df = pd.DataFrame(quebras_cenarios)

if resultados_df.empty:
    raise ValueError(
        "Nenhum cenário de robustez foi viável."
    )

print("\nRESUMO DOS CENÁRIOS")
print("Cenários válidos:", len(resultados_df))
print(
    "Cenários que atingiram o limite:",
    int(resultados_df["atingiu_limite"].sum())
)

display(
    resultados_df.groupby(
        "max_quebras"
    ).agg(
        cenarios=("cenario", "size"),
        media_quebras=(
            "quebras_selecionadas", "mean"
        ),
        pct_no_limite=(
            "atingiu_limite", "mean"
        )
    ).round(3)
)

# ------------------------------------------------------------
# 4. ESTABILIDADE DAS DATAS CANDIDATAS
# ------------------------------------------------------------

CANDIDATOS = {
    "2011-12": "2011-12-01",
    "2012-10": "2012-10-01",
    "2017-03": "2017-03-01",
    "2021-07": "2021-07-01",
    "2023-04": "2023-04-01",
    "2025-04": "2025-04-01"
}

# Uma quebra é considerada próxima quando está
# a até 3 meses de distância do candidato.
TOLERANCIA_MESES = 3

def indice_mes(data):
    return data.year * 12 + data.month

estabilidade = []

for nome, data_texto in CANDIDATOS.items():

    referencia = pd.Timestamp(data_texto)
    indice_ref = indice_mes(referencia)

    cenarios_com_quebra = set()

    distancias = []

    for _, linha in quebras_df.iterrows():

        distancia = abs(
            indice_mes(linha["data"])
            - indice_ref
        )

        if distancia <= TOLERANCIA_MESES:

            cenarios_com_quebra.add(
                linha["cenario"]
            )

            distancias.append(distancia)

    estabilidade.append({
        "candidato": nome,
        "cenarios_com_quebra": len(
            cenarios_com_quebra
        ),
        "total_cenarios": len(
            resultados_df
        ),
        "frequencia_pct": (
            100 * len(cenarios_com_quebra)
            / len(resultados_df)
        ),
        "distancia_mediana_meses": (
            float(np.median(distancias))
            if distancias else np.nan
        )
    })

estabilidade_df = pd.DataFrame(
    estabilidade
).sort_values(
    "frequencia_pct",
    ascending=False
)

print("\nESTABILIDADE DOS CANDIDATOS")
display(estabilidade_df.round(2))

# ------------------------------------------------------------
# 5. ROBUSTEZ COM FAIXA COMUM DE CARGA
# ------------------------------------------------------------

# Comparação independente em carga intermediária.
# Não refaz o modelo spline com esses dados:
# reutiliza a referência estimada na Célula 07.
#
# A restrição reduz diferenças de perfil operacional,
# mas não elimina todos os confundidores.

ARQUIVO_RESIDUOS = (
    DIR_DADOS /
    "iatan2_residuos_horarios_carga.parquet"
)

if not ARQUIVO_RESIDUOS.is_file():
    raise FileNotFoundError(
        "Resíduos da Célula 07 não encontrados."
    )

residuos = pd.read_parquet(
    ARQUIVO_RESIDUOS
)

residuos["timestamp"] = pd.to_datetime(
    residuos["timestamp"]
)

residuos["mes"] = (
    residuos["timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Faixa de comparação: 650 a 850 MW.
CARGA_MIN = 650
CARGA_MAX = 850

comum = residuos.loc[
    residuos["carga_bruta_mw"].between(
        CARGA_MIN,
        CARGA_MAX
    )
].copy()

comum_mensal = (
    comum.groupby("mes")
    .agg(
        horas=("timestamp", "size"),
        geracao_mwh=("geracao_mwh", "sum"),
        residuo_co2_t=("residuo_co2_t", "sum")
    )
    .reindex(grade)
)

comum_mensal["desvio_t_mwh"] = (
    comum_mensal["residuo_co2_t"]
    / comum_mensal["geracao_mwh"].replace(
        0, np.nan
    )
)

comum_mensal.index.name = "mes"

resultado_comum = segmentar_serie(
    comum_mensal,
    min_meses=9,
    min_observados=6,
    min_horas=100,
    max_quebras=6,
    ponderacao="geracao"
)

print(
    "\nROBUSTEZ — FAIXA DE CARGA "
    f"{CARGA_MIN}–{CARGA_MAX} MW"
)

print(
    "Horas na faixa:",
    f"{len(comum):,}"
)

if resultado_comum is not None:

    print(
        "Meses admissíveis:",
        resultado_comum["n_admissiveis"]
    )

    print(
        "Quebras selecionadas:",
        len(resultado_comum["datas"])
    )

    display(
        resultado_comum["regimes"].round(5)
    )

else:
    print(
        "Dados insuficientes para segmentação "
        "nessa faixa."
    )

# ------------------------------------------------------------
# 6. GRÁFICO COMPARATIVO
# ------------------------------------------------------------

fig, eixos = plt.subplots(
    2, 1,
    figsize=(15, 9),
    sharex=True
)

eixos[0].plot(
    mensal_base.index,
    mensal_base["desvio_t_mwh"],
    linewidth=1.5,
    label="Todas as cargas"
)

eixos[1].plot(
    comum_mensal.index,
    comum_mensal["desvio_t_mwh"],
    linewidth=1.5,
    label=f"Cargas {CARGA_MIN}–{CARGA_MAX} MW"
)

for ax in eixos:

    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1
    )

    ax.grid(alpha=0.25)
    ax.legend()
    ax.set_ylabel("Desvio (tCO₂/MWh)")

eixos[0].set_title(
    "Série completa — resíduos ajustados"
)

eixos[1].set_title(
    "Restrição a uma faixa comum de carga"
)

eixos[1].set_xlabel("Ano")

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS /
    "robustez_faixa_comum_carga.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. GRÁFICO DE ESTABILIDADE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(
    estabilidade_df["candidato"],
    estabilidade_df["frequencia_pct"]
)

ax.set_xlim(0, 100)
ax.set_xlabel(
    "Cenários com quebra próxima (%)"
)
ax.set_title(
    "Estabilidade das datas candidatas "
    "(tolerância ±3 meses)"
)

ax.grid(
    axis="x",
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    DIR_FIGURAS /
    "estabilidade_quebras_mensais.png",
    dpi=220,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 8. EXPORTAÇÃO
# ------------------------------------------------------------

resultados_df.to_csv(
    DIR_TABELAS /
    "robustez_cenarios_segmentacao.csv",
    index=False
)

quebras_df.to_csv(
    DIR_TABELAS /
    "robustez_datas_quebras.csv",
    index=False
)

estabilidade_df.to_csv(
    DIR_TABELAS /
    "robustez_estabilidade_candidatos.csv",
    index=False
)

comum_mensal.reset_index().to_csv(
    DIR_TABELAS /
    "robustez_carga_comum_mensal.csv",
    index=False
)

if resultado_comum is not None:

    resultado_comum["regimes"].to_csv(
        DIR_TABELAS /
        "robustez_regimes_carga_comum.csv",
        index=False
    )

print("\n" + "=" * 70)
print("CÉLULA 09/20 — CONCLUÍDA")
print("=" * 70)

print("Cenários executados:", len(resultados_df))
print("Datas candidatas avaliadas:", len(CANDIDATOS))
print("Tabelas:", DIR_TABELAS)
print("Figuras:", DIR_FIGURAS)

print(
    "\nIMPORTANTE: estabilidade não significa "
    "causalidade ou confirmação de manutenção."
)

## 08 — Detecção diária e refinamento horário

Refina mudanças candidatas em janelas diárias e horárias, incluindo março e julho de 2025. Usa CO₂ observado para escolher eventos: **análise retrospectiva com acesso ao alvo**.

---

**Enquadramento — Truong et al. (2020, §2.2 e §5.1.1); Jewell, Fearnhead e Witten (2022).** Trata-se de *offline changepoint detection*: toda a série, inclusive emissões futuras em relação ao instante examinado, está disponível para localizar as mudanças. A seleção de datas pelo próprio CO₂ observado introduz incerteza de pós-seleção; estimativas e testes subsequentes condicionados aos cortes **não** equivalem à detecção em operação nem a inferência seletiva ajustada.

In [ ]:
# ===== ORIGEM: CÉLULA 10 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 10/20 — DETECÇÃO DIÁRIA CORRIGIDA
# Notebook 07 — IATAN 2 | EPA 6065/2 | 2011–2025
# ============================================================
#
# Correção:
# - Excluir da detecção regimes de geração muito baixa.
# - Preservar integralmente os dados originais.
# - Calcular desvios pela razão de somas.
# - Preservar lacunas do calendário.
# - Investigar 2017, 2023 e 2025.
# - Testar sensibilidade aos limites de carga.
#
# Não utiliza heat rate.
# Não pressupõe eventos de manutenção.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

ARQUIVO_RESIDUOS = (
    DIR_DADOS /
    "iatan2_residuos_horarios_carga.parquet"
)

JANELAS = {
    "2017": ("2016-01-01", "2018-12-31"),
    "2023": ("2022-01-01", "2024-06-30"),
    "2025": ("2024-07-01", "2025-12-31")
}

CARGA_MIN_PRINCIPAL = 400
CARGAS_SENSIBILIDADE = [300, 400, 500, 650]

MIN_HORAS_DIA = 8
MIN_DIAS_SEGMENTO = 45
MIN_DIAS_OBSERVADOS = 20
MAX_QUEBRAS = 3
PESO_MAX_RELATIVO = 3.0

DIR_TABELAS.mkdir(parents=True, exist_ok=True)
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. CARREGAMENTO E AUDITORIA
# ------------------------------------------------------------

h = pd.read_parquet(ARQUIVO_RESIDUOS).copy()

colunas = [
    "timestamp",
    "carga_bruta_mw",
    "tempo_operacao_h",
    "geracao_mwh",
    "co2_t",
    "co2_t_previsto",
    "residuo_co2_t"
]

faltantes = set(colunas) - set(h.columns)

if faltantes:
    raise KeyError(
        f"Colunas ausentes: {sorted(faltantes)}"
    )

h["timestamp"] = pd.to_datetime(
    h["timestamp"], errors="coerce"
)

if h["timestamp"].isna().any():
    raise ValueError("Timestamps inválidos.")

if h["timestamp"].duplicated().any():
    raise ValueError("Timestamps duplicados.")

for coluna in colunas[1:]:
    h[coluna] = pd.to_numeric(
        h[coluna], errors="coerce"
    )

h["dia"] = h["timestamp"].dt.floor("D")

erro_identidade = (
    h["co2_t"]
    - h["co2_t_previsto"]
    - h["residuo_co2_t"]
).abs().max()

if erro_identidade > 1e-6:
    raise ValueError(
        "Inconsistência na identidade dos resíduos."
    )

print("=" * 70)
print("CÉLULA 10/20 — DETECÇÃO DIÁRIA CORRIGIDA")
print("=" * 70)

print("Horas na base original:", len(h))
print("Erro máximo da identidade:", erro_identidade)

# ------------------------------------------------------------
# 3. FILTRO OPERACIONAL
# ------------------------------------------------------------

def filtrar_operacao(
    dados,
    carga_min=400,
    carga_max=None
):
    """
    Seleciona horas operacionais adequadas à análise
    da intensidade carbônica.

    O filtro não altera os arquivos originais.
    """

    mascara = (
        dados["carga_bruta_mw"].ge(carga_min)
        & dados["tempo_operacao_h"].gt(0)
        & dados["tempo_operacao_h"].le(1)
        & dados["geracao_mwh"].gt(0)
        & dados["co2_t"].ge(0)
        & dados["co2_t_previsto"].ge(0)
        & np.isfinite(dados["residuo_co2_t"])
    )

    if carga_max is not None:
        mascara &= dados["carga_bruta_mw"].le(
            carga_max
        )

    return dados.loc[mascara].copy()

# ------------------------------------------------------------
# 4. AGREGAÇÃO DIÁRIA
# ------------------------------------------------------------

def construir_diario(dados, inicio, fim):

    inicio = pd.Timestamp(inicio)
    fim = pd.Timestamp(fim)

    grade = pd.date_range(
        inicio, fim, freq="D"
    )

    trecho = dados.loc[
        dados["dia"].between(inicio, fim)
    ]

    diario = (
        trecho.groupby("dia")
        .agg(
            horas=("timestamp", "size"),
            geracao_mwh=("geracao_mwh", "sum"),
            co2_t=("co2_t", "sum"),
            co2_t_previsto=("co2_t_previsto", "sum"),
            residuo_co2_t=("residuo_co2_t", "sum"),
            carga_media_mw=("carga_bruta_mw", "mean")
        )
        .reindex(grade)
    )

    diario.index.name = "dia"

    denominador = diario[
        "geracao_mwh"
    ].replace(0, np.nan)

    diario["desvio_t_mwh"] = (
        diario["residuo_co2_t"]
        / denominador
    )

    diario["intensidade_observada"] = (
        diario["co2_t"]
        / denominador
    )

    diario["intensidade_prevista"] = (
        diario["co2_t_previsto"]
        / denominador
    )

    diario["admissivel"] = (
        diario["horas"].ge(MIN_HORAS_DIA)
        & diario["geracao_mwh"].gt(0)
        & np.isfinite(diario["desvio_t_mwh"])
    )

    return diario

# ------------------------------------------------------------
# 5. SEGMENTAÇÃO POR PROGRAMAÇÃO DINÂMICA
# ------------------------------------------------------------

def segmentar_diario(
    tabela,
    min_dias=45,
    min_observados=20,
    max_quebras=3,
    peso_max=3.0
):

    n = len(tabela)

    y_original = tabela[
        "desvio_t_mwh"
    ].to_numpy(dtype=float)

    geracao = tabela[
        "geracao_mwh"
    ].to_numpy(dtype=float)

    admissivel = tabela[
        "admissivel"
    ].fillna(False).to_numpy(dtype=bool)

    N = int(admissivel.sum())

    if N < 2 * min_observados:
        return None

    mediana = np.median(
        geracao[admissivel]
    )

    if not np.isfinite(mediana) or mediana <= 0:
        return None

    w = np.zeros(n)

    w[admissivel] = np.minimum(
        geracao[admissivel] / mediana,
        peso_max
    )

    y = np.where(
        admissivel,
        y_original,
        0.0
    )

    W = np.r_[0, np.cumsum(w)]
    WY = np.r_[0, np.cumsum(w * y)]
    WY2 = np.r_[0, np.cumsum(w * y * y)]

    NOBS = np.r_[
        0,
        np.cumsum(admissivel.astype(int))
    ]

    def estatisticas(i, j):

        if j - i < min_dias:
            return None

        obs = int(
            NOBS[j] - NOBS[i]
        )

        if obs < min_observados:
            return None

        sw = W[j] - W[i]

        if sw <= 0:
            return None

        swy = WY[j] - WY[i]
        swy2 = WY2[j] - WY2[i]

        media = swy / sw

        sse = max(
            0.0,
            swy2 - swy**2 / sw
        )

        return media, sse, obs

    KMAX = max_quebras + 1

    custo = np.full(
        (n + 1, n + 1),
        np.inf
    )

    for i in range(n):
        for j in range(i + min_dias, n + 1):

            resultado = estatisticas(i, j)

            if resultado is not None:
                custo[i, j] = resultado[1]

    dp = np.full(
        (KMAX + 1, n + 1),
        np.inf
    )

    anterior = np.full(
        (KMAX + 1, n + 1),
        -1,
        dtype=int
    )

    dp[0, 0] = 0

    for k in range(1, KMAX + 1):

        for j in range(
            k * min_dias,
            n + 1
        ):

            candidatos = np.arange(
                (k - 1) * min_dias,
                j - min_dias + 1
            )

            valores = (
                dp[k - 1, candidatos]
                + custo[candidatos, j]
            )

            posicao = np.argmin(valores)

            if np.isfinite(valores[posicao]):

                dp[k, j] = valores[posicao]

                anterior[k, j] = (
                    candidatos[posicao]
                )

    modelos = []

    for k in range(1, KMAX + 1):

        sse = dp[k, n]

        if not np.isfinite(sse):
            continue

        bic = (
            N * np.log(
                max(sse / N, 1e-15)
            )
            + (2 * k - 1) * np.log(N)
        )

        modelos.append({
            "segmentos": k,
            "quebras": k - 1,
            "sse": sse,
            "bic_aproximado": bic
        })

    if not modelos:
        return None

    modelos = pd.DataFrame(modelos)

    k_otimo = int(
        modelos.loc[
            modelos["bic_aproximado"].idxmin(),
            "segmentos"
        ]
    )

    segmentos = []
    j = n

    for k in range(k_otimo, 0, -1):

        i = int(anterior[k, j])

        if i < 0:
            raise RuntimeError(
                "Falha na reconstrução."
            )

        segmentos.append((i, j))
        j = i

    segmentos.reverse()

    regimes = []
    ajustado = np.full(n, np.nan)

    for idx, (i, j) in enumerate(segmentos):

        media, sse, obs = estatisticas(i, j)

        ajustado[i:j] = media

        regimes.append({
            "regime_id": idx,
            "inicio": tabela.index[i],
            "fim": tabela.index[j - 1],
            "dias_calendario": j - i,
            "dias_admissiveis": obs,
            "media_desvio_t_mwh": media
        })

    regimes = pd.DataFrame(regimes)

    eventos = []

    for idx in range(1, len(segmentos)):

        i = segmentos[idx][0]

        antes = regimes.loc[
            idx - 1,
            "media_desvio_t_mwh"
        ]

        depois = regimes.loc[
            idx,
            "media_desvio_t_mwh"
        ]

        delta = depois - antes

        eventos.append({
            "data_candidata": tabela.index[i],
            "media_antes": antes,
            "media_depois": depois,
            "delta_t_mwh": delta,
            "direcao": (
                "reducao" if delta < 0
                else "aumento" if delta > 0
                else "sem_alteracao"
            )
        })

    eventos = pd.DataFrame(
        eventos,
        columns=[
            "data_candidata",
            "media_antes",
            "media_depois",
            "delta_t_mwh",
            "direcao"
        ]
    )

    return {
        "modelos": modelos,
        "regimes": regimes,
        "eventos": eventos,
        "ajustado": ajustado,
        "n_validos": N
    }

# ------------------------------------------------------------
# 6. ANÁLISE PRINCIPAL — CARGA >= 400 MW
# ------------------------------------------------------------

principal = filtrar_operacao(
    h,
    carga_min=CARGA_MIN_PRINCIPAL
)

print(
    "\nHoras após filtro principal:",
    len(principal)
)

print(
    "Percentual preservado:",
    round(100 * len(principal) / len(h), 2),
    "%"
)

resultados_principais = {}
eventos_principais = []
modelos_principais = []

for nome, (inicio, fim) in JANELAS.items():

    print("\n" + "=" * 70)
    print("JANELA:", nome)
    print("=" * 70)

    diario = construir_diario(
        principal, inicio, fim
    )

    resultado = segmentar_diario(
        diario,
        min_dias=MIN_DIAS_SEGMENTO,
        min_observados=MIN_DIAS_OBSERVADOS,
        max_quebras=MAX_QUEBRAS,
        peso_max=PESO_MAX_RELATIVO
    )

    if resultado is None:
        print("Dados insuficientes.")
        continue

    resultados_principais[nome] = {
        "diario": diario,
        "resultado": resultado
    }

    print(
        "Dias admissíveis:",
        resultado["n_validos"],
        "de", len(diario)
    )

    print("\nMODELOS")
    display(
        resultado["modelos"].round(5)
    )

    print("\nREGIMES")
    display(
        resultado["regimes"].round(5)
    )

    print("\nEVENTOS")
    display(
        resultado["eventos"].round(5)
    )

    m = resultado["modelos"].copy()
    m["janela"] = nome
    modelos_principais.append(m)

    if not resultado["eventos"].empty:

        e = resultado["eventos"].copy()
        e["janela"] = nome
        eventos_principais.append(e)

    # Gráfico: não desenhar dias inadmissíveis
    serie_grafico = diario[
        "desvio_t_mwh"
    ].where(diario["admissivel"])

    fig, ax = plt.subplots(
        figsize=(14, 5)
    )

    ax.plot(
        diario.index,
        serie_grafico,
        linewidth=1,
        alpha=0.8,
        label="Desvio diário elegível"
    )

    ax.plot(
        diario.index,
        resultado["ajustado"],
        color="black",
        linewidth=2.2,
        label="Regimes estimados"
    )

    ax.axhline(
        0,
        color="gray",
        linestyle=":",
        linewidth=1
    )

    for _, evento in resultado["eventos"].iterrows():

        ax.axvline(
            evento["data_candidata"],
            color="tab:red",
            linestyle="--",
            alpha=0.8
        )

    ax.set_title(
        f"Iatan 2 — Mudanças diárias {nome} "
        f"(carga >= {CARGA_MIN_PRINCIPAL} MW)"
    )

    ax.set_xlabel("Data")
    ax.set_ylabel("Desvio (tCO₂/MWh)")
    ax.grid(alpha=0.2)
    ax.legend()

    fig.tight_layout()

    fig.savefig(
        DIR_FIGURAS /
        f"mudancas_diarias_corrigidas_{nome}.png",
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()

    diario_saida = diario.copy()

    diario_saida["regime_estimado"] = (
        resultado["ajustado"]
    )

    diario_saida.reset_index().to_csv(
        DIR_TABELAS /
        f"serie_diaria_corrigida_{nome}.csv",
        index=False
    )

# ------------------------------------------------------------
# 7. SENSIBILIDADE AOS LIMITES DE CARGA
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SENSIBILIDADE AOS LIMITES DE CARGA")
print("=" * 70)

sensibilidade = []

configuracoes = [
    (300, None),
    (400, None),
    (500, None),
    (650, None),
    (650, 850)
]

for carga_min, carga_max in configuracoes:

    dados_filtrados = filtrar_operacao(
        h,
        carga_min=carga_min,
        carga_max=carga_max
    )

    nome_filtro = (
        f"{carga_min}+ MW"
        if carga_max is None
        else f"{carga_min}-{carga_max} MW"
    )

    for nome, (inicio, fim) in JANELAS.items():

        diario = construir_diario(
            dados_filtrados,
            inicio,
            fim
        )

        resultado = segmentar_diario(
            diario,
            min_dias=MIN_DIAS_SEGMENTO,
            min_observados=MIN_DIAS_OBSERVADOS,
            max_quebras=MAX_QUEBRAS,
            peso_max=PESO_MAX_RELATIVO
        )

        if resultado is None:

            sensibilidade.append({
                "janela": nome,
                "filtro": nome_filtro,
                "horas": int(
                    dados_filtrados["dia"]
                    .between(inicio, fim).sum()
                ),
                "dias_admissiveis": int(
                    diario["admissivel"].sum()
                ),
                "quebras": np.nan,
                "datas": "dados insuficientes"
            })

            continue

        # Uma segmentação sem quebras gera uma Series vazia com dtype object.
        # Converter explicitamente evita erro do accessor .dt nesse caso.
        datas = (
            pd.to_datetime(
                resultado["eventos"]["data_candidata"],
                errors="coerce"
            )
            .dt.strftime("%Y-%m-%d")
            .dropna()
            .tolist()
        )

        sensibilidade.append({
            "janela": nome,
            "filtro": nome_filtro,
            "horas": int(
                dados_filtrados["dia"]
                .between(inicio, fim).sum()
            ),
            "dias_admissiveis": resultado["n_validos"],
            "quebras": len(datas),
            "datas": "; ".join(datas)
        })

sensibilidade_df = pd.DataFrame(
    sensibilidade
)

display(sensibilidade_df)

# ------------------------------------------------------------
# 8. CONSOLIDAÇÃO E EXPORTAÇÃO
# ------------------------------------------------------------

if eventos_principais:

    eventos_df = pd.concat(
        eventos_principais,
        ignore_index=True
    ).sort_values(
        "data_candidata"
    )

else:

    eventos_df = pd.DataFrame(
        columns=[
            "data_candidata",
            "media_antes",
            "media_depois",
            "delta_t_mwh",
            "direcao",
            "janela"
        ]
    )

print("\nEVENTOS DIÁRIOS CORRIGIDOS")
display(eventos_df.round(5))

eventos_df.to_csv(
    DIR_TABELAS /
    "eventos_diarios_candidatos_corrigidos.csv",
    index=False
)

sensibilidade_df.to_csv(
    DIR_TABELAS /
    "sensibilidade_diaria_limites_carga.csv",
    index=False
)

if modelos_principais:

    pd.concat(
        modelos_principais,
        ignore_index=True
    ).to_csv(
        DIR_TABELAS /
        "comparacao_modelos_diarios_corrigidos.csv",
        index=False
    )

print("\n" + "=" * 70)
print("CÉLULA 10/20 — CONCLUÍDA")
print("=" * 70)

print("Eventos candidatos:", len(eventos_df))
print("Tabelas:", DIR_TABELAS)
print("Figuras:", DIR_FIGURAS)

print(
    "\nResultados exploratórios. "
    "As datas não comprovam eventos de manutenção."
)


# ===== ORIGEM: CÉLULA 11 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 11/20 — REFINAMENTO HORÁRIO CORRIGIDO
# IATAN 2 | EPA 6065 | UNIDADE 2
# ============================================================
#
# Objetivos:
# 1. Reutilizar os resíduos ajustados da Célula 10.
# 2. Investigar as mudanças de março e julho de 2025.
# 3. Estimar instantes horários candidatos.
# 4. Comparar intensidades em cargas semelhantes.
# 5. Auditar cobertura e lacunas operacionais.
#
# Não utiliza heat rate.
# Não pressupõe manutenção.
# Não altera a base histórica.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

print("=" * 72)
print("CÉLULA 11/20 — REFINAMENTO HORÁRIO CORRIGIDO")
print("=" * 72)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

ARQUIVO = (
    Path(DIR_DADOS)
    / "iatan2_residuos_horarios_carga.parquet"
)

EVENTOS = {
    "marco_2025": pd.Timestamp("2025-03-26"),
    "julho_2025": pd.Timestamp("2025-07-14"),
}

CARGA_MIN = 400
TEMPO_MIN = 0.95

DIAS_BUSCA = 7
DIAS_COMPARACAO = 21

MIN_HORAS_LADO = 72
MIN_DIAS_LADO = 7
MIN_HORAS_FAIXA = 30

FAIXAS_CARGA = [
    (400, 550),
    (550, 650),
    (650, 750),
    (750, 850),
    (850, 1000),
]

DIR_TABELAS = Path(DIR_TABELAS)
DIR_FIGURAS = Path(DIR_FIGURAS)

DIR_TABELAS.mkdir(parents=True, exist_ok=True)
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

if not ARQUIVO.exists():
    raise FileNotFoundError(
        f"Base da Célula 10 não encontrada:\n{ARQUIVO}"
    )

# ------------------------------------------------------------
# 2. CARREGAMENTO E VALIDAÇÃO
# ------------------------------------------------------------

h = pd.read_parquet(ARQUIVO).copy()

COLUNAS = [
    "timestamp",
    "carga_bruta_mw",
    "tempo_operacao_h",
    "geracao_mwh",
    "co2_t",
    "co2_t_previsto",
    "residuo_co2_t",
]

faltantes = set(COLUNAS) - set(h.columns)

if faltantes:
    raise KeyError(
        f"Colunas ausentes: {sorted(faltantes)}\n"
        f"Disponíveis: {h.columns.tolist()}"
    )

h["timestamp"] = pd.to_datetime(
    h["timestamp"], errors="coerce"
)

if h["timestamp"].isna().any():
    raise ValueError("Existem timestamps inválidos.")

if h["timestamp"].duplicated().any():
    raise ValueError("Existem timestamps duplicados.")

for c in COLUNAS[1:]:
    h[c] = pd.to_numeric(h[c], errors="coerce")

erro = (
    h["co2_t"]
    - h["co2_t_previsto"]
    - h["residuo_co2_t"]
).abs().max()

if not np.isfinite(erro) or erro > 1e-6:
    raise ValueError(
        f"Inconsistência nos resíduos: {erro}"
    )

print("Arquivo:", ARQUIVO)
print("Registros originais:", len(h))
print("Erro máximo da identidade:", erro)

# ------------------------------------------------------------
# 3. FILTRO OPERACIONAL
# ------------------------------------------------------------

h = h.loc[
    h["timestamp"].between(
        "2025-01-01",
        "2025-12-31 23:59:59"
    )
].copy()

mascara = (
    h["carga_bruta_mw"].ge(CARGA_MIN)
    & h["tempo_operacao_h"].between(
        TEMPO_MIN, 1.0
    )
    & h["geracao_mwh"].gt(0)
    & h["co2_t"].ge(0)
    & h["co2_t_previsto"].ge(0)
    & np.isfinite(h["residuo_co2_t"])
)

h = (
    h.loc[mascara]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

if h.empty:
    raise ValueError(
        "Nenhuma observação elegível em 2025."
    )

h["intensidade_observada"] = (
    h["co2_t"] / h["geracao_mwh"]
)

h["intensidade_prevista"] = (
    h["co2_t_previsto"] / h["geracao_mwh"]
)

h["desvio_ajustado"] = (
    h["residuo_co2_t"] / h["geracao_mwh"]
)

h["dia"] = h["timestamp"].dt.floor("D")

print("Horas elegíveis em 2025:", len(h))
print("Dias com observações:", h["dia"].nunique())

print("\nDISTRIBUIÇÃO DOS DESVIOS HORÁRIOS")
display(
    h["desvio_ajustado"].describe(
        percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
    ).round(5)
)

# ------------------------------------------------------------
# 4. AGREGAÇÃO PONDERADA
# ------------------------------------------------------------

def resumo_periodo(df):

    if df.empty:
        return None

    energia = df["geracao_mwh"].sum()

    if energia <= 0:
        return None

    return {
        "horas": len(df),
        "dias": df["dia"].nunique(),
        "geracao_mwh": energia,
        "intensidade_observada": (
            df["co2_t"].sum() / energia
        ),
        "intensidade_prevista": (
            df["co2_t_previsto"].sum() / energia
        ),
        "desvio_ajustado": (
            df["residuo_co2_t"].sum() / energia
        ),
        "carga_media_mw": (
            np.average(
                df["carga_bruta_mw"],
                weights=df["geracao_mwh"]
            )
        ),
    }

# ------------------------------------------------------------
# 5. DETECÇÃO DO INSTANTE HORÁRIO
# ------------------------------------------------------------

def detectar_hora(nome, data):

    inicio = data - pd.Timedelta(
        days=DIAS_BUSCA
    )

    fim = data + pd.Timedelta(
        days=DIAS_BUSCA
    )

    candidatos = pd.date_range(
        inicio,
        fim,
        freq="h"
    )

    registros = []

    for instante in candidatos:

        antes = h.loc[
            h["timestamp"].ge(
                instante - pd.Timedelta(
                    days=DIAS_COMPARACAO
                )
            )
            & h["timestamp"].lt(instante)
        ]

        depois = h.loc[
            h["timestamp"].ge(instante)
            & h["timestamp"].lt(
                instante + pd.Timedelta(
                    days=DIAS_COMPARACAO
                )
            )
        ]

        ra = resumo_periodo(antes)
        rb = resumo_periodo(depois)

        if ra is None or rb is None:
            continue

        if (
            ra["horas"] < MIN_HORAS_LADO
            or rb["horas"] < MIN_HORAS_LADO
            or ra["dias"] < MIN_DIAS_LADO
            or rb["dias"] < MIN_DIAS_LADO
        ):
            continue

        delta = (
            rb["desvio_ajustado"]
            - ra["desvio_ajustado"]
        )

        registros.append({
            "instante": instante,
            "delta_ajustado": delta,
            "desvio_antes": ra["desvio_ajustado"],
            "desvio_depois": rb["desvio_ajustado"],
            "horas_antes": ra["horas"],
            "horas_depois": rb["horas"],
            "dias_antes": ra["dias"],
            "dias_depois": rb["dias"],
            "carga_antes": ra["carga_media_mw"],
            "carga_depois": rb["carga_media_mw"],
        })

    tabela = pd.DataFrame(registros)

    if tabela.empty:
        print(f"{nome}: dados insuficientes.")
        return None

    # Critério exploratório:
    # maior redução do desvio ajustado.
    tabela = tabela.sort_values(
        ["delta_ajustado", "instante"]
    ).reset_index(drop=True)

    melhor = tabela.iloc[0]
    instante = melhor["instante"]

    # --------------------------------------------------------
    # 6. COMPARAÇÃO POR FAIXA DE CARGA
    # --------------------------------------------------------

    antes = h.loc[
        h["timestamp"].ge(
            instante - pd.Timedelta(
                days=DIAS_COMPARACAO
            )
        )
        & h["timestamp"].lt(instante)
    ]

    depois = h.loc[
        h["timestamp"].ge(instante)
        & h["timestamp"].lt(
            instante + pd.Timedelta(
                days=DIAS_COMPARACAO
            )
        )
    ]

    comparacoes = []

    for minimo, maximo in FAIXAS_CARGA:

        a = antes.loc[
            antes["carga_bruta_mw"].ge(minimo)
            & antes["carga_bruta_mw"].lt(maximo)
        ]

        b = depois.loc[
            depois["carga_bruta_mw"].ge(minimo)
            & depois["carga_bruta_mw"].lt(maximo)
        ]

        if (
            len(a) < MIN_HORAS_FAIXA
            or len(b) < MIN_HORAS_FAIXA
        ):
            continue

        ra = resumo_periodo(a)
        rb = resumo_periodo(b)

        comparacoes.append({
            "faixa_mw": f"{minimo}-{maximo}",
            "horas_antes": len(a),
            "horas_depois": len(b),
            "intensidade_antes": (
                ra["intensidade_observada"]
            ),
            "intensidade_depois": (
                rb["intensidade_observada"]
            ),
            "delta_intensidade": (
                rb["intensidade_observada"]
                - ra["intensidade_observada"]
            ),
            "delta_ajustado": (
                rb["desvio_ajustado"]
                - ra["desvio_ajustado"]
            ),
        })

    comparacao = pd.DataFrame(comparacoes)

    # --------------------------------------------------------
    # 7. APRESENTAÇÃO
    # --------------------------------------------------------

    print("\n" + "=" * 72)
    print(nome.upper())
    print("=" * 72)

    print("Data candidata diária:", data)
    print("Instante horário candidato:", instante)

    print(
        "Redução ajustada:",
        round(float(melhor["delta_ajustado"]), 5),
        "tCO2/MWh"
    )

    print("\nCOBERTURA")
    print(
        "Antes:",
        int(melhor["horas_antes"]),
        "horas /",
        int(melhor["dias_antes"]),
        "dias"
    )

    print(
        "Depois:",
        int(melhor["horas_depois"]),
        "horas /",
        int(melhor["dias_depois"]),
        "dias"
    )

    print("\nCOMPARAÇÃO POR CARGA")

    if comparacao.empty:
        print(
            "Nenhuma faixa possui observações "
            "suficientes nos dois períodos."
        )
    else:
        display(comparacao.round(5))

    # --------------------------------------------------------
    # 8. VISUALIZAÇÃO
    # --------------------------------------------------------

    janela = h.loc[
        h["timestamp"].between(
            data - pd.Timedelta(days=30),
            data + pd.Timedelta(days=30)
        )
    ].copy()

    diario = (
        janela.groupby("dia")
        .apply(
            lambda g: (
                g["residuo_co2_t"].sum()
                / g["geracao_mwh"].sum()
            ),
            include_groups=False
        )
    )

    fig, ax = plt.subplots(figsize=(15, 5))

    ax.scatter(
        janela["timestamp"],
        janela["desvio_ajustado"],
        s=5,
        alpha=0.2,
        label="Desvio horário"
    )

    ax.plot(
        diario.index,
        diario.values,
        color="black",
        linewidth=2,
        label="Desvio diário agregado"
    )

    ax.axvline(
        data,
        color="orange",
        linestyle="--",
        label="Data diária candidata"
    )

    ax.axvline(
        instante,
        color="red",
        linewidth=2,
        label="Instante horário candidato"
    )

    ax.axhline(
        0,
        color="gray",
        linestyle=":"
    )

    ax.set_title(
        f"Iatan 2 — Refinamento horário: {nome}"
    )

    ax.set_xlabel("Data")
    ax.set_ylabel(
        "Desvio ajustado (tCO₂/MWh)"
    )

    ax.grid(alpha=0.2)
    ax.legend()

    plt.tight_layout()

    fig.savefig(
        DIR_FIGURAS / f"refinamento_{nome}.png",
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()

    # --------------------------------------------------------
    # 9. EXPORTAÇÃO
    # --------------------------------------------------------

    tabela.to_csv(
        DIR_TABELAS / f"candidatos_horarios_{nome}.csv",
        index=False
    )

    comparacao.to_csv(
        DIR_TABELAS / f"comparacao_carga_{nome}.csv",
        index=False
    )

    return {
        "evento": nome,
        "data_diaria": data,
        "instante_horario": instante,
        "delta_ajustado": float(
            melhor["delta_ajustado"]
        ),
        "horas_antes": int(
            melhor["horas_antes"]
        ),
        "horas_depois": int(
            melhor["horas_depois"]
        ),
        "faixas_comparaveis": len(comparacao),
    }

# ------------------------------------------------------------
# 10. EXECUÇÃO
# ------------------------------------------------------------

resultados = []

for nome, data in EVENTOS.items():

    resultado = detectar_hora(nome, data)

    if resultado is not None:
        resultados.append(resultado)

eventos_horarios_df = pd.DataFrame(resultados)

print("\n" + "=" * 72)
print("RESUMO DOS EVENTOS HORÁRIOS CANDIDATOS")
print("=" * 72)

display(eventos_horarios_df)

eventos_horarios_df.to_csv(
    DIR_TABELAS / "eventos_horarios_candidatos.csv",
    index=False
)

print("\n" + "=" * 72)
print("CÉLULA 11/20 — CONCLUÍDA")
print("=" * 72)

print("Tabelas:", DIR_TABELAS)
print("Figuras:", DIR_FIGURAS)

print(
    "\nAs datas e horas são candidatas estatísticas, "
    "não eventos de manutenção confirmados."
)


# ===== ORIGEM: CÉLULA 12 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 12/20 — VALIDAÇÃO TEMPORAL DOS EVENTOS
# IATAN 2 | EPA 6065 | UNIDADE 2 | 2025
# ============================================================
#
# Objetivos:
# 1. Examinar as mudanças de março e julho de 2025.
# 2. Comparar janelas de 24, 48 e 72 horas.
# 3. Identificar instantes candidatos de transição.
# 4. Verificar interrupções e lacunas operacionais.
# 5. Comparar períodos em faixas comuns de carga.
# 6. Avaliar estabilidade e incerteza temporal.
#
# Importante:
# - Não utiliza heat rate.
# - Não pressupõe manutenção.
# - Não modifica as bases anteriores.
# - As datas são estimativas retrospectivas.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

print("=" * 72)
print("CÉLULA 12/20 — VALIDAÇÃO TEMPORAL")
print("=" * 72)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

BASE = (
    Path(DIR_DADOS)
    / "iatan2_residuos_horarios_carga.parquet"
)

EVENTOS = {
    "marco_2025": {
        "inicio": "2025-03-20",
        "fim": "2025-04-03",
        "referencia": "2025-03-26 08:00",
    },
    "julho_2025": {
        "inicio": "2025-07-05",
        "fim": "2025-07-22",
        "referencia": "2025-07-14 00:00",
    },
}

JANELAS_HORAS = [24, 48, 72]

CARGA_MIN = 400
TEMPO_MIN = 0.95

# Cobertura mínima de horas elegíveis em cada lado.
COBERTURA_MIN = 0.50

# Exigir observações em ambos os lados.
MIN_HORAS = {
    24: 12,
    48: 24,
    72: 36,
}

# Faixas utilizadas na análise de sensibilidade.
FAIXAS_CARGA = [
    (400, 550),
    (550, 650),
    (650, 750),
    (750, 850),
    (850, 1000),
]

DIR_TABELAS = Path(DIR_TABELAS)
DIR_FIGURAS = Path(DIR_FIGURAS)

DIR_TABELAS.mkdir(parents=True, exist_ok=True)
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

if not BASE.exists():
    raise FileNotFoundError(
        f"Base horária não encontrada: {BASE}"
    )

# ------------------------------------------------------------
# 2. LEITURA E VALIDAÇÃO
# ------------------------------------------------------------

h = pd.read_parquet(BASE).copy()

necessarias = [
    "timestamp",
    "carga_bruta_mw",
    "tempo_operacao_h",
    "geracao_mwh",
    "co2_t",
    "co2_t_previsto",
    "residuo_co2_t",
]

faltantes = [
    c for c in necessarias
    if c not in h.columns
]

if faltantes:
    raise KeyError(
        f"Colunas ausentes: {faltantes}\n"
        f"Disponíveis: {h.columns.tolist()}"
    )

h["timestamp"] = pd.to_datetime(
    h["timestamp"], errors="coerce"
)

if h["timestamp"].isna().any():
    raise ValueError("Timestamps inválidos.")

if h["timestamp"].duplicated().any():
    raise ValueError("Timestamps duplicados.")

for c in necessarias[1:]:
    h[c] = pd.to_numeric(
        h[c], errors="coerce"
    )

h = (
    h.loc[
        h["timestamp"].dt.year.eq(2025)
    ]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

h["elegivel"] = (
    h["carga_bruta_mw"].ge(CARGA_MIN)
    & h["tempo_operacao_h"].between(
        TEMPO_MIN, 1.0
    )
    & h["geracao_mwh"].gt(0)
    & h["co2_t"].ge(0)
    & h["co2_t_previsto"].ge(0)
    & np.isfinite(h["residuo_co2_t"])
)

h["desvio_ajustado"] = np.where(
    h["elegivel"],
    h["residuo_co2_t"] / h["geracao_mwh"],
    np.nan
)

h["intensidade_observada"] = np.where(
    h["elegivel"],
    h["co2_t"] / h["geracao_mwh"],
    np.nan
)

print("Horas disponíveis em 2025:", len(h))
print("Horas elegíveis:", int(h["elegivel"].sum()))

# ------------------------------------------------------------
# 3. FUNÇÕES DE AGREGAÇÃO
# ------------------------------------------------------------

def resumo(df):

    x = df.loc[df["elegivel"]].copy()

    if x.empty:
        return None

    energia = x["geracao_mwh"].sum()

    if energia <= 0:
        return None

    return {
        "n": len(x),
        "energia": energia,
        "desvio": (
            x["residuo_co2_t"].sum() / energia
        ),
        "intensidade": (
            x["co2_t"].sum() / energia
        ),
        "carga_media": (
            np.average(
                x["carga_bruta_mw"],
                weights=x["geracao_mwh"]
            )
        ),
    }


def comparar(antes, depois):

    a = resumo(antes)
    b = resumo(depois)

    if a is None or b is None:
        return None

    return {
        "horas_antes": a["n"],
        "horas_depois": b["n"],
        "desvio_antes": a["desvio"],
        "desvio_depois": b["desvio"],
        "delta_ajustado": (
            b["desvio"] - a["desvio"]
        ),
        "intensidade_antes": a["intensidade"],
        "intensidade_depois": b["intensidade"],
        "delta_intensidade": (
            b["intensidade"] - a["intensidade"]
        ),
        "carga_antes": a["carga_media"],
        "carga_depois": b["carga_media"],
    }


# ------------------------------------------------------------
# 4. BUSCA POR MUDANÇAS EM JANELAS CURTAS
# ------------------------------------------------------------

resultados = []
comparacoes_carga = []
auditorias = []

for nome, cfg in EVENTOS.items():

    inicio = pd.Timestamp(cfg["inicio"])
    fim = pd.Timestamp(cfg["fim"])
    referencia = pd.Timestamp(cfg["referencia"])

    print("\n" + "=" * 72)
    print(nome.upper())
    print("=" * 72)

    # Inclui margens suficientes para janelas de 72 h.
    margem = pd.Timedelta(hours=72)

    periodo = h.loc[
        h["timestamp"].between(
            inicio - margem,
            fim + margem
        )
    ].copy()

    # --------------------------------------------------------
    # 4.1 AUDITORIA OPERACIONAL
    # --------------------------------------------------------

    calendario = pd.DataFrame({
        "timestamp": pd.date_range(
            inicio - margem,
            fim + margem,
            freq="h"
        )
    })

    auditoria = calendario.merge(
        periodo[
            ["timestamp", "elegivel",
             "carga_bruta_mw",
             "tempo_operacao_h"]
        ],
        on="timestamp",
        how="left",
        validate="one_to_one"
    )

    auditoria["presente"] = (
        auditoria["elegivel"].notna()
    )

    auditoria["elegivel"] = (
        auditoria["elegivel"]
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )

    auditoria["dia"] = (
        auditoria["timestamp"].dt.floor("D")
    )

    diaria = (
        auditoria.groupby("dia")
        .agg(
            horas_calendario=("timestamp", "size"),
            horas_presentes=("presente", "sum"),
            horas_elegiveis=("elegivel", "sum"),
        )
        .reset_index()
    )

    diaria["cobertura_pct"] = (
        100
        * diaria["horas_elegiveis"]
        / diaria["horas_calendario"]
    )

    diaria["evento"] = nome

    auditorias.append(diaria)

    print("\nCOBERTURA OPERACIONAL DIÁRIA")
    display(
        diaria[
            ["dia", "horas_presentes",
             "horas_elegiveis", "cobertura_pct"]
        ].round(2)
    )

    # --------------------------------------------------------
    # 4.2 COMPARAÇÃO EM JANELAS DE 24, 48 E 72 H
    # --------------------------------------------------------

    for janela_h in JANELAS_HORAS:

        candidatos = pd.date_range(
            inicio,
            fim,
            freq="h"
        )

        for instante in candidatos:

            antes = periodo.loc[
                periodo["timestamp"].ge(
                    instante - pd.Timedelta(
                        hours=janela_h
                    )
                )
                & periodo["timestamp"].lt(instante)
            ]

            depois = periodo.loc[
                periodo["timestamp"].ge(instante)
                & periodo["timestamp"].lt(
                    instante + pd.Timedelta(
                        hours=janela_h
                    )
                )
            ]

            resultado = comparar(antes, depois)

            if resultado is None:
                continue

            minimo = max(
                MIN_HORAS[janela_h],
                int(np.ceil(
                    janela_h * COBERTURA_MIN
                ))
            )

            if (
                resultado["horas_antes"] < minimo
                or resultado["horas_depois"] < minimo
            ):
                continue

            resultados.append({
                "evento": nome,
                "janela_h": janela_h,
                "instante": instante,
                "distancia_referencia_h": (
                    instante - referencia
                ).total_seconds() / 3600,
                **resultado,
            })

    # --------------------------------------------------------
    # 4.3 COMPARAÇÃO POR FAIXA DE CARGA
    #     AO REDOR DA DATA DE REFERÊNCIA
    # --------------------------------------------------------

    antes_ref = periodo.loc[
        periodo["timestamp"].ge(
            referencia - pd.Timedelta(days=7)
        )
        & periodo["timestamp"].lt(referencia)
    ]

    depois_ref = periodo.loc[
        periodo["timestamp"].ge(referencia)
        & periodo["timestamp"].lt(
            referencia + pd.Timedelta(days=7)
        )
    ]

    for cmin, cmax in FAIXAS_CARGA:

        a = antes_ref.loc[
            antes_ref["carga_bruta_mw"].ge(cmin)
            & antes_ref["carga_bruta_mw"].lt(cmax)
        ]

        b = depois_ref.loc[
            depois_ref["carga_bruta_mw"].ge(cmin)
            & depois_ref["carga_bruta_mw"].lt(cmax)
        ]

        comp = comparar(a, b)

        if comp is None:
            continue

        comparacoes_carga.append({
            "evento": nome,
            "faixa_mw": f"{cmin}-{cmax}",
            **comp,
        })

# ------------------------------------------------------------
# 5. CONSOLIDAÇÃO
# ------------------------------------------------------------

df_resultados = pd.DataFrame(resultados)

if df_resultados.empty:
    raise ValueError(
        "Nenhuma janela atingiu a cobertura mínima. "
        "Inspecione a auditoria operacional."
    )

df_carga = pd.DataFrame(comparacoes_carga)
df_auditoria = pd.concat(
    auditorias,
    ignore_index=True
)

# ------------------------------------------------------------
# 6. MELHORES CANDIDATOS POR JANELA
# ------------------------------------------------------------

melhores = (
    df_resultados
    .sort_values(
        ["evento", "janela_h",
         "delta_ajustado", "instante"]
    )
    .groupby(
        ["evento", "janela_h"],
        as_index=False
    )
    .first()
)

print("\n" + "=" * 72)
print("MELHORES INSTANTES POR JANELA")
print("=" * 72)

display(
    melhores[
        [
            "evento",
            "janela_h",
            "instante",
            "delta_ajustado",
            "horas_antes",
            "horas_depois",
            "carga_antes",
            "carga_depois",
        ]
    ].round(5)
)

# ------------------------------------------------------------
# 7. ESTABILIDADE TEMPORAL
# ------------------------------------------------------------

estabilidade = (
    melhores.groupby("evento")
    .agg(
        instante_min=("instante", "min"),
        instante_max=("instante", "max"),
        delta_medio=("delta_ajustado", "mean"),
        numero_janelas=("janela_h", "nunique"),
    )
    .reset_index()
)

estabilidade["amplitude_h"] = (
    estabilidade["instante_max"]
    - estabilidade["instante_min"]
).dt.total_seconds() / 3600

print("\nESTABILIDADE ENTRE JANELAS")
display(estabilidade.round(5))

print("\nCOMPARAÇÃO POR FAIXA DE CARGA")
if df_carga.empty:
    print("Nenhuma faixa comparável.")
else:
    display(
        df_carga[
            [
                "evento",
                "faixa_mw",
                "horas_antes",
                "horas_depois",
                "delta_intensidade",
                "delta_ajustado",
            ]
        ].round(5)
    )

# ------------------------------------------------------------
# 8. VISUALIZAÇÃO DOS CANDIDATOS
# ------------------------------------------------------------

for nome, cfg in EVENTOS.items():

    inicio = pd.Timestamp(cfg["inicio"])
    fim = pd.Timestamp(cfg["fim"])
    referencia = pd.Timestamp(cfg["referencia"])

    x = h.loc[
        h["timestamp"].between(
            inicio, fim
        )
        & h["elegivel"]
    ].copy()

    candidatos = melhores.loc[
        melhores["evento"].eq(nome)
    ]

    fig, ax = plt.subplots(
        figsize=(15, 5)
    )

    ax.scatter(
        x["timestamp"],
        x["desvio_ajustado"],
        s=8,
        alpha=0.25,
        label="Desvio horário"
    )

    serie_diaria = (
        x.assign(
            dia=x["timestamp"].dt.floor("D")
        )
        .groupby("dia")
        .agg(
            residuo=("residuo_co2_t", "sum"),
            geracao=("geracao_mwh", "sum")
        )
    )

    serie_diaria["desvio"] = (
        serie_diaria["residuo"]
        / serie_diaria["geracao"]
    )

    ax.plot(
        serie_diaria.index,
        serie_diaria["desvio"],
        color="black",
        linewidth=2,
        label="Desvio diário"
    )

    ax.axvline(
        referencia,
        color="gray",
        linestyle="--",
        linewidth=2,
        label="Referência diária"
    )

    cores = {
        24: "tab:green",
        48: "tab:orange",
        72: "tab:red",
    }

    for _, r in candidatos.iterrows():

        ax.axvline(
            r["instante"],
            color=cores[r["janela_h"]],
            linestyle=":",
            linewidth=2,
            label=f'{r["janela_h"]} h'
        )

    ax.axhline(
        0,
        color="gray",
        linestyle=":"
    )

    ax.set_title(
        f"Iatan 2 — Validação temporal: {nome}"
    )

    ax.set_xlabel("Data")
    ax.set_ylabel(
        "Desvio ajustado (tCO₂/MWh)"
    )

    ax.grid(alpha=0.2)
    ax.legend()

    plt.tight_layout()

    fig.savefig(
        DIR_FIGURAS
        / f"validacao_temporal_{nome}.png",
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()

# ------------------------------------------------------------
# 9. EXPORTAÇÃO
# ------------------------------------------------------------

df_resultados.to_csv(
    DIR_TABELAS
    / "validacao_temporal_todos_candidatos.csv",
    index=False
)

melhores.to_csv(
    DIR_TABELAS
    / "validacao_temporal_melhores.csv",
    index=False
)

estabilidade.to_csv(
    DIR_TABELAS
    / "validacao_temporal_estabilidade.csv",
    index=False
)

df_carga.to_csv(
    DIR_TABELAS
    / "validacao_temporal_comparacao_carga.csv",
    index=False
)

df_auditoria.to_csv(
    DIR_TABELAS
    / "validacao_temporal_auditoria_operacional.csv",
    index=False
)

print("\n" + "=" * 72)
print("CÉLULA 12/20 — CONCLUÍDA")
print("=" * 72)

print("Candidatos avaliados:", len(df_resultados))
print("Janelas:", JANELAS_HORAS)
print("Tabelas:", DIR_TABELAS)
print("Figuras:", DIR_FIGURAS)

print(
    "\nATENÇÃO: os horários selecionados são "
    "candidatos retrospectivos, não datas "
    "de manutenção comprovadas."
)

## 09 — Magnitude e persistência

Compara regimes em faixas de carga, testa horizontes e utiliza bootstrap por blocos de dias. Preserva análises de persistência e consolida os candidatos.

---

**Comparabilidade e incerteza — Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11); Künsch (1989, pp. 1217–1218).** A estratificação por carga reduz diferenças descritivas de composição operacional nos grupos pré/pós, sem eliminar outros confundidores. O bootstrap por dias/blocos preserva parte da dependência temporal, mas seus intervalos permanecem condicionados à definição dos candidatos, das janelas e dos filtros. A fundamentação de Künsch (1989, pp. 1217–1218) pressupõe estacionariedade; a presença de mudanças de regime exige cautela adicional quanto à validade da cobertura dos intervalos em cada segmento. Não se deve interpretar a mudança estimada como efeito causal de uma manutenção.

In [ ]:
# ===== ORIGEM: CÉLULA 13 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 13/20 — MAGNITUDE E PERSISTÊNCIA
# PROJETO IATAN 2 — ORISPL 6065 / UNIDADE 2
# ============================================================
#
# Objetivos:
# 1. Comparar intensidade antes/depois de cada episódio.
# 2. Controlar a carga por estratificação.
# 3. Estimar incerteza por bootstrap de dias.
# 4. Avaliar persistência em diferentes horizontes.
# 5. Preservar resultados para as próximas células.
#
# Não pressupõe que os episódios sejam manutenções.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("=" * 72)
print("CÉLULA 13/20 — MAGNITUDE E PERSISTÊNCIA")
print("=" * 72)

# ------------------------------------------------------------
# 1. DIRETÓRIOS
# ------------------------------------------------------------

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "07_intensidade_carbonica"
)

DIR_DADOS = OUTPUT_ROOT / "dados"
DIR_TABELAS = OUTPUT_ROOT / "tabelas"
DIR_FIGURAS = OUTPUT_ROOT / "figuras"

for pasta in [DIR_DADOS, DIR_TABELAS, DIR_FIGURAS]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQUIVO_BASE = (
    DIR_DADOS / "iatan2_intensidade_horaria_2011_2025.parquet"
)

if not ARQUIVO_BASE.exists():
    raise FileNotFoundError(
        f"Base da Célula 04 não encontrada: {ARQUIVO_BASE}"
    )

df = pd.read_parquet(ARQUIVO_BASE)

print(f"Base carregada: {len(df):,} registros")
print("Colunas disponíveis:", list(df.columns))

# ------------------------------------------------------------
# 2. IDENTIFICAÇÃO SEGURA DAS COLUNAS
# ------------------------------------------------------------

def localizar_coluna(dados, candidatos, obrigatoria=True):
    normalizadas = {
        str(c).strip().lower(): c
        for c in dados.columns
    }

    for candidato in candidatos:
        chave = candidato.strip().lower()
        if chave in normalizadas:
            return normalizadas[chave]

    if obrigatoria:
        raise KeyError(
            f"Nenhuma coluna encontrada entre {candidatos}. "
            f"Disponíveis: {list(dados.columns)}"
        )

    return None


COL_TEMPO = localizar_coluna(
    df, ["timestamp", "datetime", "data_hora"]
)

COL_INTENSIDADE = localizar_coluna(
    df,
    [
        "intensidade_t_mwh",
        "intensidade_carbonica_t_mwh",
        "intensidade_co2_t_mwh",
        "intensidade"
    ]
)

COL_CARGA = localizar_coluna(
    df,
    [
        "Gross Load (MW)",
        "gross_load_mw",
        "carga_mw",
        "carga_bruta_mw"
    ]
)

COL_ELEGIVEL = localizar_coluna(
    df,
    ["elegivel", "hora_elegivel", "is_eligible"],
    obrigatoria=False
)

COL_SITUACAO = localizar_coluna(
    df, ["situacao"],
    obrigatoria=False
)

# ------------------------------------------------------------
# 3. BASE ANALÍTICA
# ------------------------------------------------------------

base = pd.DataFrame({
    "timestamp": pd.to_datetime(
        df[COL_TEMPO], errors="coerce"
    ),
    "intensidade": pd.to_numeric(
        df[COL_INTENSIDADE], errors="coerce"
    ),
    "carga_mw": pd.to_numeric(
        df[COL_CARGA], errors="coerce"
    )
})

if COL_ELEGIVEL is not None:
    elegivel = df[COL_ELEGIVEL].eq(True)
elif COL_SITUACAO is not None:
    elegivel = df[COL_SITUACAO].astype(str).eq("valida")
else:
    elegivel = pd.Series(True, index=df.index)

base = base.loc[
    elegivel.to_numpy()
    & base["timestamp"].notna()
    & base["intensidade"].notna()
    & np.isfinite(base["intensidade"])
    & (base["intensidade"] > 0)
    & base["carga_mw"].notna()
    & (base["carga_mw"] >= 400)
].copy()

base = base[
    (base["timestamp"] >= "2024-01-01")
    & (base["timestamp"] < "2026-01-01")
].sort_values("timestamp")

if base["timestamp"].duplicated().any():
    raise ValueError(
        "Há timestamps duplicados na base elegível. "
        "É necessário auditar antes de continuar."
    )

base["dia"] = base["timestamp"].dt.floor("D")

print(f"Horas elegíveis para análise: {len(base):,}")

# ------------------------------------------------------------
# 4. DEFINIÇÃO DOS EPISÓDIOS
# ------------------------------------------------------------

EVENTOS = {
    "marco_2025": {
        "referencia": pd.Timestamp("2025-03-26 08:00"),
        "exclusao_inicio": pd.Timestamp("2025-03-25"),
        "exclusao_fim": pd.Timestamp("2025-03-31"),
        "fim_analise": pd.Timestamp("2025-07-01")
    },
    "julho_2025": {
        "referencia": pd.Timestamp("2025-07-14 10:00"),
        "exclusao_inicio": pd.Timestamp("2025-07-10"),
        "exclusao_fim": pd.Timestamp("2025-07-20"),
        "fim_analise": pd.Timestamp("2025-12-31 23:00")
    }
}

# As exclusões removem dias de transição e baixa cobertura.
# Não são datas de manutenção comprovadas.

FAIXAS = [400, 550, 650, 750, 850, 1000]
JANELAS_DIAS = [15, 30, 60, 90]
N_BOOTSTRAP = 500
SEMENTE = 2026

# ------------------------------------------------------------
# 5. FUNÇÕES ESTATÍSTICAS
# ------------------------------------------------------------

def media_estratificada(antes, depois):
    """
    Diferença de intensidade após - antes,
    padronizada pela distribuição combinada de carga.

    Cada faixa precisa ter ao menos 5 horas em cada grupo.
    """

    a = antes.copy()
    d = depois.copy()

    a["faixa"] = pd.cut(
        a["carga_mw"], FAIXAS, right=False
    )
    d["faixa"] = pd.cut(
        d["carga_mw"], FAIXAS, right=False
    )

    resultados = []

    for faixa in pd.IntervalIndex.from_breaks(
        FAIXAS, closed="left"
    ):
        va = a.loc[a["faixa"] == faixa, "intensidade"]
        vd = d.loc[d["faixa"] == faixa, "intensidade"]

        if len(va) < 5 or len(vd) < 5:
            continue

        resultados.append({
            "faixa": str(faixa),
            "n_antes": len(va),
            "n_depois": len(vd),
            "media_antes": va.mean(),
            "media_depois": vd.mean(),
            "peso": len(va) + len(vd)
        })

    if not resultados:
        return np.nan, np.nan, pd.DataFrame()

    tab = pd.DataFrame(resultados)
    pesos = tab["peso"] / tab["peso"].sum()

    media_a = np.sum(pesos * tab["media_antes"])
    media_d = np.sum(pesos * tab["media_depois"])

    return media_d - media_a, media_a, tab


def bootstrap_dias(antes, depois, n_boot=500, seed=2026):
    """
    Reamostra dias completos, separadamente,
    antes e depois do episódio.

    Preserva a dependência intradiária.
    """

    rng = np.random.default_rng(seed)

    dias_a = antes["dia"].drop_duplicates().to_numpy()
    dias_d = depois["dia"].drop_duplicates().to_numpy()

    if len(dias_a) < 5 or len(dias_d) < 5:
        return np.nan, np.nan, np.nan

    grupos_a = {
        dia: grupo
        for dia, grupo in antes.groupby("dia")
    }
    grupos_d = {
        dia: grupo
        for dia, grupo in depois.groupby("dia")
    }

    estimativas = []

    for _ in range(n_boot):
        sorteio_a = rng.choice(
            dias_a, size=len(dias_a), replace=True
        )
        sorteio_d = rng.choice(
            dias_d, size=len(dias_d), replace=True
        )

        amostra_a = pd.concat(
            [grupos_a[dia] for dia in sorteio_a],
            ignore_index=True
        )
        amostra_d = pd.concat(
            [grupos_d[dia] for dia in sorteio_d],
            ignore_index=True
        )

        delta, _, _ = media_estratificada(
            amostra_a, amostra_d
        )

        if np.isfinite(delta):
            estimativas.append(delta)

    if len(estimativas) < 100:
        return np.nan, np.nan, len(estimativas)

    li, ls = np.percentile(
        estimativas, [2.5, 97.5]
    )

    return li, ls, len(estimativas)


# ------------------------------------------------------------
# 6. COMPARAÇÃO ENTRE REGIMES
# ------------------------------------------------------------

resultados = []
detalhes_faixas = []

for nome, evento in EVENTOS.items():

    referencia = evento["referencia"]
    inicio_exclusao = evento["exclusao_inicio"]
    fim_exclusao = evento["exclusao_fim"]
    fim_analise = evento["fim_analise"]

    print("\n" + "=" * 72)
    print(nome.upper())
    print("=" * 72)

    for janela in JANELAS_DIAS:

        inicio_antes = referencia - pd.Timedelta(
            days=janela
        )
        fim_depois = min(
            referencia + pd.Timedelta(days=janela),
            fim_analise + pd.Timedelta(hours=1)
        )

        antes = base[
            (base["timestamp"] >= inicio_antes)
            & (base["timestamp"] < inicio_exclusao)
        ].copy()

        depois = base[
            (base["timestamp"] >= fim_exclusao)
            & (base["timestamp"] < fim_depois)
        ].copy()

        delta, media_antes, tab = media_estratificada(
            antes, depois
        )

        if np.isfinite(delta) and media_antes > 0:
            delta_pct = 100 * delta / media_antes
        else:
            delta_pct = np.nan

        li, ls, n_validos = bootstrap_dias(
            antes,
            depois,
            n_boot=N_BOOTSTRAP,
            seed=SEMENTE + janela
        )

        resultados.append({
            "evento": nome,
            "janela_dias": janela,
            "horas_antes": len(antes),
            "horas_depois": len(depois),
            "dias_antes": antes["dia"].nunique(),
            "dias_depois": depois["dia"].nunique(),
            "media_antes_padronizada": media_antes,
            "delta_t_mwh": delta,
            "delta_pct": delta_pct,
            "ic95_inferior": li,
            "ic95_superior": ls,
            "bootstrap_validos": n_validos,
            "faixas_comparaveis": len(tab)
        })

        if not tab.empty:
            tab = tab.copy()
            tab["evento"] = nome
            tab["janela_dias"] = janela
            detalhes_faixas.append(tab)

        print(
            f"{janela:>3} dias | "
            f"antes={len(antes):>5} | "
            f"depois={len(depois):>5} | "
            f"delta={delta:+.5f} | "
            f"IC95%=[{li:+.5f}, {ls:+.5f}]"
        )

resumo = pd.DataFrame(resultados)

if detalhes_faixas:
    tabela_faixas = pd.concat(
        detalhes_faixas, ignore_index=True
    )
else:
    tabela_faixas = pd.DataFrame()

# ------------------------------------------------------------
# 7. TABELA DE RESULTADOS
# ------------------------------------------------------------

print("\nRESULTADOS COMPARATIVOS")

display(
    resumo[
        [
            "evento",
            "janela_dias",
            "horas_antes",
            "horas_depois",
            "delta_t_mwh",
            "delta_pct",
            "ic95_inferior",
            "ic95_superior",
            "faixas_comparaveis"
        ]
    ].round(5)
)

# ------------------------------------------------------------
# 8. GRÁFICO DE PERSISTÊNCIA
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1, figsize=(12, 9), sharex=True
)

for ax, (nome, grupo) in zip(
    axes, resumo.groupby("evento", sort=False)
):

    grupo = grupo.sort_values("janela_dias")

    x = grupo["janela_dias"].to_numpy()
    y = grupo["delta_t_mwh"].to_numpy()

    li = grupo["ic95_inferior"].to_numpy()
    ls = grupo["ic95_superior"].to_numpy()

    ax.plot(x, y, marker="o", linewidth=2)

    validos = (
        np.isfinite(y)
        & np.isfinite(li)
        & np.isfinite(ls)
    )

    if validos.any():
        ax.fill_between(
            x[validos],
            li[validos],
            ls[validos],
            alpha=0.2,
            label="IC 95% — bootstrap diário"
        )

    ax.axhline(0, color="black", linestyle="--")
    ax.set_title(nome.replace("_", " ").upper())
    ax.set_ylabel("Δ intensidade (tCO₂/MWh)")
    ax.grid(alpha=0.25)
    ax.legend()

axes[-1].set_xlabel(
    "Horizonte de comparação (dias)"
)

fig.suptitle(
    "Iatan 2 — Magnitude e persistência das mudanças",
    fontsize=14
)

plt.tight_layout()

ARQUIVO_FIGURA = (
    DIR_FIGURAS / "13_magnitude_persistencia.png"
)

plt.savefig(
    ARQUIVO_FIGURA, dpi=180, bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 9. SALVAMENTO
# ------------------------------------------------------------

ARQUIVO_RESUMO = (
    DIR_TABELAS / "13_magnitude_persistencia.csv"
)

ARQUIVO_FAIXAS = (
    DIR_TABELAS / "13_comparacao_por_faixa_carga.csv"
)

resumo.to_csv(
    ARQUIVO_RESUMO, index=False
)

tabela_faixas.to_csv(
    ARQUIVO_FAIXAS, index=False
)

# ------------------------------------------------------------
# 10. DIAGNÓSTICO
# ------------------------------------------------------------

print("\nDIAGNÓSTICO EXPLORATÓRIO")

for nome, grupo in resumo.groupby("evento"):

    grupo = grupo.sort_values("janela_dias")

    validos = grupo[
        grupo["delta_t_mwh"].notna()
    ]

    if validos.empty:
        print(f"{nome}: sem comparação admissível.")
        continue

    reducoes = (
        validos["delta_t_mwh"] < 0
    ).sum()

    print(
        f"{nome}: redução observada em "
        f"{reducoes}/{len(validos)} horizontes."
    )

    significativos = (
        validos["ic95_superior"] < 0
    ).sum()

    print(
        f"  IC inteiramente negativo em "
        f"{significativos}/{len(validos)} horizontes."
    )

print("\n" + "=" * 72)
print("CÉLULA 13/20 — CONCLUÍDA")
print("=" * 72)

print("Tabela:", ARQUIVO_RESUMO)
print("Comparações por carga:", ARQUIVO_FAIXAS)
print("Figura:", ARQUIVO_FIGURA)

print(
    "\nATENÇÃO: resultados observacionais. "
    "Persistência estatística não comprova manutenção "
    "nem estabelece causalidade."
)


# ===== ORIGEM: CÉLULA 14 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 14/20 — PERSISTÊNCIA TEMPORAL E ROBUSTEZ ESTATÍSTICA
# IATAN 2 | EPA 6065 | Unidade 2 | 2011–2025
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

print("=" * 72)
print("CÉLULA 14/20 — PERSISTÊNCIA TEMPORAL E ROBUSTEZ")
print("=" * 72)

# ------------------------------------------------------------
# 1. DIRETÓRIOS
# ------------------------------------------------------------

OUTPUT_ROOT = Path(OUTPUT_ROOT)

DIR_DADOS = OUTPUT_ROOT / "dados"
DIR_TABELAS = OUTPUT_ROOT / "tabelas"
DIR_FIGURAS = OUTPUT_ROOT / "figuras"

for pasta in [DIR_DADOS, DIR_TABELAS, DIR_FIGURAS]:
    pasta.mkdir(parents=True, exist_ok=True)

arquivo = (
    DIR_DADOS /
    "iatan2_intensidade_horaria_2011_2025.parquet"
)

if not arquivo.exists():
    raise FileNotFoundError(
        f"Base horária não encontrada: {arquivo}"
    )

df = pd.read_parquet(arquivo)

# ------------------------------------------------------------
# 2. CONFIGURAÇÕES
# ------------------------------------------------------------

# Datas diárias de referência.
# Não representam necessariamente o instante horário exato.
EVENTOS = {
    "marco_2025": pd.Timestamp("2025-03-26"),
    "julho_2025": pd.Timestamp("2025-07-14"),
}

CARGA_MIN = 400
FAIXAS_CARGA = [400, 500, 600, 700, 800, 900, np.inf]

JANELA_REFERENCIA_DIAS = 30
HORIZONTES = [7, 14, 30, 60, 90]

N_BOOTSTRAP = 1000
BLOCO_DIAS = 7
SEMENTE = 42

# Evita que a análise de março incorpore a mudança de julho.
LIMITE_MARCO = EVENTOS["julho_2025"]

# ------------------------------------------------------------
# 3. PREPARAÇÃO
# ------------------------------------------------------------

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="coerce"
)

for coluna in [
    "carga_bruta_mw",
    "geracao_mwh",
    "co2_t",
    "intensidade_t_mwh",
]:
    df[coluna] = pd.to_numeric(
        df[coluna], errors="coerce"
    )

elegivel = df["elegivel_intensidade"].fillna(False)

base = df.loc[
    elegivel
    & df["timestamp"].notna()
    & (df["carga_bruta_mw"] >= CARGA_MIN)
    & np.isfinite(df["intensidade_t_mwh"])
    & (df["geracao_mwh"] > 0)
    & (df["co2_t"] >= 0)
].copy()

base["data"] = base["timestamp"].dt.normalize()

base["faixa_carga"] = pd.cut(
    base["carga_bruta_mw"],
    bins=FAIXAS_CARGA,
    right=False,
    labels=False
)

print(f"Registros totais: {len(df):,}")
print(f"Horas elegíveis na análise: {len(base):,}")

# ------------------------------------------------------------
# 4. INTENSIDADE PADRONIZADA POR CARGA
# ------------------------------------------------------------

def intensidade_padronizada(antes, depois):
    """
    Compara intensidade como razão de somas,
    padronizando a distribuição de carga pelos
    pesos de geração do período anterior.
    """

    grupos = []

    for faixa in sorted(
        set(antes["faixa_carga"].dropna())
        & set(depois["faixa_carga"].dropna())
    ):
        a = antes.loc[antes["faixa_carga"] == faixa]
        d = depois.loc[depois["faixa_carga"] == faixa]

        if len(a) < 5 or len(d) < 5:
            continue

        ga = a["geracao_mwh"].sum()
        gd = d["geracao_mwh"].sum()

        if ga <= 0 or gd <= 0:
            continue

        grupos.append({
            "faixa": faixa,
            "peso": ga,
            "antes": a["co2_t"].sum() / ga,
            "depois": d["co2_t"].sum() / gd,
        })

    if not grupos:
        return np.nan, np.nan, np.nan, 0

    tabela = pd.DataFrame(grupos)
    pesos = tabela["peso"] / tabela["peso"].sum()

    ia = np.sum(pesos * tabela["antes"])
    id_ = np.sum(pesos * tabela["depois"])

    delta = id_ - ia
    delta_pct = 100 * delta / ia if ia > 0 else np.nan

    return ia, id_, delta, len(tabela)


# ------------------------------------------------------------
# 5. BOOTSTRAP EM BLOCOS DE DIAS
# ------------------------------------------------------------

def reamostrar_blocos(dados, rng, bloco_dias=7):
    """
    Reamostra blocos de dias consecutivos.

    Preserva conjuntamente as observações horárias
    dos dias selecionados.
    """

    datas = pd.date_range(
        dados["data"].min(),
        dados["data"].max(),
        freq="D"
    )

    if len(datas) == 0:
        return dados.iloc[0:0].copy()

    quantidade = len(datas)
    indices = []

    while len(indices) < quantidade:
        inicio = rng.integers(0, quantidade)
        bloco = [
            (inicio + i) % quantidade
            for i in range(bloco_dias)
        ]
        indices.extend(bloco)

    datas_sorteadas = datas[
        np.array(indices[:quantidade])
    ]

    por_dia = {
        dia: grupo
        for dia, grupo in dados.groupby("data")
    }

    partes = [
        por_dia[dia]
        for dia in datas_sorteadas
        if dia in por_dia
    ]

    if not partes:
        return dados.iloc[0:0].copy()

    return pd.concat(partes, ignore_index=True)


def bootstrap_delta(antes, depois, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    estimativas = []

    for _ in range(n):
        a = reamostrar_blocos(
            antes, rng, BLOCO_DIAS
        )
        d = reamostrar_blocos(
            depois, rng, BLOCO_DIAS
        )

        _, _, delta, _ = intensidade_padronizada(a, d)

        if np.isfinite(delta):
            estimativas.append(delta)

    if len(estimativas) < 100:
        return np.nan, np.nan, len(estimativas)

    return (
        np.percentile(estimativas, 2.5),
        np.percentile(estimativas, 97.5),
        len(estimativas)
    )


# ------------------------------------------------------------
# 6. COMPARAÇÃO POR HORIZONTE
# ------------------------------------------------------------

resultados = []

for nome, evento in EVENTOS.items():

    inicio_antes = evento - pd.Timedelta(
        days=JANELA_REFERENCIA_DIAS
    )

    antes = base.loc[
        (base["timestamp"] >= inicio_antes)
        & (base["timestamp"] < evento)
    ].copy()

    for horizonte in HORIZONTES:

        fim = evento + pd.Timedelta(days=horizonte)

        if nome == "marco_2025":
            fim = min(fim, LIMITE_MARCO)

        depois = base.loc[
            (base["timestamp"] >= evento)
            & (base["timestamp"] < fim)
        ].copy()

        ia, id_, delta, n_faixas = (
            intensidade_padronizada(antes, depois)
        )

        ic_inf, ic_sup, n_validos = bootstrap_delta(
            antes,
            depois,
            n=N_BOOTSTRAP,
            seed=SEMENTE + horizonte
        )

        resultados.append({
            "evento": nome,
            "horizonte_dias": horizonte,
            "horas_antes": len(antes),
            "horas_depois": len(depois),
            "faixas_comparaveis": n_faixas,
            "intensidade_antes": ia,
            "intensidade_depois": id_,
            "delta_t_mwh": delta,
            "delta_pct": (
                100 * delta / ia
                if np.isfinite(ia) and ia > 0
                else np.nan
            ),
            "ic95_inferior": ic_inf,
            "ic95_superior": ic_sup,
            "bootstrap_validos": n_validos,
        })

resumo = pd.DataFrame(resultados)

# ------------------------------------------------------------
# 7. BLOCOS SUCESSIVOS APÓS CADA EVENTO
# ------------------------------------------------------------

blocos_resultado = []

for nome, evento in EVENTOS.items():

    antes = base.loc[
        (base["timestamp"] >= evento - pd.Timedelta(days=30))
        & (base["timestamp"] < evento)
    ].copy()

    limite = (
        LIMITE_MARCO
        if nome == "marco_2025"
        else pd.Timestamp("2026-01-01")
    )

    for tamanho in [7, 14, 30]:

        inicio = evento
        numero = 1

        while inicio < limite:

            fim = min(
                inicio + pd.Timedelta(days=tamanho),
                limite
            )

            # Não analisar blocos finais incompletos.
            if (fim - inicio).days < tamanho:
                break

            depois = base.loc[
                (base["timestamp"] >= inicio)
                & (base["timestamp"] < fim)
            ].copy()

            ia, id_, delta, n_faixas = (
                intensidade_padronizada(antes, depois)
            )

            blocos_resultado.append({
                "evento": nome,
                "bloco_dias": tamanho,
                "numero_bloco": numero,
                "inicio": inicio,
                "fim_exclusivo": fim,
                "horas": len(depois),
                "faixas_comparaveis": n_faixas,
                "delta_t_mwh": delta,
                "delta_pct": (
                    100 * delta / ia
                    if np.isfinite(ia) and ia > 0
                    else np.nan
                ),
            })

            inicio = fim
            numero += 1

blocos = pd.DataFrame(blocos_resultado)

# ------------------------------------------------------------
# 8. FIGURAS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1, figsize=(13, 9), sharex=False
)

for ax, (nome, grupo) in zip(
    axes,
    resumo.groupby("evento", sort=False)
):
    grupo = grupo.sort_values("horizonte_dias")

    x = grupo["horizonte_dias"].to_numpy()
    y = grupo["delta_t_mwh"].to_numpy()

    ax.plot(x, y, marker="o", linewidth=2)
    ax.fill_between(
        x,
        grupo["ic95_inferior"].to_numpy(),
        grupo["ic95_superior"].to_numpy(),
        alpha=0.2
    )

    ax.axhline(0, color="black", linestyle="--")
    ax.set_title(nome.upper())
    ax.set_ylabel("Δ intensidade (tCO₂/MWh)")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Horizonte acumulado (dias)")

plt.suptitle(
    "Iatan 2 — Persistência e bootstrap em blocos",
    fontsize=14
)
plt.tight_layout()

fig.savefig(
    DIR_FIGURAS / "14_persistencia_bootstrap_blocos.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

fig, axes = plt.subplots(
    2, 1, figsize=(13, 8)
)

for ax, nome in zip(axes, EVENTOS):

    dados = blocos.loc[
        (blocos["evento"] == nome)
        & (blocos["bloco_dias"] == 14)
    ]

    ax.plot(
        dados["inicio"],
        dados["delta_t_mwh"],
        marker="o"
    )

    ax.axhline(0, color="black", linestyle="--")
    ax.set_title(nome.upper())
    ax.set_ylabel("Δ intensidade (tCO₂/MWh)")
    ax.grid(alpha=0.25)

plt.suptitle(
    "Iatan 2 — Evolução em blocos sucessivos de 14 dias"
)
plt.tight_layout()

fig.savefig(
    DIR_FIGURAS / "14_blocos_sucessivos_14_dias.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 9. EXPORTAÇÃO
# ------------------------------------------------------------

arquivo_resumo = (
    DIR_TABELAS /
    "14_persistencia_bootstrap.csv"
)

arquivo_blocos = (
    DIR_TABELAS /
    "14_persistencia_blocos_sucessivos.csv"
)

resumo.to_csv(
    arquivo_resumo,
    index=False,
    encoding="utf-8-sig"
)

blocos.to_csv(
    arquivo_blocos,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 10. RESULTADOS
# ------------------------------------------------------------

print("\nRESULTADOS POR HORIZONTE")

display(
    resumo[
        [
            "evento",
            "horizonte_dias",
            "horas_antes",
            "horas_depois",
            "delta_t_mwh",
            "delta_pct",
            "ic95_inferior",
            "ic95_superior",
        ]
    ].round(5)
)

print("\nBLOCOS SUCESSIVOS DE 14 DIAS")

display(
    blocos.loc[
        blocos["bloco_dias"] == 14,
        [
            "evento",
            "numero_bloco",
            "inicio",
            "horas",
            "delta_t_mwh",
            "delta_pct",
        ]
    ].round(5)
)

print("\n" + "=" * 72)
print("CÉLULA 14/20 — CONCLUÍDA")
print("=" * 72)
print("Resumo:", arquivo_resumo)
print("Blocos:", arquivo_blocos)
print("Figuras:", DIR_FIGURAS)


# ===== ORIGEM: CÉLULA 15 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 15/20 — CONSOLIDAÇÃO E CARACTERIZAÇÃO DOS EVENTOS
# PROJETO IATAN | EPA 6065 | UNIDADE 2
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

print("=" * 72)
print("CÉLULA 15/20 — CONSOLIDAÇÃO DOS EVENTOS")
print("=" * 72)

# ------------------------------------------------------------
# 1. DIRETÓRIOS E ARQUIVOS
# ------------------------------------------------------------

OUTPUT_ROOT = Path(OUTPUT_ROOT)

DIR_DADOS = OUTPUT_ROOT / "dados"
DIR_TABELAS = OUTPUT_ROOT / "tabelas"
DIR_FIGURAS = OUTPUT_ROOT / "figuras"
DIR_RELATORIOS = OUTPUT_ROOT / "relatorios"

for pasta in [
    DIR_DADOS, DIR_TABELAS, DIR_FIGURAS, DIR_RELATORIOS
]:
    pasta.mkdir(parents=True, exist_ok=True)

arquivo_base = (
    DIR_DADOS /
    "iatan2_intensidade_horaria_2011_2025.parquet"
)

arquivo_13 = DIR_TABELAS / "13_magnitude_persistencia.csv"
arquivo_14 = DIR_TABELAS / "14_persistencia_bootstrap.csv"
arquivo_blocos = (
    DIR_TABELAS / "14_persistencia_blocos_sucessivos.csv"
)

arquivos = [
    arquivo_base, arquivo_13, arquivo_14, arquivo_blocos
]

for arquivo in arquivos:
    if not arquivo.exists():
        raise FileNotFoundError(
            f"Arquivo necessário não encontrado: {arquivo}"
        )

base = pd.read_parquet(arquivo_base)
r13 = pd.read_csv(arquivo_13)
r14 = pd.read_csv(arquivo_14)
blocos = pd.read_csv(arquivo_blocos)

base["timestamp"] = pd.to_datetime(
    base["timestamp"], errors="coerce"
)

for coluna in [
    "carga_bruta_mw",
    "geracao_mwh",
    "co2_t",
    "intensidade_t_mwh",
]:
    base[coluna] = pd.to_numeric(
        base[coluna], errors="coerce"
    )

base["data"] = base["timestamp"].dt.normalize()

print(f"Base histórica: {len(base):,} registros")
print("Resultados anteriores carregados.")

# ------------------------------------------------------------
# 2. EVENTOS CANDIDATOS
# ------------------------------------------------------------

# Referências diárias utilizadas na Célula 14.
# Os instantes horários devem ser confirmados com a Célula 11.

EVENTOS = {
    "marco_2025": {
        "data_referencia": pd.Timestamp("2025-03-26"),
        "instante_horario": pd.NaT,
    },
    "julho_2025": {
        "data_referencia": pd.Timestamp("2025-07-14"),
        "instante_horario": pd.NaT,
    },
}

# Procurar arquivos de resultados da Célula 11.
# Não inferir automaticamente instantes a partir
# de nomes ou de arquivos não identificados.

candidatos_11 = [
    caminho for caminho in [DIR_TABELAS / "eventos_horarios_candidatos.csv"]
    if caminho.is_file()
]

print("\nARQUIVOS DA CÉLULA 11")

if candidatos_11:
    for caminho in candidatos_11:
        print("-", caminho.name)
else:
    print("Nenhum CSV da Célula 11 localizado.")

# Exibir esquema dos arquivos para conferência.
for caminho in candidatos_11:
    try:
        temp = pd.read_csv(caminho, nrows=5)
        print(f"\n{caminho.name}")
        print("Colunas:", list(temp.columns))
        display(temp.head(3))
    except Exception as erro:
        print("Erro de leitura:", erro)

# ------------------------------------------------------------
# 3. CONSOLIDAÇÃO DAS MAGNITUDES
# ------------------------------------------------------------

def extrair_horizonte(tabela, evento, horizonte=30):
    linha = tabela.loc[
        (tabela["evento"] == evento)
        & (tabela["horizonte_dias"] == horizonte)
    ]

    if linha.empty:
        return None

    return linha.iloc[0]


resultados = []

for nome, info in EVENTOS.items():

    linha14 = extrair_horizonte(r14, nome, 30)

    if linha14 is None:
        raise ValueError(
            f"Resultado de 30 dias ausente: {nome}"
        )

    linha13 = r13.loc[
        (r13["evento"] == nome)
        & (r13["janela_dias"] == 30)
    ]

    linha13 = (
        linha13.iloc[0]
        if not linha13.empty
        else None
    )

    resultados.append({
        "evento": nome,
        "data_referencia": info["data_referencia"],
        "instante_horario_confirmado": pd.NaT,
        "delta_30d_t_mwh": linha14["delta_t_mwh"],
        "delta_30d_pct": linha14["delta_pct"],
        "ic95_bloco_inferior": linha14["ic95_inferior"],
        "ic95_bloco_superior": linha14["ic95_superior"],
        "horas_antes": linha14["horas_antes"],
        "horas_depois": linha14["horas_depois"],
        "delta_13_30d_t_mwh": (
            linha13["delta_t_mwh"]
            if linha13 is not None
            else np.nan
        ),
        "classificacao": "pendente",
    })

eventos = pd.DataFrame(resultados)

# ------------------------------------------------------------
# 4. PERSISTÊNCIA POR BLOCOS
# ------------------------------------------------------------

blocos["inicio"] = pd.to_datetime(
    blocos["inicio"], errors="coerce"
)

blocos14 = blocos.loc[
    (blocos["bloco_dias"] == 14)
    & (blocos["horas"] > 0)
    & blocos["delta_t_mwh"].notna()
].copy()

persistencia = []

for nome, grupo in blocos14.groupby("evento"):

    grupo = grupo.sort_values("inicio")

    negativos = int((grupo["delta_t_mwh"] < 0).sum())
    positivos = int((grupo["delta_t_mwh"] > 0).sum())

    primeiro_positivo = grupo.loc[
        grupo["delta_t_mwh"] > 0, "inicio"
    ]

    persistencia.append({
        "evento": nome,
        "blocos_observados": len(grupo),
        "blocos_negativos": negativos,
        "blocos_positivos": positivos,
        "primeiro_bloco_positivo": (
            primeiro_positivo.min()
            if not primeiro_positivo.empty
            else pd.NaT
        ),
        "ultimo_bloco_negativo": (
            grupo.loc[
                grupo["delta_t_mwh"] < 0,
                "inicio"
            ].max()
        ),
    })

persistencia = pd.DataFrame(persistencia)

eventos = eventos.merge(
    persistencia,
    on="evento",
    how="left"
)

for idx, linha in eventos.iterrows():

    if linha["blocos_positivos"] == 0:
        classificacao = (
            "reducao_persistente_no_periodo_observado"
        )
    else:
        classificacao = (
            "reducao_com_recuperacao_posterior"
        )

    eventos.loc[idx, "classificacao"] = classificacao

# ------------------------------------------------------------
# 5. INVESTIGAÇÃO OPERACIONAL DE OUTUBRO
# ------------------------------------------------------------

inicio_outubro = pd.Timestamp("2025-09-15")
fim_outubro = pd.Timestamp("2025-11-16")

outubro = base.loc[
    (base["timestamp"] >= inicio_outubro)
    & (base["timestamp"] < fim_outubro)
].copy()

outubro["elegivel"] = (
    outubro["elegivel_intensidade"]
    .fillna(False)
    .astype(bool)
)

auditoria = (
    outubro.groupby("data")
    .agg(
        horas_registradas=("timestamp", "size"),
        horas_elegiveis=("elegivel", "sum"),
        geracao_mwh=("geracao_mwh", "sum"),
        co2_t=("co2_t", "sum"),
        carga_media_mw=("carga_bruta_mw", "mean"),
    )
    .reset_index()
)

auditoria["intensidade_agregada"] = np.where(
    auditoria["geracao_mwh"] > 0,
    auditoria["co2_t"] / auditoria["geracao_mwh"],
    np.nan
)

auditoria["sem_horas_elegiveis"] = (
    auditoria["horas_elegiveis"] == 0
)

# Identificar sequências consecutivas sem horas elegíveis.

auditoria["grupo_sequencia"] = (
    auditoria["sem_horas_elegiveis"]
    != auditoria["sem_horas_elegiveis"].shift()
).cumsum()

paradas = (
    auditoria.loc[auditoria["sem_horas_elegiveis"]]
    .groupby("grupo_sequencia")
    .agg(
        inicio=("data", "min"),
        fim=("data", "max"),
        dias=("data", "size"),
    )
    .reset_index(drop=True)
)

print("\nPERÍODOS SEM HORAS ELEGÍVEIS")

if paradas.empty:
    print("Nenhuma sequência identificada.")
else:
    display(paradas)

# ------------------------------------------------------------
# 6. VERIFICAR RECUPERAÇÃO EM NOVEMBRO
# ------------------------------------------------------------

recuperacao = blocos14.loc[
    blocos14["evento"] == "julho_2025",
    [
        "inicio",
        "horas",
        "delta_t_mwh",
        "delta_pct",
    ]
].copy()

recuperacao["situacao"] = np.select(
    [
        recuperacao["delta_t_mwh"] < 0,
        recuperacao["delta_t_mwh"] > 0,
    ],
    [
        "abaixo_da_referencia",
        "acima_da_referencia",
    ],
    default="igual_ou_indefinido"
)

print("\nEVOLUÇÃO APÓS JULHO")
display(recuperacao.round(5))

# ------------------------------------------------------------
# 7. GRÁFICOS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1, figsize=(14, 9)
)

axes[0].bar(
    auditoria["data"],
    auditoria["horas_elegiveis"],
    width=0.85
)

axes[0].set_title(
    "Iatan 2 — Horas elegíveis por dia "
    "(setembro–novembro de 2025)"
)
axes[0].set_ylabel("Horas elegíveis")
axes[0].set_ylim(0, 24)
axes[0].grid(alpha=0.25)

axes[1].plot(
    auditoria["data"],
    auditoria["geracao_mwh"],
    linewidth=1.5
)

axes[1].set_title(
    "Geração bruta diária — investigação operacional"
)
axes[1].set_ylabel("Geração (MWh/dia)")
axes[1].set_xlabel("Data")
axes[1].grid(alpha=0.25)

plt.tight_layout()

fig.savefig(
    DIR_FIGURAS /
    "15_auditoria_operacional_outubro.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 8. EXPORTAÇÃO
# ------------------------------------------------------------

arquivo_eventos = (
    DIR_TABELAS /
    "15_eventos_consolidados.csv"
)

arquivo_auditoria = (
    DIR_TABELAS /
    "15_auditoria_operacional.csv"
)

arquivo_paradas = (
    DIR_TABELAS /
    "15_periodos_sem_horas_elegiveis.csv"
)

eventos.to_csv(
    arquivo_eventos,
    index=False,
    encoding="utf-8-sig"
)

auditoria.to_csv(
    arquivo_auditoria,
    index=False,
    encoding="utf-8-sig"
)

paradas.to_csv(
    arquivo_paradas,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 9. RELATÓRIO METODOLÓGICO
# ------------------------------------------------------------

relatorio = {
    "projeto": "Iatan 2",
    "planta_epa": 6065,
    "unidade": "2",
    "periodo_historico": "2011-2025",
    "eventos_principais": [
        "marco_2025",
        "julho_2025"
    ],
    "criterios": {
        "controle_carga": True,
        "bootstrap_blocos_dias": 7,
        "comparacao_principal_dias": 30,
        "persistencia_blocos_dias": 14,
    },
    "ressalvas": [
        "Eventos estatisticos nao comprovam manutencao.",
        "Instantes horarios precisam ser confirmados.",
        "Mudancas retrospectivas podem causar vazamento.",
        "Recuperacao de novembro exige investigacao.",
        "Dias sem horas elegiveis nao comprovam parada.",
    ],
}

arquivo_relatorio = (
    DIR_RELATORIOS /
    "15_caracterizacao_eventos.json"
)

with open(
    arquivo_relatorio, "w", encoding="utf-8"
) as f:
    json.dump(
        relatorio,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 10. RESULTADOS
# ------------------------------------------------------------

print("\nEVENTOS CONSOLIDADOS")

display(
    eventos[
        [
            "evento",
            "data_referencia",
            "delta_30d_t_mwh",
            "delta_30d_pct",
            "ic95_bloco_inferior",
            "ic95_bloco_superior",
            "blocos_negativos",
            "blocos_positivos",
            "classificacao",
        ]
    ].round(5)
)

print("\n" + "=" * 72)
print("CÉLULA 15/20 — CONCLUÍDA")
print("=" * 72)

print("Eventos:", arquivo_eventos)
print("Auditoria:", arquivo_auditoria)
print("Períodos:", arquivo_paradas)
print("Relatório:", arquivo_relatorio)

print(
    "\nATENÇÃO: instantes horários ainda não "
    "foram registrados automaticamente."
)

## 10 — Paradas operacionais

Classifica estados operacionais a partir de `opTime`, constrói paradas e compara IC antes/depois em cargas semelhantes. Dados ausentes não significam desligamento.

---

**Interpretação operacional — EPA (2022, p. 8, *Hourly Data*); Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11).** Paradas, partidas e mudanças de carga podem acompanhar mudanças em desempenho de unidades térmicas. Aqui, os episódios são **inferidos de registros de operação**, não extraídos de um histórico certificado de manutenção. O estudo estima associações entre eventos e intensidade, sem atribuir causas específicas a cada ocorrência.

**Resultado da execução otimizada:** identificaram-se **247 episódios de desligamento**, dos quais **178 duraram ao menos 24 horas**. A classificação histórico-estatística das 178 paradas, apresentada na próxima seção, não constitui registro certificado de manutenção.

In [ ]:
# ===== ORIGEM: CÉLULA 16 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 16/20 — PARADAS OPERACIONAIS HISTÓRICAS
# VERSÃO CORRIGIDA
# IATAN 2 | 2011–2025
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

print("=" * 72)
print("CÉLULA 16/20 — PARADAS OPERACIONAIS HISTÓRICAS")
print("=" * 72)

# ------------------------------------------------------------
# 1. Recuperar base
# ------------------------------------------------------------

base = pd.read_parquet(ARQUIVO_ANALITICO).copy()  # base explícita; sem depender de variáveis residuais

obrigatorias = [
    "timestamp",
    "tempo_operacao_h",
    "geracao_mwh"
]

faltantes = [
    c for c in obrigatorias if c not in base.columns
]

if faltantes:
    raise ValueError(f"Colunas ausentes: {faltantes}")

base["timestamp"] = pd.to_datetime(base["timestamp"])

base = (
    base.sort_values("timestamp")
        .reset_index(drop=True)
)

if base["timestamp"].duplicated().any():
    raise ValueError("Existem timestamps duplicados.")

# ------------------------------------------------------------
# 2. Classificação operacional
# ------------------------------------------------------------

tempo = pd.to_numeric(
    base["tempo_operacao_h"], errors="coerce"
)

geracao = pd.to_numeric(
    base["geracao_mwh"], errors="coerce"
)

base["estado_operacional"] = np.select(
    [
        tempo.eq(0),
        tempo.gt(0) & geracao.gt(0),
        tempo.gt(0) & geracao.eq(0)
    ],
    [
        "desligada",
        "operando",
        "operacao_sem_geracao"
    ],
    default="indeterminado"
)

print("\nCLASSIFICAÇÃO OPERACIONAL")
display(
    base["estado_operacional"]
    .value_counts()
    .rename_axis("estado")
    .reset_index(name="horas")
)

# ------------------------------------------------------------
# 3. Detectar sequências contínuas de desligamento
# ------------------------------------------------------------

base["desligada"] = (
    base["estado_operacional"] == "desligada"
)

# Nova sequência se houver mudança de estado ou
# descontinuidade temporal.

nova_sequencia = (
    base["desligada"].ne(base["desligada"].shift())
    |
    base["timestamp"].diff().ne(pd.Timedelta(hours=1))
)

base["sequencia_id"] = nova_sequencia.cumsum()

paradas = (
    base.loc[base["desligada"]]
    .groupby("sequencia_id")
    .agg(
        inicio=("timestamp", "min"),
        ultima_hora=("timestamp", "max"),
        duracao_horas=("timestamp", "size")
    )
    .reset_index(drop=True)
)

paradas["fim_exclusivo"] = (
    paradas["ultima_hora"] + pd.Timedelta(hours=1)
)

paradas["duracao_dias"] = (
    paradas["duracao_horas"] / 24
)

paradas["ano"] = paradas["inicio"].dt.year

paradas.insert(
    0, "parada_id",
    np.arange(1, len(paradas) + 1)
)

# ------------------------------------------------------------
# 4. Resumo anual
# ------------------------------------------------------------

anos = pd.Index(
    range(2011, 2026), name="ano"
)

resumo = (
    paradas.groupby("ano")
    .agg(
        total_paradas=("parada_id", "size"),
        paradas_24h=(
            "duracao_horas",
            lambda s: int((s >= 24).sum())
        ),
        paradas_72h=(
            "duracao_horas",
            lambda s: int((s >= 72).sum())
        ),
        paradas_7dias=(
            "duracao_horas",
            lambda s: int((s >= 168).sum())
        ),
        horas_paradas=("duracao_horas", "sum"),
        maior_parada_h=("duracao_horas", "max")
    )
    .reindex(anos, fill_value=0)
    .reset_index()
)

print("\nRESUMO ANUAL")
display(resumo)

print("\n20 MAIORES PARADAS")
display(
    paradas.nlargest(
        20, "duracao_horas"
    )[
        [
            "parada_id",
            "inicio",
            "fim_exclusivo",
            "duracao_horas",
            "duracao_dias"
        ]
    ]
)

# ------------------------------------------------------------
# 5. Paradas anteriores às mudanças investigadas
# ------------------------------------------------------------

eventos_referencia = {
    "mudanca_2017": "2017-03-01",
    "mudanca_marco_2025": "2025-03-26",
    "mudanca_julho_2025": "2025-07-14"
}

print("\nPARADAS ANTERIORES ÀS MUDANÇAS")

for nome, data in eventos_referencia.items():

    data = pd.Timestamp(data)

    anteriores = paradas.loc[
        (paradas["fim_exclusivo"] <= data)
        &
        (
            paradas["fim_exclusivo"]
            >= data - pd.Timedelta(days=30)
        )
    ].copy()

    anteriores["horas_ate_evento"] = (
        (data - anteriores["fim_exclusivo"])
        / pd.Timedelta(hours=1)
    )

    print("\n" + "-" * 60)
    print(nome.upper(), "|", data.date())

    if anteriores.empty:
        print("Nenhuma parada terminando nos 30 dias anteriores.")
    else:
        display(
            anteriores[
                [
                    "inicio",
                    "fim_exclusivo",
                    "duracao_horas",
                    "horas_ate_evento"
                ]
            ].sort_values("fim_exclusivo")
        )

# ------------------------------------------------------------
# 6. Figuras
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1, figsize=(14, 8),
    sharex=True
)

axes[0].bar(
    resumo["ano"],
    resumo["paradas_24h"]
)

axes[0].set_title(
    "Iatan 2 — Frequência anual de paradas ≥ 24 h"
)

axes[0].set_ylabel("Número de paradas")
axes[0].grid(alpha=0.25)

axes[1].bar(
    resumo["ano"],
    resumo["horas_paradas"]
)

axes[1].set_title(
    "Iatan 2 — Horas anuais de desligamento"
)

axes[1].set_ylabel("Horas")
axes[1].set_xlabel("Ano")
axes[1].set_xticks(resumo["ano"])
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. Exportação
# ------------------------------------------------------------

saida = (
    PROJECT_ROOT
    / "outputs"
    / "07_intensidade_carbonica"
)

dir_tabelas = saida / "tabelas"
dir_dados = saida / "dados"

dir_tabelas.mkdir(parents=True, exist_ok=True)
dir_dados.mkdir(parents=True, exist_ok=True)

paradas.to_csv(
    dir_tabelas / "16_paradas_operacionais.csv",
    index=False
)

resumo.to_csv(
    dir_tabelas / "16_resumo_anual_paradas.csv",
    index=False
)

base.to_parquet(
    dir_dados / "16_estados_operacionais_horarios.parquet",
    index=False
)

print("\n" + "=" * 72)
print("CÉLULA 16/20 — CONCLUÍDA")
print("=" * 72)

print("Paradas identificadas:", len(paradas))
print(
    "Paradas ≥ 24 h:",
    int((paradas["duracao_horas"] >= 24).sum())
)
print(
    "Paradas ≥ 72 h:",
    int((paradas["duracao_horas"] >= 72).sum())
)
print(
    "Paradas ≥ 7 dias:",
    int((paradas["duracao_horas"] >= 168).sum())
)

print(
    "\nNota: desligamento observado não comprova manutenção."
)


# ===== ORIGEM: CÉLULA 17 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 17/20 — PARADAS E MUDANÇAS DE INTENSIDADE
# IATAN 2 | 2011–2025
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

print("=" * 72)
print("CÉLULA 17/20 — ASSOCIAÇÃO HISTÓRICA")
print("=" * 72)

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

RAIZ = PROJECT_ROOT

OUT = OUTPUT_ROOT
TABELAS = OUT / "tabelas"
FIGURAS = OUT / "figuras"

TABELAS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

ANOS = range(2011, 2026)

CARGA_MIN = 400
FAIXAS = np.array([400, 500, 600, 700, 800, 1000])
JANELAS = [15, 30, 60, 90]

MIN_HORAS_LADO = 48
MIN_HORAS_FAIXA = 12
MIN_FAIXAS = 2
MIN_COBERTURA = 0.50

N_BOOT = 400
BLOCO_DIAS = 7
SEMENTE = 2026

# Limiar descritivo, não prova de causalidade
LIMIAR_RELEVANCIA = 0.02  # tCO2/MWh

# ------------------------------------------------------------
# 2. LOCALIZAR BASE HORÁRIA
# ------------------------------------------------------------

base_path = ARQUIVO_ANALITICO
if not base_path.is_file():
    raise FileNotFoundError(f"Execute a etapa de padronização: {base_path}")
df = pd.read_parquet(base_path)

print("Base:", base_path)
print("Registros:", f"{len(df):,}")

# ------------------------------------------------------------
# 3. PADRONIZAR COLUNAS
# ------------------------------------------------------------

def obter_coluna(opcoes):
    mapa = {
        str(c).strip().lower(): c
        for c in df.columns
    }
    for nome in opcoes:
        if nome.lower() in mapa:
            return mapa[nome.lower()]
    return None

def serie_numerica(opcoes):
    c = obter_coluna(opcoes)
    if c is None:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[c], errors="coerce")

col_tempo = obter_coluna(["timestamp", "datetime"])

if col_tempo is None:
    raise ValueError("Coluna timestamp não encontrada.")

dados = pd.DataFrame(index=df.index)

dados["timestamp"] = pd.to_datetime(
    df[col_tempo], errors="coerce"
)

dados["carga"] = serie_numerica([
    "carga_bruta_mw", "Gross Load (MW)"
])

dados["tempo"] = serie_numerica([
    "tempo_operacao_h", "Operating Time"
])

dados["co2"] = serie_numerica([
    "co2_t"
])

if dados["co2"].notna().sum() == 0:
    dados["co2"] = (
        serie_numerica([
            "co2_short_tons",
            "CO2 Mass (short tons)"
        ]) * 0.90718474
    )

dados["geracao"] = dados["carga"] * dados["tempo"]

dados["intensidade"] = (
    dados["co2"] / dados["geracao"].replace(0, np.nan)
)

dados = (
    dados.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .drop_duplicates("timestamp")
    .reset_index(drop=True)
)

dados = dados[
    dados["timestamp"].dt.year.isin(ANOS)
].copy()

# Grade horária para preservar intervalos desligados
grade = pd.DataFrame({
    "timestamp": pd.date_range(
        "2011-01-01",
        "2025-12-31 23:00",
        freq="h"
    )
})

dados = grade.merge(dados, on="timestamp", how="left")

# Não confundir ausência de dados com desligamento
dados["observado"] = dados["tempo"].notna()

dados["ligado"] = (
    dados["observado"]
    & (dados["tempo"] > 0)
    & (dados["carga"] > 0)
)

dados["desligado"] = (
    dados["observado"]
    & (dados["tempo"] == 0)
)

dados["elegivel"] = (
    dados["ligado"]
    & (dados["carga"] >= CARGA_MIN)
    & dados["intensidade"].between(0.3, 1.8)
)

print("Horas observadas:", int(dados["observado"].sum()))
print("Horas elegíveis:", int(dados["elegivel"].sum()))

# ------------------------------------------------------------
# 4. IDENTIFICAR PARADAS CONTÍNUAS
# ------------------------------------------------------------

# Um registro ausente interrompe a contagem da parada.
estado = np.select(
    [dados["desligado"], dados["ligado"]],
    ["desligado", "ligado"],
    default="indeterminado"
)

dados["estado"] = estado

grupo = (
    dados["estado"]
    != dados["estado"].shift()
).cumsum()

paradas = (
    dados.loc[dados["desligado"]]
    .groupby(grupo[dados["desligado"]])
    .agg(
        inicio=("timestamp", "min"),
        ultima_hora=("timestamp", "max"),
        duracao_h=("timestamp", "size")
    )
    .reset_index(drop=True)
)

paradas["fim"] = (
    paradas["ultima_hora"]
    + pd.Timedelta(hours=1)
)

paradas = (
    paradas[paradas["duracao_h"] >= 24]
    .sort_values("inicio")
    .reset_index(drop=True)
)

paradas.insert(
    0, "parada_id",
    np.arange(1, len(paradas) + 1)
)

paradas["ano"] = paradas["inicio"].dt.year

paradas["categoria"] = pd.cut(
    paradas["duracao_h"],
    bins=[24, 72, 168, np.inf],
    labels=["24–71 h", "72–167 h", "≥168 h"],
    right=False
)

print("Paradas >= 24 h:", len(paradas))

# ------------------------------------------------------------
# 5. BASE ELEGÍVEL PARA COMPARAÇÕES
# ------------------------------------------------------------

validos = dados.loc[
    dados["elegivel"],
    ["timestamp", "carga", "intensidade"]
].copy()

validos["dia"] = validos["timestamp"].dt.floor("D")

validos["faixa"] = pd.cut(
    validos["carga"],
    bins=FAIXAS,
    right=False,
    labels=False
)

validos = validos.dropna(subset=["faixa"])
validos["faixa"] = validos["faixa"].astype(int)

# ------------------------------------------------------------
# 6. EFEITO AJUSTADO POR FAIXAS DE CARGA
# ------------------------------------------------------------

def efeito_ajustado(antes, depois):

    a = antes.groupby("faixa", observed=True).agg(
        media_a=("intensidade", "mean"),
        n_a=("intensidade", "size")
    )

    b = depois.groupby("faixa", observed=True).agg(
        media_b=("intensidade", "mean"),
        n_b=("intensidade", "size")
    )

    comp = a.join(b, how="inner")

    comp = comp[
        (comp["n_a"] >= MIN_HORAS_FAIXA)
        & (comp["n_b"] >= MIN_HORAS_FAIXA)
    ]

    if len(comp) < MIN_FAIXAS:
        return np.nan, np.nan, len(comp)

    # Pesos de sobreposição: mesmas faixas nos dois lados
    pesos = np.minimum(
        comp["n_a"].to_numpy(),
        comp["n_b"].to_numpy()
    ).astype(float)

    media_a = np.average(comp["media_a"], weights=pesos)
    media_b = np.average(comp["media_b"], weights=pesos)

    delta = media_b - media_a

    percentual = (
        100 * delta / media_a
        if media_a > 0 else np.nan
    )

    return delta, percentual, len(comp)

# ------------------------------------------------------------
# 7. BOOTSTRAP EM BLOCOS DE DIAS
# ------------------------------------------------------------

rng = np.random.default_rng(SEMENTE)

def _agregar_por_dia_e_faixa(amostra):
    """Agrega cada dia e faixa UMA VEZ para acelerar o bootstrap.

    Mantém a mesma unidade de reamostragem original: dias observados
    em blocos circulares. Guarda totais e contagens, sem replicar linhas.
    """
    dias = pd.DatetimeIndex(amostra["dia"].unique()).sort_values()
    n_dias = len(dias)
    n_faixas = len(FAIXAS) - 1

    contagens = np.zeros((n_dias, n_faixas), dtype=np.int64)
    somas = np.zeros((n_dias, n_faixas), dtype=np.float64)

    if not n_dias:
        return contagens, somas

    idx_dia = dias.get_indexer(pd.DatetimeIndex(amostra["dia"]))
    idx_faixa = amostra["faixa"].to_numpy(dtype=np.intp)
    intensidades = amostra["intensidade"].to_numpy(dtype=np.float64)

    np.add.at(contagens, (idx_dia, idx_faixa), 1)
    np.add.at(somas, (idx_dia, idx_faixa), intensidades)
    return contagens, somas


def _indices_bootstrap_circular(n_dias, bloco=BLOCO_DIAS):
    """Mesmo sorteio de blocos circulares do procedimento original.

    A amostra tem exatamente n_dias, preservando repetição de dias
    e pesos de cada observação horária pertencente aos dias sorteados.
    """
    n_blocos = (n_dias + bloco - 1) // bloco
    inicios = rng.integers(0, n_dias, size=n_blocos)
    indices = (inicios[:, None] + np.arange(bloco)) % n_dias
    return indices.ravel()[:n_dias]


def _delta_agregado(cont_a, soma_a, cont_b, soma_b):
    """Mesmo efeito_ajustado, usando somas e contagens de cada faixa."""
    suporte = (
        (cont_a >= MIN_HORAS_FAIXA)
        & (cont_b >= MIN_HORAS_FAIXA)
    )
    if int(suporte.sum()) < MIN_FAIXAS:
        return np.nan

    pesos = np.minimum(cont_a[suporte], cont_b[suporte]).astype(float)
    media_a = soma_a[suporte] / cont_a[suporte]
    media_b = soma_b[suporte] / cont_b[suporte]
    return float(np.average(media_b - media_a, weights=pesos))


def intervalo_bootstrap(antes, depois):
    """400 reamostragens/7 dias, sem concatenar milhares de DataFrames.

    Preserva o algoritmo de reamostragem, os limites de suporte,
    a semente e a fórmula do efeito do código original.
    """
    ca_dia, sa_dia = _agregar_por_dia_e_faixa(antes)
    cb_dia, sb_dia = _agregar_por_dia_e_faixa(depois)

    na, nb = ca_dia.shape[0], cb_dia.shape[0]
    if not na or not nb:
        return np.nan, np.nan, 0

    valores = []
    for _ in range(N_BOOT):
        ia = _indices_bootstrap_circular(na)
        ib = _indices_bootstrap_circular(nb)

        delta = _delta_agregado(
            ca_dia[ia].sum(axis=0), sa_dia[ia].sum(axis=0),
            cb_dia[ib].sum(axis=0), sb_dia[ib].sum(axis=0)
        )
        if np.isfinite(delta):
            valores.append(delta)

    if len(valores) < max(100, N_BOOT // 4):
        return np.nan, np.nan, len(valores)

    li, ls = np.percentile(valores, [2.5, 97.5])
    return float(li), float(ls), len(valores)

# ------------------------------------------------------------
# 8. AVALIAR TODAS AS PARADAS
# ------------------------------------------------------------

resultados = []

for i, parada in paradas.iterrows():

    inicio = parada["inicio"]
    fim = parada["fim"]

    anterior = (
        paradas.iloc[i - 1]["fim"]
        if i > 0 else pd.Timestamp("2011-01-01")
    )

    proxima = (
        paradas.iloc[i + 1]["inicio"]
        if i + 1 < len(paradas)
        else pd.Timestamp("2026-01-01")
    )

    for horizonte in JANELAS:

        janela = pd.Timedelta(days=horizonte)

        # Janelas encerradas/iniciadas no próprio desligamento
        a_inicio = max(inicio - janela, anterior)
        b_fim = min(fim + janela, proxima)

        antes = validos[
            (validos["timestamp"] >= a_inicio)
            & (validos["timestamp"] < inicio)
        ]

        depois = validos[
            (validos["timestamp"] >= fim)
            & (validos["timestamp"] < b_fim)
        ]

        # Cobertura operacional da janela disponível
        # Horas de operação podem ser menores que horas calendário.
        n_a = len(antes)
        n_b = len(depois)

        if (
            n_a < MIN_HORAS_LADO
            or n_b < MIN_HORAS_LADO
        ):
            resultados.append({
                "parada_id": parada["parada_id"],
                "ano": parada["ano"],
                "inicio": inicio,
                "fim": fim,
                "duracao_h": parada["duracao_h"],
                "categoria": str(parada["categoria"]),
                "horizonte_dias": horizonte,
                "horas_antes": n_a,
                "horas_depois": n_b,
                "delta_t_mwh": np.nan,
                "delta_pct": np.nan,
                "ic95_inferior": np.nan,
                "ic95_superior": np.nan,
                "status": "amostra_insuficiente"
            })
            continue

        delta, pct, faixas = efeito_ajustado(
            antes, depois
        )

        if not np.isfinite(delta):
            li, ls = np.nan, np.nan
            status = "sem_suporte_carga"
        else:
            li, ls, n_validos = intervalo_bootstrap(
                antes, depois
            )

            if not np.isfinite(li):
                status = "bootstrap_insuficiente"
            elif (
                ls < 0
                and delta <= -LIMIAR_RELEVANCIA
            ):
                status = "reducao_sustentada"
            elif (
                li > 0
                and delta >= LIMIAR_RELEVANCIA
            ):
                status = "aumento_sustentado"
            else:
                status = "sem_evidencia_conclusiva"

        resultados.append({
            "parada_id": parada["parada_id"],
            "ano": parada["ano"],
            "inicio": inicio,
            "fim": fim,
            "duracao_h": parada["duracao_h"],
            "categoria": str(parada["categoria"]),
            "horizonte_dias": horizonte,
            "horas_antes": n_a,
            "horas_depois": n_b,
            "faixas_comparaveis": faixas,
            "delta_t_mwh": delta,
            "delta_pct": pct,
            "ic95_inferior": li,
            "ic95_superior": ls,
            "status": status
        })

    if (i + 1) % 25 == 0:
        print(
            f"Paradas avaliadas: {i+1}/{len(paradas)}"
        )

res = pd.DataFrame(resultados)

# ------------------------------------------------------------
# 9. RESUMO POR PARADA
# ------------------------------------------------------------

resumo = (
    res.groupby("parada_id")
    .agg(
        ano=("ano", "first"),
        inicio=("inicio", "first"),
        fim=("fim", "first"),
        duracao_h=("duracao_h", "first"),
        categoria=("categoria", "first"),
        horizontes_validos=(
            "delta_t_mwh",
            lambda x: x.notna().sum()
        ),
        horizontes_reducao=(
            "status",
            lambda x: (x == "reducao_sustentada").sum()
        ),
        horizontes_aumento=(
            "status",
            lambda x: (x == "aumento_sustentado").sum()
        )
    )
    .reset_index()
)

def classificar(r):

    if r["horizontes_validos"] == 0:
        return "inconclusivo"

    if r["horizontes_reducao"] >= 3:
        return "reducao_persistente"

    if r["horizontes_aumento"] >= 3:
        return "aumento_persistente"

    return "sem_mudanca_persistente_confirmada"

resumo["classificacao"] = resumo.apply(
    classificar, axis=1
)

# ------------------------------------------------------------
# 10. IDENTIFICAR MUDANÇAS SEM PARADA PROLONGADA
# ------------------------------------------------------------

# Datas candidatas identificadas nas células anteriores.
# São referências exploratórias, não manutenção confirmada.

candidatos = {
    "2017-03": pd.Timestamp("2017-03-01"),
    "2023-04": pd.Timestamp("2023-04-01"),
    "2025-03": pd.Timestamp("2025-03-26"),
    "2025-07": pd.Timestamp("2025-07-14")
}

comparacao_eventos = []

for nome, data in candidatos.items():

    anteriores = paradas[
        (paradas["fim"] <= data)
        & (paradas["fim"] >= data - pd.Timedelta(days=30))
    ]

    maiores = anteriores.sort_values(
        "duracao_h", ascending=False
    )

    maior = (
        float(maiores.iloc[0]["duracao_h"])
        if len(maiores) else 0.0
    )

    comparacao_eventos.append({
        "evento": nome,
        "data_referencia": data,
        "paradas_24h_30d": len(anteriores),
        "maior_parada_h_30d": maior,
        "parada_72h": maior >= 72,
        "parada_168h": maior >= 168
    })

eventos = pd.DataFrame(comparacao_eventos)

# ------------------------------------------------------------
# 11. RESUMOS ANUAIS
# ------------------------------------------------------------

anual = (
    resumo.groupby("ano")
    .agg(
        paradas=("parada_id", "count"),
        reducoes_persistentes=(
            "classificacao",
            lambda x: (x == "reducao_persistente").sum()
        ),
        aumentos_persistentes=(
            "classificacao",
            lambda x: (x == "aumento_persistente").sum()
        ),
        inconclusivos=(
            "classificacao",
            lambda x: (x == "inconclusivo").sum()
        )
    )
    .reindex(ANOS, fill_value=0)
    .reset_index()
)

# ------------------------------------------------------------
# 12. SALVAR RESULTADOS
# ------------------------------------------------------------

res.to_csv(
    TABELAS / "17_paradas_comparacoes.csv",
    index=False
)

resumo.to_csv(
    TABELAS / "17_paradas_classificadas.csv",
    index=False
)

anual.to_csv(
    TABELAS / "17_resumo_anual.csv",
    index=False
)

eventos.to_csv(
    TABELAS / "17_eventos_sem_parada_prolongada.csv",
    index=False
)

# ------------------------------------------------------------
# 13. VISUALIZAÇÃO
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    anual["ano"],
    anual["paradas"],
    marker="o",
    label="Paradas >= 24 h"
)

ax.plot(
    anual["ano"],
    anual["reducoes_persistentes"],
    marker="s",
    label="Seguidas de redução persistente"
)

ax.plot(
    anual["ano"],
    anual["aumentos_persistentes"],
    marker="^",
    label="Seguidas de aumento persistente"
)

ax.set_title(
    "Iatan 2 — Paradas e alterações posteriores de intensidade"
)
ax.set_xlabel("Ano")
ax.set_ylabel("Número de eventos")
ax.set_xticks(list(ANOS))
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

plt.savefig(
    FIGURAS / "17_paradas_e_mudancas.png",
    dpi=180
)
plt.show()

# ------------------------------------------------------------
# 14. DIAGNÓSTICO FINAL
# ------------------------------------------------------------

print("\nRESUMO HISTÓRICO")
display(anual)

print("\nCLASSIFICAÇÃO DAS PARADAS")
display(
    resumo["classificacao"]
    .value_counts()
    .rename_axis("classificacao")
    .reset_index(name="quantidade")
)

print("\nEVENTOS CANDIDATOS")
display(eventos)

print("\nPARADAS COM REDUÇÃO PERSISTENTE")
display(
    resumo[
        resumo["classificacao"]
        == "reducao_persistente"
    ].sort_values("inicio")
)

print("\n" + "=" * 72)
print("CÉLULA 17/20 — CONCLUÍDA")
print("=" * 72)

print("Arquivos:", TABELAS)
print("Figura:", FIGURAS / "17_paradas_e_mudancas.png")

print(
    "\nINTERPRETAÇÃO: associação temporal observacional. "
    "Não identifica manutenção nem demonstra causalidade."
)

## 11 — Associação entre paradas e mudanças

Consolida associação temporal observacional entre eventos candidatos e paradas. Não identifica causas nem comprova manutenção.

---

**Limitação de identificação — Truong et al. (2020, §2.2 e §5.1.1); Jewell et al. (2022, §1, pp. 1082–1084).** Uma coincidência temporal entre uma parada operacional e uma quebra detectada na série de intensidade é evidência **observacional**, não identificação causal. A seleção de eventos após examinar as emissões limita a interpretação de significância estatística e a capacidade de antecipar mudanças.

Cada parada de ao menos 24 h recebe uma classificação em `17_paradas_classificadas.csv`, acompanhada das comparações pré/pós em `17_paradas_comparacoes.csv`. Os filtros de potência e suporte amostral desta análise são mais restritivos que os critérios gerais de intensidade. As classes dependem desses critérios e da reamostragem em blocos.

**Eventos candidatos:** quatro paradas ≥24 h terminaram nos 30 dias anteriores à mudança de **26/03/2025**; nenhuma terminou nos 30 dias anteriores à mudança de **14/07/2025**. A proximidade de eventos não prova manutenção nem causalidade.

In [ ]:
# ===== ORIGEM: CÉLULA 18 DO NOTEBOOK EXPLORATÓRIO =====

# ================================================================
# CÉLULA 18/20 — VALIDAÇÃO INTEGRADA DOS EVENTOS
# IATAN 2 | 2011–2025
# VERSÃO CORRIGIDA
# ================================================================
#
# OBJETIVOS
# 1. Reutilizar as paradas identificadas na Célula 16.
# 2. Investigar a relação temporal entre paradas e mudanças.
# 3. Comparar a intensidade antes/depois em cargas equivalentes.
# 4. Identificar eventos sem parada prolongada anterior.
# 5. Preservar a distinção entre associação e causalidade.
#
# NÃO REALIZA
# - Identificação comprovada de manutenção.
# - Análise de heat rate.
# - Alterações nas Células 01–17.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("=" * 76)
print("CÉLULA 18/20 — VALIDAÇÃO INTEGRADA DOS EVENTOS")
print("=" * 76)

# ================================================================
# 1. DIRETÓRIOS
# ================================================================

PROJECT_ROOT_18 = PROJECT_ROOT

OUTPUT_ROOT_18 = OUTPUT_ROOT

DIR_DADOS_18 = OUTPUT_ROOT_18 / "dados"
DIR_TABELAS_18 = OUTPUT_ROOT_18 / "tabelas"
DIR_FIGURAS_18 = OUTPUT_ROOT_18 / "figuras"

for pasta in [
    DIR_DADOS_18,
    DIR_TABELAS_18,
    DIR_FIGURAS_18
]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQUIVO_BASE_18 = (
    DIR_DADOS_18
    / "16_estados_operacionais_horarios.parquet"
)

ARQUIVO_PARADAS_18 = (
    DIR_TABELAS_18
    / "16_paradas_operacionais.csv"
)

for arquivo in [ARQUIVO_BASE_18, ARQUIVO_PARADAS_18]:
    if not arquivo.exists():
        raise FileNotFoundError(
            f"Arquivo necessário não encontrado: {arquivo}"
        )

# ================================================================
# 2. CARREGAMENTO E VALIDAÇÃO
# ================================================================

base18 = pd.read_parquet(ARQUIVO_BASE_18).copy()
paradas18 = pd.read_csv(ARQUIVO_PARADAS_18).copy()

COLUNAS_BASE_18 = [
    "timestamp",
    "carga_bruta_mw",
    "intensidade_t_mwh",
    "tempo_operacao_h",
    "elegivel_intensidade"
]

COLUNAS_PARADAS_18 = [
    "inicio",
    "fim_exclusivo",
    "duracao_horas"
]

faltantes_base = [
    c for c in COLUNAS_BASE_18
    if c not in base18.columns
]

faltantes_paradas = [
    c for c in COLUNAS_PARADAS_18
    if c not in paradas18.columns
]

if faltantes_base:
    raise ValueError(
        f"Colunas ausentes na base: {faltantes_base}"
    )

if faltantes_paradas:
    raise ValueError(
        f"Colunas ausentes nas paradas: {faltantes_paradas}"
    )

base18["timestamp"] = pd.to_datetime(
    base18["timestamp"], errors="coerce"
)

for coluna in [
    "carga_bruta_mw",
    "intensidade_t_mwh",
    "tempo_operacao_h"
]:
    base18[coluna] = pd.to_numeric(
        base18[coluna], errors="coerce"
    )

base18 = (
    base18.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

if base18["timestamp"].duplicated().any():
    raise ValueError(
        "Timestamps duplicados na base horária."
    )

paradas18["inicio"] = pd.to_datetime(
    paradas18["inicio"], errors="coerce"
)

paradas18["fim_exclusivo"] = pd.to_datetime(
    paradas18["fim_exclusivo"], errors="coerce"
)

paradas18["duracao_horas"] = pd.to_numeric(
    paradas18["duracao_horas"], errors="coerce"
)

if paradas18[
    ["inicio", "fim_exclusivo", "duracao_horas"]
].isna().any().any():
    raise ValueError(
        "Existem datas ou durações inválidas nas paradas."
    )

paradas18 = (
    paradas18.sort_values("inicio")
    .reset_index(drop=True)
)

if "parada_id" not in paradas18.columns:
    paradas18.insert(
        0, "parada_id",
        np.arange(1, len(paradas18) + 1)
    )

print(f"Registros horários: {len(base18):,}")
print(f"Paradas da Célula 16: {len(paradas18):,}")
print(
    "Paradas >= 24 h:",
    int((paradas18["duracao_horas"] >= 24).sum())
)

# ================================================================
# 3. PREPARAÇÃO DAS HORAS ELEGÍVEIS
# ================================================================

CARGA_MIN_18 = 400

FAIXAS_CARGA_18 = [
    400, 500, 600, 700, 800, 900, np.inf
]

MIN_HORAS_FAIXA_18 = 12
MIN_FAIXAS_18 = 2

elegivel_original = (
    base18["elegivel_intensidade"]
    .astype("string")
    .str.lower()
    .isin(["true", "1", "sim", "yes"])
)

base18["elegivel_18"] = (
    elegivel_original.fillna(False)
    & base18["tempo_operacao_h"].gt(0)
    & base18["carga_bruta_mw"].ge(CARGA_MIN_18)
    & np.isfinite(base18["intensidade_t_mwh"])
    & base18["intensidade_t_mwh"].gt(0)
)

base18["faixa_carga_18"] = pd.cut(
    base18["carga_bruta_mw"],
    bins=FAIXAS_CARGA_18,
    right=False
)

base18["dia_18"] = (
    base18["timestamp"].dt.floor("D")
)

print(
    "Horas elegíveis para comparação:",
    int(base18["elegivel_18"].sum())
)

# ================================================================
# 4. EVENTOS CANDIDATOS
# ================================================================
#
# Datas de referência das análises anteriores.
# Não representam necessariamente o instante horário exato.
#
# A Célula 18 não substitui o refinamento da Célula 11.
# ================================================================

eventos18 = pd.DataFrame([
    {
        "evento": "2017-03",
        "data_referencia": "2017-03-01"
    },
    {
        "evento": "2023-04",
        "data_referencia": "2023-04-01"
    },
    {
        "evento": "2025-03",
        "data_referencia": "2025-03-26"
    },
    {
        "evento": "2025-07",
        "data_referencia": "2025-07-14"
    }
])

eventos18["data_referencia"] = pd.to_datetime(
    eventos18["data_referencia"]
)

# ================================================================
# 5. FUNÇÃO DE COMPARAÇÃO AJUSTADA POR CARGA
# ================================================================

def comparar_carga_18(data_corte, horizonte_dias):

    janela = pd.Timedelta(days=horizonte_dias)

    antes = base18.loc[
        base18["elegivel_18"]
        & (
            base18["timestamp"]
            >= data_corte - janela
        )
        & (base18["timestamp"] < data_corte)
    ].copy()

    depois = base18.loc[
        base18["elegivel_18"]
        & (base18["timestamp"] >= data_corte)
        & (
            base18["timestamp"]
            < data_corte + janela
        )
    ].copy()

    comparacoes = []

    for faixa in base18["faixa_carga_18"].cat.categories:

        a = antes.loc[
            antes["faixa_carga_18"] == faixa,
            "intensidade_t_mwh"
        ]

        d = depois.loc[
            depois["faixa_carga_18"] == faixa,
            "intensidade_t_mwh"
        ]

        if (
            len(a) < MIN_HORAS_FAIXA_18
            or len(d) < MIN_HORAS_FAIXA_18
        ):
            continue

        comparacoes.append({
            "faixa": str(faixa),
            "n_antes": len(a),
            "n_depois": len(d),
            "media_antes": a.mean(),
            "media_depois": d.mean()
        })

    if len(comparacoes) < MIN_FAIXAS_18:
        return {
            "delta_t_mwh": np.nan,
            "delta_pct": np.nan,
            "faixas_comparaveis": len(comparacoes),
            "horas_antes": len(antes),
            "horas_depois": len(depois)
        }

    tabela = pd.DataFrame(comparacoes)

    pesos = np.minimum(
        tabela["n_antes"].to_numpy(),
        tabela["n_depois"].to_numpy()
    ).astype(float)

    media_antes = np.average(
        tabela["media_antes"],
        weights=pesos
    )

    media_depois = np.average(
        tabela["media_depois"],
        weights=pesos
    )

    delta = media_depois - media_antes

    return {
        "delta_t_mwh": float(delta),
        "delta_pct": (
            float(100 * delta / media_antes)
            if media_antes > 0 else np.nan
        ),
        "faixas_comparaveis": len(tabela),
        "horas_antes": len(antes),
        "horas_depois": len(depois)
    }

# ================================================================
# 6. RELAÇÃO TEMPORAL ENTRE EVENTOS E PARADAS
# ================================================================

JANELAS_18 = [15, 30, 60, 90]

resultados18 = []
relacoes_paradas18 = []

for _, evento in eventos18.iterrows():

    nome = evento["evento"]
    data = evento["data_referencia"]

    # Paradas encerradas nos 30 dias anteriores.
    anteriores = paradas18.loc[
        (paradas18["fim_exclusivo"] <= data)
        & (
            paradas18["fim_exclusivo"]
            >= data - pd.Timedelta(days=30)
        )
    ].copy()

    # Paradas que incluem o instante candidato.
    concomitantes = paradas18.loc[
        (paradas18["inicio"] <= data)
        & (paradas18["fim_exclusivo"] > data)
    ].copy()

    # Paradas iniciadas nos 30 dias posteriores.
    posteriores = paradas18.loc[
        (paradas18["inicio"] > data)
        & (
            paradas18["inicio"]
            <= data + pd.Timedelta(days=30)
        )
    ].copy()

    for tipo, conjunto in [
        ("anterior", anteriores),
        ("concomitante", concomitantes),
        ("posterior", posteriores)
    ]:
        for _, parada in conjunto.iterrows():

            relacoes_paradas18.append({
                "evento": nome,
                "data_referencia": data,
                "parada_id": parada["parada_id"],
                "relacao_temporal": tipo,
                "inicio_parada": parada["inicio"],
                "fim_parada": parada["fim_exclusivo"],
                "duracao_horas": parada["duracao_horas"],
                "distancia_fim_parada_h": (
                    data - parada["fim_exclusivo"]
                ).total_seconds() / 3600
            })

    maior_anterior = (
        anteriores["duracao_horas"].max()
        if len(anteriores) else 0
    )

    if len(anteriores):
        ultima = anteriores.sort_values(
            "fim_exclusivo"
        ).iloc[-1]

        distancia_ultima_h = (
            data - ultima["fim_exclusivo"]
        ).total_seconds() / 3600
    else:
        distancia_ultima_h = np.nan

    for horizonte in JANELAS_18:

        comparacao = comparar_carga_18(
            data, horizonte
        )

        delta = comparacao["delta_t_mwh"]

        if not np.isfinite(delta):
            classificacao = "evidencia_insuficiente"

        elif len(concomitantes):
            classificacao = (
                "evento_durante_parada"
            )

        elif len(anteriores) and delta < 0:
            classificacao = (
                "reducao_compativel_com_parada_anterior"
            )

        elif not len(anteriores) and delta < 0:
            classificacao = (
                "reducao_sem_parada_prolongada_anterior"
            )

        elif delta > 0:
            classificacao = (
                "aumento_de_intensidade"
            )

        else:
            classificacao = (
                "sem_mudanca_detectada"
            )

        resultados18.append({
            "evento": nome,
            "data_referencia": data,
            "horizonte_dias": horizonte,
            "paradas_anteriores_30d": len(anteriores),
            "paradas_concomitantes": len(concomitantes),
            "paradas_posteriores_30d": len(posteriores),
            "maior_parada_anterior_h": maior_anterior,
            "distancia_ultima_parada_h": distancia_ultima_h,
            **comparacao,
            "classificacao_exploratoria": classificacao
        })

resultado18 = pd.DataFrame(resultados18)

relacoes18 = pd.DataFrame(
    relacoes_paradas18,
    columns=[
        "evento",
        "data_referencia",
        "parada_id",
        "relacao_temporal",
        "inicio_parada",
        "fim_parada",
        "duracao_horas",
        "distancia_fim_parada_h"
    ]
)

# ================================================================
# 7. RESULTADOS
# ================================================================

print("\n" + "=" * 76)
print("RESULTADOS POR EVENTO E HORIZONTE")
print("=" * 76)

display(
    resultado18.round(5)
)

print("\nRELAÇÃO TEMPORAL COM AS PARADAS")

if relacoes18.empty:
    print("Nenhuma parada relacionada às janelas avaliadas.")
else:
    display(
        relacoes18.sort_values(
            ["evento", "inicio_parada"]
        )
    )

# ================================================================
# 8. VISUALIZAÇÃO INTEGRADA
# ================================================================

fig, axes = plt.subplots(
    len(eventos18),
    1,
    figsize=(15, 14)
)

for ax, (_, evento) in zip(
    axes, eventos18.iterrows()
):

    nome = evento["evento"]
    data = evento["data_referencia"]

    inicio = data - pd.Timedelta(days=45)
    fim = data + pd.Timedelta(days=45)

    amostra = base18.loc[
        base18["elegivel_18"]
        & (base18["timestamp"] >= inicio)
        & (base18["timestamp"] < fim)
    ].copy()

    serie_diaria = (
        amostra.set_index("timestamp")[
            "intensidade_t_mwh"
        ]
        .resample("D")
        .median()
    )

    ax.plot(
        serie_diaria.index,
        serie_diaria.values,
        linewidth=1.5,
        marker=".",
        markersize=3,
        label="Intensidade diária (mediana)"
    )

    paradas_janela = paradas18.loc[
        (paradas18["inicio"] < fim)
        & (paradas18["fim_exclusivo"] > inicio)
        & (paradas18["duracao_horas"] >= 24)
    ]

    for _, parada in paradas_janela.iterrows():

        ax.axvspan(
            parada["inicio"],
            parada["fim_exclusivo"],
            color="gray",
            alpha=0.22
        )

    ax.axvline(
        data,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label="Data candidata"
    )

    ax.set_title(
        f"Iatan 2 — Evento {nome}"
    )

    ax.set_ylabel("tCO₂/MWh")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")

plt.tight_layout()

arquivo_figura18 = (
    DIR_FIGURAS_18
    / "18_validacao_integrada_eventos.png"
)

plt.savefig(
    arquivo_figura18,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ================================================================
# 9. EXPORTAÇÃO
# ================================================================

arquivo_resultado18 = (
    DIR_TABELAS_18
    / "18_validacao_integrada_eventos.csv"
)

arquivo_relacoes18 = (
    DIR_TABELAS_18
    / "18_relacoes_temporais_paradas.csv"
)

resultado18.to_csv(
    arquivo_resultado18,
    index=False,
    encoding="utf-8-sig"
)

relacoes18.to_csv(
    arquivo_relacoes18,
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 10. AUDITORIA FINAL
# ================================================================

print("\n" + "=" * 76)
print("CÉLULA 18/20 — CONCLUÍDA")
print("=" * 76)

print("Eventos avaliados:", len(eventos18))
print("Horizontes por evento:", JANELAS_18)
print("Comparações:", len(resultado18))
print("Relações com paradas:", len(relacoes18))

print("\nArquivos gerados:")
print(arquivo_resultado18)
print(arquivo_relacoes18)
print(arquivo_figura18)

print(
    "\nATENÇÃO: os resultados representam associações "
    "temporais exploratórias, não comprovação de manutenção."
)


# ===== ORIGEM: CÉLULA 19 DO NOTEBOOK EXPLORATÓRIO =====

# ================================================================
# CÉLULA 19/20 — CONSOLIDAÇÃO E AUDITORIA CIENTÍFICA
# IATAN 2 | 2011–2025
# ================================================================
#
# OBJETIVOS
# 1. Consolidar resultados das Células 16, 17 e 18.
# 2. Distinguir redução observada de evidência estatística.
# 3. Corrigir a classificação das paradas anteriores.
# 4. Auditar persistência, cobertura e comparabilidade.
# 5. Exportar tabelas para a síntese da Célula 20.
#
# IMPORTANTE
# - Não atribui causalidade à manutenção.
# - Não recalcula intervalos de confiança.
# - Não transforma datas retrospectivas em preditores.
# - Não realiza análise de heat rate.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("=" * 76)
print("CÉLULA 19/20 — CONSOLIDAÇÃO E AUDITORIA CIENTÍFICA")
print("=" * 76)

# ================================================================
# 1. DIRETÓRIOS E ARQUIVOS
# ================================================================

ROOT19 = OUTPUT_ROOT

TAB19 = ROOT19 / "tabelas"
FIG19 = ROOT19 / "figuras"

TAB19.mkdir(parents=True, exist_ok=True)
FIG19.mkdir(parents=True, exist_ok=True)

arquivos19 = {
    "paradas": TAB19 / "16_paradas_operacionais.csv",
    "historico": TAB19 / "17_paradas_comparacoes.csv",
    "classificadas": TAB19 / "17_paradas_classificadas.csv",
    "eventos": TAB19 / "18_validacao_integrada_eventos.csv",
    "relacoes": TAB19 / "18_relacoes_temporais_paradas.csv",
}

for nome, caminho in arquivos19.items():
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo necessário ausente ({nome}): {caminho}"
        )

paradas19 = pd.read_csv(arquivos19["paradas"])
historico19 = pd.read_csv(arquivos19["historico"])
classificadas19 = pd.read_csv(arquivos19["classificadas"])
eventos19 = pd.read_csv(arquivos19["eventos"])
relacoes19 = pd.read_csv(arquivos19["relacoes"])

print("Arquivos carregados:")
for nome, caminho in arquivos19.items():
    print(f"  {nome}: {len(pd.read_csv(caminho)):,} registros")

# ================================================================
# 2. PADRONIZAÇÃO E VALIDAÇÃO
# ================================================================

for coluna in ["inicio", "fim_exclusivo"]:
    paradas19[coluna] = pd.to_datetime(
        paradas19[coluna], errors="coerce"
    )

eventos19["data_referencia"] = pd.to_datetime(
    eventos19["data_referencia"], errors="coerce"
)

for coluna in ["inicio_parada", "fim_parada", "data_referencia"]:
    if coluna in relacoes19.columns:
        relacoes19[coluna] = pd.to_datetime(
            relacoes19[coluna], errors="coerce"
        )

for coluna in [
    "horizonte_dias",
    "delta_t_mwh",
    "delta_pct",
    "faixas_comparaveis",
    "horas_antes",
    "horas_depois"
]:
    eventos19[coluna] = pd.to_numeric(
        eventos19[coluna], errors="coerce"
    )

if eventos19[
    ["evento", "data_referencia", "horizonte_dias"]
].duplicated().any():
    raise ValueError(
        "Comparações duplicadas na Célula 18."
    )

if eventos19["data_referencia"].isna().any():
    raise ValueError("Datas candidatas inválidas.")

# ================================================================
# 3. AUDITORIA DOS EVENTOS CANDIDATOS
# ================================================================

LIMIAR_PARADA_19 = 24
LIMIAR_LONGA_19 = 72
LIMIAR_EXTENSA_19 = 168

resumos19 = []

for nome, grupo in eventos19.groupby("evento", sort=False):

    grupo = grupo.sort_values("horizonte_dias")
    data = grupo["data_referencia"].iloc[0]

    anteriores = paradas19.loc[
        (paradas19["fim_exclusivo"] <= data)
        & (
            paradas19["fim_exclusivo"]
            >= data - pd.Timedelta(days=30)
        )
    ].copy()

    posteriores = paradas19.loc[
        (paradas19["inicio"] > data)
        & (
            paradas19["inicio"]
            <= data + pd.Timedelta(days=30)
        )
    ].copy()

    concomitantes = paradas19.loc[
        (paradas19["inicio"] <= data)
        & (paradas19["fim_exclusivo"] > data)
    ].copy()

    anteriores_24 = anteriores.loc[
        anteriores["duracao_horas"] >= LIMIAR_PARADA_19
    ]

    anteriores_72 = anteriores.loc[
        anteriores["duracao_horas"] >= LIMIAR_LONGA_19
    ]

    anteriores_168 = anteriores.loc[
        anteriores["duracao_horas"] >= LIMIAR_EXTENSA_19
    ]

    maior_parada = (
        float(anteriores["duracao_horas"].max())
        if len(anteriores) else 0.0
    )

    if len(anteriores_24):
        ultima_24 = anteriores_24.sort_values(
            "fim_exclusivo"
        ).iloc[-1]

        distancia_24 = (
            data - ultima_24["fim_exclusivo"]
        ).total_seconds() / 3600
    else:
        distancia_24 = np.nan

    deltas = grupo["delta_pct"].dropna()

    # Verifica apenas consistência descritiva dos sinais.
    horizontes_negativos = int(
        (grupo["delta_t_mwh"] < 0).sum()
    )

    horizontes_positivos = int(
        (grupo["delta_t_mwh"] > 0).sum()
    )

    horizontes_validos = int(
        grupo["delta_t_mwh"].notna().sum()
    )

    linha30 = grupo.loc[
        grupo["horizonte_dias"] == 30
    ]

    if len(linha30):
        delta30 = float(linha30["delta_pct"].iloc[0])
        delta_abs30 = float(
            linha30["delta_t_mwh"].iloc[0]
        )
    else:
        delta30 = np.nan
        delta_abs30 = np.nan

    if len(concomitantes):
        associacao = "data_candidata_durante_parada"

    elif len(anteriores_72):
        associacao = (
            "mudanca_com_parada_72h_anterior"
        )

    elif len(anteriores_24):
        associacao = (
            "mudanca_com_parada_24h_anterior"
        )

    else:
        associacao = (
            "sem_parada_24h_anterior"
        )

    if horizontes_validos == 0:
        persistencia = "nao_avaliavel"

    elif horizontes_negativos == horizontes_validos:
        persistencia = "reducao_em_todos_horizontes"

    elif horizontes_positivos == horizontes_validos:
        persistencia = "aumento_em_todos_horizontes"

    else:
        persistencia = "sinal_variavel"

    resumos19.append({
        "evento": nome,
        "data_referencia": data,
        "horizontes_validos": horizontes_validos,
        "horizontes_negativos": horizontes_negativos,
        "horizontes_positivos": horizontes_positivos,
        "delta_30d_t_mwh": delta_abs30,
        "delta_30d_pct": delta30,
        "delta_min_pct": (
            float(deltas.min()) if len(deltas) else np.nan
        ),
        "delta_max_pct": (
            float(deltas.max()) if len(deltas) else np.nan
        ),
        "paradas_anteriores_total": len(anteriores),
        "paradas_anteriores_24h": len(anteriores_24),
        "paradas_anteriores_72h": len(anteriores_72),
        "paradas_anteriores_168h": len(anteriores_168),
        "maior_parada_anterior_h": maior_parada,
        "distancia_ultima_parada_24h_h": distancia_24,
        "paradas_posteriores_30d": len(posteriores),
        "paradas_concomitantes": len(concomitantes),
        "associacao_temporal": associacao,
        "persistencia_descritiva": persistencia
    })

consolidado19 = pd.DataFrame(resumos19)

# ================================================================
# 4. COBERTURA E SUPORTE DE CARGA
# ================================================================

auditoria19 = eventos19.copy()

auditoria19["cobertura_antes"] = (
    auditoria19["horas_antes"]
    / (auditoria19["horizonte_dias"] * 24)
)

auditoria19["cobertura_depois"] = (
    auditoria19["horas_depois"]
    / (auditoria19["horizonte_dias"] * 24)
)

# Cobertura aqui é a proporção de horas elegíveis.
# Não equivale à disponibilidade da unidade.

auditoria19["suporte_carga_adequado"] = (
    auditoria19["faixas_comparaveis"] >= 2
)

auditoria19["amostra_minima"] = (
    (auditoria19["horas_antes"] >= 48)
    & (auditoria19["horas_depois"] >= 48)
)

auditoria19["comparacao_valida"] = (
    auditoria19["delta_t_mwh"].notna()
    & auditoria19["suporte_carga_adequado"]
    & auditoria19["amostra_minima"]
)

# Não interpretar horizontes sobrepostos como
# confirmações estatísticas independentes.
auditoria19["horizontes_sobrepostos"] = True

# ================================================================
# 5. AUDITORIA HISTÓRICA DA CÉLULA 17
# ================================================================

total_paradas19 = len(paradas19)

paradas24_19 = int(
    (paradas19["duracao_horas"] >= 24).sum()
)

paradas72_19 = int(
    (paradas19["duracao_horas"] >= 72).sum()
)

paradas168_19 = int(
    (paradas19["duracao_horas"] >= 168).sum()
)

contagem_classificacao19 = (
    classificadas19["classificacao"]
    .value_counts()
    .rename_axis("classificacao")
    .reset_index(name="quantidade")
)

# Os resultados da Célula 17 são exploratórios.
# A classificação por múltiplos horizontes não deve
# ser interpretada como teste estatístico independente.

# ================================================================
# 6. TABELA DE EVIDÊNCIAS
# ================================================================

evidencias19 = []

for _, linha in consolidado19.iterrows():

    nome = linha["evento"]

    if nome in ["2025-03", "2025-07"]:
        origem = (
            "mudanca_2025_investigada_nas_celulas_anteriores"
        )
    else:
        origem = "data_exploratoria"

    evidencias19.append({
        "evento": nome,
        "origem_da_data": origem,
        "delta_30d_pct": linha["delta_30d_pct"],
        "persistencia_descritiva":
            linha["persistencia_descritiva"],
        "associacao_temporal":
            linha["associacao_temporal"],
        "evidencia_estatistica": (
            "consultar_resultados_bootstrap_anteriores"
        ),
        "manutencao_confirmada": False,
        "causalidade_demonstrada": False,
        "uso_preditivo_direto_autorizado": False
    })

evidencias19 = pd.DataFrame(evidencias19)

# ================================================================
# 7. VISUALIZAÇÃO
# ================================================================

grafico19 = auditoria19.pivot(
    index="horizonte_dias",
    columns="evento",
    values="delta_pct"
)

ax = grafico19.plot(
    figsize=(11, 6),
    marker="o",
    linewidth=2
)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(
    "Iatan 2 — Mudanças de intensidade por horizonte"
)
ax.set_xlabel("Horizonte de comparação (dias)")
ax.set_ylabel("Variação de intensidade (%)")
ax.grid(alpha=0.25)

plt.tight_layout()

arquivo_figura19 = (
    FIG19 / "19_consolidacao_eventos.png"
)

plt.savefig(
    arquivo_figura19,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ================================================================
# 8. EXPORTAÇÃO
# ================================================================

arquivos_saida19 = {
    "consolidado": (
        TAB19 / "19_eventos_consolidados.csv"
    ),
    "auditoria": (
        TAB19 / "19_auditoria_comparacoes.csv"
    ),
    "evidencias": (
        TAB19 / "19_matriz_evidencias.csv"
    ),
    "classificacoes_historicas": (
        TAB19 / "19_classificacoes_historicas.csv"
    )
}

consolidado19.to_csv(
    arquivos_saida19["consolidado"],
    index=False,
    encoding="utf-8-sig"
)

auditoria19.to_csv(
    arquivos_saida19["auditoria"],
    index=False,
    encoding="utf-8-sig"
)

evidencias19.to_csv(
    arquivos_saida19["evidencias"],
    index=False,
    encoding="utf-8-sig"
)

contagem_classificacao19.to_csv(
    arquivos_saida19["classificacoes_historicas"],
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 9. RELATÓRIO FINAL
# ================================================================

print("\n" + "=" * 76)
print("RESUMO HISTÓRICO")
print("=" * 76)

print(f"Paradas totais: {total_paradas19}")
print(f"Paradas >= 24 h: {paradas24_19}")
print(f"Paradas >= 72 h: {paradas72_19}")
print(f"Paradas >= 168 h: {paradas168_19}")

print("\nCLASSIFICAÇÕES HISTÓRICAS DA CÉLULA 17")
display(contagem_classificacao19)

print("\nEVENTOS CONSOLIDADOS")
display(consolidado19.round(4))

print("\nMATRIZ DE EVIDÊNCIAS")
display(evidencias19)

print("\nAUDITORIA DAS COMPARAÇÕES")
display(
    auditoria19[
        [
            "evento",
            "horizonte_dias",
            "delta_pct",
            "cobertura_antes",
            "cobertura_depois",
            "faixas_comparaveis",
            "comparacao_valida"
        ]
    ].round(4)
)

print("\nARQUIVOS EXPORTADOS")
for caminho in arquivos_saida19.values():
    print(caminho)

print("Figura:", arquivo_figura19)

print("\n" + "=" * 76)
print("CÉLULA 19/20 — CONCLUÍDA")
print("=" * 76)

print(
    "\nCONCLUSÃO METODOLÓGICA: "
    "as mudanças observadas são associações temporais. "
    "Não constituem evidência causal de manutenção."
)

## 12 — Refinamento horário e reconciliação das datas de 2025

Os métodos de refinamento têm **funções-objetivo diferentes**. A busca preliminar da seção 08 seleciona a maior redução do desvio entre CO₂ observado e estimado por unidade de geração, com janelas móveis e ponderação pela energia. O instante preliminar de julho está registrado separadamente do refinamento final na tabela 19C. O método 19A utiliza a IC em faixas de carga e escolhe a menor soma de erros quadráticos (SSE), apontando **14/07/2025 às 10h**. Já o 19B utiliza uma **amostra fixa para cada janela**, remove diferenças médias por faixa de carga e compara SSE com e sem mudança; todas as janelas de ±10, ±14, ±21 e ±30 dias apontaram **14/07/2025 às 10h**.

**Regra explícita:** para os cenários M1/M2 e 22D, adota-se o instante consistente entre 19A e 19B, **não** o de maior ganho de RMSE em 2025. A divergência entre métodos será preservada e registrada em CSV, com verificação automática das datas utilizadas. A estabilidade condicional às janelas **não prova o instante físico exato**, não é um intervalo de confiança e não confirma manutenção. Todas as datas foram obtidas retrospectivamente com emissões observadas.

---

**Justificativa do refinamento — Bai e Perron (2003, §3, pp. 3–6); Truong et al. (2020, §2.2 e §5.1.1); Jewell et al. (2022, §1, pp. 1082–1084).** A minimização da SSE é compatível com a família de métodos de segmentação por mudanças de parâmetro; as diferentes funções-objetivo das rotinas 19A/19B e da busca preliminar podem produzir datas distintas. A estabilidade do corte 14/07/2025 em janelas fixas é um **diagnóstico de sensibilidade condicionado ao método**, não um intervalo de confiança para a data verdadeira. Os testes de inferência pós-seleção de Jewell et al. **não foram aplicados**.

In [ ]:
# ===== ORIGEM: CÉLULA 20 DO NOTEBOOK EXPLORATÓRIO =====

# ================================================================
# CÉLULA 19A — REFINAMENTO HORÁRIO DAS MUDANÇAS DE 2025
# IATAN 2 | EPA CEMS | 2011–2025
# ================================================================
#
# OBJETIVOS
# 1. Refinar o instante das mudanças de março e julho/2025.
# 2. Controlar diferenças de carga operacional.
# 3. Comparar regimes antes/depois da mudança.
# 4. Relacionar cronologicamente mudanças e paradas.
# 5. Avaliar a sensibilidade da data estimada.
#
# LIMITAÇÕES
# - Detecção retrospectiva e exploratória.
# - Não demonstra causalidade nem manutenção.
# - Não produz intervalos de confiança para a data.
# - Não autoriza utilizar datas futuras como preditores.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("=" * 76)
print("CÉLULA 19A — REFINAMENTO HORÁRIO DOS EVENTOS DE 2025")
print("=" * 76)

# ================================================================
# 1. ARQUIVOS E PARÂMETROS
# ================================================================

ROOT_19A = OUTPUT_ROOT

DADOS_19A = ROOT_19A / "dados"
TABELAS_19A = ROOT_19A / "tabelas"
FIGURAS_19A = ROOT_19A / "figuras"

for pasta in [TABELAS_19A, FIGURAS_19A]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQ_BASE_19A = (
    DADOS_19A / "16_estados_operacionais_horarios.parquet"
)

ARQ_PARADAS_19A = (
    TABELAS_19A / "16_paradas_operacionais.csv"
)

for arq in [ARQ_BASE_19A, ARQ_PARADAS_19A]:
    if not arq.exists():
        raise FileNotFoundError(arq)

base19a = pd.read_parquet(ARQ_BASE_19A).copy()
paradas19a = pd.read_csv(ARQ_PARADAS_19A).copy()

base19a["timestamp"] = pd.to_datetime(
    base19a["timestamp"], errors="coerce"
)

paradas19a["inicio"] = pd.to_datetime(
    paradas19a["inicio"], errors="coerce"
)

paradas19a["fim_exclusivo"] = pd.to_datetime(
    paradas19a["fim_exclusivo"], errors="coerce"
)

for coluna in [
    "carga_bruta_mw",
    "intensidade_t_mwh",
    "tempo_operacao_h"
]:
    base19a[coluna] = pd.to_numeric(
        base19a[coluna], errors="coerce"
    )

paradas19a["duracao_horas"] = pd.to_numeric(
    paradas19a["duracao_horas"], errors="coerce"
)

base19a = (
    base19a.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

if base19a["timestamp"].duplicated().any():
    raise ValueError("Existem timestamps duplicados.")

# Parâmetros explícitos.
CARGA_MIN_19A = 400
INT_MIN_19A = 0.30
INT_MAX_19A = 1.80

FAIXAS_19A = [400, 500, 600, 700, 800, 900, np.inf]

# Região local onde procuraremos a mudança.
RAIO_BUSCA_DIAS_19A = 10

# Dias de dados disponíveis antes/depois de cada candidato.
JANELA_AJUSTE_DIAS_19A = 14

# Suporte mínimo em cada lado.
MIN_HORAS_LADO_19A = 72
MIN_HORAS_FAIXA_19A = 12
MIN_FAIXAS_19A = 2

# Mínimo de dias distintos com observações em cada lado.
MIN_DIAS_LADO_19A = 5

# Eventos previamente investigados.
EVENTOS_19A = {
    "2025-03": pd.Timestamp("2025-03-26 00:00"),
    "2025-07": pd.Timestamp("2025-07-14 00:00")
}

# ================================================================
# 2. PREPARAÇÃO DOS DADOS
# ================================================================

elegivel_original19a = (
    base19a["elegivel_intensidade"]
    .astype("string")
    .str.lower()
    .isin(["true", "1", "sim", "yes"])
)

base19a["elegivel_19a"] = (
    elegivel_original19a.fillna(False)
    & base19a["tempo_operacao_h"].gt(0)
    & base19a["carga_bruta_mw"].ge(CARGA_MIN_19A)
    & base19a["intensidade_t_mwh"].between(
        INT_MIN_19A, INT_MAX_19A
    )
)

base19a["faixa_19a"] = pd.cut(
    base19a["carga_bruta_mw"],
    bins=FAIXAS_19A,
    right=False
)

base19a["dia_19a"] = (
    base19a["timestamp"].dt.floor("D")
)

# ================================================================
# 3. COMPARAÇÃO AJUSTADA POR CARGA
# ================================================================

def comparar_corte_19a(corte, janela_dias):

    janela = pd.Timedelta(days=janela_dias)

    antes = base19a.loc[
        base19a["elegivel_19a"]
        & (base19a["timestamp"] >= corte - janela)
        & (base19a["timestamp"] < corte)
    ]

    depois = base19a.loc[
        base19a["elegivel_19a"]
        & (base19a["timestamp"] >= corte)
        & (base19a["timestamp"] < corte + janela)
    ]

    if (
        len(antes) < MIN_HORAS_LADO_19A
        or len(depois) < MIN_HORAS_LADO_19A
    ):
        return None

    if (
        antes["dia_19a"].nunique() < MIN_DIAS_LADO_19A
        or depois["dia_19a"].nunique() < MIN_DIAS_LADO_19A
    ):
        return None

    comparacoes = []

    for faixa in base19a["faixa_19a"].cat.categories:

        a = antes.loc[
            antes["faixa_19a"] == faixa,
            "intensidade_t_mwh"
        ]

        d = depois.loc[
            depois["faixa_19a"] == faixa,
            "intensidade_t_mwh"
        ]

        if (
            len(a) < MIN_HORAS_FAIXA_19A
            or len(d) < MIN_HORAS_FAIXA_19A
        ):
            continue

        comparacoes.append({
            "faixa": str(faixa),
            "n_antes": len(a),
            "n_depois": len(d),
            "media_antes": float(a.mean()),
            "media_depois": float(d.mean()),
            "sq_antes": float(
                ((a - a.mean()) ** 2).sum()
            ),
            "sq_depois": float(
                ((d - d.mean()) ** 2).sum()
            )
        })

    if len(comparacoes) < MIN_FAIXAS_19A:
        return None

    tab = pd.DataFrame(comparacoes)

    pesos = np.minimum(
        tab["n_antes"].to_numpy(),
        tab["n_depois"].to_numpy()
    ).astype(float)

    media_antes = np.average(
        tab["media_antes"],
        weights=pesos
    )

    media_depois = np.average(
        tab["media_depois"],
        weights=pesos
    )

    delta = media_depois - media_antes

    # Soma dos erros quadráticos dentro de cada
    # faixa e regime (antes/depois).
    sq_total = (
        tab["sq_antes"].sum()
        + tab["sq_depois"].sum()
    )

    return {
        "corte": corte,
        "sse": float(sq_total),
        "delta_t_mwh": float(delta),
        "delta_pct": (
            float(100 * delta / media_antes)
            if media_antes > 0 else np.nan
        ),
        "n_antes": int(len(antes)),
        "n_depois": int(len(depois)),
        "dias_antes": int(
            antes["dia_19a"].nunique()
        ),
        "dias_depois": int(
            depois["dia_19a"].nunique()
        ),
        "faixas_comparaveis": len(tab)
    }

# ================================================================
# 4. BUSCA HORÁRIA DO PONTO DE MUDANÇA
# ================================================================

resultados_busca19a = []
resultados_eventos19a = []

for nome, referencia in EVENTOS_19A.items():

    print("\n" + "-" * 76)
    print("EVENTO:", nome)
    print("Referência inicial:", referencia)

    inicio_busca = (
        referencia
        - pd.Timedelta(days=RAIO_BUSCA_DIAS_19A)
    )

    fim_busca = (
        referencia
        + pd.Timedelta(days=RAIO_BUSCA_DIAS_19A)
    )

    candidatos = pd.date_range(
        inicio_busca,
        fim_busca,
        freq="h"
    )

    registros = []

    for corte in candidatos:

        comparacao = comparar_corte_19a(
            corte,
            JANELA_AJUSTE_DIAS_19A
        )

        if comparacao is not None:
            comparacao["evento"] = nome
            registros.append(comparacao)

    if not registros:
        print("Nenhum candidato com suporte suficiente.")
        continue

    busca = pd.DataFrame(registros)

    # Como o objetivo é localizar reduções,
    # avaliamos apenas candidatos com delta negativo.
    reducoes = busca.loc[
        busca["delta_t_mwh"] < 0
    ].copy()

    if reducoes.empty:
        print("Nenhuma redução identificada.")
        continue

    melhor = reducoes.sort_values(
        ["sse", "corte"]
    ).iloc[0]

    corte_estimado = melhor["corte"]

    resultados_busca19a.append(busca)

    # ============================================================
    # 5. CRONOLOGIA DAS PARADAS
    # ============================================================

    anteriores = paradas19a.loc[
        (paradas19a["fim_exclusivo"] <= corte_estimado)
        & (
            paradas19a["fim_exclusivo"]
            >= corte_estimado - pd.Timedelta(days=30)
        )
    ].copy()

    concomitantes = paradas19a.loc[
        (paradas19a["inicio"] <= corte_estimado)
        & (paradas19a["fim_exclusivo"] > corte_estimado)
    ].copy()

    posteriores = paradas19a.loc[
        (paradas19a["inicio"] > corte_estimado)
        & (
            paradas19a["inicio"]
            <= corte_estimado + pd.Timedelta(days=30)
        )
    ].copy()

    anteriores24 = anteriores.loc[
        anteriores["duracao_horas"] >= 24
    ]

    anteriores72 = anteriores.loc[
        anteriores["duracao_horas"] >= 72
    ]

    if len(anteriores24):
        ultima24 = anteriores24.sort_values(
            "fim_exclusivo"
        ).iloc[-1]

        distancia24 = (
            corte_estimado - ultima24["fim_exclusivo"]
        ).total_seconds() / 3600
    else:
        distancia24 = np.nan

    if len(concomitantes):
        cronologia = "corte_durante_parada"
    elif len(anteriores72):
        cronologia = "parada_72h_anterior"
    elif len(anteriores24):
        cronologia = "parada_24h_anterior"
    else:
        cronologia = "sem_parada_24h_anterior"

    resultados_eventos19a.append({
        "evento": nome,
        "referencia_inicial": referencia,
        "corte_estimado": corte_estimado,
        "deslocamento_referencia_h": (
            corte_estimado - referencia
        ).total_seconds() / 3600,
        "delta_t_mwh": melhor["delta_t_mwh"],
        "delta_pct": melhor["delta_pct"],
        "sse": melhor["sse"],
        "n_antes": melhor["n_antes"],
        "n_depois": melhor["n_depois"],
        "dias_antes": melhor["dias_antes"],
        "dias_depois": melhor["dias_depois"],
        "faixas_comparaveis": melhor["faixas_comparaveis"],
        "paradas_anteriores_30d": len(anteriores),
        "paradas_anteriores_24h": len(anteriores24),
        "paradas_anteriores_72h": len(anteriores72),
        "paradas_concomitantes": len(concomitantes),
        "paradas_posteriores_30d": len(posteriores),
        "distancia_ultima_parada_24h_h": distancia24,
        "classificacao_cronologica": cronologia
    })

    print("Corte estimado:", corte_estimado)
    print(
        "Variação ajustada:",
        f"{melhor['delta_pct']:.2f}%"
    )
    print("Cronologia:", cronologia)

# ================================================================
# 6. RESULTADOS CONSOLIDADOS
# ================================================================

resultado19a = pd.DataFrame(
    resultados_eventos19a
)

if resultados_busca19a:
    busca19a = pd.concat(
        resultados_busca19a,
        ignore_index=True
    )
else:
    busca19a = pd.DataFrame()

print("\n" + "=" * 76)
print("RESULTADOS DO REFINAMENTO HORÁRIO")
print("=" * 76)

if resultado19a.empty:
    print("Nenhum evento refinado com suporte suficiente.")
else:
    display(resultado19a.round(5))

# ================================================================
# 7. GRÁFICOS DE DIAGNÓSTICO
# ================================================================

if not resultado19a.empty:

    fig, axes = plt.subplots(
        len(resultado19a),
        1,
        figsize=(15, 5 * len(resultado19a)),
        squeeze=False
    )

    for ax, (_, linha) in zip(
        axes.flatten(),
        resultado19a.iterrows()
    ):

        nome = linha["evento"]
        corte = linha["corte_estimado"]

        inicio = corte - pd.Timedelta(days=20)
        fim = corte + pd.Timedelta(days=20)

        amostra = base19a.loc[
            base19a["elegivel_19a"]
            & (base19a["timestamp"] >= inicio)
            & (base19a["timestamp"] < fim)
        ]

        diaria = (
            amostra.set_index("timestamp")[
                "intensidade_t_mwh"
            ]
            .resample("D")
            .median()
        )

        ax.plot(
            diaria.index,
            diaria.values,
            marker=".",
            linewidth=1.5,
            label="Mediana diária"
        )

        ax.axvline(
            linha["referencia_inicial"],
            color="gray",
            linestyle="--",
            label="Referência inicial"
        )

        ax.axvline(
            corte,
            color="red",
            linestyle="-",
            linewidth=1.5,
            label="Corte estimado"
        )

        proximas = paradas19a.loc[
            (paradas19a["inicio"] < fim)
            & (paradas19a["fim_exclusivo"] > inicio)
            & (paradas19a["duracao_horas"] >= 24)
        ]

        for _, parada in proximas.iterrows():
            ax.axvspan(
                parada["inicio"],
                parada["fim_exclusivo"],
                color="gray",
                alpha=0.20
            )

        ax.set_title(
            f"Iatan 2 — Refinamento horário {nome}"
        )
        ax.set_ylabel("tCO₂/MWh")
        ax.grid(alpha=0.25)
        ax.legend(loc="best")

    plt.tight_layout()

    arq_figura19a = (
        FIGURAS_19A
        / "19A_refinamento_horario_2025.png"
    )

    plt.savefig(
        arq_figura19a,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

# ================================================================
# 8. EXPORTAÇÃO
# ================================================================

arq_resultado19a = (
    TABELAS_19A
    / "19A_eventos_refinados.csv"
)

arq_busca19a = (
    TABELAS_19A
    / "19A_busca_horaria.csv"
)

resultado19a.to_csv(
    arq_resultado19a,
    index=False,
    encoding="utf-8-sig"
)

busca19a.to_csv(
    arq_busca19a,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 76)
print("CÉLULA 19A — CONCLUÍDA")
print("=" * 76)

print("Resultados:", arq_resultado19a)
print("Busca horária:", arq_busca19a)

if not resultado19a.empty:
    print("Figura:", arq_figura19a)

print(
    "\nATENÇÃO: datas estimadas retrospectivamente. "
    "Não demonstram manutenção ou causalidade."
)


# ===== ORIGEM: CÉLULA 21 DO NOTEBOOK EXPLORATÓRIO =====

# ================================================================
# CÉLULA 19B — VALIDAÇÃO ROBUSTA DOS PONTOS DE MUDANÇA
# IATAN 2 | EPA CEMS | 2025
# ================================================================
#
# OBJETIVOS
# 1. Detectar cortes em amostra temporal fixa por evento.
# 2. Ajustar a intensidade pelas faixas de carga.
# 3. Comparar modelo sem quebra e modelo com quebra.
# 4. Avaliar sensibilidade a diferentes janelas.
# 5. Auditar proximidade com paradas operacionais.
#
# LIMITAÇÕES
# - Análise retrospectiva e exploratória.
# - Não identifica a causa física da mudança.
# - Não confirma intervenções de manutenção.
# - Não produz inferência seletiva para a data escolhida.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("=" * 76)
print("CÉLULA 19B — VALIDAÇÃO ROBUSTA DOS PONTOS DE MUDANÇA")
print("=" * 76)

# ================================================================
# 1. CONFIGURAÇÃO
# ================================================================

ROOT_19B = OUTPUT_ROOT

DADOS_19B = ROOT_19B / "dados"
TABELAS_19B = ROOT_19B / "tabelas"
FIGURAS_19B = ROOT_19B / "figuras"

for pasta in [TABELAS_19B, FIGURAS_19B]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQ_BASE_19B = (
    DADOS_19B / "16_estados_operacionais_horarios.parquet"
)

ARQ_PARADAS_19B = (
    TABELAS_19B / "16_paradas_operacionais.csv"
)

ARQ_19A = (
    TABELAS_19B / "19A_eventos_refinados.csv"
)

for arquivo in [ARQ_BASE_19B, ARQ_PARADAS_19B, ARQ_19A]:
    if not arquivo.exists():
        raise FileNotFoundError(arquivo)

base19b = pd.read_parquet(ARQ_BASE_19B).copy()
paradas19b = pd.read_csv(ARQ_PARADAS_19B).copy()
refinados19b = pd.read_csv(ARQ_19A).copy()

base19b["timestamp"] = pd.to_datetime(
    base19b["timestamp"], errors="coerce"
)

for coluna in [
    "carga_bruta_mw",
    "intensidade_t_mwh",
    "tempo_operacao_h"
]:
    base19b[coluna] = pd.to_numeric(
        base19b[coluna], errors="coerce"
    )

for coluna in ["inicio", "fim_exclusivo"]:
    paradas19b[coluna] = pd.to_datetime(
        paradas19b[coluna], errors="coerce"
    )

paradas19b["duracao_horas"] = pd.to_numeric(
    paradas19b["duracao_horas"], errors="coerce"
)

refinados19b["referencia_inicial"] = pd.to_datetime(
    refinados19b["referencia_inicial"]
)

refinados19b["corte_estimado"] = pd.to_datetime(
    refinados19b["corte_estimado"]
)

base19b = (
    base19b.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

if base19b["timestamp"].duplicated().any():
    raise ValueError("Timestamps duplicados na base.")

# ================================================================
# 2. PARÂMETROS
# ================================================================

CARGA_MIN_19B = 400
INT_MIN_19B = 0.30
INT_MAX_19B = 1.80

FAIXAS_19B = [
    400, 500, 600, 700, 800, 900, np.inf
]

# A janela de cada experimento permanece fixa.
JANELAS_FIXAS_19B = [10, 14, 21, 30]

# Os cortes são procurados em torno da referência inicial.
RAIO_BUSCA_DIAS_19B = 7

MIN_HORAS_LADO_19B = 48
MIN_DIAS_LADO_19B = 3
MIN_FAIXAS_19B = 2
MIN_HORAS_FAIXA_19B = 12

# ================================================================
# 3. HORAS ELEGÍVEIS
# ================================================================

elegivel_original19b = (
    base19b["elegivel_intensidade"]
    .astype("string")
    .str.lower()
    .isin(["true", "1", "sim", "yes"])
)

base19b["elegivel_19b"] = (
    elegivel_original19b.fillna(False)
    & base19b["tempo_operacao_h"].gt(0)
    & base19b["carga_bruta_mw"].ge(CARGA_MIN_19B)
    & base19b["intensidade_t_mwh"].between(
        INT_MIN_19B, INT_MAX_19B
    )
)

base19b["faixa_19b"] = pd.cut(
    base19b["carga_bruta_mw"],
    bins=FAIXAS_19B,
    right=False,
    labels=False
)

# ================================================================
# 4. FUNÇÃO DE AJUSTE POR CARGA EM AMOSTRA FIXA
# ================================================================
#
# Primeiro removemos diferenças médias entre faixas de carga.
# Depois testamos uma mudança de nível no resíduo ajustado.
#
# A amostra de cada janela é definida uma única vez.
# Todos os candidatos são avaliados sobre as mesmas horas.
# ================================================================

def preparar_amostra_fixa_19b(referencia, janela_dias):

    inicio = referencia - pd.Timedelta(days=janela_dias)
    fim = referencia + pd.Timedelta(days=janela_dias)

    amostra = base19b.loc[
        base19b["elegivel_19b"]
        & (base19b["timestamp"] >= inicio)
        & (base19b["timestamp"] < fim)
    ].copy()

    amostra = amostra.dropna(
        subset=["faixa_19b", "intensidade_t_mwh"]
    )

    if amostra.empty:
        return None

    # Ajuste de nível médio por faixa de carga.
    media_faixa = (
        amostra.groupby("faixa_19b")[
            "intensidade_t_mwh"
        ].transform("mean")
    )

    amostra["residuo_carga"] = (
        amostra["intensidade_t_mwh"] - media_faixa
    )

    return amostra


def avaliar_cortes_fixos_19b(amostra, referencia):

    candidatos = pd.date_range(
        referencia - pd.Timedelta(
            days=RAIO_BUSCA_DIAS_19B
        ),
        referencia + pd.Timedelta(
            days=RAIO_BUSCA_DIAS_19B
        ),
        freq="h"
    )

    y = amostra["residuo_carga"].to_numpy(
        dtype=float
    )

    timestamps = amostra["timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    faixas = amostra["faixa_19b"].to_numpy()

    # SSE do modelo sem quebra.
    sse_sem_quebra = float(
        np.sum((y - y.mean()) ** 2)
    )

    resultados = []

    for corte in candidatos:

        mascara_antes = (
            timestamps < np.datetime64(corte)
        )

        mascara_depois = ~mascara_antes

        n_antes = int(mascara_antes.sum())
        n_depois = int(mascara_depois.sum())

        if (
            n_antes < MIN_HORAS_LADO_19B
            or n_depois < MIN_HORAS_LADO_19B
        ):
            continue

        antes = amostra.loc[mascara_antes]
        depois = amostra.loc[mascara_depois]

        if (
            antes["timestamp"].dt.floor("D").nunique()
            < MIN_DIAS_LADO_19B
            or depois["timestamp"].dt.floor("D").nunique()
            < MIN_DIAS_LADO_19B
        ):
            continue

        # Exigir faixas de carga presentes dos dois lados.
        faixas_comuns = 0

        for faixa in np.unique(faixas):

            na = np.sum(
                mascara_antes & (faixas == faixa)
            )

            nd = np.sum(
                mascara_depois & (faixas == faixa)
            )

            if (
                na >= MIN_HORAS_FAIXA_19B
                and nd >= MIN_HORAS_FAIXA_19B
            ):
                faixas_comuns += 1

        if faixas_comuns < MIN_FAIXAS_19B:
            continue

        media_antes = float(y[mascara_antes].mean())
        media_depois = float(y[mascara_depois].mean())

        sse_com_quebra = float(
            np.sum(
                (y[mascara_antes] - media_antes) ** 2
            )
            + np.sum(
                (y[mascara_depois] - media_depois) ** 2
            )
        )

        melhoria = (
            100 * (sse_sem_quebra - sse_com_quebra)
            / sse_sem_quebra
            if sse_sem_quebra > 0 else np.nan
        )

        resultados.append({
            "corte": corte,
            "n_antes": n_antes,
            "n_depois": n_depois,
            "faixas_comuns": faixas_comuns,
            "delta_residuo": (
                media_depois - media_antes
            ),
            "sse_sem_quebra": sse_sem_quebra,
            "sse_com_quebra": sse_com_quebra,
            "melhoria_sse_pct": melhoria
        })

    return pd.DataFrame(resultados)

# ================================================================
# 5. AVALIAÇÃO DOS EVENTOS E JANELAS
# ================================================================

resultados19b = []
buscas19b = []

for _, evento in refinados19b.iterrows():

    nome = evento["evento"]
    referencia = evento["referencia_inicial"]

    print("\n" + "-" * 76)
    print("EVENTO:", nome)

    for janela in JANELAS_FIXAS_19B:

        amostra = preparar_amostra_fixa_19b(
            referencia, janela
        )

        if amostra is None:
            continue

        busca = avaliar_cortes_fixos_19b(
            amostra, referencia
        )

        if busca.empty:
            print(
                f"Janela {janela} dias: "
                "sem candidatos válidos."
            )
            continue

        busca["evento"] = nome
        busca["janela_dias"] = janela

        buscas19b.append(busca)

        # A minimização do SSE não é condicionada ao sinal.
        melhor = busca.sort_values(
            ["sse_com_quebra", "corte"]
        ).iloc[0]

        corte = melhor["corte"]

        anteriores = paradas19b.loc[
            (paradas19b["fim_exclusivo"] <= corte)
            & (
                paradas19b["fim_exclusivo"]
                >= corte - pd.Timedelta(days=30)
            )
        ]

        anteriores24 = anteriores.loc[
            anteriores["duracao_horas"] >= 24
        ]

        anteriores72 = anteriores.loc[
            anteriores["duracao_horas"] >= 72
        ]

        concomitantes = paradas19b.loc[
            (paradas19b["inicio"] <= corte)
            & (paradas19b["fim_exclusivo"] > corte)
        ]

        if len(anteriores24):
            ultima24 = anteriores24.sort_values(
                "fim_exclusivo"
            ).iloc[-1]

            distancia24 = (
                corte - ultima24["fim_exclusivo"]
            ).total_seconds() / 3600
        else:
            distancia24 = np.nan

        resultados19b.append({
            "evento": nome,
            "janela_dias": janela,
            "referencia_inicial": referencia,
            "corte_estimado": corte,
            "deslocamento_h": (
                corte - referencia
            ).total_seconds() / 3600,
            "delta_residuo_t_mwh": (
                melhor["delta_residuo"]
            ),
            "melhoria_sse_pct": (
                melhor["melhoria_sse_pct"]
            ),
            "n_antes": melhor["n_antes"],
            "n_depois": melhor["n_depois"],
            "faixas_comuns": melhor["faixas_comuns"],
            "paradas_anteriores_24h": len(anteriores24),
            "paradas_anteriores_72h": len(anteriores72),
            "paradas_concomitantes": len(concomitantes),
            "distancia_ultima_parada_24h_h": distancia24
        })

        print(
            f"Janela ±{janela:2d} dias | "
            f"Corte: {corte} | "
            f"Delta ajustado: "
            f"{melhor['delta_residuo']:+.5f} t/MWh | "
            f"Melhoria SSE: "
            f"{melhor['melhoria_sse_pct']:.2f}%"
        )

# ================================================================
# 6. CONSOLIDAÇÃO
# ================================================================

resultado19b = pd.DataFrame(resultados19b)

if buscas19b:
    busca19b = pd.concat(
        buscas19b,
        ignore_index=True
    )
else:
    busca19b = pd.DataFrame()

print("\n" + "=" * 76)
print("RESULTADOS DA VALIDAÇÃO EM AMOSTRA FIXA")
print("=" * 76)

if resultado19b.empty:
    print("Nenhum resultado válido.")
else:
    display(resultado19b.round(5))

# ================================================================
# 7. ESTABILIDADE TEMPORAL
# ================================================================

estabilidade19b = []

if not resultado19b.empty:

    for nome, grupo in resultado19b.groupby("evento"):

        cortes = grupo["corte_estimado"].sort_values()

        amplitude_h = (
            cortes.max() - cortes.min()
        ).total_seconds() / 3600

        estabilidade19b.append({
            "evento": nome,
            "janelas_avaliadas": len(grupo),
            "primeiro_corte": cortes.min(),
            "ultimo_corte": cortes.max(),
            "amplitude_cortes_h": amplitude_h,
            "todas_reducoes": bool(
                (grupo["delta_residuo_t_mwh"] < 0).all()
            ),
            "melhoria_sse_media_pct": (
                grupo["melhoria_sse_pct"].mean()
            )
        })

estabilidade19b = pd.DataFrame(estabilidade19b)

print("\nESTABILIDADE ENTRE JANELAS")
display(estabilidade19b)

# ================================================================
# 8. VISUALIZAÇÃO
# ================================================================

if not resultado19b.empty:

    fig, axes = plt.subplots(
        len(refinados19b),
        1,
        figsize=(14, 5 * len(refinados19b)),
        squeeze=False
    )

    for ax, (_, evento) in zip(
        axes.flatten(),
        refinados19b.iterrows()
    ):

        nome = evento["evento"]

        dados = busca19b.loc[
            (busca19b["evento"] == nome)
        ]

        for janela, grupo in dados.groupby("janela_dias"):

            ax.plot(
                grupo["corte"],
                grupo["melhoria_sse_pct"],
                label=f"Janela ±{janela} dias",
                linewidth=1.4
            )

        ax.axvline(
            evento["corte_estimado"],
            color="black",
            linestyle="--",
            label="Corte da Célula 19A"
        )

        ax.set_title(
            f"Iatan 2 — Estabilidade da quebra {nome}"
        )

        ax.set_ylabel("Melhoria SSE (%)")
        ax.grid(alpha=0.25)
        ax.legend()

    plt.tight_layout()

    arq_figura19b = (
        FIGURAS_19B
        / "19B_estabilidade_pontos_mudanca.png"
    )

    plt.savefig(
        arq_figura19b,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

# ================================================================
# 9. EXPORTAÇÃO
# ================================================================

ARQ_RESULTADOS_19B = (
    TABELAS_19B
    / "19B_resultados_amostra_fixa.csv"
)

ARQ_BUSCA_19B = (
    TABELAS_19B
    / "19B_busca_cortes.csv"
)

ARQ_ESTABILIDADE_19B = (
    TABELAS_19B
    / "19B_estabilidade.csv"
)

resultado19b.to_csv(
    ARQ_RESULTADOS_19B,
    index=False,
    encoding="utf-8-sig"
)

busca19b.to_csv(
    ARQ_BUSCA_19B,
    index=False,
    encoding="utf-8-sig"
)

estabilidade19b.to_csv(
    ARQ_ESTABILIDADE_19B,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 76)
print("CÉLULA 19B — CONCLUÍDA")
print("=" * 76)

print("Resultados:", ARQ_RESULTADOS_19B)
print("Busca:", ARQ_BUSCA_19B)
print("Estabilidade:", ARQ_ESTABILIDADE_19B)

if not resultado19b.empty:
    print("Figura:", arq_figura19b)

print(
    "\nATENÇÃO: a melhoria de SSE é descritiva. "
    "Não corresponde a um p-valor nem demonstra causalidade."
)

# ================================================================
# 19C — Reconciliação metodológica dos instantes candidatos
# ================================================================
# A primeira busca e 19A/19B respondem a objetivos distintos.
# Os cortes usados na avaliação retrospectiva são os de 19A/19B,
# não escolhidos pelo menor RMSE de 2025.
from IPython.display import display

_nomes_eventos_19c = {
    "2025-03": "marco_2025",
    "2025-07": "julho_2025",
}
_registros_19c = []
cortes_reconciliados_2025 = {}
for _evento_19c, _nome_preliminar in _nomes_eventos_19c.items():
    _preliminar = eventos_horarios_df.loc[
        eventos_horarios_df["evento"].eq(_nome_preliminar), "instante_horario"
    ]
    _refinado = resultado19a.loc[
        resultado19a["evento"].eq(_evento_19c), "corte_estimado"
    ]
    _fixos = resultado19b.loc[
        resultado19b["evento"].eq(_evento_19c), ["janela_dias", "corte_estimado"]
    ]
    if len(_preliminar) != 1 or len(_refinado) != 1 or len(_fixos) != 4:
        raise AssertionError(f"Resultados incompletos para {_evento_19c} em 19C.")
    _pre = pd.Timestamp(_preliminar.iloc[0])
    _ref = pd.Timestamp(_refinado.iloc[0])
    _datas_fixas = pd.to_datetime(_fixos["corte_estimado"], errors="raise")
    if _datas_fixas.nunique() != 1 or _ref != _datas_fixas.iloc[0]:
        raise AssertionError(
            f"19A e 19B não concordam para {_evento_19c}; "
            "não utilizar automaticamente as datas anteriores."
        )
    if set(_fixos["janela_dias"].astype(int)) != {10, 14, 21, 30}:
        raise AssertionError(f"Janelas 19B inesperadas para {_evento_19c}.")
    cortes_reconciliados_2025[_evento_19c] = _ref
    _registros_19c.append({
        "evento": _evento_19c,
        "preliminar_maior_reducao_residuo": _pre,
        "refinamento_19A_SSE_carga": _ref,
        "validacao_19B_SSE_amostra_fixa": _datas_fixas.iloc[0],
        "janelas_19B_dias": "10;14;21;30",
        "diferenca_preliminar_versus_final_h": (_ref - _pre).total_seconds() / 3600,
        "instante_usado_22C_22D": _ref,
        "criterio": "concordancia_19A_19B_sem_selecao_por_RMSE_2025",
        "carater": "retrospectivo_nao_causal",
    })
reconciliacao_cortes_2025 = pd.DataFrame(_registros_19c)
reconciliacao_cortes_2025.to_csv(
    DIR_TABELAS / "19C_reconciliacao_datas_2025.csv",
    index=False, encoding="utf-8-sig"
)
print("19C — RECONCILIAÇÃO DOS MÉTODOS DE REFINAMENTO")
display(reconciliacao_cortes_2025)
print("Instantes para avaliação retrospectiva:", cortes_reconciliados_2025)
print("A estabilidade entre janelas não constitui intervalo de confiança da data.")

## 13 — Atributos operacionais e catálogo

Gera o catálogo de eventos e a base horária para os modelos. Rótulos retrospectivos permanecem separados das features de previsão.

In [ ]:
# ===== ORIGEM: CÉLULA 22 DO NOTEBOOK EXPLORATÓRIO =====

# ================================================================
# CÉLULA 20/20 — ATRIBUTOS OPERACIONAIS E CATÁLOGO DE EVENTOS
# IATAN 2 | EPA CEMS | 2011–2025
# ================================================================
#
# OBJETIVOS
# 1. Construir variáveis horárias históricas sem vazamento futuro.
# 2. Caracterizar paradas, partidas e retornos à operação.
# 3. Classificar eventos retrospectivos de intensidade de CO2.
# 4. Analisar classes por duração das paradas.
# 5. Exportar dados e documentação para o Notebook 07.
#
# CUIDADOS
# - Não presumir que parada significa manutenção.
# - Não atribuir causalidade a mudanças de intensidade.
# - Não usar rótulos retrospectivos como preditores.
# - As variáveis são conhecidas ao final da hora t.
# - Para previsão antecipada, aplicar defasagem apropriada.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("=" * 78)
print("CÉLULA 20/20 — ATRIBUTOS OPERACIONAIS E CATÁLOGO DE EVENTOS")
print("=" * 78)

# ================================================================
# 1. DIRETÓRIOS E ARQUIVOS
# ================================================================

PROJECT_ROOT_20 = PROJECT_ROOT

ROOT_20 = OUTPUT_ROOT

DADOS_20 = ROOT_20 / "dados"
TABELAS_20 = ROOT_20 / "tabelas"
FIGURAS_20 = ROOT_20 / "figuras"

for pasta in [DADOS_20, TABELAS_20, FIGURAS_20]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQUIVOS_20 = {
    "base": DADOS_20 / "16_estados_operacionais_horarios.parquet",
    "paradas": TABELAS_20 / "16_paradas_operacionais.csv",
    "classificacoes": TABELAS_20 / "17_paradas_classificadas.csv",
    "comparacoes": TABELAS_20 / "17_paradas_comparacoes.csv",
    "eventos_refinados": TABELAS_20 / "19A_eventos_refinados.csv",
    "validacao_fixa": TABELAS_20 / "19B_resultados_amostra_fixa.csv",
}

for nome, arquivo in ARQUIVOS_20.items():
    if not arquivo.exists():
        raise FileNotFoundError(
            f"Arquivo obrigatório ausente ({nome}): {arquivo}"
        )

base20 = pd.read_parquet(ARQUIVOS_20["base"]).copy()
paradas20 = pd.read_csv(ARQUIVOS_20["paradas"]).copy()
classes20 = pd.read_csv(ARQUIVOS_20["classificacoes"]).copy()
comparacoes20 = pd.read_csv(ARQUIVOS_20["comparacoes"]).copy()
refinados20 = pd.read_csv(ARQUIVOS_20["eventos_refinados"]).copy()
validacao20 = pd.read_csv(ARQUIVOS_20["validacao_fixa"]).copy()

print("Arquivos carregados:")
for nome, arquivo in ARQUIVOS_20.items():
    print(f"  {nome}: {arquivo.name}")

# ================================================================
# 2. PADRONIZAÇÃO E AUDITORIA
# ================================================================

base20["timestamp"] = pd.to_datetime(
    base20["timestamp"], errors="coerce"
)

base20 = (
    base20.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

if base20["timestamp"].duplicated().any():
    raise ValueError("Timestamps duplicados na base.")

if not base20["timestamp"].diff().dropna().eq(
    pd.Timedelta(hours=1)
).all():
    raise ValueError(
        "A base não possui grade horária contínua. "
        "Revisar a Célula 16."
    )

for coluna in [
    "tempo_operacao_h",
    "carga_bruta_mw",
    "geracao_mwh",
    "co2_t",
    "intensidade_t_mwh"
]:
    base20[coluna] = pd.to_numeric(
        base20[coluna], errors="coerce"
    )

for coluna in ["inicio", "fim_exclusivo"]:
    paradas20[coluna] = pd.to_datetime(
        paradas20[coluna], errors="coerce"
    )

paradas20["duracao_horas"] = pd.to_numeric(
    paradas20["duracao_horas"], errors="coerce"
)

refinados20["corte_estimado"] = pd.to_datetime(
    refinados20["corte_estimado"], errors="coerce"
)

validacao20["corte_estimado"] = pd.to_datetime(
    validacao20["corte_estimado"], errors="coerce"
)

# ================================================================
# 3. ESTADOS OPERACIONAIS
# ================================================================

# Reutilizar a classificação da Célula 16.
estados_validos20 = [
    "operando",
    "desligada",
    "operacao_sem_geracao"
]

base20["estado_20"] = (
    base20["estado_operacional"]
    .astype("string")
    .str.lower()
)

base20["estado_conhecido_20"] = (
    base20["estado_20"].isin(estados_validos20)
)

base20["desligada_20"] = (
    base20["estado_20"] == "desligada"
)

base20["operando_20"] = (
    base20["estado_20"] == "operando"
)

# Não tratar observações indeterminadas como desligadas.
base20["desligada_20"] = (
    base20["desligada_20"].fillna(False)
)

base20["operando_20"] = (
    base20["operando_20"].fillna(False)
)

# ================================================================
# 4. ATRIBUTOS HORÁRIOS DE PARADAS E PARTIDAS
# ================================================================

n = len(base20)

desligada = base20["desligada_20"].to_numpy(dtype=bool)
operando = base20["operando_20"].to_numpy(dtype=bool)
conhecido = base20["estado_conhecido_20"].to_numpy(dtype=bool)

inicio_parada = np.zeros(n, dtype=bool)
inicio_partida = np.zeros(n, dtype=bool)

horas_parada_atual = np.zeros(n, dtype=float)
horas_operando_atual = np.zeros(n, dtype=float)

duracao_ultima_parada = np.full(n, np.nan)
horas_desde_ultima_parada = np.full(n, np.nan)

ultima_duracao = np.nan
ultima_finalizacao = None

for i in range(n):

    if i == 0:
        anterior_desligada = False
        anterior_operando = False
        anterior_conhecido = False
    else:
        anterior_desligada = desligada[i - 1]
        anterior_operando = operando[i - 1]
        anterior_conhecido = conhecido[i - 1]

    # Início de parada.
    if desligada[i]:

        if i == 0 or (
            anterior_conhecido and not anterior_desligada
        ):
            inicio_parada[i] = True
            horas_parada_atual[i] = 1

        elif anterior_desligada:
            horas_parada_atual[i] = (
                horas_parada_atual[i - 1] + 1
            )

    # Retorno à operação após parada.
    if operando[i]:

        if i > 0 and anterior_desligada:

            inicio_partida[i] = True

            ultima_duracao = horas_parada_atual[i - 1]
            ultima_finalizacao = i

        if i > 0 and anterior_operando:
            horas_operando_atual[i] = (
                horas_operando_atual[i - 1] + 1
            )
        else:
            horas_operando_atual[i] = 1

    if conhecido[i]:

        duracao_ultima_parada[i] = ultima_duracao

        if ultima_finalizacao is not None:
            horas_desde_ultima_parada[i] = (
                i - ultima_finalizacao
            )

base20["inicio_parada_20"] = inicio_parada.astype(int)
base20["inicio_partida_20"] = inicio_partida.astype(int)

base20["horas_parada_atual_20"] = horas_parada_atual
base20["horas_operando_desde_partida_20"] = (
    horas_operando_atual
)

base20["duracao_ultima_parada_h_20"] = (
    duracao_ultima_parada
)

base20["horas_desde_ultima_parada_20"] = (
    horas_desde_ultima_parada
)

# ================================================================
# 5. ATRIBUTOS DE JANELAS HISTÓRICAS
# ================================================================

# Todas as janelas incluem apenas horas até t.
# A disponibilidade para previsão em t+h exige defasagem.

for dias in [7, 30, 90]:

    janela = dias * 24

    base20[f"horas_paradas_{dias}d_20"] = (
        base20["desligada_20"]
        .astype(int)
        .rolling(
            window=janela,
            min_periods=1
        )
        .sum()
    )

    base20[f"numero_partidas_{dias}d_20"] = (
        base20["inicio_partida_20"]
        .rolling(
            window=janela,
            min_periods=1
        )
        .sum()
    )

    base20[f"numero_paradas_{dias}d_20"] = (
        base20["inicio_parada_20"]
        .rolling(
            window=janela,
            min_periods=1
        )
        .sum()
    )

# ================================================================
# 6. INDICADORES DE PARADAS PROLONGADAS
# ================================================================

# Uma parada só é reconhecida como >= X horas
# depois que sua duração já é observável.
#
# Evitamos usar a duração final de uma parada
# enquanto ela ainda está em andamento.

for limiar in [4, 24, 72, 168]:

    base20[f"parada_atual_ge_{limiar}h_20"] = (
        base20["horas_parada_atual_20"] >= limiar
    ).astype(int)

    base20[f"ultima_parada_ge_{limiar}h_20"] = (
        base20["duracao_ultima_parada_h_20"] >= limiar
    ).astype(int)

# ================================================================
# 7. CATÁLOGO HISTÓRICO DE PARADAS
# ================================================================

catalogo20 = paradas20.copy()

# As classificações da Célula 17 são retrospectivas.
# São adequadas à investigação científica,
# mas não devem entrar diretamente no treinamento.

if "parada_id" not in catalogo20.columns:
    catalogo20["parada_id"] = np.arange(
        1, len(catalogo20) + 1
    )

if "parada_id" not in classes20.columns:
    raise KeyError(
        "A Célula 17 não possui coluna parada_id."
    )

if "classificacao" not in classes20.columns:
    raise KeyError(
        "A Célula 17 não possui coluna classificacao."
    )

catalogo20 = catalogo20.merge(
    classes20[
        ["parada_id", "classificacao"]
    ],
    on="parada_id",
    how="left",
    validate="one_to_one"
)

catalogo20["faixa_duracao"] = pd.cut(
    catalogo20["duracao_horas"],
    bins=[0, 4, 24, 72, 168, np.inf],
    labels=[
        "ate_4h",
        "4_a_24h",
        "24_a_72h",
        "72_a_168h",
        "acima_168h"
    ],
    right=False
)

catalogo20["tipo_evento_ic"] = np.select(
    [
        catalogo20["classificacao"].eq(
            "reducao_persistente"
        ),
        catalogo20["classificacao"].eq(
            "aumento_persistente"
        ),
        catalogo20["classificacao"].eq(
            "sem_mudanca_persistente_confirmada"
        )
    ],
    [
        "parada_seguida_reducao",
        "parada_seguida_aumento",
        "sem_mudanca_persistente_confirmada"
    ],
    default="inconclusivo_ou_nao_avaliado"
)

catalogo20["uso_preditivo_autorizado"] = False

# ================================================================
# 8. EVENTOS DE MUDANÇA DE 2025
# ================================================================

eventos_ic20 = refinados20.copy()

# Recuperar estabilidade da Célula 19B.
estabilidade20 = (
    validacao20.groupby("evento")
    .agg(
        numero_janelas=("janela_dias", "nunique"),
        primeiro_corte=("corte_estimado", "min"),
        ultimo_corte=("corte_estimado", "max"),
        melhoria_sse_media_pct=(
            "melhoria_sse_pct", "mean"
        )
    )
    .reset_index()
)

estabilidade20["amplitude_cortes_h"] = (
    (
        estabilidade20["ultimo_corte"]
        - estabilidade20["primeiro_corte"]
    )
    .dt.total_seconds() / 3600
)

eventos_ic20 = eventos_ic20.merge(
    estabilidade20,
    on="evento",
    how="left",
    validate="one_to_one"
)

eventos_ic20["tipo_evento_ic"] = np.where(
    eventos_ic20["paradas_anteriores_24h"] > 0,
    "reducao_apos_parada_prolongada",
    "reducao_sem_parada_prolongada_anterior"
)

eventos_ic20["causalidade_confirmada"] = False
eventos_ic20["manutencao_confirmada"] = False
eventos_ic20["uso_preditivo_autorizado"] = False

# ================================================================
# 9. PROPORÇÕES POR DURAÇÃO DAS PARADAS
# ================================================================

# Somente paradas >=24h foram avaliadas na Célula 17.
# As demais permanecem como não avaliadas.

avaliadas20 = catalogo20.loc[
    catalogo20["duracao_horas"] >= 24
].copy()

proporcoes20 = (
    avaliadas20.groupby(
        ["faixa_duracao", "tipo_evento_ic"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
)

proporcoes20["total"] = proporcoes20.sum(axis=1)

colunas_classes20 = [
    c for c in proporcoes20.columns
    if c != "total"
]

for coluna in colunas_classes20:
    proporcoes20[f"pct_{coluna}"] = (
        100 * proporcoes20[coluna]
        / proporcoes20["total"]
    )

proporcoes20 = proporcoes20.reset_index()

# ================================================================
# 10. AUDITORIA DE VAZAMENTO TEMPORAL
# ================================================================

# O dataset preditivo não deve conter:
# - cortes identificados retrospectivamente;
# - classificações de redução ou aumento;
# - magnitudes futuras de mudança;
# - resultados de bootstrap retrospectivo.

colunas_excluir20 = [
    "estado_20",
    "estado_conhecido_20",
    "desligada_20",
    "operando_20",
    "inicio_parada_20",
    "inicio_partida_20",
    "horas_parada_atual_20",
    "horas_operando_desde_partida_20",
    "duracao_ultima_parada_h_20",
    "horas_desde_ultima_parada_20"
]

# As colunas acima serão preservadas na base enriquecida;
# não são rótulos retrospectivos de intensidade.

colunas_atributos20 = [
    c for c in base20.columns
    if c.endswith("_20")
]

base_preditiva20 = base20[
    [
        "timestamp",
        "facility_id",
        "unit_id",
        "tempo_operacao_h",
        "carga_bruta_mw",
        "geracao_mwh",
        "co2_t",
        "intensidade_t_mwh"
    ]
    + colunas_atributos20
].copy()

# Não incluir classificação de mudanças de intensidade.
proibidas20 = [
    c for c in base_preditiva20.columns
    if (
        "tipo_evento_ic" in c
        or "corte_estimado" in c
        or "causalidade" in c
        or "melhoria_sse" in c
    )
]

if proibidas20:
    raise ValueError(
        f"Colunas retrospectivas encontradas: {proibidas20}"
    )

# ================================================================
# 11. GRÁFICO DE PROPORÇÕES
# ================================================================

grafico20 = (
    avaliadas20.groupby(
        ["faixa_duracao", "tipo_evento_ic"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
)

grafico_pct20 = (
    grafico20.div(
        grafico20.sum(axis=1),
        axis=0
    ) * 100
)

ax = grafico_pct20.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

ax.set_title(
    "Iatan 2 — Classificação exploratória após paradas"
)
ax.set_xlabel("Duração da parada")
ax.set_ylabel("Participação (%)")
ax.legend(
    title="Classificação retrospectiva",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

ARQ_FIGURA_20 = (
    FIGURAS_20
    / "20_proporcoes_por_duracao_parada.png"
)

plt.savefig(
    ARQ_FIGURA_20,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ================================================================
# 12. EXPORTAÇÃO
# ================================================================

ARQ_BASE_20 = (
    DADOS_20
    / "20_base_horaria_atributos_operacionais.parquet"
)

ARQ_CATALOGO_20 = (
    TABELAS_20
    / "20_catalogo_historico_paradas.csv"
)

ARQ_EVENTOS_20 = (
    TABELAS_20
    / "20_catalogo_mudancas_ic_2025.csv"
)

ARQ_PROPORCOES_20 = (
    TABELAS_20
    / "20_proporcoes_por_duracao.csv"
)

base_preditiva20.to_parquet(
    ARQ_BASE_20,
    index=False
)

catalogo20.to_csv(
    ARQ_CATALOGO_20,
    index=False,
    encoding="utf-8-sig"
)

eventos_ic20.to_csv(
    ARQ_EVENTOS_20,
    index=False,
    encoding="utf-8-sig"
)

proporcoes20.to_csv(
    ARQ_PROPORCOES_20,
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 13. RELATÓRIO FINAL
# ================================================================

print("\n" + "=" * 78)
print("BASE HORÁRIA ENRIQUECIDA")
print("=" * 78)

print("Registros:", len(base_preditiva20))
print("Colunas:", len(base_preditiva20.columns))

display(base_preditiva20.head(10))

print("\n" + "=" * 78)
print("CATÁLOGO DE EVENTOS DE INTENSIDADE")
print("=" * 78)

display(
    eventos_ic20[
        [
            "evento",
            "corte_estimado",
            "delta_pct",
            "paradas_anteriores_24h",
            "tipo_evento_ic",
            "numero_janelas",
            "amplitude_cortes_h"
        ]
    ]
)

print("\n" + "=" * 78)
print("PROPORÇÕES POR DURAÇÃO")
print("=" * 78)

display(proporcoes20.round(2))

print("\nARQUIVOS GERADOS")
for arquivo in [
    ARQ_BASE_20,
    ARQ_CATALOGO_20,
    ARQ_EVENTOS_20,
    ARQ_PROPORCOES_20,
    ARQ_FIGURA_20
]:
    print(arquivo)

print("\n" + "=" * 78)
print("CÉLULA 20/20 — CONCLUÍDA")
print("=" * 78)

print(
    "\nA base horária contém atributos operacionais históricos. "
    "As classificações retrospectivas permanecem em arquivos "
    "separados e não devem ser utilizadas como preditores."
)

## 14 — Comparação complementar e atualização mensal

Executa o estudo complementar 21 (com e sem histórico de CO₂) e o experimento 22A. A comparação principal utiliza somente geração; modelos com lags de CO₂ têm condição informacional diferente.

**Limitação 22A:** a atualização mensal usa janela móvel de 24 meses, enquanto o M0 estático utiliza histórico muito mais longo. O resultado não isola apenas o efeito de atualizar o modelo. O estudo E2 (lags de CO₂) usa emissões passadas; sua comparação com M2 exige explicitar esta diferença de informação.

---

**Base computacional — Chen e Guestrin (2016, §§2.1–2.2, p. 786); Cerqueira, Torgo e Mozetič (2020).** XGBoost fundamenta o estimador por árvores impulsionadas; a avaliação temporal requer distinguir dados disponíveis no instante da previsão de variáveis retrospectivas. Como o experimento com *lags* de CO₂ exige valores passados do alvo e o protocolo de atualização mensal altera também o histórico de treinamento, essas comparações têm condições informacionais e contrafactuais distintas.

In [ ]:
# ===== ORIGEM: CÉLULA 23 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 21 — EXPERIMENTO COMPLEMENTAR
# CONTRIBUIÇÃO DOS ATRIBUTOS OPERACIONAIS
# IATAN 2 | 2011–2025
# ============================================================
#
# E0: Geração
# E1: Geração + Operacional
# E2: Geração + Histórico
# E3: Geração + Histórico + Operacional
#
# ALVO: emissões horárias de CO2 (toneladas métricas)
#
# Tarefa: estimação contemporânea de emissões,
# condicionada à geração observada na própria hora.
#
# Os lags de CO2 usam somente horas anteriores.
# Não utiliza rótulos retrospectivos de mudanças de IC.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from xgboost import XGBRegressor
from IPython.display import display

print("=" * 75)
print("EXPERIMENTO 21 — CONTRIBUIÇÃO OPERACIONAL")
print("=" * 75)

# ------------------------------------------------------------
# 1. DIRETÓRIOS
# ------------------------------------------------------------

ROOT21 = OUTPUT_ROOT

DADOS21 = ROOT21 / "dados"
TABELAS21 = ROOT21 / "tabelas"
FIGURAS21 = ROOT21 / "figuras"

for pasta in [TABELAS21, FIGURAS21]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQUIVO21 = (
    DADOS21
    / "20_base_horaria_atributos_operacionais.parquet"
)

if not ARQUIVO21.exists():
    raise FileNotFoundError(
        "Execute primeiro a Célula 20. "
        f"Arquivo não encontrado: {ARQUIVO21}"
    )

df21 = pd.read_parquet(ARQUIVO21).copy()

df21["timestamp"] = pd.to_datetime(
    df21["timestamp"], errors="coerce"
)

df21 = (
    df21.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

if df21["timestamp"].duplicated().any():
    raise ValueError("Existem timestamps duplicados.")

if not df21["timestamp"].diff().dropna().eq(
    pd.Timedelta(hours=1)
).all():
    raise ValueError(
        "A grade temporal não é contínua."
    )

print("Registros carregados:", len(df21))

# ------------------------------------------------------------
# 2. DEFINIÇÃO DO ALVO E DAS VARIÁVEIS
# ------------------------------------------------------------

ALVO21 = "co2_t"

GERACAO21 = ["geracao_mwh"]

OPERACIONAL21 = [
    "duracao_ultima_parada_h_20",
    "horas_desde_ultima_parada_20",
    "horas_paradas_30d_20",
    "horas_operando_desde_partida_20"
]

COLUNAS_NECESSARIAS21 = (
    ["timestamp", ALVO21]
    + GERACAO21
    + OPERACIONAL21
)

faltantes21 = [
    c for c in COLUNAS_NECESSARIAS21
    if c not in df21.columns
]

if faltantes21:
    raise KeyError(
        f"Colunas não encontradas: {faltantes21}"
    )

for coluna in COLUNAS_NECESSARIAS21:
    if coluna != "timestamp":
        df21[coluna] = pd.to_numeric(
            df21[coluna], errors="coerce"
        )

# ------------------------------------------------------------
# 3. ATRIBUTOS HISTÓRICOS
# ------------------------------------------------------------
#
# As defasagens são calculadas na grade horária completa.
# Nunca deslocamos apenas as horas elegíveis, pois isso
# alteraria o significado de lag 1, lag 24 etc.
#
# A variável co2_t da hora atual não entra como preditor.
# ------------------------------------------------------------

df21["co2_lag_1h"] = df21[ALVO21].shift(1)
df21["co2_lag_24h"] = df21[ALVO21].shift(24)

df21["geracao_lag_1h"] = (
    df21["geracao_mwh"].shift(1)
)

df21["geracao_lag_24h"] = (
    df21["geracao_mwh"].shift(24)
)

df21["co2_media_24h"] = (
    df21[ALVO21]
    .shift(1)
    .rolling(24, min_periods=12)
    .mean()
)

df21["geracao_media_24h"] = (
    df21["geracao_mwh"]
    .shift(1)
    .rolling(24, min_periods=12)
    .mean()
)

HISTORICO21 = [
    "co2_lag_1h",
    "co2_lag_24h",
    "geracao_lag_1h",
    "geracao_lag_24h",
    "co2_media_24h",
    "geracao_media_24h"
]

# ------------------------------------------------------------
# 4. CONFIGURAÇÕES EXPERIMENTAIS
# ------------------------------------------------------------

CONFIGURACOES21 = {
    "E0_Geracao": GERACAO21,
    "E1_Geracao_Operacional": (
        GERACAO21 + OPERACIONAL21
    ),
    "E2_Geracao_Historico": (
        GERACAO21 + HISTORICO21
    ),
    "E3_Geracao_Historico_Operacional": (
        GERACAO21 + HISTORICO21 + OPERACIONAL21
    )
}

# ------------------------------------------------------------
# 5. AMOSTRA COMUM
# ------------------------------------------------------------
#
# Utilizar exatamente as mesmas linhas em todos os modelos.
#
# Excluímos horas sem geração positiva e sem alvo válido.
# Valores ausentes em atributos históricos/operacionais
# permanecem como NaN, tratados pelo XGBoost.
#
# A ausência de histórico antes da primeira parada
# não deve ser preenchida artificialmente com zero.
# ------------------------------------------------------------

df21 = df21.replace(
    [np.inf, -np.inf], np.nan
)

df21["elegivel_modelagem21"] = (
    df21["geracao_mwh"].gt(0)
    & df21[ALVO21].notna()
    & df21[ALVO21].ge(0)
)

dados21 = df21.loc[
    df21["elegivel_modelagem21"]
].copy()

# ------------------------------------------------------------
# 6. DIVISÃO CRONOLÓGICA
# ------------------------------------------------------------

treino21 = dados21.loc[
    dados21["timestamp"] < "2024-01-01"
].copy()

validacao21 = dados21.loc[
    (dados21["timestamp"] >= "2024-01-01")
    & (dados21["timestamp"] < "2025-01-01")
].copy()

teste21 = dados21.loc[
    (dados21["timestamp"] >= "2025-01-01")
    & (dados21["timestamp"] < "2026-01-01")
].copy()

for nome, conjunto in [
    ("Treino", treino21),
    ("Validação", validacao21),
    ("Teste", teste21)
]:
    if conjunto.empty:
        raise ValueError(
            f"Conjunto {nome} está vazio."
        )

    print(
        f"{nome}: {len(conjunto):,} horas | "
        f"{conjunto['timestamp'].min()} até "
        f"{conjunto['timestamp'].max()}"
    )

# ------------------------------------------------------------
# 7. MODELO E MÉTRICAS
# ------------------------------------------------------------

PARAMETROS21 = {
    "n_estimators": 350,
    "max_depth": 4,
    "learning_rate": 0.04,
    "subsample": 0.85,
    "colsample_bytree": 0.9,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist"
}

def metricas21(y, pred):

    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y, pred)
        ),
        "R2": r2_score(y, pred)
    }

modelos21 = {}
resultados21 = []
previsoes21 = []

# ------------------------------------------------------------
# 8. TREINAMENTO DOS QUATRO MODELOS
# ------------------------------------------------------------

for nome, atributos in CONFIGURACOES21.items():

    print("\n" + "-" * 75)
    print("MODELO:", nome)
    print("Atributos:", atributos)

    modelo = XGBRegressor(**PARAMETROS21)

    modelo.fit(
        treino21[atributos],
        treino21[ALVO21]
    )

    modelos21[nome] = modelo

    for periodo, conjunto in [
        ("validacao_2024", validacao21),
        ("teste_2025", teste21)
    ]:

        y_real = conjunto[ALVO21].to_numpy()

        y_pred = modelo.predict(
            conjunto[atributos]
        )

        # Emissões físicas não negativas.
        y_pred = np.maximum(y_pred, 0)

        m = metricas21(y_real, y_pred)

        resultados21.append({
            "modelo": nome,
            "periodo": periodo,
            "n_atributos": len(atributos),
            "n_horas": len(conjunto),
            **m
        })

        previsoes21.append(
            pd.DataFrame({
                "timestamp": conjunto["timestamp"].to_numpy(),
                "periodo": periodo,
                "modelo": nome,
                "co2_real_t": y_real,
                "co2_previsto_t": y_pred,
                "erro_t": y_pred - y_real
            })
        )

        print(
            f"{periodo}: "
            f"MAE={m['MAE']:.4f} | "
            f"RMSE={m['RMSE']:.4f} | "
            f"R2={m['R2']:.4f}"
        )

resultados21 = pd.DataFrame(resultados21)

previsoes21 = pd.concat(
    previsoes21,
    ignore_index=True
)

# ------------------------------------------------------------
# 9. COMPARAÇÃO NO TESTE DE 2025
# ------------------------------------------------------------

teste_resultados21 = (
    resultados21.loc[
        resultados21["periodo"] == "teste_2025"
    ]
    .set_index("modelo")
)

rmse_e0 = teste_resultados21.loc[
    "E0_Geracao", "RMSE"
]

rmse_e1 = teste_resultados21.loc[
    "E1_Geracao_Operacional", "RMSE"
]

rmse_e2 = teste_resultados21.loc[
    "E2_Geracao_Historico", "RMSE"
]

rmse_e3 = teste_resultados21.loc[
    "E3_Geracao_Historico_Operacional", "RMSE"
]

ganho_operacional_sem_historico21 = (
    100 * (rmse_e0 - rmse_e1) / rmse_e0
)

ganho_operacional_com_historico21 = (
    100 * (rmse_e2 - rmse_e3) / rmse_e2
)

print("\n" + "=" * 75)
print("RESULTADOS DO TESTE — 2025")
print("=" * 75)

display(
    teste_resultados21[
        ["n_atributos", "n_horas", "MAE", "RMSE", "R2"]
    ].sort_values("RMSE").round(5)
)

print("\nCONTRIBUIÇÃO DOS ATRIBUTOS OPERACIONAIS")

print(
    "E1 vs E0 — redução do RMSE: "
    f"{ganho_operacional_sem_historico21:+.2f}%"
)

print(
    "E3 vs E2 — redução do RMSE: "
    f"{ganho_operacional_com_historico21:+.2f}%"
)

# ------------------------------------------------------------
# 10. ANÁLISE DOS EVENTOS DE 2025
# ------------------------------------------------------------
#
# Avaliar os erros em janelas próximas às mudanças
# identificadas no Notebook 07.
#
# Essas datas são usadas somente para diagnóstico
# retrospectivo, nunca como atributos do modelo.
# ------------------------------------------------------------

EVENTOS21 = {
    "marco_2025": pd.Timestamp("2025-03-26 08:00"),
    "julho_2025": pd.Timestamp("2025-07-14 10:00")
}

metricas_eventos21 = []

for evento, corte in EVENTOS21.items():

    inicio = corte - pd.Timedelta(days=14)
    fim = corte + pd.Timedelta(days=14)

    for nome in CONFIGURACOES21:

        amostra = previsoes21.loc[
            (previsoes21["periodo"] == "teste_2025")
            & (previsoes21["modelo"] == nome)
            & (previsoes21["timestamp"] >= inicio)
            & (previsoes21["timestamp"] < fim)
        ]

        if len(amostra) < 24:
            continue

        m = metricas21(
            amostra["co2_real_t"],
            amostra["co2_previsto_t"]
        )

        metricas_eventos21.append({
            "evento": evento,
            "modelo": nome,
            "n_horas": len(amostra),
            **m
        })

metricas_eventos21 = pd.DataFrame(
    metricas_eventos21
)

print("\n" + "=" * 75)
print("DESEMPENHO PRÓXIMO ÀS MUDANÇAS DE 2025")
print("=" * 75)

display(
    metricas_eventos21.sort_values(
        ["evento", "RMSE"]
    ).round(5)
)

# ------------------------------------------------------------
# 11. IMPORTÂNCIA DOS ATRIBUTOS OPERACIONAIS
# ------------------------------------------------------------

importancias21 = []

for nome in [
    "E1_Geracao_Operacional",
    "E3_Geracao_Historico_Operacional"
]:

    modelo = modelos21[nome]
    atributos = CONFIGURACOES21[nome]

    for atributo, importancia in zip(
        atributos,
        modelo.feature_importances_
    ):

        importancias21.append({
            "modelo": nome,
            "atributo": atributo,
            "importancia_xgboost": float(importancia)
        })

importancias21 = pd.DataFrame(importancias21)

print("\nIMPORTÂNCIA DOS ATRIBUTOS")
display(
    importancias21.sort_values(
        ["modelo", "importancia_xgboost"],
        ascending=[True, False]
    ).round(5)
)

# ------------------------------------------------------------
# 12. GRÁFICOS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

ordem21 = list(CONFIGURACOES21.keys())

valores_rmse21 = [
    teste_resultados21.loc[nome, "RMSE"]
    for nome in ordem21
]

valores_mae21 = [
    teste_resultados21.loc[nome, "MAE"]
    for nome in ordem21
]

rotulos21 = ["E0", "E1", "E2", "E3"]

axes[0].bar(rotulos21, valores_rmse21)
axes[0].set_title("RMSE — Teste 2025")
axes[0].set_ylabel("tCO2")
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(rotulos21, valores_mae21)
axes[1].set_title("MAE — Teste 2025")
axes[1].set_ylabel("tCO2")
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()

ARQ_FIG21 = (
    FIGURAS21
    / "21_comparacao_atributos_operacionais.png"
)

plt.savefig(
    ARQ_FIG21,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 13. EXPORTAÇÃO
# ------------------------------------------------------------

ARQ_RESULT21 = (
    TABELAS21 / "21_resultados_modelos.csv"
)

ARQ_PREV21 = (
    DADOS21 / "21_previsoes_modelos.parquet"
)

ARQ_EVENTOS21 = (
    TABELAS21 / "21_metricas_eventos_2025.csv"
)

ARQ_IMPORT21 = (
    TABELAS21 / "21_importancia_atributos.csv"
)

resultados21.to_csv(
    ARQ_RESULT21,
    index=False,
    encoding="utf-8-sig"
)

previsoes21.to_parquet(
    ARQ_PREV21,
    index=False
)

metricas_eventos21.to_csv(
    ARQ_EVENTOS21,
    index=False,
    encoding="utf-8-sig"
)

importancias21.to_csv(
    ARQ_IMPORT21,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("EXPERIMENTO 21 — CONCLUÍDO")
print("=" * 75)

print("Resultados:", ARQ_RESULT21)
print("Previsões:", ARQ_PREV21)
print("Eventos:", ARQ_EVENTOS21)
print("Importâncias:", ARQ_IMPORT21)
print("Figura:", ARQ_FIG21)

print(
    "\nInterpretação: diferenças de desempenho "
    "indicam contribuição preditiva dos atributos, "
    "não causalidade operacional."
)


# ===== ORIGEM: CÉLULA 24 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 22A — ADAPTAÇÃO TEMPORAL DO MODELO DE EMISSÕES
# IATAN 2 | EPA CEMS | 2011–2025
# ============================================================
#
# QUESTÃO:
# A atualização mensal melhora a estimação de CO2 em 2025?
#
# MODELO A — ESTÁTICO
# Treinamento: 2011–2023
# Sem atualizações durante 2025.
#
# MODELO B — ADAPTATIVO
# Treinamento em janela móvel de 24 meses.
# Reajuste no início de cada mês de 2025.
#
# ENTRADA EXCLUSIVA:
#   geracao_mwh
#
# ALVO:
#   co2_t
#
# Não utiliza emissões como atributos.
# Não utiliza datas de mudança para treinamento.
# Não utiliza informações futuras.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from xgboost import XGBRegressor
from IPython.display import display

print("=" * 75)
print("EXPERIMENTO 22A — MODELO ESTÁTICO VS ADAPTATIVO")
print("=" * 75)

# ------------------------------------------------------------
# 1. DIRETÓRIOS
# ------------------------------------------------------------

ROOT22 = OUTPUT_ROOT

DADOS22 = ROOT22 / "dados"
TABELAS22 = ROOT22 / "tabelas"
FIGURAS22 = ROOT22 / "figuras"

for pasta in [TABELAS22, FIGURAS22]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQUIVO22 = (
    DADOS22
    / "20_base_horaria_atributos_operacionais.parquet"
)

if not ARQUIVO22.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO22}"
    )

df22 = pd.read_parquet(ARQUIVO22).copy()

df22["timestamp"] = pd.to_datetime(
    df22["timestamp"], errors="coerce"
)

df22 = (
    df22.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. SELEÇÃO EXCLUSIVA DAS VARIÁVEIS
# ------------------------------------------------------------

df22 = df22[
    ["timestamp", "geracao_mwh", "co2_t"]
].copy()

df22["geracao_mwh"] = pd.to_numeric(
    df22["geracao_mwh"], errors="coerce"
)

df22["co2_t"] = pd.to_numeric(
    df22["co2_t"], errors="coerce"
)

df22 = df22.replace(
    [np.inf, -np.inf], np.nan
)

df22 = df22.loc[
    df22["geracao_mwh"].gt(0)
    & df22["co2_t"].notna()
    & df22["co2_t"].ge(0)
].copy()

print("Horas elegíveis:", len(df22))

# ------------------------------------------------------------
# 3. PARÂMETROS FIXOS DO XGBOOST
# ------------------------------------------------------------

PARAMETROS22 = {
    "n_estimators": 350,
    "max_depth": 4,
    "learning_rate": 0.04,
    "subsample": 0.85,
    "colsample_bytree": 0.9,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist"
}

def novo_modelo22():
    return XGBRegressor(**PARAMETROS22)

def calcular_metricas22(y, pred):
    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "R2": r2_score(y, pred),
        "Vies_medio": float(np.mean(pred - y))
    }

# ------------------------------------------------------------
# 4. MODELO A — ESTÁTICO
# ------------------------------------------------------------

treino_estatico22 = df22.loc[
    df22["timestamp"] < "2024-01-01"
].copy()

teste22 = df22.loc[
    (df22["timestamp"] >= "2025-01-01")
    & (df22["timestamp"] < "2026-01-01")
].copy()

if treino_estatico22.empty or teste22.empty:
    raise ValueError(
        "Treinamento ou teste sem observações."
    )

modelo_estatico22 = novo_modelo22()

modelo_estatico22.fit(
    treino_estatico22[["geracao_mwh"]],
    treino_estatico22["co2_t"]
)

teste22["pred_estatico"] = np.maximum(
    modelo_estatico22.predict(
        teste22[["geracao_mwh"]]
    ),
    0
)

print(
    "Modelo estático treinado com",
    len(treino_estatico22),
    "observações."
)

# ------------------------------------------------------------
# 5. MODELO B — ATUALIZAÇÃO MENSAL
# ------------------------------------------------------------
#
# No início de cada mês:
# - selecionar os 24 meses anteriores;
# - treinar o modelo;
# - estimar as horas do mês corrente.
#
# O modelo não conhece emissões do mês que estima.
# ------------------------------------------------------------

JANELA_MESES22 = 24

meses22 = pd.date_range(
    start="2025-01-01",
    end="2025-12-01",
    freq="MS"
)

previsoes_adaptativas22 = []
registro_atualizacoes22 = []

for inicio_mes in meses22:

    fim_mes = inicio_mes + pd.DateOffset(months=1)

    inicio_treino = (
        inicio_mes
        - pd.DateOffset(months=JANELA_MESES22)
    )

    treino_mes = df22.loc[
        (df22["timestamp"] >= inicio_treino)
        & (df22["timestamp"] < inicio_mes)
    ].copy()

    teste_mes = df22.loc[
        (df22["timestamp"] >= inicio_mes)
        & (df22["timestamp"] < fim_mes)
    ].copy()

    if teste_mes.empty:
        continue

    if len(treino_mes) < 500:
        raise ValueError(
            f"Treino insuficiente em {inicio_mes}: "
            f"{len(treino_mes)} observações."
        )

    modelo_mes = novo_modelo22()

    modelo_mes.fit(
        treino_mes[["geracao_mwh"]],
        treino_mes["co2_t"]
    )

    teste_mes["pred_adaptativo"] = np.maximum(
        modelo_mes.predict(
            teste_mes[["geracao_mwh"]]
        ),
        0
    )

    previsoes_adaptativas22.append(
        teste_mes[
            ["timestamp", "pred_adaptativo"]
        ]
    )

    registro_atualizacoes22.append({
        "mes": inicio_mes,
        "inicio_treino": inicio_treino,
        "fim_treino_exclusivo": inicio_mes,
        "n_treino": len(treino_mes),
        "n_teste": len(teste_mes)
    })

    print(
        f"{inicio_mes:%Y-%m}: "
        f"treino={len(treino_mes):,} | "
        f"teste={len(teste_mes):,}"
    )

previsoes_adaptativas22 = pd.concat(
    previsoes_adaptativas22,
    ignore_index=True
)

atualizacoes22 = pd.DataFrame(
    registro_atualizacoes22
)

teste22 = teste22.merge(
    previsoes_adaptativas22,
    on="timestamp",
    how="left",
    validate="one_to_one"
)

if teste22["pred_adaptativo"].isna().any():
    raise ValueError(
        "Há previsões adaptativas ausentes."
    )

# ------------------------------------------------------------
# 6. MÉTRICAS GERAIS
# ------------------------------------------------------------

resultados22 = []

for nome, coluna in [
    ("A_Estatico", "pred_estatico"),
    ("B_Adaptativo", "pred_adaptativo")
]:

    m = calcular_metricas22(
        teste22["co2_t"],
        teste22[coluna]
    )

    resultados22.append({
        "modelo": nome,
        "n_horas": len(teste22),
        **m
    })

resultados22 = pd.DataFrame(resultados22)

print("\n" + "=" * 75)
print("RESULTADOS GERAIS — TESTE 2025")
print("=" * 75)

display(resultados22.round(5))

rmse_a22 = resultados22.loc[
    resultados22["modelo"] == "A_Estatico",
    "RMSE"
].iloc[0]

rmse_b22 = resultados22.loc[
    resultados22["modelo"] == "B_Adaptativo",
    "RMSE"
].iloc[0]

ganho22 = 100 * (rmse_a22 - rmse_b22) / rmse_a22

print(
    f"\nRedução do RMSE com atualização mensal: "
    f"{ganho22:+.2f}%"
)

# ------------------------------------------------------------
# 7. RESULTADOS MENSAIS
# ------------------------------------------------------------

teste22["mes"] = (
    teste22["timestamp"].dt.to_period("M").astype(str)
)

resultados_mensais22 = []

for mes, grupo in teste22.groupby("mes"):

    for nome, coluna in [
        ("A_Estatico", "pred_estatico"),
        ("B_Adaptativo", "pred_adaptativo")
    ]:

        m = calcular_metricas22(
            grupo["co2_t"],
            grupo[coluna]
        )

        resultados_mensais22.append({
            "mes": mes,
            "modelo": nome,
            "n_horas": len(grupo),
            **m
        })

resultados_mensais22 = pd.DataFrame(
    resultados_mensais22
)

print("\nRESULTADOS MENSAIS")
display(
    resultados_mensais22.pivot(
        index="mes",
        columns="modelo",
        values="RMSE"
    ).round(3)
)

# ------------------------------------------------------------
# 8. ANÁLISE PRÓXIMA ÀS MUDANÇAS DE 2025
# ------------------------------------------------------------
#
# As datas são utilizadas apenas para avaliação.
# Não são fornecidas aos modelos.
# ------------------------------------------------------------

EVENTOS22 = {
    "marco_2025": pd.Timestamp("2025-03-26 08:00"),
    "julho_2025": pd.Timestamp("2025-07-14 10:00")
}

resultados_eventos22 = []

for evento, corte in EVENTOS22.items():

    for fase, inicio, fim in [
        (
            "antes_14d",
            corte - pd.Timedelta(days=14),
            corte
        ),
        (
            "depois_14d",
            corte,
            corte + pd.Timedelta(days=14)
        )
    ]:

        amostra = teste22.loc[
            (teste22["timestamp"] >= inicio)
            & (teste22["timestamp"] < fim)
        ]

        if len(amostra) < 24:
            continue

        for nome, coluna in [
            ("A_Estatico", "pred_estatico"),
            ("B_Adaptativo", "pred_adaptativo")
        ]:

            m = calcular_metricas22(
                amostra["co2_t"],
                amostra[coluna]
            )

            resultados_eventos22.append({
                "evento": evento,
                "fase": fase,
                "modelo": nome,
                "n_horas": len(amostra),
                **m
            })

resultados_eventos22 = pd.DataFrame(
    resultados_eventos22
)

print("\nDESEMPENHO ANTES E DEPOIS DAS MUDANÇAS")
display(
    resultados_eventos22.sort_values(
        ["evento", "fase", "RMSE"]
    ).round(4)
)

# ------------------------------------------------------------
# 9. GRÁFICOS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1,
    figsize=(13, 9)
)

pivot22 = resultados_mensais22.pivot(
    index="mes",
    columns="modelo",
    values="RMSE"
)

pivot22.plot(
    ax=axes[0],
    marker="o"
)

axes[0].set_title(
    "Iatan 2 — RMSE mensal em 2025"
)
axes[0].set_ylabel("RMSE (tCO2)")
axes[0].set_xlabel("Mês")
axes[0].grid(alpha=0.25)

teste22["erro_estatico"] = (
    teste22["pred_estatico"] - teste22["co2_t"]
)

teste22["erro_adaptativo"] = (
    teste22["pred_adaptativo"] - teste22["co2_t"]
)

erros_diarios22 = (
    teste22.set_index("timestamp")[
        ["erro_estatico", "erro_adaptativo"]
    ]
    .resample("D")
    .mean()
)

erros_diarios22.plot(
    ax=axes[1],
    alpha=0.8
)

for corte in EVENTOS22.values():
    axes[1].axvline(
        corte,
        linestyle="--",
        alpha=0.6
    )

axes[1].axhline(0, linewidth=1)

axes[1].set_title(
    "Viés diário das estimativas — 2025"
)
axes[1].set_ylabel(
    "Predito − observado (tCO2)"
)
axes[1].set_xlabel("Data")
axes[1].grid(alpha=0.25)

plt.tight_layout()

ARQ_FIG22 = (
    FIGURAS22
    / "22A_estatico_vs_atualizacao_mensal.png"
)

plt.savefig(
    ARQ_FIG22,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 10. EXPORTAÇÃO
# ------------------------------------------------------------

ARQ_RESULT22 = (
    TABELAS22
    / "22A_resultados_gerais.csv"
)

ARQ_MENSAL22 = (
    TABELAS22
    / "22A_resultados_mensais.csv"
)

ARQ_EVENTOS22 = (
    TABELAS22
    / "22A_resultados_eventos.csv"
)

ARQ_ATUALIZACOES22 = (
    TABELAS22
    / "22A_registro_atualizacoes.csv"
)

ARQ_PREV22 = (
    DADOS22
    / "22A_previsoes_2025.parquet"
)

resultados22.to_csv(
    ARQ_RESULT22,
    index=False,
    encoding="utf-8-sig"
)

resultados_mensais22.to_csv(
    ARQ_MENSAL22,
    index=False,
    encoding="utf-8-sig"
)

resultados_eventos22.to_csv(
    ARQ_EVENTOS22,
    index=False,
    encoding="utf-8-sig"
)

atualizacoes22.to_csv(
    ARQ_ATUALIZACOES22,
    index=False,
    encoding="utf-8-sig"
)

teste22.to_parquet(
    ARQ_PREV22,
    index=False
)

print("\n" + "=" * 75)
print("EXPERIMENTO 22A — CONCLUÍDO")
print("=" * 75)

print("Resultados:", ARQ_RESULT22)
print("Resultados mensais:", ARQ_MENSAL22)
print("Eventos:", ARQ_EVENTOS22)
print("Atualizações:", ARQ_ATUALIZACOES22)
print("Previsões:", ARQ_PREV22)
print("Figura:", ARQ_FIG22)

print(
    "\nTodas as predições utilizaram exclusivamente "
    "a geração horária como entrada."
)

## 15 — Auditoria de paradas e indicadores informados

Reproduz a auditoria 22B.1 e executa **somente a versão corrigida** do 22B. A versão 22B inicial foi arquivada por ter subcontado retornos históricos.

**Limitação 22B:** no experimento retrospectivo, os eventos do treino são retornos após paradas >=72 h, mas os dois eventos do teste são mudanças de IC selecionadas com CO₂ observado. A célula adicional 15A testa um controle com a **mesma definição operacional de evento** no treino e no teste, sem substituir os resultados originais.

---

**Cuidados com avaliação retrospectiva — Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3); Jewell et al. (2022, §1, pp. 1082–1084).** A escolha de eventos de teste a partir de mudanças detectadas nas emissões permite uma comparação condicionada, mas não um cenário prospectivo equivalente aos eventos de treinamento. A contraprova operacional da seção 15A foi incluída para testar uma definição mais homogênea, sem apagar o experimento original.

In [ ]:
# ===== ORIGEM: CÉLULA 26 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 22B.1 — AUDITORIA DAS PARADAS OPERACIONAIS
# IATAN 2 | 2011–2025
# ============================================================
#
# OBJETIVO:
# Investigar a diferença entre:
# - 96 paradas >=72h identificadas no estudo multianual;
# - 5 retornos históricos encontrados na Célula 22B.
#
# NÃO TREINA MODELOS.
# NÃO UTILIZA CO2 OU INTENSIDADE CARBÔNICA.
#
# RESULTADOS:
# 22B1_auditoria_paradas.csv
# 22B1_resumo_auditoria.csv
# 22B1_comparacao_catalogo.csv (quando disponível)
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = OUTPUT_ROOT

DADOS = ROOT / "dados"
TABELAS = ROOT / "tabelas"
TABELAS.mkdir(parents=True, exist_ok=True)

ARQ_BASE = DADOS / "20_base_horaria_atributos_operacionais.parquet"
ARQ_CATALOGO = TABELAS / "16_paradas_operacionais.csv"

if not ARQ_BASE.exists():
    raise FileNotFoundError(ARQ_BASE)

df = pd.read_parquet(ARQ_BASE)

COLUNAS = [
    "timestamp",
    "tempo_operacao_h",
    "geracao_mwh"
]

ausentes = [c for c in COLUNAS if c not in df.columns]

if ausentes:
    raise KeyError(f"Colunas ausentes: {ausentes}")

df = df[COLUNAS].copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

if df["timestamp"].duplicated().any():
    raise ValueError(
        "Timestamps duplicados: verificar a unidade e a base."
    )

print("=" * 75)
print("AUDITORIA DAS PARADAS — IATAN 2")
print("=" * 75)

print("Período:", df.timestamp.min(), "até", df.timestamp.max())
print("Registros:", len(df))

# ============================================================
# 1. ESTADO OPERACIONAL
# ============================================================
#
# desligada: tempo_operacao_h == 0
# operando:  tempo_operacao_h > 0
# indefinido: tempo_operacao_h ausente
#
# Geração zero, isoladamente, não confirma desligamento.
# ============================================================

df["estado"] = np.select(
    [
        df["tempo_operacao_h"].eq(0),
        df["tempo_operacao_h"].gt(0)
    ],
    [
        "desligada",
        "operando"
    ],
    default="indefinido"
)

print("\nESTADOS OPERACIONAIS")
display(df["estado"].value_counts(dropna=False))

# ============================================================
# 2. RECONSTRUÇÃO DAS SEQUÊNCIAS
# ============================================================
#
# Uma sequência termina quando:
# - o estado operacional muda;
# - existe uma lacuna temporal.
#
# Não se assume continuidade através de horas ausentes.
# ============================================================

delta = df["timestamp"].diff().dt.total_seconds() / 3600

novo_bloco = (
    df["estado"].ne(df["estado"].shift())
    | delta.ne(1)
)

df["bloco_id"] = novo_bloco.cumsum()

blocos = (
    df.groupby("bloco_id")
    .agg(
        inicio=("timestamp", "min"),
        fim=("timestamp", "max"),
        estado=("estado", "first"),
        duracao_h=("timestamp", "size")
    )
    .reset_index()
)

paradas = blocos.loc[
    blocos["estado"].eq("desligada")
].copy()

# ============================================================
# 3. LOCALIZAR O RETORNO DE CADA PARADA
# ============================================================
#
# Distinguimos:
# - primeira hora imediatamente posterior;
# - retorno operacional confirmado;
# - retorno com geração positiva.
#
# Isso permite saber onde os eventos são descartados.
# ============================================================

por_tempo = df.set_index("timestamp")

registros = []

for _, parada in paradas.iterrows():

    fim = parada["fim"]
    instante_seguinte = fim + pd.Timedelta(hours=1)

    existe_hora_seguinte = (
        instante_seguinte in por_tempo.index
    )

    retorno_operacional = False
    retorno_com_geracao = False
    retorno_apos_lacuna = False

    primeira_hora_operacao = pd.NaT
    primeira_hora_geracao = pd.NaT

    if existe_hora_seguinte:

        linha = por_tempo.loc[instante_seguinte]

        retorno_operacional = (
            linha["estado"] == "operando"
        )

        retorno_com_geracao = (
            retorno_operacional
            and pd.notna(linha["geracao_mwh"])
            and linha["geracao_mwh"] > 0
        )

        if retorno_operacional:
            primeira_hora_operacao = instante_seguinte

        if retorno_com_geracao:
            primeira_hora_geracao = instante_seguinte

    # Identificar a primeira observação posterior.
    pos = df["timestamp"].searchsorted(
        instante_seguinte
    )

    if pos < len(df):

        proxima = df.iloc[pos]

        retorno_apos_lacuna = (
            proxima["timestamp"] > instante_seguinte
            and proxima["estado"] == "operando"
        )

    registros.append({
        "bloco_id": parada["bloco_id"],
        "inicio_parada": parada["inicio"],
        "fim_parada": parada["fim"],
        "duracao_h": parada["duracao_h"],
        "ano_inicio": parada["inicio"].year,
        "ano_fim": parada["fim"].year,
        "hora_seguinte_presente": existe_hora_seguinte,
        "retorno_operacional_imediato": retorno_operacional,
        "retorno_com_geracao_imediata": retorno_com_geracao,
        "retorno_apos_lacuna": retorno_apos_lacuna,
        "primeira_hora_operacao": primeira_hora_operacao,
        "primeira_hora_geracao": primeira_hora_geracao
    })

auditoria = pd.DataFrame(registros)

# ============================================================
# 4. RESUMO POR LIMIAR DE DURAÇÃO
# ============================================================

resumo = []

for limite in [1, 4, 12, 24, 72, 168]:

    subset = auditoria.loc[
        auditoria["duracao_h"] >= limite
    ]

    subset_treino = subset.loc[
        subset["fim_parada"] < "2024-01-01"
    ]

    resumo.append({
        "limiar_h": limite,
        "paradas_2011_2025": len(subset),
        "paradas_ate_2023": len(subset_treino),
        "retornos_operacionais_ate_2023": int(
            subset_treino[
                "retorno_operacional_imediato"
            ].sum()
        ),
        "retornos_com_geracao_ate_2023": int(
            subset_treino[
                "retorno_com_geracao_imediata"
            ].sum()
        ),
        "retornos_apos_lacuna_ate_2023": int(
            subset_treino[
                "retorno_apos_lacuna"
            ].sum()
        )
    })

resumo = pd.DataFrame(resumo)

print("\nRESUMO POR DURAÇÃO")
display(resumo)

# ============================================================
# 5. INSPECIONAR ESPECIFICAMENTE PARADAS >=72H
# ============================================================

longas = auditoria.loc[
    auditoria["duracao_h"] >= 72
].copy()

longas["situacao_retorno"] = np.select(
    [
        longas["retorno_com_geracao_imediata"],
        longas["retorno_operacional_imediato"],
        longas["retorno_apos_lacuna"],
        ~longas["hora_seguinte_presente"]
    ],
    [
        "retorno_com_geracao",
        "retorno_sem_geracao",
        "retorno_apos_lacuna",
        "hora_seguinte_ausente"
    ],
    default="outro"
)

print("\nSITUAÇÃO DOS RETORNOS — PARADAS >=72H")
display(
    longas["situacao_retorno"].value_counts()
)

print("\nPARADAS LONGAS E SEUS RETORNOS")
display(
    longas[
        [
            "inicio_parada",
            "fim_parada",
            "duracao_h",
            "situacao_retorno"
        ]
    ].sort_values("inicio_parada")
)

# ============================================================
# 6. COMPARAÇÃO COM O CATÁLOGO ORIGINAL
# ============================================================
#
# O catálogo original pode ter estrutura diferente.
# Primeiro inspecionamos suas colunas e contagem.
# Não presumimos nomes ou equivalência entre eventos.
# ============================================================

if ARQ_CATALOGO.exists():

    catalogo = pd.read_csv(ARQ_CATALOGO)

    print("\nCATÁLOGO ORIGINAL ENCONTRADO")
    print("Registros:", len(catalogo))
    print("Colunas:", list(catalogo.columns))

    display(catalogo.head(5))

    catalogo.to_csv(
        TABELAS / "22B1_comparacao_catalogo.csv",
        index=False,
        encoding="utf-8-sig"
    )

else:

    print("\nCatálogo original não encontrado:")
    print(ARQ_CATALOGO)

# ============================================================
# 7. SALVAR RESULTADOS
# ============================================================

auditoria.to_csv(
    TABELAS / "22B1_auditoria_paradas.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo.to_csv(
    TABELAS / "22B1_resumo_auditoria.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("AUDITORIA CONCLUÍDA")
print("=" * 75)

print("Total de paradas reconstruídas:", len(auditoria))
print("Paradas >=72h:", len(longas))

print(
    "\nArquivos salvos em:",
    TABELAS
)


# ===== ORIGEM: CÉLULA 27 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 22B — VERSÃO CORRIGIDA
# INTERVENÇÕES INFORMADAS E ESTIMAÇÃO HORÁRIA DE CO2
# IATAN 2 | EPA CEMS | 2011–2025
# ============================================================
#
# HIPÓTESE:
# Informar possíveis intervenções operacionais melhora
# a estimação de CO2 após mudanças de intensidade carbônica?
#
# M0: geração
# M1: geração + intervenção recente
# M2: geração + intervenção recente + tempo desde evento
#
# TREINO: 2011–2023
# VALIDAÇÃO DIAGNÓSTICA: 2024
# TESTE: 2025
#
# Nenhum atributo utiliza CO2 ou intensidade carbônica.
#
# ATENÇÃO:
# As datas de 2025 foram identificadas retrospectivamente
# a partir de mudanças de IC. O teste é exploratório e
# não representa detecção prospectiva de intervenções.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from IPython.display import display

# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

ROOT22B = OUTPUT_ROOT

DADOS22B = ROOT22B / "dados"
TABELAS22B = ROOT22B / "tabelas"
FIGURAS22B = ROOT22B / "figuras"

for pasta in [TABELAS22B, FIGURAS22B]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQ_BASE22B = (
    DADOS22B / "20_base_horaria_atributos_operacionais.parquet"
)

ARQ_PARADAS22B = (
    TABELAS22B / "16_paradas_operacionais.csv"
)

for arquivo in [ARQ_BASE22B, ARQ_PARADAS22B]:
    if not arquivo.exists():
        raise FileNotFoundError(arquivo)

JANELA_INTERVENCAO_H = 90 * 24
LIMIAR_PARADA_H = 72

EVENTOS_2025 = [
    pd.Timestamp("2025-03-26 08:00:00"),
    pd.Timestamp("2025-07-14 10:00:00")
]

# ============================================================
# 2. CARREGAMENTO
# ============================================================

df22b = pd.read_parquet(ARQ_BASE22B)
paradas22b = pd.read_csv(ARQ_PARADAS22B)

df22b["timestamp"] = pd.to_datetime(
    df22b["timestamp"]
)

df22b = (
    df22b.sort_values("timestamp")
    .reset_index(drop=True)
)

if df22b["timestamp"].duplicated().any():
    raise ValueError(
        "Existem timestamps duplicados na base horária."
    )

paradas22b["fim_exclusivo"] = pd.to_datetime(
    paradas22b["fim_exclusivo"]
)

# ============================================================
# 3. IDENTIFICAR RETORNOS HISTÓRICOS
# ============================================================
#
# O retorno ocorre na primeira hora após o encerramento
# da parada, independentemente da geração elétrica.
#
# Essa hora deve apresentar tempo_operacao_h > 0.
#
# Não exigimos geração positiva na hora de retorno.
# ============================================================

paradas_longas22b = paradas22b.loc[
    paradas22b["duracao_horas"] >= LIMIAR_PARADA_H
].copy()

estados22b = df22b.set_index("timestamp")

eventos_historicos22b = []

for _, parada in paradas_longas22b.iterrows():

    retorno = parada["fim_exclusivo"]

    if retorno not in estados22b.index:
        continue

    tempo_operacao = estados22b.loc[
        retorno, "tempo_operacao_h"
    ]

    if (
        pd.notna(tempo_operacao)
        and tempo_operacao > 0
        and retorno < pd.Timestamp("2025-01-01")
    ):
        eventos_historicos22b.append(retorno)

eventos_historicos22b = sorted(
    set(eventos_historicos22b)
)

eventos_treino22b = [
    t for t in eventos_historicos22b
    if t < pd.Timestamp("2024-01-01")
]

eventos_validacao22b = [
    t for t in eventos_historicos22b
    if pd.Timestamp("2024-01-01") <= t
    < pd.Timestamp("2025-01-01")
]

print("=" * 78)
print("EXPERIMENTO 22B — VERSÃO CORRIGIDA")
print("=" * 78)

print(
    "Paradas >=72h no catálogo:",
    len(paradas_longas22b)
)

print(
    "Retornos históricos até 2023:",
    len(eventos_treino22b)
)

print(
    "Retornos em 2024:",
    len(eventos_validacao22b)
)

print(
    "Eventos informados em 2025:",
    len(EVENTOS_2025)
)

# Conferência com auditoria anterior.
if len(eventos_treino22b) != 76:
    print(
        "\nATENÇÃO: esperávamos 76 retornos até 2023. "
        "Verifique os critérios e o catálogo."
    )

# ============================================================
# 4. CRIAÇÃO DOS ATRIBUTOS DE INTERVENÇÃO
# ============================================================
#
# Para cada hora:
# - identifica o evento mais recente;
# - calcula as horas desde esse evento;
# - ativa o indicador por até 90 dias;
# - reinicia o relógio quando ocorre outro evento.
#
# Fora da janela:
# indicador = 0
# horas_desde = 0
#
# O modelo recebe apenas informações disponíveis
# até o instante da previsão.
# ============================================================

def atributos_intervencao_22b(timestamps, eventos):

    tempos = pd.to_datetime(timestamps)
    t = tempos.to_numpy(dtype="datetime64[ns]")

    indicador = np.zeros(len(t), dtype=np.int8)
    horas = np.zeros(len(t), dtype=np.float64)

    if len(eventos) == 0:
        return indicador, horas

    e = np.array(
        sorted(set(eventos)),
        dtype="datetime64[ns]"
    )

    pos = np.searchsorted(
        e,
        t,
        side="right"
    ) - 1

    valido = pos >= 0

    distancia = np.full(
        len(t),
        np.inf,
        dtype=float
    )

    distancia[valido] = (
        (t[valido] - e[pos[valido]])
        / np.timedelta64(1, "h")
    ).astype(float)

    recente = (
        valido
        & (distancia >= 0)
        & (distancia <= JANELA_INTERVENCAO_H)
    )

    indicador[recente] = 1
    horas[recente] = distancia[recente]

    return indicador, horas

df22b["intervencao_22b"] = 0
df22b["horas_desde_intervencao_22b"] = 0.0

# Treinamento: retornos de paradas até 2023.
# Validação: retornos históricos até 2024.
# Teste: somente os dois eventos informados de 2025.

periodos22b = [
    (
        df22b["timestamp"] < "2024-01-01",
        eventos_treino22b
    ),
    (
        df22b["timestamp"].between(
            "2024-01-01",
            "2024-12-31 23:00:00"
        ),
        eventos_historicos22b
    ),
    (
        df22b["timestamp"].dt.year.eq(2025),
        EVENTOS_2025
    )
]

for mascara, eventos in periodos22b:

    ind, horas = atributos_intervencao_22b(
        df22b.loc[mascara, "timestamp"],
        eventos
    )

    df22b.loc[
        mascara, "intervencao_22b"
    ] = ind

    df22b.loc[
        mascara, "horas_desde_intervencao_22b"
    ] = horas

# ============================================================
# 5. PREPARAÇÃO DOS DADOS
# ============================================================

colunas_obrigatorias = [
    "timestamp",
    "geracao_mwh",
    "co2_t",
    "intervencao_22b",
    "horas_desde_intervencao_22b"
]

ausentes = [
    c for c in colunas_obrigatorias
    if c not in df22b.columns
]

if ausentes:
    raise KeyError(f"Colunas ausentes: {ausentes}")

amostra22b = df22b.loc[
    df22b["geracao_mwh"].notna()
    & df22b["co2_t"].notna()
    & (df22b["geracao_mwh"] > 0)
    & (df22b["co2_t"] >= 0)
].copy()

treino22b = amostra22b.loc[
    amostra22b["timestamp"] < "2024-01-01"
].copy()

validacao22b = amostra22b.loc[
    amostra22b["timestamp"].dt.year.eq(2024)
].copy()

teste22b = amostra22b.loc[
    amostra22b["timestamp"].dt.year.eq(2025)
].copy()

FEATURES22B = {
    "M0_Geracao": [
        "geracao_mwh"
    ],
    "M1_Indicador": [
        "geracao_mwh",
        "intervencao_22b"
    ],
    "M2_Tempo": [
        "geracao_mwh",
        "intervencao_22b",
        "horas_desde_intervencao_22b"
    ]
}

PARAMS22B = dict(
    n_estimators=350,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

# ============================================================
# 6. TREINAMENTO
# ============================================================

modelos22b = {}
resultados22b = []
resultados_validacao22b = []

def calcular_metricas22b(y, pred):

    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y, pred)
        ),
        "R2": r2_score(y, pred),
        "vies": float(np.mean(pred - y))
    }

for nome, atributos in FEATURES22B.items():

    modelo = XGBRegressor(**PARAMS22B)

    modelo.fit(
        treino22b[atributos],
        treino22b["co2_t"]
    )

    modelos22b[nome] = modelo

    # Validação 2024
    pred_val = np.maximum(
        modelo.predict(validacao22b[atributos]),
        0
    )

    resultados_validacao22b.append({
        "modelo": nome,
        **calcular_metricas22b(
            validacao22b["co2_t"],
            pred_val
        )
    })

    # Teste 2025
    pred_teste = np.maximum(
        modelo.predict(teste22b[atributos]),
        0
    )

    teste22b[f"pred_{nome}"] = pred_teste

    resultados22b.append({
        "modelo": nome,
        **calcular_metricas22b(
            teste22b["co2_t"],
            pred_teste
        )
    })

resultados22b = pd.DataFrame(resultados22b)
resultados_validacao22b = pd.DataFrame(
    resultados_validacao22b
)

# ============================================================
# 7. RESULTADOS GERAIS
# ============================================================

print("\nRESULTADOS — VALIDAÇÃO 2024")
display(
    resultados_validacao22b.round(4)
)

print("\nRESULTADOS — TESTE 2025")
display(
    resultados22b.round(4)
)

rmse_m0 = resultados22b.loc[
    resultados22b["modelo"].eq("M0_Geracao"),
    "RMSE"
].iloc[0]

resultados22b["ganho_rmse_pct"] = (
    100 * (
        rmse_m0 - resultados22b["RMSE"]
    ) / rmse_m0
)

print("\nGANHO EM RELAÇÃO AO MODELO M0")
display(
    resultados22b[
        ["modelo", "RMSE", "ganho_rmse_pct"]
    ].round(3)
)

# ============================================================
# 8. ANÁLISE ANTES E DEPOIS DAS MUDANÇAS DE IC
# ============================================================

resultados_eventos22b = []

for evento, data in [
    ("Marco", EVENTOS_2025[0]),
    ("Julho", EVENTOS_2025[1])
]:

    periodos = [
        (
            "Antes_14d",
            data - pd.Timedelta(days=14),
            data
        ),
        (
            "Depois_14d",
            data,
            data + pd.Timedelta(days=14)
        ),
        (
            "Depois_30d",
            data,
            data + pd.Timedelta(days=30)
        )
    ]

    for fase, inicio, fim in periodos:

        parte = teste22b.loc[
            (teste22b["timestamp"] >= inicio)
            & (teste22b["timestamp"] < fim)
        ]

        if len(parte) < 2:
            continue

        for nome in FEATURES22B:

            pred = parte[f"pred_{nome}"]

            resultados_eventos22b.append({
                "evento": evento,
                "fase": fase,
                "modelo": nome,
                "n_horas": len(parte),
                **calcular_metricas22b(
                    parte["co2_t"],
                    pred
                )
            })

resultados_eventos22b = pd.DataFrame(
    resultados_eventos22b
)

print("\nRESULTADOS POR EVENTO")
display(
    resultados_eventos22b.round(3)
)

# ============================================================
# 9. GRÁFICO — VIÉS DIÁRIO
# ============================================================

fig, ax = plt.subplots(
    figsize=(14, 5)
)

for nome in FEATURES22B:

    serie = (
        teste22b
        .set_index("timestamp")[f"pred_{nome}"]
        - teste22b
        .set_index("timestamp")["co2_t"]
    )

    serie.resample("D").mean().plot(
        ax=ax,
        label=nome,
        alpha=0.8
    )

for data in EVENTOS_2025:

    ax.axvline(
        data,
        color="black",
        linestyle="--",
        alpha=0.6
    )

ax.axhline(
    0,
    color="gray",
    linewidth=1
)

ax.set_title(
    "Iatan 2 — Intervenções informadas: experimento corrigido"
)

ax.set_ylabel(
    "Viés diário (tCO2)"
)

ax.set_xlabel("Data")
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

ARQ_FIG22B = (
    FIGURAS22B
    / "22B_intervencoes_informadas_corrigido.png"
)

plt.savefig(
    ARQ_FIG22B,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 10. EXPORTAÇÃO
# ============================================================

resultados22b.to_csv(
    TABELAS22B / "22B_resultados_gerais_corrigido.csv",
    index=False,
    encoding="utf-8-sig"
)

resultados_validacao22b.to_csv(
    TABELAS22B / "22B_validacao_2024_corrigido.csv",
    index=False,
    encoding="utf-8-sig"
)

resultados_eventos22b.to_csv(
    TABELAS22B / "22B_resultados_eventos_corrigido.csv",
    index=False,
    encoding="utf-8-sig"
)

teste22b.to_parquet(
    DADOS22B / "22B_previsoes_2025_corrigido.parquet",
    index=False
)

print("\n" + "=" * 78)
print("EXPERIMENTO 22B CORRIGIDO — CONCLUÍDO")
print("=" * 78)

print("Treino:", len(treino22b), "horas")
print("Validação:", len(validacao22b), "horas")
print("Teste:", len(teste22b), "horas")
print("Eventos de treino:", len(eventos_treino22b))
print("Eventos de validação:", len(eventos_validacao22b))
print("Eventos informados em 2025:", EVENTOS_2025)
print("Figura:", ARQ_FIG22B)

## 15A — Contraprova do 22B com eventos operacionais comparáveis

Reaproveita os modelos treinados no 22B, mas substitui **somente os eventos de teste** por retornos após paradas >=72 h identificados pela operação, mesma definição usada no treino. Não implica que essas paradas sejam manutenções ou mudanças de IC. Resultados complementares, não substitutos do 22B retrospectivo.

---

**Controle de definição — Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3).** Uma avaliação comparável depende de regras de construção das amostras aplicadas de maneira coerente. O critério de paradas ≥72 h é uma **escolha operacional deste trabalho**; não é uma duração prescrita pelas referências e não comprova intervenção de manutenção.

In [ ]:
# 15A — Controle adicional: mesma definição de evento em treino e teste (22B)
# Mantém intacto o experimento 22B retrospectivo já executado.
# Aqui, treino E teste usam exclusivamente retornos após paradas >=72 h,
# identificados por registros operacionais, e NÃO por mudanças de IC.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_22b_necessarias = ["fim_exclusivo", "duracao_horas"]
if any(c not in paradas22b.columns for c in _22b_necessarias):
    raise KeyError("Catálogo de paradas incompleto para o controle 22B operacional.")

_22b_estados = df22b.set_index("timestamp")["tempo_operacao_h"]
_22b_retornos_2025 = []
for _, _parada in paradas22b.loc[paradas22b["duracao_horas"].ge(LIMIAR_PARADA_H)].iterrows():
    _retorno = pd.Timestamp(_parada["fim_exclusivo"])
    if not (pd.Timestamp("2025-01-01") <= _retorno < pd.Timestamp("2026-01-01")):
        continue
    if _retorno in _22b_estados.index:
        _op = _22b_estados.loc[_retorno]
        if pd.notna(_op) and _op > 0:
            _22b_retornos_2025.append(_retorno)
_22b_retornos_2025 = sorted(set(_22b_retornos_2025))

_22b_teste_op = teste22b.copy()
_22b_ind, _22b_horas = atributos_intervencao_22b(
    _22b_teste_op["timestamp"], _22b_retornos_2025
)
_22b_teste_op["intervencao_22b"] = _22b_ind
_22b_teste_op["horas_desde_intervencao_22b"] = _22b_horas

_22b_controle = []
for _nome, _features in FEATURES22B.items():
    _prev = np.maximum(modelos22b[_nome].predict(_22b_teste_op[_features]), 0)
    _22b_teste_op[f"pred_{_nome}_retornos"] = _prev
    _22b_controle.append({
        "modelo": _nome,
        "definicao_evento_treino": "retorno_apos_parada_ge_72h",
        "definicao_evento_teste": "retorno_apos_parada_ge_72h",
        "n_eventos_teste": len(_22b_retornos_2025),
        "n_horas": len(_22b_teste_op),
        **calcular_metricas22b(_22b_teste_op["co2_t"], _prev),
    })
_22b_controle = pd.DataFrame(_22b_controle)
_22b_controle.to_csv(
    DIR_TABELAS / "15A_22B_controle_eventos_operacionais_2025.csv",
    index=False, encoding="utf-8-sig"
)
_22b_teste_op[["timestamp", "co2_t", "geracao_mwh"] +
             [f"pred_{n}_retornos" for n in FEATURES22B]].to_parquet(
    DIR_DADOS / "15A_22B_previsoes_eventos_operacionais_2025.parquet", index=False
)
print("CONTROLE 22B — EVENTOS HOMOGÊNEOS (RETORNOS OPERACIONAIS)")
print("Retornos de paradas >=72h em 2025:", len(_22b_retornos_2025))
print("Datas:", [str(x) for x in _22b_retornos_2025])
print("Este controle não pressupõe que os retornos tenham alterado a IC.")
display(_22b_controle.round(4))
print("22B original com mudanças de IC continua EXPLORATÓRIO; não comparar como teste prospectivo.")

## 16 — Fator histórico e correção informada

Reestima o fator histórico na mudança candidata de 2017, verifica persistência em 2018 e compara M0, M1 e M2 em 2025. O M2 é uma **pós-correção de M0**, não um novo conjunto de features.

**Sensibilidade:** o intervalo março–dezembro/2017 inclui outra mudança candidata em 28/09/2017. A nova célula 16A recalcula fatores em janelas alternativas. Nenhum fator é escolhido pelo erro de 2025.
**Diferença entre modelos de referência:** o fator de 2017 é estimado contra um XGBoost treinado até 2015; o M0 aplicado em 2025 é treinado até 2023, incorporando dados posteriores à mudança histórica. Logo, o fator pode corrigir viés entre modelos e regimes, não sendo identificável como efeito causal isolado de uma manutenção. A verificação em 2018 avalia a continuidade de um regime, não um novo evento independente.

---

### Fundamento e formulação do fator histórico (contribuição experimental)

**Literatura de apoio:** Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11) fundamentam a relevância da carga para o fator de emissões; Gama et al. (2014, pp. 44:2 e 44:4) discutem modelos submetidos a mudança da relação entrada–saída; Chen e Guestrin (2016, §§2.1–2.2, p. 786) apresentam o XGBoost de referência. **A expressão seguinte não é reproduzida desses trabalhos: foi definida e implementada neste estudo** como regra empírica de calibração.

O modelo histórico é treinado em **2011–2015**. Para observações pré e pós-mudança candidata de 2017, define-se a log-razão \(r_t=\ln(CO_{2,t}/\widehat{CO}_{2,t}^{\mathrm{hist}})\). Restringe-se a comparação à interseção das faixas de geração entre os dois períodos e dividem-se as cargas em até **oito faixas quantílicas**. Para cada faixa válida \(b\):

\[
\Delta_b=\operatorname{mediana}(r_{t\in\mathrm{pós},b})-\operatorname{mediana}(r_{t\in\mathrm{pré},b}),\quad w_b=\min(n_{\mathrm{pré},b},n_{\mathrm{pós},b}).
\]

O fator único é \(F=\exp\left(\frac{\sum_b w_b\Delta_b}{\sum_b w_b}\right)\). O modelo de referência de 2025 usa outra janela de treinamento (**até 2023**). A aplicação ao M0 é um **transporte hipotético de uma correção histórica entre períodos e modelos**, e pode captar diferenças de viés além de possíveis mudanças físicas.

Para as duas datas \(\tau_1\) e \(\tau_2\), retrospectivamente identificadas, as previsões são \(\widehat y_t^{M1}=\widehat y_t^{M0}F^{\mathbf 1(t\ge\tau_1)}\) e \(\widehat y_t^{M2}=\widehat y_t^{M0}F^{\mathbf 1(t\ge\tau_1)+\mathbf 1(t\ge\tau_2)}\). **Aplicar o mesmo F novamente após julho é uma hipótese experimental de efeitos proporcionais sucessivos, não uma lei físico-operacional nem a confirmação de duas manutenções.**

In [ ]:
# ===== ORIGEM: CÉLULA 28 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 22C — CORREÇÃO INFORMADA POR MUDANÇAS CANDIDATAS
# IATAN 2 | EPA CEMS | 2011–2025
# ============================================================
#
# QUESTÃO:
# Uma redução histórica na relação geração–CO2 pode
# ajudar a estimar emissões após intervenções informadas?
#
# ETAPA HISTÓRICA:
# - Treinar XGBoost com 2011–2015.
# - Comparar resíduos relativos em 2016/início de 2017
#   e após a mudança candidata de março de 2017.
# - Aprender um fator multiplicativo histórico.
# - Avaliar sua persistência em 2018.
#
# ETAPA 2025:
# - Treinar M0 com 2011–2023.
# - Aplicar o fator histórico após eventos informados.
# - Não recalibrar usando emissões de 2025.
#
# M0: geração, sem correção.
# M1: uma correção após o primeiro evento.
# M2: correção cumulativa após os dois eventos.
#
# CO2 NÃO É ENTRADA DO XGBOOST.
# O fator histórico é estimado com CO2 de 2016–2017.
#
# Datas dos eventos são retrospectivas, não confirmam
# intervenções físicas ou relações causais.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from IPython.display import display

# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

ROOT22C = OUTPUT_ROOT

DADOS22C = ROOT22C / "dados"
TABELAS22C = ROOT22C / "tabelas"
FIGURAS22C = ROOT22C / "figuras"

for pasta in [TABELAS22C, FIGURAS22C]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQ22C = (
    DADOS22C
    / "20_base_horaria_atributos_operacionais.parquet"
)

if not ARQ22C.exists():
    raise FileNotFoundError(ARQ22C)

# Data candidata baseada no episódio B do gráfico mensal.
# Não representa manutenção confirmada.
EVENTO_HIST22C = pd.Timestamp("2017-03-01 00:00")

# Datas retrospectivas identificadas no Notebook 07.
EVENTOS_2025_22C = [cortes_reconciliados_2025["2025-03"],
                    cortes_reconciliados_2025["2025-07"]]

PARAMS22C = dict(
    n_estimators=350,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

# ============================================================
# 2. CARREGAR E VALIDAR BASE
# ============================================================

df22c = pd.read_parquet(ARQ22C)

obrigatorias = ["timestamp", "geracao_mwh", "co2_t"]
faltantes = [c for c in obrigatorias if c not in df22c.columns]

if faltantes:
    raise KeyError(f"Colunas ausentes: {faltantes}")

df22c["timestamp"] = pd.to_datetime(df22c["timestamp"])

df22c = (
    df22c.sort_values("timestamp")
    .reset_index(drop=True)
)

if df22c["timestamp"].duplicated().any():
    raise ValueError("Há timestamps duplicados na base.")

# Mesmo critério de elegibilidade dos experimentos anteriores.
base22c = df22c.loc[
    df22c["geracao_mwh"].notna()
    & df22c["co2_t"].notna()
    & (df22c["geracao_mwh"] > 0)
    & (df22c["co2_t"] >= 0)
].copy()

def metricas22c(real, previsto):
    return {
        "MAE": mean_absolute_error(real, previsto),
        "RMSE": np.sqrt(mean_squared_error(real, previsto)),
        "R2": r2_score(real, previsto),
        "vies": float(np.mean(previsto - real))
    }

def prever22c(modelo, dados):
    return np.maximum(
        modelo.predict(dados[["geracao_mwh"]]),
        0
    )

print("=" * 75)
print("EXPERIMENTO 22C — APRENDIZADO HISTÓRICO")
print("=" * 75)

print("Horas elegíveis:", len(base22c))
print("Evento histórico:", EVENTO_HIST22C)
print("Eventos de 2025:", EVENTOS_2025_22C)

# ============================================================
# 3. MODELO HISTÓRICO — TREINO 2011–2015
# ============================================================

treino_hist22c = base22c.loc[
    base22c["timestamp"] < "2016-01-01"
].copy()

pre_hist22c = base22c.loc[
    (base22c["timestamp"] >= "2016-01-01")
    & (base22c["timestamp"] < EVENTO_HIST22C)
].copy()

pos_hist22c = base22c.loc[
    (base22c["timestamp"] >= EVENTO_HIST22C)
    & (base22c["timestamp"] < "2018-01-01")
].copy()

valid_hist22c = base22c.loc[
    base22c["timestamp"].dt.year.eq(2018)
].copy()

for nome, dados in [
    ("Treino histórico", treino_hist22c),
    ("Antes do evento", pre_hist22c),
    ("Depois do evento", pos_hist22c),
    ("Verificação 2018", valid_hist22c)
]:
    if len(dados) < 100:
        raise ValueError(
            f"{nome} possui somente {len(dados)} horas. "
            "Verifique a cobertura temporal."
        )

modelo_hist22c = XGBRegressor(**PARAMS22C)

modelo_hist22c.fit(
    treino_hist22c[["geracao_mwh"]],
    treino_hist22c["co2_t"]
)

pre_hist22c["pred_base"] = prever22c(
    modelo_hist22c, pre_hist22c
)

pos_hist22c["pred_base"] = prever22c(
    modelo_hist22c, pos_hist22c
)

valid_hist22c["pred_base"] = prever22c(
    modelo_hist22c, valid_hist22c
)

# ============================================================
# 4. ESTIMAR MUDANÇA HISTÓRICA CONTROLANDO GERAÇÃO
# ============================================================
#
# Comparar CO2 observado / CO2 estimado antes e depois.
#
# Para reduzir o efeito de diferentes distribuições de carga:
# 1. restringimos à faixa comum de geração;
# 2. comparamos resíduos relativos em faixas de carga;
# 3. agregamos as diferenças com pesos por disponibilidade.
#
# O fator não é estimado com dados de 2025.
# ============================================================

def preparar_residuos22c(dados):

    aux = dados.loc[
        (dados["co2_t"] > 0)
        & (dados["pred_base"] > 1)
    ].copy()

    aux["log_razao"] = np.log(
        aux["co2_t"] / aux["pred_base"]
    )

    return aux

pre_cal22c = preparar_residuos22c(pre_hist22c)
pos_cal22c = preparar_residuos22c(pos_hist22c)

# Faixa de geração comum entre os períodos.
limite_inf22c = max(
    pre_cal22c["geracao_mwh"].quantile(0.05),
    pos_cal22c["geracao_mwh"].quantile(0.05)
)

limite_sup22c = min(
    pre_cal22c["geracao_mwh"].quantile(0.95),
    pos_cal22c["geracao_mwh"].quantile(0.95)
)

if limite_inf22c >= limite_sup22c:
    raise ValueError(
        "Não existe faixa de geração comparável "
        "entre os períodos históricos."
    )

pre_cal22c = pre_cal22c.loc[
    pre_cal22c["geracao_mwh"].between(
        limite_inf22c, limite_sup22c
    )
].copy()

pos_cal22c = pos_cal22c.loc[
    pos_cal22c["geracao_mwh"].between(
        limite_inf22c, limite_sup22c
    )
].copy()

if min(len(pre_cal22c), len(pos_cal22c)) < 100:
    raise ValueError(
        "A faixa comum de geração possui poucos registros."
    )

# Oito faixas quantílicas de geração.
geracao_combinada22c = pd.concat([
    pre_cal22c["geracao_mwh"],
    pos_cal22c["geracao_mwh"]
])

limites22c = np.unique(
    np.quantile(
        geracao_combinada22c,
        np.linspace(0, 1, 9)
    )
)

if len(limites22c) < 4:
    raise ValueError(
        "Variação de geração insuficiente para "
        "comparação estratificada."
    )

pre_cal22c["faixa"] = pd.cut(
    pre_cal22c["geracao_mwh"],
    bins=limites22c,
    include_lowest=True
)

pos_cal22c["faixa"] = pd.cut(
    pos_cal22c["geracao_mwh"],
    bins=limites22c,
    include_lowest=True
)

registros_faixas22c = []

for faixa in pre_cal22c["faixa"].cat.categories:

    a = pre_cal22c.loc[
        pre_cal22c["faixa"] == faixa, "log_razao"
    ]

    b = pos_cal22c.loc[
        pos_cal22c["faixa"] == faixa, "log_razao"
    ]

    if len(a) < 15 or len(b) < 15:
        continue

    diferenca = float(b.median() - a.median())

    registros_faixas22c.append({
        "faixa_geracao": str(faixa),
        "n_antes": len(a),
        "n_depois": len(b),
        "delta_log": diferenca,
        "fator_faixa": float(np.exp(diferenca)),
        "peso": min(len(a), len(b))
    })

faixas22c = pd.DataFrame(registros_faixas22c)

if len(faixas22c) < 3:
    raise ValueError(
        "Menos de três faixas comparáveis. "
        "Não é seguro estimar o fator histórico."
    )

delta_historico22c = np.average(
    faixas22c["delta_log"],
    weights=faixas22c["peso"]
)

FATOR_HIST22C = float(np.exp(delta_historico22c))

EFEITO_HIST_PCT22C = 100 * (FATOR_HIST22C - 1)

print("\nAPRENDIZADO HISTÓRICO")
print("Horas antes:", len(pre_cal22c))
print("Horas depois:", len(pos_cal22c))
print("Faixas comparáveis:", len(faixas22c))
print(
    "Faixa comum de geração:",
    round(limite_inf22c, 2),
    "a",
    round(limite_sup22c, 2),
    "MWh"
)

print(f"Fator histórico aprendido: {FATOR_HIST22C:.5f}")
print(f"Alteração relativa estimada: {EFEITO_HIST_PCT22C:+.2f}%")

display(faixas22c.round(4))

if FATOR_HIST22C >= 1:
    print(
        "\nATENÇÃO: o fator estimado não indica redução. "
        "A hipótese de intervenção redutora não foi "
        "confirmada por esta comparação histórica. "
        "Não vamos forçar artificialmente um fator menor que 1."
    )

# ============================================================
# 5. VERIFICAÇÃO HISTÓRICA EM 2018
# ============================================================
#
# 2018 não participa do cálculo do fator.
# Verificamos se a correção aprendida permanece útil.
# ============================================================

valid_hist22c["pred_corrigida"] = (
    valid_hist22c["pred_base"] * FATOR_HIST22C
)

validacao_hist22c = pd.DataFrame([
    {
        "modelo": "Base_historica",
        **metricas22c(
            valid_hist22c["co2_t"],
            valid_hist22c["pred_base"]
        )
    },
    {
        "modelo": "Correcao_historica",
        **metricas22c(
            valid_hist22c["co2_t"],
            valid_hist22c["pred_corrigida"]
        )
    }
])

print("\nVERIFICAÇÃO HISTÓRICA — 2018")
display(validacao_hist22c.round(4))

# ============================================================
# 6. MODELO DE REFERÊNCIA PARA 2025
# ============================================================
#
# Mesma estrutura do M0 dos experimentos 22A e 22B.
# XGBoost treinado até 2023, usando somente geração.
# ============================================================

treino_2025_22c = base22c.loc[
    base22c["timestamp"] < "2024-01-01"
].copy()

validacao_2024_22c = base22c.loc[
    base22c["timestamp"].dt.year.eq(2024)
].copy()

teste22c = base22c.loc[
    base22c["timestamp"].dt.year.eq(2025)
].copy()

modelo_2025_22c = XGBRegressor(**PARAMS22C)

modelo_2025_22c.fit(
    treino_2025_22c[["geracao_mwh"]],
    treino_2025_22c["co2_t"]
)

teste22c["pred_M0"] = prever22c(
    modelo_2025_22c,
    teste22c
)

# ============================================================
# 7. INFORMAR OS EVENTOS REDUTORES DE 2025
# ============================================================
#
# M1: aplicar uma única redução após março.
#
# M2: aplicar uma redução após março e outra após julho,
#     de forma cumulativa.
#
# As datas são informadas externamente ao estimador.
# Não utilizamos a magnitude observada da queda em 2025.
# ============================================================

eventos_np22c = np.array(
    sorted(EVENTOS_2025_22C),
    dtype="datetime64[ns]"
)

tempos_np22c = teste22c["timestamp"].to_numpy(
    dtype="datetime64[ns]"
)

teste22c["n_intervencoes"] = np.searchsorted(
    eventos_np22c,
    tempos_np22c,
    side="right"
)

# M1: uma única correção, mantida após março.
teste22c["pred_M1"] = (
    teste22c["pred_M0"]
    * np.where(
        teste22c["n_intervencoes"] >= 1,
        FATOR_HIST22C,
        1.0
    )
)

# M2: correção cumulativa, reiniciada a cada evento.
teste22c["pred_M2"] = (
    teste22c["pred_M0"]
    * np.power(
        FATOR_HIST22C,
        teste22c["n_intervencoes"]
    )
)

# ============================================================
# 8. RESULTADOS GERAIS DE 2025
# ============================================================

NOMES22C = {
    "M0_Geracao": "pred_M0",
    "M1_Uma_intervencao": "pred_M1",
    "M2_Duas_intervencoes": "pred_M2"
}

resultados22c = []

for nome, coluna in NOMES22C.items():

    resultados22c.append({
        "modelo": nome,
        **metricas22c(
            teste22c["co2_t"],
            teste22c[coluna]
        )
    })

resultados22c = pd.DataFrame(resultados22c)

rmse_ref22c = resultados22c.loc[
    resultados22c["modelo"] == "M0_Geracao",
    "RMSE"
].iloc[0]

resultados22c["ganho_RMSE_pct"] = (
    100 * (
        rmse_ref22c - resultados22c["RMSE"]
    ) / rmse_ref22c
)

print("\nRESULTADOS GERAIS — TESTE 2025")
display(resultados22c.round(4))

# A síntese final verifica o M0 nas comparações com a mesma amostra.

# ============================================================
# 9. RESULTADOS ANTES E DEPOIS DOS EVENTOS
# ============================================================

eventos_resultados22c = []

for evento, data in [
    ("Marco", EVENTOS_2025_22C[0]),
    ("Julho", EVENTOS_2025_22C[1])
]:

    janelas = [
        (
            "Antes_14d",
            data - pd.Timedelta(days=14),
            data
        ),
        (
            "Depois_14d",
            data,
            data + pd.Timedelta(days=14)
        ),
        (
            "Depois_30d",
            data,
            data + pd.Timedelta(days=30)
        )
    ]

    for fase, inicio, fim in janelas:

        parte = teste22c.loc[
            (teste22c["timestamp"] >= inicio)
            & (teste22c["timestamp"] < fim)
        ]

        if len(parte) < 2:
            continue

        for nome, coluna in NOMES22C.items():

            eventos_resultados22c.append({
                "evento": evento,
                "fase": fase,
                "modelo": nome,
                "n_horas": len(parte),
                **metricas22c(
                    parte["co2_t"],
                    parte[coluna]
                )
            })

resultados_eventos22c = pd.DataFrame(
    eventos_resultados22c
)

print("\nRESULTADOS POR EVENTO")
display(resultados_eventos22c.round(3))

# ============================================================
# 10. GRÁFICO DE VIÉS DIÁRIO — 2025
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))

serie22c = teste22c.set_index("timestamp")

for nome, coluna in NOMES22C.items():

    erro_diario = (
        serie22c[coluna] - serie22c["co2_t"]
    ).resample("D").mean()

    ax.plot(
        erro_diario.index,
        erro_diario.values,
        label=nome,
        linewidth=1.4,
        alpha=0.85
    )

for data in EVENTOS_2025_22C:
    ax.axvline(
        data,
        color="black",
        linestyle="--",
        alpha=0.65
    )

ax.axhline(0, color="gray", linewidth=1)

ax.set_title(
    "Iatan 2 — Mudanças candidatas: "
    "efeito aprendido em 2017"
)

ax.set_xlabel("Data")
ax.set_ylabel("Viés diário (tCO2)")
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

ARQ_FIG22C = (
    FIGURAS22C
    / "22C_intervencoes_redutoras_2025.png"
)

plt.savefig(
    ARQ_FIG22C,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 11. EXPORTAR RESULTADOS
# ============================================================

faixas22c.to_csv(
    TABELAS22C / "22C_faixas_historicas.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "evento_historico": EVENTO_HIST22C,
    "fator_aprendido": FATOR_HIST22C,
    "efeito_pct": EFEITO_HIST_PCT22C,
    "horas_pre": len(pre_cal22c),
    "horas_pos": len(pos_cal22c),
    "n_faixas": len(faixas22c)
}]).to_csv(
    TABELAS22C / "22C_fator_historico.csv",
    index=False,
    encoding="utf-8-sig"
)

validacao_hist22c.to_csv(
    TABELAS22C / "22C_validacao_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

resultados22c.to_csv(
    TABELAS22C / "22C_resultados_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

resultados_eventos22c.to_csv(
    TABELAS22C / "22C_resultados_eventos.csv",
    index=False,
    encoding="utf-8-sig"
)

teste22c.to_parquet(
    DADOS22C / "22C_previsoes_2025.parquet",
    index=False
)

print("\n" + "=" * 75)
print("EXPERIMENTO 22C CONCLUÍDO")
print("=" * 75)

print("Fator histórico:", round(FATOR_HIST22C, 5))
print("Efeito histórico (%):", round(EFEITO_HIST_PCT22C, 3))
print("Horas de treino 2025:", len(treino_2025_22c))
print("Horas de teste 2025:", len(teste22c))
print("Resultados salvos em:", ROOT22C)

## 16A — Sensibilidade do fator histórico de 2017

Repete o estimador original com três cortes temporais: março–dezembro, março–setembro e fevereiro–setembro. O fator principal de 22C é reproduzido como controle; 2018 é verificação de continuidade e 2025 é apenas diagnóstico retrospectivo. **Nenhum cenário é escolhido pela performance de 2025.**

---

**Sensibilidade da calibração — Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11); Gama et al. (2014, pp. 44:2 e 44:4).** A escolha das janelas históricas e a presença de outras mudanças candidatas afetam a interpretação do fator; por isso o notebook estima fatores alternativos. Não há seleção desses fatores usando o erro de 2025. Essa análise constitui uma verificação interna de estabilidade, **não um intervalo de confiança** para o efeito causal ou para a transferência a outras usinas.

In [ ]:
# 16A — Sensibilidade histórica do fator F (2017)
# Estima fatores SEM usar CO2 de 2025; a avaliação de 2025 é apenas diagnóstica.
# O fator principal de 22C não é escolhido/reajustado com base no teste.
import numpy as np
import pandas as pd
from IPython.display import display


def estimar_fator_2017_sensibilidade(corte, fim_pos):
    """Replica o estimador 22C: log-resíduos, interseção p5–p95,
    8 faixas quantílicas, mediana por faixa e média ponderada dos deltas.
    """
    corte, fim_pos = pd.Timestamp(corte), pd.Timestamp(fim_pos)
    antes = base22c.loc[
        (base22c["timestamp"] >= pd.Timestamp("2016-01-01"))
        & (base22c["timestamp"] < corte)
    ].copy()
    depois = base22c.loc[
        (base22c["timestamp"] >= corte)
        & (base22c["timestamp"] < fim_pos)
    ].copy()
    if min(len(antes), len(depois)) < 100:
        raise ValueError(f"Períodos insuficientes: corte={corte}, fim={fim_pos}")
    for parte in (antes, depois):
        parte["pred_base"] = prever22c(modelo_hist22c, parte)
    a = preparar_residuos22c(antes)
    b = preparar_residuos22c(depois)
    li = max(a["geracao_mwh"].quantile(.05), b["geracao_mwh"].quantile(.05))
    ls = min(a["geracao_mwh"].quantile(.95), b["geracao_mwh"].quantile(.95))
    if not np.isfinite(li) or not np.isfinite(ls) or li >= ls:
        raise ValueError("Sem faixa comum de geração para comparação histórica.")
    a = a.loc[a["geracao_mwh"].between(li, ls)].copy()
    b = b.loc[b["geracao_mwh"].between(li, ls)].copy()
    if min(len(a), len(b)) < 100:
        raise ValueError("Menos de 100 observações em uma das janelas comparáveis.")
    cortes = np.unique(np.quantile(
        pd.concat([a["geracao_mwh"], b["geracao_mwh"]]), np.linspace(0, 1, 9)
    ))
    if len(cortes) < 4:
        raise ValueError("Faixas quantílicas insuficientes.")
    a["faixa"] = pd.cut(a["geracao_mwh"], bins=cortes, include_lowest=True)
    b["faixa"] = pd.cut(b["geracao_mwh"], bins=cortes, include_lowest=True)
    deltas, pesos = [], []
    for faixa in a["faixa"].cat.categories:
        va = a.loc[a["faixa"] == faixa, "log_razao"]
        vb = b.loc[b["faixa"] == faixa, "log_razao"]
        if len(va) < 15 or len(vb) < 15:
            continue
        deltas.append(float(vb.median() - va.median()))
        pesos.append(min(len(va), len(vb)))
    if len(deltas) < 3:
        raise ValueError("Menos de três faixas comparáveis.")
    return {
        "fator": float(np.exp(np.average(deltas, weights=pesos))),
        "n_antes": len(a), "n_depois": len(b), "n_faixas": len(deltas),
        "carga_min_mwh": float(li), "carga_max_mwh": float(ls),
    }

_cenarios_fator_2017 = [
    ("Principal_22C_mar_dez", "2017-03-01", "2018-01-01"),
    ("Ate_segunda_quebra_set", "2017-03-01", "2017-09-28"),
    ("Corte_diario_fev_ate_set", "2017-02-15", "2017-09-28"),
]
_sens_2017 = []
for nome, corte, fim in _cenarios_fator_2017:
    resultado = estimar_fator_2017_sensibilidade(corte, fim)
    f = resultado["fator"]
    # Verificação histórica de 2018 não é uma intervenção independente.
    val = metricas22c(valid_hist22c["co2_t"], valid_hist22c["pred_base"] * f)
    n_eventos = teste22c["n_intervencoes"].to_numpy(dtype=int)
    pred = teste22c["pred_M0"].to_numpy() * np.power(f, n_eventos)
    teste_diag = metricas22c(teste22c["co2_t"], pred)
    _sens_2017.append({
        "cenario": nome, "corte": corte, "fim_pos_exclusivo": fim,
        **resultado, "efeito_pct": 100 * (f - 1),
        "RMSE_2018": val["RMSE"], "MAE_2018": val["MAE"],
        "RMSE_2025_diagnostico": teste_diag["RMSE"],
        "MAE_2025_diagnostico": teste_diag["MAE"],
        "vies_2025_diagnostico": teste_diag["vies"],
    })
sensibilidade_fator_2017 = pd.DataFrame(_sens_2017)
_f_reproduzido = float(sensibilidade_fator_2017.iloc[0]["fator"])
if not np.isclose(_f_reproduzido, FATOR_HIST22C, rtol=0, atol=1e-10):
    raise AssertionError(
        f"Reimplementação do estimador divergiu do 22C: "
        f"{_f_reproduzido:.12f} versus {FATOR_HIST22C:.12f}"
    )
sensibilidade_fator_2017["delta_fator_vs_principal"] = (
    sensibilidade_fator_2017["fator"] - FATOR_HIST22C
)
sensibilidade_fator_2017.to_csv(
    DIR_TABELAS / "16A_sensibilidade_fator_historico_2017.csv",
    index=False, encoding="utf-8-sig"
)
print("SENSIBILIDADE DO FATOR HISTÓRICO — 2017")
print("A segunda mudança candidata (28/09/2017) é excluída nos cenários alternativos.")
print("2018: continuidade histórica; 2025: comparação retrospectiva, NÃO seleção de F.")
display(sensibilidade_fator_2017.round(5))
if (sensibilidade_fator_2017["delta_fator_vs_principal"].abs() > 0.02).any():
    print("ATENÇÃO: há fatores alternativos que diferem >0,02 do principal; revisar interpretação.")

## 17 — Robustez da correção

Executa controles de correção uniforme, deslocamentos temporais e bootstrap. Datas de 2025 selecionadas retrospectivamente, não validadas prospectivamente.

**Mudança adicional:** a detecção retrospectiva aponta possível recuperação da IC a partir de 29/10/2025. O M2 permanece com F² até dezembro; a célula 17A examina o erro em quatro regimes, sem introduzir uma terceira correção calibrada no teste.
**Interpretação do bootstrap:** os intervalos de 95% são percentis de 1.000 reamostragens de blocos calendários de sete dias e medem **diferenças de RMSE** entre M2 e cada controle, mantendo fixos os modelos, o fator histórico e as datas retrospectivamente selecionadas. Não incluem incerteza de seleção dos pontos de mudança, de estimação de F ou de transferência entre usinas. **Não são intervalos de predição para emissões horárias**, nem justificam afirmar superioridade estatística das datas exatas quando o intervalo inclui zero.

---

**Base de inferência — Künsch (1989, pp. 1217–1218); Jewell et al. (2022, §1, pp. 1082–1084).** Reamostrar blocos temporais é mais apropriado do que reamostrar horas independentemente quando há autocorrelação. Os blocos calendários de sete dias são uma **escolha de desenho do estudo**, não um valor derivado automaticamente de Künsch. A distribuição bootstrap aqui condiciona-se aos modelos, ao fator e às datas selecionadas; não corrige a escolha retrospectiva dos cortes nem oferece inferência seletiva conforme Jewell et al. O trabalho de Künsch trata observações estacionárias; quebras de regime e dependência residual podem violar pressupostos necessários à cobertura nominal. Portanto, os intervalos percentis de 95% são **estimativas condicionais**, não garantia de cobertura em presença de não estacionariedade.

In [ ]:
# ===== ORIGEM: CÉLULA 29 DO NOTEBOOK EXPLORATÓRIO =====

# ============================================================
# CÉLULA 22D — TESTE DE ROBUSTEZ DAS INTERVENÇÕES
# IATAN 2 | EPA CEMS | TESTE 2025
# ============================================================
#
# OBJETIVO
# Verificar se o ganho do Experimento 22C decorre:
#
# (a) do conhecimento das datas das intervenções; ou
# (b) de uma simples correção multiplicativa do viés.
#
# Nenhum modelo será treinado nesta célula.
#
# O fator foi aprendido exclusivamente com dados históricos.
# CO2 de 2025 será utilizado SOMENTE para avaliação.
#
# ATENÇÃO:
# As datas de intervenção foram identificadas
# retrospectivamente usando a intensidade carbônica.
# Este experimento não representa validação prospectiva.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. DIRETÓRIOS
# ============================================================

ROOT22D = OUTPUT_ROOT

DADOS22D = ROOT22D / "dados"
TABELAS22D = ROOT22D / "tabelas"
FIGURAS22D = ROOT22D / "figuras"

TABELAS22D.mkdir(parents=True, exist_ok=True)
FIGURAS22D.mkdir(parents=True, exist_ok=True)

ARQ_PREVISOES = (
    DADOS22D / "22C_previsoes_2025.parquet"
)

ARQ_FATOR = (
    TABELAS22D / "22C_fator_historico.csv"
)

for arquivo in [ARQ_PREVISOES, ARQ_FATOR]:
    if not arquivo.exists():
        raise FileNotFoundError(arquivo)

# ============================================================
# 2. CARREGAR RESULTADOS DO EXPERIMENTO 22C
# ============================================================

df22d = pd.read_parquet(ARQ_PREVISOES)

df22d["timestamp"] = pd.to_datetime(
    df22d["timestamp"]
)

df22d = (
    df22d.sort_values("timestamp")
    .reset_index(drop=True)
)

colunas = [
    "timestamp",
    "geracao_mwh",
    "co2_t",
    "pred_M0"
]

faltantes = [
    c for c in colunas
    if c not in df22d.columns
]

if faltantes:
    raise KeyError(
        f"Colunas ausentes: {faltantes}"
    )

if df22d["timestamp"].duplicated().any():
    raise ValueError(
        "Existem timestamps duplicados."
    )

if not df22d["timestamp"].dt.year.eq(2025).all():
    raise ValueError(
        "O arquivo deve conter somente o teste de 2025."
    )

if df22d[colunas].isna().any().any():
    raise ValueError(
        "Existem valores ausentes nas colunas obrigatórias."
    )

fatores22d = pd.read_csv(ARQ_FATOR, float_precision="round_trip")

FATOR22D = float(
    fatores22d["fator_aprendido"].iloc[0]
)

if not np.isfinite(FATOR22D) or FATOR22D <= 0:
    raise ValueError(
        "Fator histórico inválido."
    )

print("=" * 75)
print("EXPERIMENTO 22D — TESTE DE ROBUSTEZ")
print("=" * 75)

print("Horas de teste:", len(df22d))
print(f"Fator histórico: {FATOR22D:.5f}")
print(
    f"Redução por evento: "
    f"{100 * (1 - FATOR22D):.3f}%"
)
print(
    f"Redução acumulada de dois eventos: "
    f"{100 * (1 - FATOR22D**2):.3f}%"
)

# ============================================================
# 3. DATAS DE REFERÊNCIA
# ============================================================

EVENTO_MARCO, EVENTO_JULHO = EVENTOS_2025_22C
EVENTOS_CANDIDATOS = [EVENTO_MARCO, EVENTO_JULHO]

DESLOCAMENTO = pd.Timedelta(days=30)

# ============================================================
# 4. CENÁRIOS
# ============================================================
#
# C1: uma redução aplicada durante todo o ano.
#
# C2: duas reduções aplicadas durante todo o ano.
#
# M1: uma redução após março.
#
# M2: duas reduções nas datas identificadas.
#
# P1/P2: datas deslocadas em 30 dias.
#
# P3: calendário alternativo.
#
# Todos utilizam o MESMO fator histórico.
# ============================================================

CENARIOS22D = {
    "M0_Sem_correcao": [],

    "C1_Anual_1fator": [
        pd.Timestamp("2025-01-01")
    ],

    "C2_Anual_2fatores": [
        pd.Timestamp("2025-01-01"),
        pd.Timestamp("2025-01-01")
    ],

    "M1_Marco": [
        EVENTO_MARCO
    ],

    "M2_Marco_Julho": [
        EVENTO_MARCO,
        EVENTO_JULHO
    ],

    "P1_Antecipado_30d": [
        EVENTO_MARCO - DESLOCAMENTO,
        EVENTO_JULHO - DESLOCAMENTO
    ],

    "P2_Atrasado_30d": [
        EVENTO_MARCO + DESLOCAMENTO,
        EVENTO_JULHO + DESLOCAMENTO
    ],

    "P3_Calendario": [
        pd.Timestamp("2025-04-01"),
        pd.Timestamp("2025-08-01")
    ]
}

# ============================================================
# 5. APLICAR AS CORREÇÕES
# ============================================================
#
# O modelo base já foi treinado no Experimento 22C.
#
# A previsão corrigida é:
#
# pred_corrigida = pred_M0 * fator ** n_eventos
#
# n_eventos considera somente datas anteriores
# ou iguais ao timestamp da previsão.
#
# Não há consulta ao CO2 observado nesta etapa.
# ============================================================

tempos22d = df22d["timestamp"].to_numpy(
    dtype="datetime64[ns]"
)

def calcular_previsao_cenario(eventos):

    if len(eventos) == 0:
        n_eventos = np.zeros(
            len(df22d),
            dtype=int
        )

    else:
        datas = np.array(
            sorted(eventos),
            dtype="datetime64[ns]"
        )

        n_eventos = np.searchsorted(
            datas,
            tempos22d,
            side="right"
        )

    previsao = (
        df22d["pred_M0"].to_numpy()
        * np.power(FATOR22D, n_eventos)
    )

    return previsao, n_eventos

COLUNAS_PREV22D = {}

for nome, eventos in CENARIOS22D.items():

    coluna = f"pred_{nome}"

    pred, contagem = calcular_previsao_cenario(
        eventos
    )

    df22d[coluna] = pred

    COLUNAS_PREV22D[nome] = coluna

# Conferência: o M2 deve reproduzir o Experimento 22C.

if "pred_M2" in df22d.columns:

    diferenca = np.max(
        np.abs(
            df22d["pred_M2"]
            - df22d["pred_M2_Marco_Julho"]
        )
    )

    print(
        "\nDiferença máxima em relação ao 22C:",
        round(float(diferenca), 8)
    )

    if diferenca > 0.001:
        raise ValueError(
            "O M2 não reproduziu o Experimento 22C. "
            "Verifique o fator e as datas."
        )

# ============================================================
# 6. MÉTRICAS
# ============================================================

def metricas22d(real, previsto):

    real = np.asarray(real)
    previsto = np.asarray(previsto)

    return {
        "MAE": mean_absolute_error(real, previsto),

        "RMSE": np.sqrt(
            mean_squared_error(real, previsto)
        ),

        "R2": r2_score(real, previsto),

        "vies": float(
            np.mean(previsto - real)
        )
    }

resultados22d = []

for nome, coluna in COLUNAS_PREV22D.items():

    resultados22d.append({
        "cenario": nome,
        "n_eventos": len(CENARIOS22D[nome]),
        **metricas22d(
            df22d["co2_t"],
            df22d[coluna]
        )
    })

resultados22d = pd.DataFrame(resultados22d)

RMSE_M0 = resultados22d.loc[
    resultados22d["cenario"] == "M0_Sem_correcao",
    "RMSE"
].iloc[0]

RMSE_M2 = resultados22d.loc[
    resultados22d["cenario"] == "M2_Marco_Julho",
    "RMSE"
].iloc[0]

resultados22d["ganho_vs_M0_pct"] = (
    100 * (
        RMSE_M0 - resultados22d["RMSE"]
    ) / RMSE_M0
)

resultados22d["diferenca_RMSE_vs_M2"] = (
    resultados22d["RMSE"] - RMSE_M2
)

print("\nRESULTADOS GERAIS — 2025")

display(
    resultados22d.round(4)
)

# ============================================================
# 7. ANÁLISE POR PERÍODO
# ============================================================
#
# Comparar o ano completo pode esconder comportamentos.
#
# Por isso, avaliamos:
# - antes da primeira mudança;
# - após a primeira mudança;
# - após a segunda mudança;
# - 30 dias após março;
# - 30 dias após julho.
# ============================================================

t = df22d["timestamp"]

JANELAS22D = {
    "Ano_completo":
        t.notna(),

    "Antes_marco":
        t < EVENTO_MARCO,

    "Apos_marco":
        t >= EVENTO_MARCO,

    "Apos_julho":
        t >= EVENTO_JULHO,

    "Marco_30d":
        (t >= EVENTO_MARCO)
        & (t < EVENTO_MARCO + pd.Timedelta(days=30)),

    "Julho_30d":
        (t >= EVENTO_JULHO)
        & (t < EVENTO_JULHO + pd.Timedelta(days=30))
}

resultados_periodos22d = []

for janela, mascara in JANELAS22D.items():

    parte = df22d.loc[mascara]

    if len(parte) < 2:
        continue

    for nome, coluna in COLUNAS_PREV22D.items():

        resultados_periodos22d.append({
            "janela": janela,
            "cenario": nome,
            "n_horas": len(parte),
            **metricas22d(
                parte["co2_t"],
                parte[coluna]
            )
        })

resultados_periodos22d = pd.DataFrame(
    resultados_periodos22d
)

tabela_rmse22d = (
    resultados_periodos22d
    .pivot(
        index="cenario",
        columns="janela",
        values="RMSE"
    )
    .reindex(CENARIOS22D.keys())
)

print("\nRMSE POR PERÍODO")
display(tabela_rmse22d.round(3))

# ============================================================
# 8. BOOTSTRAP EXPLORATÓRIO POR BLOCOS DE 7 DIAS
# ============================================================
#
# Objetivo:
# Avaliar a estabilidade da diferença de RMSE entre
# M2 e os controles.
#
# Reamostragem de blocos calendários de sete dias, com reposição.
# Condicionada às datas e ao fator já escolhidos retrospectivamente.
# Não propaga incerteza de seleção de datas ou de estimação do fator.
# NÃO fornece intervalos de predição de CO2 por hora.
# Não é validação causal nem prospectiva.
#
# Ganho positivo = M2 possui RMSE menor.
# ============================================================

df22d["bloco_7d"] = (
    (
        df22d["timestamp"]
        - pd.Timestamp("2025-01-01")
    ).dt.days // 7
)

nomes22d = list(COLUNAS_PREV22D.keys())

blocos22d = sorted(
    df22d["bloco_7d"].unique()
)

SSE22D = []
N22D = []

for bloco in blocos22d:

    parte = df22d.loc[
        df22d["bloco_7d"] == bloco
    ]

    N22D.append(len(parte))

    SSE22D.append([
        np.sum(
            (
                parte[coluna].to_numpy()
                - parte["co2_t"].to_numpy()
            ) ** 2
        )
        for coluna in COLUNAS_PREV22D.values()
    ])

SSE22D = np.asarray(SSE22D)
N22D = np.asarray(N22D)

rng22d = np.random.default_rng(42)

N_BOOT22D = 1000

indices22d = rng22d.integers(
    low=0,
    high=len(blocos22d),
    size=(N_BOOT22D, len(blocos22d))
)

sse_boot22d = SSE22D[indices22d].sum(axis=1)
n_boot22d = N22D[indices22d].sum(axis=1)

rmse_boot22d = np.sqrt(
    sse_boot22d / n_boot22d[:, None]
)

idx_m2 = nomes22d.index("M2_Marco_Julho")

intervalos22d = []

for i, nome in enumerate(nomes22d):

    if nome == "M2_Marco_Julho":
        continue

    # Positivo significa vantagem de M2.
    ganho = (
        rmse_boot22d[:, i]
        - rmse_boot22d[:, idx_m2]
    )

    intervalo_inf, intervalo_sup = np.quantile(
        ganho,
        [0.025, 0.975]
    )

    intervalo22d = {
        "controle": nome,

        "vantagem_M2_RMSE": float(
            resultados22d.loc[
                resultados22d["cenario"] == nome,
                "RMSE"
            ].iloc[0] - RMSE_M2
        ),

        "IC95_inf": intervalo_inf,
        "IC95_sup": intervalo_sup
    }

    intervalos22d.append(intervalo22d)

intervalos22d = pd.DataFrame(
    intervalos22d
)

print("\nBOOTSTRAP — VANTAGEM DO M2 SOBRE OS CONTROLES")
print("Valores positivos favorecem M2.")
print("IC95 condicional para diferença de RMSE, NÃO intervalo de predição de CO2.")
print("Fator e datas fixos; incertezas de estimação/seleção não incluídas.")

display(intervalos22d.round(3))

# ============================================================
# 9. GRÁFICO — RMSE GERAL
# ============================================================

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(
    resultados22d["cenario"],
    resultados22d["RMSE"]
)

ax.axhline(
    RMSE_M0,
    linestyle="--",
    color="gray",
    label="Referência M0"
)

ax.set_title(
    "Iatan 2 — Robustez da correção por mudanças candidatas"
)

ax.set_ylabel("RMSE (tCO2)")
ax.set_xlabel("Cenário")

ax.tick_params(
    axis="x",
    rotation=40
)

ax.grid(
    axis="y",
    alpha=0.25
)

ax.legend()

plt.tight_layout()

ARQ_FIG_RMSE = (
    FIGURAS22D / "22D_comparacao_RMSE.png"
)

plt.savefig(
    ARQ_FIG_RMSE,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 10. GRÁFICO — VIÉS DIÁRIO
# ============================================================

CENARIOS_GRAFICO = [
    "M0_Sem_correcao",
    "C2_Anual_2fatores",
    "M2_Marco_Julho",
    "P1_Antecipado_30d",
    "P2_Atrasado_30d"
]

fig, ax = plt.subplots(figsize=(14, 5))

serie22d = df22d.set_index("timestamp")

for nome in CENARIOS_GRAFICO:

    coluna = COLUNAS_PREV22D[nome]

    vies_diario = (
        serie22d[coluna]
        - serie22d["co2_t"]
    ).resample("D").mean()

    ax.plot(
        vies_diario.index,
        vies_diario.values,
        label=nome,
        linewidth=1.3,
        alpha=0.85
    )

for data in EVENTOS_CANDIDATOS:

    ax.axvline(
        data,
        linestyle="--",
        color="black",
        alpha=0.6
    )

ax.axhline(
    0,
    color="gray",
    linewidth=1
)

ax.set_title(
    "Iatan 2 — Viés diário e sensibilidade às datas"
)

ax.set_ylabel("Viés diário (tCO2)")
ax.set_xlabel("Data")
ax.grid(alpha=0.25)
ax.legend(fontsize=9)

plt.tight_layout()

ARQ_FIG_VIES = (
    FIGURAS22D / "22D_vies_diario_controles.png"
)

plt.savefig(
    ARQ_FIG_VIES,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 11. EXPORTAÇÃO
# ============================================================

resultados22d.to_csv(
    TABELAS22D / "22D_resultados_gerais.csv",
    index=False,
    encoding="utf-8-sig"
)

resultados_periodos22d.to_csv(
    TABELAS22D / "22D_resultados_periodos.csv",
    index=False,
    encoding="utf-8-sig"
)

intervalos22d.to_csv(
    TABELAS22D / "22D_bootstrap_robustez.csv",
    index=False,
    encoding="utf-8-sig"
)

df22d.to_parquet(
    DADOS22D / "22D_previsoes_controles_2025.parquet",
    index=False
)

# ============================================================
# 12. DIAGNÓSTICO FINAL
# ============================================================

melhor22d = resultados22d.loc[
    resultados22d["RMSE"].idxmin()
]

print("\n" + "=" * 75)
print("EXPERIMENTO 22D CONCLUÍDO")
print("=" * 75)

print("Melhor cenário:", melhor22d["cenario"])
print("RMSE:", round(melhor22d["RMSE"], 4))

print(
    "Ganho sobre M0 (%):",
    round(melhor22d["ganho_vs_M0_pct"], 3)
)

print(
    "\nM2 foi o melhor cenário?",
    melhor22d["cenario"] == "M2_Marco_Julho"
)

print("\nResultados salvos em:")
print(ROOT22D)

## 17A — Regimes de 2025, faixa de carga e qualidade CEMS

Compara M0/M2 antes de março, março–julho, julho–outubro e após 29/out/2025; examina cargas dentro/fora da faixa histórica de calibração, flags de medição e exclusões diagnósticas de extremos. Os filtros não alteram o resultado principal nem o treinamento.

---

**Interpretação das verificações — EPA (2022, p. 8, *Hourly Data*); Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11).** Subconjuntos por carga e pelos indicadores efetivamente disponíveis na extração CAMPD avaliam a sensibilidade da conclusão a condições operacionais e a atributos registrados. **Esses marcadores não permitem certificar, hora a hora, medição direta versus substituição:** para isso seriam necessários os indicadores dos arquivos FTP ou os códigos MODC do FACT/FACT API (EPA, 2022, p. 8). A melhora em uma subamostra **não** autoriza generalização automática para todas as horas do calendário, outras unidades ou futuras intervenções.

In [ ]:
# 17A — Avaliação adicional: regimes de 2025, carga e qualidade CEMS
# Não reestima M0, M2 ou F com o CO2 de 2025. Apenas audita os erros.
import numpy as np
import pandas as pd
from IPython.display import display

_aud_2025 = pd.read_parquet(DIR_DADOS / "22C_previsoes_2025.parquet").copy()
_aud_2025["timestamp"] = pd.to_datetime(_aud_2025["timestamp"])
_aud_2025 = _aud_2025.sort_values("timestamp").reset_index(drop=True)
if _aud_2025["timestamp"].duplicated().any():
    raise ValueError("Duplicação de timestamp no teste 2025.")

# A mudança de 29/10/2025 é um candidato retrospectivo obtido com CO2;
# ela serve APENAS para avaliar resíduos, não para alterar previsões.
_regimes_2025 = [
    ("R0_antes_marco", "2025-01-01", "2025-03-26 08:00"),
    ("R1_marco_julho", "2025-03-26 08:00", "2025-07-14 10:00"),
    ("R2_julho_outubro", "2025-07-14 10:00", "2025-10-29"),
    ("R3_apos_29_outubro", "2025-10-29", "2026-01-01"),
]

def metricas_por_grupo_07(dados, dimensao, grupo):
    if dados.empty:
        return None
    r = dados["co2_t"].to_numpy(dtype=float)
    g = dados["geracao_mwh"].to_numpy(dtype=float)
    saida = {"dimensao": dimensao, "grupo": str(grupo), "n_horas": len(dados),
             "ic_observada_agregada_t_mwh": float(r.sum() / g.sum()) if g.sum() > 0 else np.nan}
    for nome, coluna in [("M0", "pred_M0"), ("M2", "pred_M2")]:
        p = dados[coluna].to_numpy(dtype=float)
        erro = p - r
        saida[f"RMSE_{nome}"] = float(np.sqrt(np.mean(erro**2)))
        saida[f"MAE_{nome}"] = float(np.mean(np.abs(erro)))
        saida[f"vies_{nome}"] = float(np.mean(erro))
        saida[f"R2_{nome}"] = (
            float(1 - np.sum(erro**2) / np.sum((r - r.mean())**2))
            if len(r) > 1 and np.sum((r - r.mean())**2) > 0 else np.nan
        )
    saida["ganho_RMSE_M2_pct"] = (
        100 * (saida["RMSE_M0"] - saida["RMSE_M2"]) / saida["RMSE_M0"]
        if saida["RMSE_M0"] > 0 else np.nan
    )
    return saida

_metricas_regimes = []
for nome, inicio, fim in _regimes_2025:
    fatia = _aud_2025.loc[
        (_aud_2025["timestamp"] >= pd.Timestamp(inicio))
        & (_aud_2025["timestamp"] < pd.Timestamp(fim))
    ]
    r = metricas_por_grupo_07(fatia, "regime_2025", nome)
    if r is not None:
        _metricas_regimes.append(r)
regimes_2025 = pd.DataFrame(_metricas_regimes)
if int(regimes_2025["n_horas"].sum()) != len(_aud_2025):
    raise AssertionError("Regimes não cobrem todas as horas elegíveis de 2025.")
regimes_2025.to_csv(
    DIR_TABELAS / "17A_metricas_por_regime_2025.csv", index=False, encoding="utf-8-sig"
)
print("DESEMPENHO POR QUATRO REGIMES DE 2025")
display(regimes_2025.round(4))

# Faixa calibrada em 2016–2017 (sem uso de 2025 para definir limites).
_carga_inf = float(limite_inf22c)
_carga_sup = float(limite_sup22c)
if not 0 < _carga_inf < _carga_sup:
    raise ValueError("Faixa histórica de calibração inválida.")
_aud_2025["faixa_geracao_calibracao"] = np.select(
    [_aud_2025["geracao_mwh"] < _carga_inf,
     _aud_2025["geracao_mwh"] > _carga_sup],
    ["Abaixo_da_faixa", "Acima_da_faixa"], default="Dentro_da_faixa"
)
_tabelas_faixa = []
for nome, grupo in _aud_2025.groupby("faixa_geracao_calibracao", sort=False):
    r = metricas_por_grupo_07(grupo, "faixa_geracao_historica", nome)
    if r is not None:
        _tabelas_faixa.append(r)
faixas_geracao_2025 = pd.DataFrame(_tabelas_faixa)
faixas_geracao_2025.to_csv(
    DIR_TABELAS / "17A_metricas_faixas_geracao_2025.csv", index=False, encoding="utf-8-sig"
)
print(f"FAIXA HISTÓRICA DE CALIBRAÇÃO: {_carga_inf:.1f}–{_carga_sup:.1f} MWh")
display(faixas_geracao_2025.round(4))

# Anexar indicadores originais de medição por timestamp; não imputar flags.
_origem_medicao = pd.read_parquet(
    DIR_DADOS / "iatan2_intensidade_horaria_2011_2025.parquet",
    columns=["timestamp", "indicador_medicao_co2"]
)
_origem_medicao["timestamp"] = pd.to_datetime(_origem_medicao["timestamp"])
if _origem_medicao["timestamp"].duplicated().any():
    raise ValueError("Indicadores CEMS com timestamps duplicados.")
_aud_2025 = _aud_2025.merge(
    _origem_medicao, on="timestamp", how="left", validate="one_to_one"
)
_aud_2025["indicador_medicao_co2"] = (
    _aud_2025["indicador_medicao_co2"].astype("string").fillna("(ausente)")
)
_qualidade = []
for nome, grupo in _aud_2025.groupby("indicador_medicao_co2", dropna=False):
    r = metricas_por_grupo_07(grupo, "indicador_medicao_co2", nome)
    if r is not None:
        _qualidade.append(r)
qualidade_2025 = pd.DataFrame(_qualidade)
qualidade_2025.to_csv(
    DIR_TABELAS / "17A_metricas_qualidade_cems_2025.csv", index=False, encoding="utf-8-sig"
)
print("DESEMPENHO POR INDICADOR DE QUALIDADE DA MEDIÇÃO")
display(qualidade_2025.round(4))

# Sensibilidades de avaliação, sem remoção no treinamento nem no resultado principal.
_treino_ic = base22c.loc[base22c["timestamp"] < "2024-01-01"].copy()
_treino_ic["ic"] = _treino_ic["co2_t"] / _treino_ic["geracao_mwh"]
_limite_ic = float(_treino_ic["ic"].quantile(.99))
_limite_carga = float(_treino_ic["geracao_mwh"].quantile(.05))
_aud_2025["ic_observada"] = _aud_2025["co2_t"] / _aud_2025["geracao_mwh"]
_subamostras = [
    ("Principal_todas_horas", np.ones(len(_aud_2025), dtype=bool)),
    ("Somente_measured", _aud_2025["indicador_medicao_co2"].eq("Measured").fillna(False)),
    ("Sem_IC_extrema_p99_treino", _aud_2025["ic_observada"].le(_limite_ic)),
    ("Sem_carga_inferior_p05_treino", _aud_2025["geracao_mwh"].ge(_limite_carga)),
    ("Measured_sem_extremos", (
        _aud_2025["indicador_medicao_co2"].eq("Measured").fillna(False)
        & _aud_2025["ic_observada"].le(_limite_ic)
        & _aud_2025["geracao_mwh"].ge(_limite_carga)
    )),
]
_sensibilidade_amostra = []
for nome, mascara in _subamostras:
    dados = _aud_2025.loc[np.asarray(mascara, dtype=bool)]
    r = metricas_por_grupo_07(dados, "sensibilidade_amostra", nome)
    if r is not None:
        _sensibilidade_amostra.append(r)
sensibilidade_amostra_2025 = pd.DataFrame(_sensibilidade_amostra)
sensibilidade_amostra_2025.to_csv(
    DIR_TABELAS / "17A_sensibilidade_amostra_2025.csv", index=False, encoding="utf-8-sig"
)
print(f"LIMITES FIXADOS PELO TREINO: IC p99={_limite_ic:.4f} t/MWh; carga p05={_limite_carga:.2f} MWh")
display(sensibilidade_amostra_2025.round(4))
print("Os filtros que usam IC observada são APENAS diagnósticos retrospectivos.")

## 21 — Síntese, rastreabilidade e figura final

Padroniza as tabelas 22A–22D e o controle 22B operacional, verifica consistência do M0, gera figura com a terceira mudança candidata assinalada apenas como diagnóstico e salva relatório com limitações, versões e SHA-256 da base. As métricas são horárias; a figura mostra médias diárias das horas elegíveis suavizadas por sete dias.
**Fechamento científico:** A associação das 178 paradas prolongadas foi concluída (10 reduções persistentes, 8 aumentos, 63 sem persistência confirmada e 97 inconclusivas), com critérios próprios de elegibilidade. A melhoria de M2 é retrospectiva e condicionada às datas detectadas com CO₂ observado; não há comprovação de manutenção nem demonstração de efeito causal. A comparação 22A envolve janelas de treino distintas; o experimento complementar E2 pressupõe CO₂ anterior disponível; e o controle 22B operacional utiliza eventos definidos de forma homogênea. A calibração do fator foi feita com modelo histórico diferente do M0. O relatório final registra essas ressalvas e os intervalos bootstrap como **incerteza condicional da diferença de RMSE**, não como intervalos de predição.

---

**Síntese interpretativa — Gama et al. (2014, pp. 44:2 e 44:4); Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3); Jewell et al. (2022, §1, pp. 1082–1084).** Mudanças de regime podem prejudicar o desempenho de modelos fixos, mas avaliar correções com datas detectadas no próprio conjunto de teste é distinto de demonstrar adaptação online. O uso do termo *intervenção* refere-se à **data candidata empregada no cenário de pós-correção**, não a uma ação comprovada, exógena ou causal.

In [ ]:
# 21 — Síntese final padronizada, visualização e auditoria de reprodutibilidade
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

prev_final = pd.read_parquet(DIR_DADOS / "22C_previsoes_2025.parquet").copy()
colunas_final = ["timestamp", "geracao_mwh", "co2_t", "pred_M0", "pred_M2"]
if any(c not in prev_final for c in colunas_final):
    raise KeyError("Colunas ausentes na previsão 22C.")
prev_final = prev_final[colunas_final]
prev_final["timestamp"] = pd.to_datetime(prev_final["timestamp"])
if prev_final["timestamp"].duplicated().any():
    raise ValueError("Timestamps duplicados na avaliação final.")
if prev_final[colunas_final].isna().any().any():
    raise ValueError("Previsões finais incompletas; não excluir horas silenciosamente.")
prev_final = prev_final.sort_values("timestamp").set_index("timestamp")
if not (prev_final.index.year == 2025).all():
    raise ValueError("A tabela final deve conter apenas 2025.")


def metricas_21(real, pred):
    r = np.asarray(real, dtype=float)
    p = np.asarray(pred, dtype=float)
    if not (len(r) == len(p) and len(r) > 1 and np.isfinite(r).all() and np.isfinite(p).all()):
        raise ValueError("Métricas exigem amostras finitas, não vazias e alinhadas.")
    erro = p - r
    sst = np.sum((r - r.mean()) ** 2)
    return {"MAE": float(np.mean(np.abs(erro))),
            "RMSE": float(np.sqrt(np.mean(erro ** 2))),
            "R2": float(1 - np.sum(erro ** 2) / sst) if sst > 0 else np.nan,
            "vies": float(np.mean(erro))}

m0_final = metricas_21(prev_final["co2_t"], prev_final["pred_M0"])
m2_final = metricas_21(prev_final["co2_t"], prev_final["pred_M2"])
if m0_final["RMSE"] <= 0:
    raise ValueError("RMSE de referência não positivo.")
resumo_final = pd.DataFrame([
    {"modelo": "M0 — XGBoost geração", "n_horas": len(prev_final), **m0_final},
    {"modelo": "M2 — correção histórica retrospectiva", "n_horas": len(prev_final), **m2_final},
])
resumo_final["ganho_rmse_vs_m0_pct"] = (
    100 * (m0_final["RMSE"] - resumo_final["RMSE"]) / m0_final["RMSE"]
)
resumo_final.to_csv(DIR_TABELAS / "21_sintese_final_M0_M2.csv", index=False, encoding="utf-8-sig")
print(f"SÍNTESE HORÁRIA DE 2025 — mesmas {len(prev_final):,} horas para M0 e M2")
display(resumo_final.round(4))

# Comparação em esquema comum: nenhuma coluna 'NaN' causada por concatenação heterogênea.
# R² por subamostra não deve ser comparado com R² de todo o ano.
comparacoes_21 = []
fontes_21 = [
    ("22A", "22A_resultados_gerais.csv", "modelo", "Vies_medio", len(teste22)),
    ("22B_retro", "22B_resultados_gerais_corrigido.csv", "modelo", "vies", len(teste22b)),
    ("22B_operacional", "15A_22B_controle_eventos_operacionais_2025.csv", "modelo", "vies", len(_22b_teste_op)),
    ("22C", "22C_resultados_2025.csv", "modelo", "vies", len(teste22c)),
    ("22D", "22D_resultados_gerais.csv", "cenario", "vies", len(df22d)),
]
notas_21 = {
    "22A": "Atualização mensal; histórico de treino diferente do M0 estático",
    "22B_retro": "Treino: retornos >=72h; teste: datas de mudanças de IC retrospectivas",
    "22B_operacional": "Treino e teste: retornos >=72h definidos operacionalmente",
    "22C": "Fator histórico; datas de 2025 escolhidas retrospectivamente",
    "22D": "Controles e deslocamentos retrospectivos; não seleção prospectiva",
}
for experimento, arquivo, campo_modelo, campo_vies, n_esperado in fontes_21:
    tab = pd.read_csv(DIR_TABELAS / arquivo)
    obrig = {campo_modelo, "MAE", "RMSE", "R2", campo_vies}
    falt = obrig - set(tab.columns)
    if falt:
        raise KeyError(f"{arquivo} sem campos: {sorted(falt)}")
    for _, r in tab.iterrows():
        n_h = int(r["n_horas"]) if "n_horas" in tab.columns and pd.notna(r["n_horas"]) else n_esperado
        comparacoes_21.append({
            "experimento": experimento, "modelo": str(r[campo_modelo]),
            "n_horas": n_h, "MAE": float(r["MAE"]),
            "RMSE": float(r["RMSE"]), "R2": float(r["R2"]),
            "vies": float(r[campo_vies]), "nota_metodologica": notas_21[experimento]
        })
comparacao_final = pd.DataFrame(comparacoes_21)
comparacao_final["ganho_RMSE_vs_M0_pct"] = (
    100 * (m0_final["RMSE"] - comparacao_final["RMSE"]) / m0_final["RMSE"]
)
comparacao_final.to_csv(
    DIR_TABELAS / "21_comparacao_experimentos_padronizada.csv",
    index=False, encoding="utf-8-sig"
)
print("COMPARAÇÃO PADRONIZADA — métricas sobre horas elegíveis de 2025")
display(comparacao_final.round(4))

# Consistência do M0 em todos os experimentos que usam a mesma amostra.
_m0_linhas = comparacao_final.loc[
    ((comparacao_final["modelo"] == "M0_Geracao") |
     (comparacao_final["modelo"] == "M0_Sem_correcao") |
     (comparacao_final["modelo"] == "A_Estatico"))
    & (comparacao_final["n_horas"] == len(prev_final))
]
if not np.allclose(_m0_linhas["RMSE"], m0_final["RMSE"], atol=0.02, rtol=0):
    raise AssertionError("O M0 difere entre experimentos; revisar bases e parâmetros.")

# Figura única: médias das horas elegíveis por dia, suavizadas por 7 dias.
# NÃO representa totais diários de emissões.
diario_21 = prev_final.resample("D").agg({
    "co2_t": "mean", "pred_M0": "mean", "pred_M2": "mean", "geracao_mwh": "mean"
})
suave_21 = diario_21.rolling(7, min_periods=3).mean().where(diario_21["co2_t"].notna())
fig, ax1 = plt.subplots(figsize=(16, 7))
ax1.plot(suave_21.index, suave_21["pred_M2"], color="#EA580C", lw=2.6,
         label="M2 — corrigido", zorder=4)
ax1.plot(suave_21.index, suave_21["pred_M0"], color="#475569", lw=2.1,
         linestyle=(0, (3, 3)), label="M0 — original", zorder=6)
ax1.plot(suave_21.index, suave_21["co2_t"], color="#2563EB", lw=2.2,
         label="CO₂ observado", zorder=8)
ax1.set_ylabel("CO₂ médio nas horas elegíveis (tCO₂/h)")
ax1.set_xlabel("Data (2025)")
ax1.grid(alpha=.17)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b/%Y"))
ax2 = ax1.twinx()
ax2.plot(suave_21.index, suave_21["geracao_mwh"], color="#059669", lw=1.8,
         linestyle="--", alpha=.75, label="Geração elétrica")
ax2.set_ylabel("Geração média nas horas elegíveis (MWh/h)")
for data in ("2025-03-26 08:00", "2025-07-14 10:00"):
    ax1.axvline(pd.Timestamp(data), color="#64748B", linestyle=":", lw=1.4, alpha=.7)
ax1.axvline(pd.Timestamp("2025-10-29"), color="#7C3AED", linestyle="-.",
            lw=1.3, alpha=.8, label="29/out — mudança adicional (diagnóstico)")
ax1.set_title("Iatan 2 — CO₂, modelos M0/M2 e geração (2025) | médias móveis de 7 dias")
linhas = [x for x in ax1.get_lines() + ax2.get_lines() if not x.get_label().startswith("_")]
ax1.legend(linhas, [x.get_label() for x in linhas], loc="upper right", ncol=2, fontsize=9)
ax1.text(.01, .02,
         f"RMSE M0={m0_final['RMSE']:.2f} | M2={m2_final['RMSE']:.2f} tCO₂ | ganho={100*(m0_final['RMSE']-m2_final['RMSE'])/m0_final['RMSE']:.2f}%",
         transform=ax1.transAxes, fontsize=9,
         bbox={"facecolor": "white", "alpha": .85, "edgecolor": "none"})
fig.autofmt_xdate()
fig.tight_layout()
nome_fig = "21_grafico_final_M0_M2_real_geracao_2025"
fig.savefig(DIR_FIGURAS / f"{nome_fig}.png", dpi=250, bbox_inches="tight")
fig.savefig(DIR_FIGURAS / f"{nome_fig}.svg", bbox_inches="tight")
plt.show()
plt.close(fig)

# Registro de proveniência: versão da base, parâmetros, ressalvas e arquivos.
_arquivo_raw = DIR_DADOS / "iatan2_hourly_raw_2011_2025.parquet"
_hash_raw = None
if _arquivo_raw.exists():
    _digest_raw = hashlib.sha256()
    with _arquivo_raw.open("rb") as _f_raw:
        for _bloco in iter(lambda: _f_raw.read(1024 * 1024), b""):
            _digest_raw.update(_bloco)
    _hash_raw = _digest_raw.hexdigest()
_fatores_alternativos = sensibilidade_fator_2017[["cenario", "fator", "RMSE_2018", "RMSE_2025_diagnostico"]].to_dict(orient="records")
relatorio_21 = {
    "notebook": "07_mudancas_intensidade_carbonica_reprodutivel",
    "unidade": "Iatan 2", "facility_id": PLANT_ID, "unit_id": UNIT_ID,
    "n_horas_2025": len(prev_final),
    "rmse_m0": m0_final["RMSE"], "rmse_m2": m2_final["RMSE"],
    "ganho_rmse_pct": 100*(m0_final["RMSE"]-m2_final["RMSE"])/m0_final["RMSE"],
    "fator_historico_principal": float(FATOR_HIST22C),
    "fatores_alternativos_2017": _fatores_alternativos,
    "cortes_2025": [str(cortes_reconciliados_2025["2025-03"]),
                    str(cortes_reconciliados_2025["2025-07"])],
    "reconciliacao_metodos_2025": [
        {k: (v.isoformat() if isinstance(v, pd.Timestamp) else v)
         for k, v in linha.items()}
        for linha in reconciliacao_cortes_2025.to_dict(orient="records")
    ],
    "regra_cortes": "19A e 19B concordantes; sem escolha por RMSE 2025",
    "fator_2017_modelo_referencia": "XGBoost treinado 2011-2015",
    "modelo_M0_2025_treinamento": "2011-2023, inclui anos posteriores a 2017",
    "bootstrap_22D": {
        "tipo": "reamostragem de blocos calendarios de 7 dias com reposicao",
        "n_replicacoes": int(N_BOOT22D),
        "parametros_fixos": ["fator historico", "datas retrospectivas", "modelos"],
        "alvo": "intervalo condicional para diferenca de RMSE entre modelos",
        "nao_e": "intervalo de predicao de CO2 horario",
        "nao_inclui": ["incerteza de selecao dos cortes", "incerteza do fator", "generalizacao para outra usina"]
    },
    "terceira_mudanca_diagnostica": "2025-10-29",
    "sha256_base_raw": _hash_raw,
    "ambiente": AMBIENTE,
    "limites": [
        "Mudanças de IC detectadas retrospectivamente com CO2 de 2025",
        "Não há confirmação documental de manutenção nas datas candidatas",
        "Fator histórico de 2017 pode misturar mais de um regime e corrigir vies entre modelos",
        "Corte preliminar de julho difere de 19A/19B devido a criterios de otimizacao distintos",
        "Estabilidade entre janelas de 19B nao e IC95 da data",
        "Verificação de 2018 não constitui intervenção independente",
        "22A altera também o tamanho da janela de treino",
        "22B original mistura definições de evento; 15A usa retornos homogêneos",
        "Correção M2 mantém F² após outubro apesar de possível recuperação da IC",
        "Bootstrap 22D condicional: nao e intervalo de predicao nem propaga incerteza de deteccao/F",
        "Não há validação prospectiva ou em outra usina",
    ],
    "gerado_em_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
(DIR_RELATORIOS / "21_resumo_reprodutibilidade.json").write_text(
    json.dumps(relatorio_21, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Arquivos finais:", DIR_TABELAS / "21_comparacao_experimentos_padronizada.csv",
      DIR_FIGURAS / f"{nome_fig}.png")


from iatan.carbon_tracking import log_notebook07
print("Registro no MLflow:", log_notebook07(PROJECT_ROOT, OUTPUT_ROOT))

## Leitura dos resultados

A tabela `22C_fator_historico.csv` registra o fator estimado sem dados de 2025. A tabela `19C_reconciliacao_datas_2025.csv` conserva o refinamento preliminar, os dois critérios de mínimos quadrados e os cortes usados. As previsões e os resultados de M0/M1/M2 permitem avaliar o efeito do ajuste no modelo de geração deste notebook. As previsões A–D do 06 são analisadas no 08.

O CO₂ de 2025 participou da localização dos cortes. Os intervalos de 22D mantêm fixos fator, datas e modelos. As séries operacionais e o teste condicional não identificam causas físicas ou manutenção nem permitem inferir desempenho prospectivo.

## Referências científicas e documentação técnica

As referências a seguir foram utilizadas para **fundamentar conceitos, algoritmos ou escolhas de avaliação**, não para atribuir a trabalhos anteriores a fórmula própria de correção do experimento 22C. As citações autor–ano aparecem junto das etapas correspondentes.

1. **AKPAN, P. U.; FULS, W. F. (2021).** Cycling of coal fired power plants: A generic CO₂ emissions factor model for predicting CO₂ emissions. *Energy*, 214, 119026. DOI: [10.1016/j.energy.2020.119026](https://doi.org/10.1016/j.energy.2020.119026). **Uso:** relação entre fator de emissão, carga e comportamento cíclico de termelétricas a carvão; **não** demonstra que os eventos identificados nesta unidade sejam manutenções.
2. **U.S. ENVIRONMENTAL PROTECTION AGENCY — EPA (2022, p. 8, *Hourly Data*).** *CAMD's Power Sector Emissions Data Guide*. Washington, DC: U.S. EPA. [Documento e acesso](https://www.epa.gov/power-sector/power-sector-emission-data-guide). **Uso:** EPA/CEMS, variáveis horárias, tratamento de `opTime` e distinção entre os produtos CAMPD, FTP e FACT quanto à disponibilidade de informação sobre valores substituídos; o CAMPD isoladamente não permite verificar integralmente os códigos MODC.
3. **BAI, J.; PERRON, P. (2003).** Computation and analysis of multiple structural change models. *Journal of Applied Econometrics*, 18(1), 1–22. DOI: [10.1002/jae.659](https://doi.org/10.1002/jae.659). **Uso:** princípio de otimização global de múltiplas mudanças por minimização de custo; o código implementa adaptação própria com ponderação e critério aproximado.
4. **TRUONG, C.; OUDRE, L.; VAYATIS, N. (2020).** Selective review of offline change point detection methods. *Signal Processing*, 167, 107299. DOI: [10.1016/j.sigpro.2019.107299](https://doi.org/10.1016/j.sigpro.2019.107299). **Uso:** enquadramento da detecção *offline*, funções de custo, busca e penalização.
5. **SCHWARZ, G. (1978).** Estimating the dimension of a model. *The Annals of Statistics*, 6(2), 461–464. DOI: [10.1214/aos/1176344136](https://doi.org/10.1214/aos/1176344136). **Uso:** fundamento do BIC; aqui é empregada uma **aproximação exploratória ponderada**, não o BIC exato sob modelo probabilístico completo.
6. **CHEN, T.; GUESTRIN, C. (2016).** XGBoost: A scalable tree boosting system. In: *Proceedings of KDD '16*, pp. 785–794. DOI: [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). **Uso:** modelos XGBoost de referência.
7. **GAMA, J.; ŽLIOBAITĖ, I.; BIFET, A.; PECHENIZKIY, M.; BOUCHACHIA, A. (2014).** A survey on concept drift adaptation. *ACM Computing Surveys*, 46(4), artigo 44. DOI: [10.1145/2523813](https://doi.org/10.1145/2523813). **Uso:** motivação conceitual da adaptação à mudança da relação entre variáveis e emissões; não implica validação prospectiva neste experimento.
8. **CERQUEIRA, V.; TORGO, L.; MOZETIČ, I. (2020).** Evaluating time series forecasting models: An empirical study on performance estimation methods. *Machine Learning*, 109, 1997–2028. DOI: [10.1007/s10994-020-05910-7](https://doi.org/10.1007/s10994-020-05910-7). **Uso:** avaliação com ordenação temporal e restrições de informação no instante preditivo.
9. **KÜNSCH, H. R. (1989).** The jackknife and the bootstrap for general stationary observations. *The Annals of Statistics*, 17(3), 1217–1241. DOI: [10.1214/aos/1176347265](https://doi.org/10.1214/aos/1176347265). **Uso:** justificativa para reamostragem por blocos; não determina automaticamente que sete dias seja a escolha ótima.
10. **JEWELL, S.; FEARNHEAD, P.; WITTEN, D. (2022).** Testing for a change in mean after changepoint detection. *Journal of the Royal Statistical Society: Series B*, 84(4), 1082–1104. DOI: [10.1111/rssb.12501](https://doi.org/10.1111/rssb.12501). **Uso:** advertência sobre inferência depois de selecionar pontos de mudança com os dados; os procedimentos de inferência seletiva do artigo **não foram implementados** aqui.

**Fonte de dados primários — citação sugerida pela própria EPA (2022, p. 8):** UNITED STATES ENVIRONMENTAL PROTECTION AGENCY (EPA). *Power Sector Emissions Data*. Washington, DC: Office of Atmospheric Programs, Clean Air Markets Division. Disponível em: [EPA Clean Air Markets Program Data — CAMPD](https://campd.epa.gov). Essa fonte identifica o **conjunto de dados consultado**, distinto do guia técnico de 2022; não é um décimo primeiro artigo científico.

**Delimitação das contribuições:** a literatura sustenta a relevância da carga, da segmentação, da adaptação a mudanças e da inferência dependente no tempo. A combinação específica de corte de 2017, log-razões ajustadas por faixa de geração, pesos, fator \(F\) e cenários cumulativos M1/M2 é uma **proposição empírica deste notebook**. Os resultados são específicos da Iatan 2 e das horas elegíveis, sem identificação causal ou validade prospectiva demonstrada.